# 统计检验

In [ ]:
import pandas as pd
import numpy as np
import os
start = '2025-01-20 12:00:00'
end = '2025-03-20 12:00:00'

In [ ]:
# df_final_features= pd.read_csv('Time_Series_Features_1.csv', encoding='utf-8-sig')
OUTPUT_PATH = r'C:\tongji\0 code\00_data\01_EDA'
RESULT_PATH = r'C:\tongji\0 code\00_data\02_stimulate_image'

df_features= pd.read_csv(OUTPUT_PATH+os.sep+'Time_Series_Features_2024-12-05 00 to 2025-11-27 00_15min.csv', encoding='utf-8-sig')
df_features.rename(columns={'Unnamed: 0': 'timestamp','origin_ratio':'retweet_ratio_post'}, inplace=True)
df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])
df_features['comp_ratio_post'] = 1-df_features['comp_ratio_post']
df_features['comp_ratio_comment'] = 1-df_features['comp_ratio_comment']
df_features = df_features.drop(['total_medium_post','total_medium_comment','unique_users_post', 'unique_users_comment', 'semantic_shift_cross'],axis=1) 
df_features = df_features.fillna(0)


df_official=pd.read_csv(OUTPUT_PATH + os.sep + "temp_all_official_standart_time_type.csv", encoding='utf-8-sig')
df_official['timestamp'] = pd.to_datetime(df_official['timestamp'])

def slice_data(df,start,end):
    if 'timestamp' not in df.columns:
        df['timestamp'] = pd.to_datetime(df.index)
    slice = (df['timestamp'] >= start) & (df['timestamp'] <= end)
    return df[slice]



In [ ]:
df_feature_slice = slice_data(df_features,start,end) 

In [ ]:
feature_list_simple = list(df_features.columns)
feature_list_simple.remove('timestamp')

## step1 分布检验

In [ ]:
from step1_1 import run_step1_distribution_test

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import nbinom, poisson, beta as beta_dist, kstest, chi2
from scipy.optimize import minimize
from sklearn.mixture import GaussianMixture
from scipy.special import gammaln
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False



def get_feature_type(FEATURE_CONFIG, feature_name):
    """获取特征类型"""
    for ftype, features in FEATURE_CONFIG.items():
        if feature_name in features:
            return ftype
    return 'continuous'


# ==================== 分布检验函数 ====================

def fit_negative_binomial(data):
    """
    拟合负二项分布，返回参数(r, p)和拟合优度
    使用最大似然估计
    """
    data = data[data >= 0].astype(int)
    if len(data) == 0:
        return None, None, None, None
    
    mean_val = data.mean()
    var_val = data.var()
    
    # 矩估计初始值
    if var_val <= mean_val:
        # 方差小于等于均值，负二项不适用
        return None, None, None, "方差<=均值，不适合负二项分布"
    
    r_init = mean_val ** 2 / (var_val - mean_val)
    p_init = mean_val / var_val
    
    # 约束优化
    def neg_log_likelihood(params):
        r, p = params
        if r <= 0 or p <= 0 or p >= 1:
            return 1e10
        try:
            ll = np.sum(nbinom.logpmf(data, r, p))
            return -ll
        except:
            return 1e10
    
    try:
        result = minimize(neg_log_likelihood, [r_init, p_init], 
                         method='L-BFGS-B',
                         bounds=[(0.01, 1000), (0.001, 0.999)])
        r_fit, p_fit = result.x
        
        # KS检验
        ks_stat, ks_p = kstest(data, lambda x: nbinom.cdf(x, r_fit, p_fit))
        
        # 计算AIC
        k = 2  # 参数个数
        ll = -result.fun
        aic = 2 * k - 2 * ll
        
        return r_fit, p_fit, {'ks_stat': ks_stat, 'ks_p': ks_p, 'aic': aic, 'll': ll}, None
    except Exception as e:
        return None, None, None, str(e)


def fit_poisson(data):
    """
    拟合泊松分布，返回参数lambda和拟合优度
    """
    data = data[data >= 0].astype(int)
    if len(data) == 0:
        return None, None, None
    
    lambda_fit = data.mean()
    
    # KS检验
    ks_stat, ks_p = kstest(data, lambda x: poisson.cdf(x, lambda_fit))
    
    # 计算AIC
    ll = np.sum(poisson.logpmf(data, lambda_fit))
    aic = 2 * 1 - 2 * ll  # 1个参数
    
    # 离散系数 (判断是否过度离散)
    dispersion = data.var() / data.mean() if data.mean() > 0 else np.inf
    
    return lambda_fit, {'ks_stat': ks_stat, 'ks_p': ks_p, 'aic': aic, 'll': ll, 
                        'dispersion': dispersion}, None


def fit_beta(data):
    """
    拟合Beta分布，返回参数(a, b)和拟合优度
    """
    # 处理边界值
    data_clean = data[(data > 0) & (data < 1)]
    
    if len(data_clean) < 10:
        # 如果有效数据太少，尝试轻微收缩
        data_clean = np.clip(data, 0.001, 0.999)
    
    if len(data_clean) == 0:
        return None, None, None, "无有效数据"
    
    try:
        # MLE拟合
        a_fit, b_fit, loc, scale = beta_dist.fit(data_clean, floc=0, fscale=1)
        
        # KS检验
        ks_stat, ks_p = kstest(data_clean, lambda x: beta_dist.cdf(x, a_fit, b_fit))
        
        # 计算AIC
        ll = np.sum(beta_dist.logpdf(data_clean, a_fit, b_fit))
        aic = 2 * 2 - 2 * ll  # 2个参数
        
        return a_fit, b_fit, {'ks_stat': ks_stat, 'ks_p': ks_p, 'aic': aic, 'll': ll}, None
    except Exception as e:
        return None, None, None, str(e)


def fit_gmm(data, n_components=2):
    """
    拟合高斯混合模型，返回参数和拟合优度
    """
    data_clean = data[~np.isnan(data) & ~np.isinf(data)]
    
    if len(data_clean) < 30:
        return None, None, "数据不足"
    
    try:
        gmm = GaussianMixture(n_components=n_components, random_state=42, n_init=5)
        gmm.fit(data_clean.reshape(-1, 1))
        
        # BIC/AIC
        bic = gmm.bic(data_clean.reshape(-1, 1))
        aic = gmm.aic(data_clean.reshape(-1, 1))
        
        params = {
            'weights': gmm.weights_.tolist(),
            'means': gmm.means_.flatten().tolist(),
            'stds': np.sqrt(gmm.covariances_.flatten()).tolist(),
            'aic': aic,
            'bic': bic
        }
        
        return gmm, params, None
    except Exception as e:
        return None, None, str(e)

def fit_lognormal(data):
    """
    拟合对数正态分布 (Log-Normal)
    适用于：右偏的连续数据，或者这就必须大于0的数据
    """
    # 对数正态分布要求 x > 0
    data_clean = data[data > 0]
    
    if len(data_clean) < 10:
        return None, None, "有效数据(>0)不足"

    try:
        # s 是形状参数 (sigma), scale = exp(mu)
        # floc=0 固定位置参数为0（标准对数正态）
        s_fit, loc_fit, scale_fit = stats.lognorm.fit(data_clean, floc=0)
        mu_fit = np.log(scale_fit)
        
        # KS检验
        ks_stat, ks_p = kstest(data_clean, lambda x: stats.lognorm.cdf(x, s_fit, loc=0, scale=scale_fit))
        
        # AIC
        ll = np.sum(stats.lognorm.logpdf(data_clean, s_fit, loc=0, scale=scale_fit))
        k = 2 # mu, sigma
        aic = 2 * k - 2 * ll
        
        return (s_fit, scale_fit), {'ks_stat': ks_stat, 'ks_p': ks_p, 'aic': aic, 'll': ll}, None
    except Exception as e:
        return None, None, str(e)


def fit_powerlaw(data):
    """
    拟合幂律分布 (使用 Pareto 分布近似)
    适用于：长尾分布，二八定律数据
    """
    # Pareto通常要求 x >= scale (通常 scale >= 1)
    # 为了拟合，我们取数据中的正值
    # data_clean = data[data >= 0]
    data_clean = data
    
    if len(data_clean) < 10:
        return None, None, "有效数据(>0)不足"

    try:
        # b 是形状参数 (alpha)
        # 很多时候我们将 location 固定，或者让 fit 自动寻找
        b_fit, loc_fit, scale_fit = stats.pareto.fit(data_clean, floc=0)
        
        # KS检验
        ks_stat, ks_p = kstest(data_clean, lambda x: stats.pareto.cdf(x, b_fit, loc=loc_fit, scale=scale_fit))
        
        # AIC
        ll = np.sum(stats.pareto.logpdf(data_clean, b_fit, loc=loc_fit, scale=scale_fit))
        k = 2 # b, scale (假设 loc 固定或作为参数)
        aic = 2 * k - 2 * ll
        
        return (b_fit, loc_fit, scale_fit), {'ks_stat': ks_stat, 'ks_p': ks_p, 'aic': aic, 'll': ll}, None
    except Exception as e:
        return None, None, str(e)


def fit_zinb(data):
    """
    拟合零膨胀负二项分布 (ZINB)
    适用于：计数数据，且0特别多，方差>>均值
    """
    # data = data[data >= 0].astype(int)
    data_clean = data

    if len(data) == 0:
        return None, None, "无数据"
    
    # 初始参数猜测
    mean_val = data.mean()
    var_val = data.var()
    zero_frac = (data == 0).mean()
    
    # 如果方差小于均值，NB甚至都不适用，ZINB更不适用
    if var_val <= mean_val:
        return None, None, "方差<=均值，不适合ZINB"

    # 初始化 pi (零膨胀系数), r, p
    pi_init = zero_frac if zero_frac > 0 else 0.1
    r_init = mean_val**2 / (var_val - mean_val)
    p_init = mean_val / var_val
    
    # 负对数似然函数
    def zinb_nll(params):
        pi, r, p = params
        if not (0 <= pi < 1) or r <= 0 or not (0 < p < 1):
            return 1e10
        
        # PMF for ZINB:
        # P(Y=0) = pi + (1-pi) * NB(0)
        # P(Y=k) = (1-pi) * NB(k)  for k > 0
        
        nb_logpmf = nbinom.logpmf(data, r, p)
        nb_pmf_0 = np.exp(nbinom.logpmf(0, r, p))
        
        # 针对 0 值
        ll_0 = np.log(pi + (1 - pi) * nb_pmf_0 + 1e-10)
        
        # 针对 非0 值
        ll_non0 = np.log(1 - pi + 1e-10) + nb_logpmf
        
        # 组合
        log_likelihoods = np.where(data == 0, ll_0, ll_non0)
        return -np.sum(log_likelihoods)

    try:
        result = minimize(zinb_nll, [pi_init, r_init, p_init], 
                          method='L-BFGS-B',
                          bounds=[(0.0001, 0.999), (0.01, 1000), (0.001, 0.999)])
        
        pi_fit, r_fit, p_fit = result.x
        
        # ZINB CDF 用于 KS 检验
        # CDF(k) = pi + (1-pi) * NB_CDF(k)  (for k >= 0)
        def zinb_cdf(k, pi, r, p):
            vals = np.array(k)
            cdf_vals = pi + (1 - pi) * nbinom.cdf(vals, r, p)
            # 修正小于0的情况 (虽然理论上没有)
            cdf_vals = np.where(vals < 0, 0, cdf_vals)
            return cdf_vals

        ks_stat, ks_p = kstest(data, lambda x: zinb_cdf(x, pi_fit, r_fit, p_fit))
        
        ll = -result.fun
        k = 3 # pi, r, p
        aic = 2 * k - 2 * ll
        
        return (pi_fit, r_fit, p_fit), {'ks_stat': ks_stat, 'ks_p': ks_p, 'aic': aic, 'll': ll}, None
        
    except Exception as e:
        return None, None, str(e)
# ==================== 综合分布检验 ====================
def comprehensive_distribution_test(df, feature_name, FEATURE_CONFIG, output_dir):
    """
    综合分布检验：包含 Poisson, NB, ZINB, LogNorm, Pareto(PowerLaw), Beta, GMM
    """
    os.makedirs(output_dir, exist_ok=True)
    
    if feature_name not in df.columns:
        print(f"❌ 特征 {feature_name} 不存在")
        return None
    
    series = df[feature_name].replace([np.inf, -np.inf], np.nan).dropna()
    data = series.values
    
    if len(data) < 30:
        print(f"⚠️ {feature_name}: 数据量不足 ({len(data)})")
        return None
    
    ftype = get_feature_type(FEATURE_CONFIG, feature_name)
    
    results = {
        'feature': feature_name, 'type': ftype,
        'n_samples': len(data), 'mean': data.mean(),
        'zero_ratio': (data == 0).mean()
    }

    # 修改：3行3列布局
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle(f'{feature_name} 综合分布检验', fontsize=16, fontweight='bold')
    plt.subplots_adjust(hspace=0.4, wspace=0.3)
    
    # 辅助：获取直方图数据用于统一Y轴
    counts, bin_edges = np.histogram(data, bins=50, density=True)
    max_density = max(counts) * 1.1

    # ========== 0. 原始数据 (0,0) ==========
    axes[0, 0].hist(data, bins=50, density=True, alpha=0.7, edgecolor='black', color='gray')
    axes[0, 0].set_title(f'原始分布\nZero Ratio: {results["zero_ratio"]:.2%}')
    axes[0, 0].set_ylabel('Density')
    
    # ========== 1. Poisson (0,1) ==========
    if ftype == 'count':
        lmb, metrics, _ = fit_poisson(data)
        if lmb is not None:
            results['poisson'] = {'lambda': lmb, **metrics}
            x_range = np.arange(0, np.percentile(data, 99))
            axes[0, 1].hist(data, bins=50, density=True, alpha=0.3, color='gray')
            axes[0, 1].plot(x_range, poisson.pmf(x_range, lmb), 'o-', markersize=3, label=f'Pois({lmb:.1f})', color='orange')
            axes[0, 1].set_title(f'Poisson\nAIC={metrics["aic"]:.0f}, p={metrics["ks_p"]:.3f}')
            axes[0, 1].legend()
    else:
        axes[0, 1].text(0.5, 0.5, '仅限计数数据', ha='center')
        
    # ========== 2. Negative Binomial (0,2) ==========
    if ftype == 'count':
        r, p, metrics, _ = fit_negative_binomial(data)
        if r is not None:
            results['nb'] = {'r': r, 'p': p, **metrics}
            x_range = np.arange(0, np.percentile(data, 99))
            axes[0, 2].hist(data, bins=50, density=True, alpha=0.3, color='gray')
            axes[0, 2].plot(x_range, nbinom.pmf(x_range, r, p), 'o-', markersize=3, label=f'NB(r={r:.1f})', color='green')
            axes[0, 2].set_title(f'NegBinomial\nAIC={metrics["aic"]:.0f}, p={metrics["ks_p"]:.3f}')
            axes[0, 2].legend()
    else:
        axes[0, 2].text(0.5, 0.5, '仅限计数数据', ha='center')

    # ========== 3. ZINB (1,0) ==========
    if ftype == 'count':
        params, metrics, err = fit_zinb(data)
        if params is not None:
            pi_val, r_val, p_val = params
            results['zinb'] = {'pi': pi_val, 'r': r_val, 'p': p_val, **metrics}
            
            x_range = np.arange(0, np.percentile(data, 99))
            # ZINB PMF
            pmf_vals = (1 - pi_val) * nbinom.pmf(x_range, r_val, p_val)
            pmf_vals[0] += pi_val
            
            axes[1, 0].hist(data, bins=50, density=True, alpha=0.3, color='gray')
            axes[1, 0].plot(x_range, pmf_vals, 'o-', markersize=3, label=f'ZINB(π={pi_val:.2f})', color='purple')
            axes[1, 0].set_title(f'ZINB\nAIC={metrics["aic"]:.0f}, p={metrics["ks_p"]:.3f}')
            axes[1, 0].legend()
        else:
            axes[1, 0].text(0.5, 0.5, f'ZINB Fit Failed:\n{err}', ha='center')
    else:
        axes[1, 0].text(0.5, 0.5, '仅限计数数据', ha='center')

    # ========== 4. Log-Normal (1,1) ==========
    # 适用于连续或计数（忽略0值拟合趋势）
    if data.max() > 0:
        params, metrics, err = fit_lognormal(data)
        if params is not None:
            s_val, scale_val = params
            results['lognorm'] = {'sigma': s_val, 'scale': scale_val, **metrics}
            
            x_range = np.linspace(0.01, np.percentile(data, 99), 200)
            pdf_vals = stats.lognorm.pdf(x_range, s_val, loc=0, scale=scale_val)
            
            axes[1, 1].hist(data, bins=50, density=True, alpha=0.3, color='gray')
            axes[1, 1].plot(x_range, pdf_vals, 'r-', lw=2, label=f'LogN(σ={s_val:.2f})')
            axes[1, 1].set_title(f'Log-Normal\nAIC={metrics["aic"]:.0f}, p={metrics["ks_p"]:.3f}')
            axes[1, 1].legend()
    else:
        axes[1, 1].text(0.5, 0.5, '无正值数据', ha='center')

    # ========== 5. Power Law / Pareto (1,2) ==========
    if data.max() > 0:
        params, metrics, err = fit_powerlaw(data)
        if params is not None:
            b_val, loc_val, scale_val = params
            results['powerlaw'] = {'b': b_val, **metrics}
            
            x_range = np.linspace(max(0.1, loc_val), np.percentile(data, 99), 200)
            pdf_vals = stats.pareto.pdf(x_range, b_val, loc=loc_val, scale=scale_val)
            
            axes[1, 2].hist(data, bins=50, density=True, alpha=0.3, color='gray')
            axes[1, 2].plot(x_range, pdf_vals, 'b-', lw=2, label=f'Pareto(α={b_val:.2f})')
            axes[1, 2].set_title(f'PowerLaw(Pareto)\nAIC={metrics["aic"]:.0f}, p={metrics["ks_p"]:.3f}')
            axes[1, 2].legend()
    else:
        axes[1, 2].text(0.5, 0.5, '无正值数据', ha='center')

    # ========== 6. Beta (2,0) ==========
    # 仅当数据在 [0,1] 之间时
    if (data.min() >= 0) and (data.max() <= 1) and (data.std() > 0):
        a, b, metrics, _ = fit_beta(data)
        if a is not None:
            results['beta'] = {'a': a, 'b': b, **metrics}
            x_range = np.linspace(0.01, 0.99, 200)
            axes[2, 0].hist(data, bins=50, density=True, alpha=0.3, color='gray')
            axes[2, 0].plot(x_range, beta_dist.pdf(x_range, a, b), 'r-', label=f'Beta')
            axes[2, 0].set_title(f'Beta\nAIC={metrics["aic"]:.0f}')
    else:
        axes[2, 0].text(0.5, 0.5, '数据不在[0,1]', ha='center')

    # ========== 7. GMM (2,1) ==========
    gmm, params, err = fit_gmm(data)
    if gmm is not None:
        results['gmm'] = params
        x_range = np.linspace(data.min(), data.max(), 200)
        logprob = gmm.score_samples(x_range.reshape(-1, 1))
        axes[2, 1].hist(data, bins=50, density=True, alpha=0.3, color='gray')
        axes[2, 1].plot(x_range, np.exp(logprob), 'k-', lw=2, label='GMM')
        axes[2, 1].set_title(f'GMM(2)\nAIC={params["aic"]:.0f}')
    else:
        axes[2, 1].text(0.5, 0.5, 'GMM Failed', ha='center')

    # ========== 8. Q-Q Plot (2,2) ==========
    stats.probplot(data, dist="norm", plot=axes[2, 2])
    axes[2, 2].set_title('Q-Q Plot (vs Norm)')

    # ========== 判定最优分布 ==========
    best_aic = np.inf
    best_dist = 'Unknown'
    
    # 比较列表
    candidates = [
        ('poisson', 'Poisson'), ('nb', 'NegBinomial'), ('zinb', 'ZINB'),
        ('lognorm', 'LogNormal'), ('powerlaw', 'PowerLaw'), 
        ('beta', 'Beta'), ('gmm', 'GMM')
    ]
    
    for key, name in candidates:
        if key in results and 'aic' in results[key]:
            # 优先选择通过 KS 检验的分布 (p > 0.05)
            # 如果没有通过的，则单纯比 AIC
            current_aic = results[key]['aic']
            if current_aic < best_aic:
                best_aic = current_aic
                best_dist = name

    results['best_distribution'] = best_dist
    results['best_aic'] = best_aic
    
    print(f"📊 {feature_name}: 最优分布 -> {best_dist} (AIC={best_aic:.1f})")
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{feature_name}_分布综合.png', dpi=100)
    plt.close()
    
    return results


# ==================== 批量检验所有特征 ====================

def run_step1_distribution_test_v2(df_feature, feature_list=None, FEATURE_CONFIG = {}, output_dir = None):
    """
    步骤1：批量进行分布检验
    
    Parameters:
    -----------
    df_feature : pd.DataFrame, 特征数据
    feature_list : list, 要检验的特征列表，None则使用全部
    output_dir : str, 输出目录
    
    Returns:
    --------
    dict : 所有特征的检验结果
    """
    os.makedirs(output_dir, exist_ok=True)
    
    if feature_list is None:
        feature_list = []
        for ftype, features in FEATURE_CONFIG.items():
            feature_list.extend(features)
    
    all_results = {}
    
    print("="*70)
    print("📌 步骤1：分布检验")
    print("="*70)
    
    for feature in feature_list:
        if feature in df_feature.columns:
            result = comprehensive_distribution_test(df_feature, feature, FEATURE_CONFIG, output_dir)
            if result:
                all_results[feature] = result
    
    # 生成汇总表
    summary_data = []
    for feat, res in all_results.items():
        row = {
            '特征': feat,
            '类型': res['type'],
            '样本量': res['n_samples'],
            '零值比例': f"{res['zero_ratio']:.1%}",
            '最优分布': res.get('best_distribution', 'Unknown'),
            '最优AIC': res.get('best_aic', np.nan)
        }
        
        # 添加各分布检验结果
        if 'poisson' in res:
            row['泊松_D'] = f"{res['poisson']['ks_stat']:.4f}"
            row['泊松_p'] = f"{res['poisson']['ks_p']:.2e}"
            row['泊松_通过'] = '✅' if res['poisson']['ks_p']>0.05 else '❌'
        else:
            row['泊松_D'] = '-'
            row['泊松_p'] = '-'
            row['泊松_通过'] = '-'
        
        if 'negative_binomial' in res:
            row['负二项_D'] = f"{res['negative_binomial']['ks_stat']:.4f}"
            row['负二项_p'] = f"{res['negative_binomial']['ks_p']:.2e}"
            row['负二项_通过'] = '✅' if res['negative_binomial']['ks_p']>0.05 else '❌'
        else:
            row['负二项_D'] = '-'
            row['负二项_p'] = '-'
            row['负二项_通过'] = '-'
        
        if 'beta' in res:
            row['Beta_D'] = f"{res['beta']['ks_stat']:.4f}"
            row['Beta_p'] = f"{res['beta']['ks_p']:.2e}"
            row['Beta_通过'] = '✅' if res['beta']['ks_p']>0.05 else '❌'
        else:
            row['Beta_D'] = '-'
            row['Beta_p'] = '-'
            row['Beta_通过'] = '-'
        
        
        if 'zinb' in res:
            row['ZINB_D'] = f"{res['zinb']['ks_stat']:.4f}"
            row['ZINB_p'] = f"{res['zinb']['ks_p']:.2e}"
            row['ZINB_通过'] = '✅' if res['zinb']['ks_p'] > 0.05 else '❌'
            row['ZINB_AIC'] = f"{res['zinb']['aic']:.1f}"
        else:
            row['ZINB_D'] = '-'
            row['ZINB_p'] = '-'
            row['ZINB_通过'] = '-'

        if 'lognorm' in res:
            row['LogN_D'] = f"{res['lognorm']['ks_stat']:.4f}"
            row['LogN_p'] = f"{res['lognorm']['ks_p']:.2e}"
            row['LogN_通过'] = '✅' if res['lognorm']['ks_p'] > 0.05 else '❌'
        else:
            row['LogN_D'] = '-'
            row['LogN_p'] = '-'
            row['LogN_通过'] = '-'
        if 'powerlaw' in res:
            row['Power_D'] = f"{res['powerlaw']['ks_stat']:.4f}"
            row['Power_p'] = f"{res['powerlaw']['ks_p']:.2e}"
            row['Power_通过'] = '✅' if res['powerlaw']['ks_p'] > 0.05 else '❌'
        else:
            row['Power_D'] = '-'
            row['Power_p'] = '-'
            row['Power_通过'] = '-'
            
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(f'{output_dir}/分布检验汇总.csv', index=False, encoding='utf-8-sig')
    
    print("\n" + "="*70)
    print("✅ 步骤1完成！")
    print(f"📁 结果保存至: {output_dir}")
    print("="*70)
    
    # 打印汇总表
    print("\n📋 分布检验汇总表:")
    print(summary_df.to_string(index=False))
    
    return all_results, summary_df


In [ ]:
# ==================== 特征分类配置 ====================
FEATURE_CONFIG = {
    # 计数型 - 检验泊松/负二项
    'count': [
        'total_volume_post', 'total_volume_comment',
        'total_short_post', 'total_long_post',
        'total_short_comment', 'total_long_comment',
        'vis_abs_redundancy_post',
        'senti_symbol_post', 'senti_symbol_comment'
    ],
    # 比率型 [0,1] - 检验Beta
    'ratio': [
        'gini_post', 'gini_comment',
        'retweet_ratio_post', 'neg_ratio_post', 'neg_ratio_comment',
        'vis_concentration_post', 'comp_ratio_post', 'comp_ratio_comment'
    ],
    # 连续型 - 检验GMM/正态
    'continuous': [
        'semantic_shift_post', 'semantic_shift_comment'
    ]
}


### run

In [ ]:
feature_list_simple = ['total_volume_post', 'total_volume_comment',
        'gini_post',
       'gini_comment', 'senti_symbol_post', 'senti_symbol_comment',
       'comp_ratio_post', 'comp_ratio_comment', 'retweet_ratio_post',
       'total_short_post', 'total_long_post',
       'total_short_comment',  'total_long_comment',
       'neg_ratio_post', 'neg_ratio_comment', 'semantic_shift_post',
       'semantic_shift_comment', 
       'vis_abs_redundancy_post', 'vis_concentration_post']
feature_list = feature_list_simple

In [ ]:
step1_results, step1_summary = run_step1_distribution_test(
    df_feature=df_feature_slice, 
    feature_list=feature_list_simple, 
    FEATURE_CONFIG=FEATURE_CONFIG,
    output_dir=os.path.join(RESULT_PATH, 'step1_distribution')
)

In [ ]:
# step1_results_v2, step1_summary_v2 = run_step1_distribution_test_v2(
#     df_feature=df_feature_slice, 
#     feature_list=feature_list_simple, 
#     FEATURE_CONFIG=FEATURE_CONFIG,
#     output_dir=os.path.join(RESULT_PATH, 'step1_distribution_v2')
# )

## step2 数据变换

In [ ]:
# from step1_2 import run_step2_transform_v3

### 论文图

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib as mpl

mpl.rcParams['font.family'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False


def plot_transform_comparison_two_features(
    data_dict,
    output_path=None
):
    """
    图5.3: 两个典型特征变换前后对比 (4行×2列)
    (已针对学术论文排版优化字体大小)
    """
    features = list(data_dict.keys())
    assert len(features) == 2, "需要恰好两个特征用于对比"
    
    # 【修改】：排版改为 4行2列，调整画布长宽比例让每张子图有足够的空间
    fig, axes = plt.subplots(4, 2, figsize=(12, 18))
    fig.suptitle('图 5.3  变换效果对比', fontsize=22, fontweight='bold', y=1.01)
    
    # 效果评级颜色
    grade_colors = {
        'A': '#59A14F',   # 绿
        'B': '#F28E2B',   # 橙
        'C': '#E15759',   # 红
    }
    grade_labels = {
        'A': '优秀',
        'B': '良好',
        'C': '受限',
    }
    
    # 统一设置论文常用的字体大小变量
    TITLE_FONT = 14
    LABEL_FONT = 13
    TICK_FONT = 12
    LEGEND_FONT = 11
    
    for feat_idx, feat_name in enumerate(features):
        info = data_dict[feat_name]
        original = info['original']
        transformed = info['transformed']
        method = info['method']
        orig_skew = info['orig_skew']
        trans_skew = info['trans_skew']
        zero_ratio = info['zero_ratio']
        grade = info['grade']
        subtitle = info.get('subtitle', f'({chr(97 + feat_idx)})')
        
        color_grade = grade_colors.get(grade, 'gray')
        label_grade = grade_labels.get(grade, grade)
        
        # 【修改】：计算当前特征占用的基础行号 (特征0占第0,1行；特征1占第2,3行)
        base_row = feat_idx * 2 
        
        # ── 图1 (左上): 原始直方图 ──
        ax = axes[base_row, 0]
        ax.hist(original, bins=60, density=True, alpha=0.7,
                edgecolor='white', linewidth=0.3, color='#4E79A7')
        ax.axvline(np.mean(original), color='#E15759', linestyle='--',
                   linewidth=1.5, label=f'$\\bar{{x}}$={np.mean(original):.2f}')
        ax.axvline(np.median(original), color='#F28E2B', linestyle=':',
                   linewidth=1.5, label=f'中位数={np.median(original):.2f}')
        
        ax.set_title(f'{subtitle} {feat_name} - 原始分布\n'
                     f'($\\gamma_1$={orig_skew:.2f}, 零值率={zero_ratio:.1f}%)',
                     fontsize=TITLE_FONT)
        ax.set_ylabel('概率密度', fontsize=LABEL_FONT)
        ax.legend(fontsize=LEGEND_FONT, framealpha=0.8)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)
        ax.grid(True, alpha=0.2)
        
        # ── 图2 (右上): 变换后直方图 ──
        ax = axes[base_row, 1]
        ax.hist(transformed, bins=60, density=True, alpha=0.7,
                edgecolor='white', linewidth=0.3, color=color_grade)
        ax.axvline(np.mean(transformed), color='#E15759', linestyle='--',
                   linewidth=1.5, label=f'$\\bar{{x}}$={np.mean(transformed):.2f}')
        
        # 叠加正态参考密度
        x_norm = np.linspace(transformed.min(), transformed.max(), 200)
        mu_t, sig_t = np.mean(transformed), np.std(transformed)
        ax.plot(x_norm, stats.norm.pdf(x_norm, mu_t, sig_t),
                'k--', linewidth=1.5, alpha=0.5, label='正态参考')
        
        skew_change = abs(orig_skew) - abs(trans_skew)
        ax.set_title(f'{method} 变换后分布\n'
                     f'$\\gamma_1$={trans_skew:.2f}, '
                     f'$\\Delta|\\gamma_1|$={skew_change:.2f}, '
                     f'评级: {label_grade}',
                     fontsize=TITLE_FONT, color=color_grade if grade == 'C' else 'black')
        ax.legend(fontsize=LEGEND_FONT, framealpha=0.8)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)
        ax.grid(True, alpha=0.2)
        
        # ── 图3 (左下): 原始Q-Q图 ──
        ax = axes[base_row + 1, 0]
        (osm, osr), (slope, intercept, r_val) = stats.probplot(original, dist="norm")
        ax.scatter(osm, osr, s=10, alpha=0.4, color='#4E79A7', edgecolors='none') 
        x_line = np.array([osm.min(), osm.max()])
        ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2)
        ax.set_title(f'原始 Q-Q图\n$R^2$={r_val**2:.4f}', fontsize=TITLE_FONT)
        ax.set_xlabel('理论分位数', fontsize=LABEL_FONT)
        ax.set_ylabel('样本分位数', fontsize=LABEL_FONT)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)
        ax.grid(True, alpha=0.2)
        
        # ── 图4 (右下): 变换后Q-Q图 ──
        ax = axes[base_row + 1, 1]
        (osm2, osr2), (slope2, intercept2, r_val2) = stats.probplot(
            transformed, dist="norm")
        ax.scatter(osm2, osr2, s=10, alpha=0.4, color=color_grade, edgecolors='none')
        x_line2 = np.array([osm2.min(), osm2.max()])
        ax.plot(x_line2, slope2 * x_line2 + intercept2, 'r-', linewidth=2)
        
        # 如果是零膨胀受限型, 标注截断线区域
        if grade == 'C' and zero_ratio > 50:
            zero_transformed = transformed[original == 0]
            if len(zero_transformed) > 0:
                zero_level = np.median(zero_transformed)
                ax.axhline(zero_level, color='#E15759', linestyle='--',
                           linewidth=1.5, alpha=0.7)
                ax.annotate('零值截断线', xy=(osm2.min() + 0.5, zero_level),
                            fontsize=LEGEND_FONT, color='#E15759',
                            bbox=dict(boxstyle='round,pad=0.3',
                                      facecolor='white', edgecolor='#E15759',
                                      alpha=0.8))
        
        ax.set_title(f'变换后 Q-Q图\n$R^2$={r_val2**2:.4f}', fontsize=TITLE_FONT)
        ax.set_xlabel('理论分位数', fontsize=LABEL_FONT)
        ax.set_ylabel('样本分位数', fontsize=LABEL_FONT)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)
        ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white') 
        print(f'已保存: {output_path}')
    
    plt.show()
    plt.close()

### 主流程

In [ ]:
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import logit, expit
from scipy.stats import boxcox, yeojohnson
import warnings
from statsmodels.tsa.ar_model import AutoReg

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== 变换函数库 ====================

def safe_log1p(x):
    """安全的log1p变换"""
    x = np.array(x, dtype=float)
    x = np.where(x < 0, 0, x)
    x = np.where(np.isinf(x), np.nan, x)
    return np.log1p(x)


def safe_sqrt(x):
    """安全的平方根变换"""
    x = np.array(x, dtype=float)
    x = np.where(x < 0, 0, x)
    x = np.where(np.isinf(x), np.nan, x)
    return np.sqrt(x)


def safe_logit(x, epsilon=1e-6):
    """安全的Logit变换（带边界保护）"""
    x = np.array(x, dtype=float)
    x = np.clip(x, epsilon, 1 - epsilon)
    x = np.where(np.isnan(x), 0.5, x)
    x = np.where(np.isinf(x), 0.5, x)
    return logit(x)


def standardize(x):
    """Z-score标准化（不改变偏度）"""
    x = np.array(x, dtype=float)
    valid = x[~np.isnan(x) & ~np.isinf(x)]
    if len(valid) == 0:
        return x
    mean_val = valid.mean()
    std_val = valid.std()
    if std_val == 0:
        std_val = 1
    return (x - mean_val) / std_val


def safe_boxcox(x, shift=1e-6):
    """
    安全的Box-Cox变换
    要求数据严格为正
    """
    x = np.array(x, dtype=float)
    valid_mask = ~np.isnan(x) & ~np.isinf(x)
    
    if valid_mask.sum() < 30:
        return standardize(x), None
    
    # Box-Cox要求正值，平移数据
    x_valid = x[valid_mask]
    x_positive = x_valid - x_valid.min() + shift
    
    try:
        transformed, lambda_opt = boxcox(x_positive)
        
        # 应用到全部数据
        x_all_positive = x - x[valid_mask].min() + shift
        result = np.full_like(x, np.nan)
        
        if lambda_opt == 0:
            result = np.log(x_all_positive)
        else:
            result = (np.power(x_all_positive, lambda_opt) - 1) / lambda_opt
        
        result[~valid_mask] = np.nan
        return result, lambda_opt
    except Exception as e:
        print(f"      Box-Cox失败: {e}")
        return standardize(x), None


def safe_yeo_johnson(x):
    """
    Yeo-Johnson变换
    支持负值和零值，比Box-Cox更通用
    """
    x = np.array(x, dtype=float)
    valid_mask = ~np.isnan(x) & ~np.isinf(x)
    
    if valid_mask.sum() < 30:
        return standardize(x), None
    
    try:
        x_valid = x[valid_mask]
        transformed, lambda_opt = yeojohnson(x_valid)
        
        # 应用到全部数据
        result = np.full_like(x, np.nan)
        result[valid_mask] = transformed
        
        return result, lambda_opt
    except Exception as e:
        print(f"      Yeo-Johnson失败: {e}")
        return standardize(x), None



# ==================== 智能变换选择器（更新版） ====================

def choose_best_transform_v2(data, feature_name, ftype, zero_ratio):
    """
    根据数据特性智能选择最佳变换（更新版）
    
    决策逻辑：
    1. 偏度在[-0.5, 0.5]：直接标准化
    2. 偏度在[-1, 1]：优先标准化，除非明确需要
    3. 偏度 > 1：尝试log1p, sqrt, yeo-johnson
    4. 比率型：避免logit（边界敏感）
    5. 高零膨胀：log1p + 零值指示
    """
    valid_data = data[~np.isnan(data) & ~np.isinf(data)]
    
    if len(valid_data) < 30:
        return 'none', data, {'reason': '数据不足'}
    
    original_skew = stats.skew(valid_data)
    
    # ========== 规则1: 偏度已经很好 ==========
    if abs(original_skew) <= 0.5:
        return 'standardize', standardize(data), {
            'reason': f'原始偏度{original_skew:.2f}已接近正态(±0.5)，仅标准化',
            'original_skew': original_skew,
            'transformed_skew': original_skew  # 标准化不改变偏度
        }
    
    # ========== 规则2: 偏度可接受 ==========
    if abs(original_skew) <= 1.0:
        return 'standardize', standardize(data), {
            'reason': f'原始偏度{original_skew:.2f}可接受(±1.0)，仅标准化以避免过度校正',
            'original_skew': original_skew,
            'transformed_skew': original_skew
        }
    
    # ========== 规则3: 需要变换（偏度 > 1） ==========
    candidates = {}
    
    # 候选1: sqrt (最温和)
    if valid_data.min() >= 0:
        sqrt_data = safe_sqrt(valid_data)
        sqrt_skew = stats.skew(sqrt_data[~np.isnan(sqrt_data)])
        candidates['sqrt'] = {
            'data': safe_sqrt(data),
            'skew': sqrt_skew,
            'improvement': abs(original_skew) - abs(sqrt_skew)
        }
    
    # 候选2: log1p (中等强度)
    if valid_data.min() >= 0:
        log_data = safe_log1p(valid_data)
        log_skew = stats.skew(log_data[~np.isnan(log_data)])
        candidates['log1p'] = {
            'data': safe_log1p(data),
            'skew': log_skew,
            'improvement': abs(original_skew) - abs(log_skew)
        }

    # 候选3: Logit (新增，仅限低零值 Ratio)
    # 理由：高零值数据的 0 会被映射为 logit(epsilon)≈-14，成为离群值，破坏正态性。
    if ftype == 'ratio' and zero_ratio < 0.05 and valid_data.min() >= 0 and valid_data.max() <= 1:
        logit_data = safe_logit(data)
        logit_valid = logit_data[~np.isnan(logit_data) & ~np.isinf(logit_data)]
        if len(logit_valid) > 0:
            candidates['logit'] = {
                'data': logit_data,
                'skew': stats.skew(logit_valid)
            }
    
    # 候选4: Yeo-Johnson (自适应)
    yj_data, yj_lambda = safe_yeo_johnson(valid_data)
    if yj_lambda is not None:
        yj_full, _ = safe_yeo_johnson(data)
        yj_skew = stats.skew(yj_data[~np.isnan(yj_data)])
        candidates['yeo_johnson'] = {
            'data': yj_full,
            'skew': yj_skew,
            'lambda': yj_lambda,
            'improvement': abs(original_skew) - abs(yj_skew)
        }
    
    # 候选5: Box-Cox (仅对严格正值)
    if valid_data.min() > 0:
        bc_data, bc_lambda = safe_boxcox(valid_data)
        if bc_lambda is not None:
            bc_full, _ = safe_boxcox(data)
            bc_skew = stats.skew(bc_data[~np.isnan(bc_data)])
            candidates['boxcox'] = {
                'data': bc_full,
                'skew': bc_skew,
                'lambda': bc_lambda,
                'improvement': abs(original_skew) - abs(bc_skew)
            }
    
    # ========== 选择最佳变换 ==========
    if not candidates:
        return 'standardize', standardize(data), {
            'reason': '无可用变换，使用标准化',
            'original_skew': original_skew,
            'transformed_skew': original_skew
        }
    
    # 选择标准：
    # 1. 偏度改善最大
    # 2. 但不能过度校正（变换后偏度反向且绝对值更大）
    
    best_transform = None
    best_improvement = -np.inf
    
    for name, info in candidates.items():
        # 检查是否过度校正
        new_skew = info['skew']
        if new_skew * original_skew < 0 and abs(new_skew) > 0.5:
            # 偏度反向且绝对值超过0.5，视为过度校正
            continue
        
        if info['improvement'] > best_improvement:
            best_improvement = info['improvement']
            best_transform = name
    
    if best_transform is None:
        # 所有变换都过度校正，使用最温和的sqrt
        if 'sqrt' in candidates:
            best_transform = 'sqrt'
        else:
            return 'standardize', standardize(data), {
                'reason': '所有变换都过度校正，使用标准化',
                'original_skew': original_skew,
                'transformed_skew': original_skew
            }
    
    result = candidates[best_transform]
    
    return best_transform, result['data'], {
        'reason': f'偏度从{original_skew:.2f}改善到{result["skew"]:.2f}',
        'original_skew': original_skew,
        'transformed_skew': result['skew'],
        'improvement': result['improvement'],
        'lambda': result.get('lambda')
    }


# ==================== 主变换流程（V3版） ====================

def apply_transformations_v3(df, step1_results, TRANSFORM_OVERRIDES_V2, output_dir):
    """
    步骤2（V3版）：智能变换
    
    核心改进：
    1. 偏度≤1的特征只做标准化
    2. 完全避免Logit（边界敏感导致过度校正）
    3. 优先使用温和变换（sqrt > log1p > yeo-johnson）
    4. 特殊规则覆盖问题特征
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    df_transformed = df.copy()
    transform_info = {}
    
    print("="*70)
    print("📌 步骤2（V3版）：智能数据变换")
    print("="*70)
    
    print("\n📋 特殊规则覆盖:")
    for feat, config in TRANSFORM_OVERRIDES_V2.items():
        print(f"   • {feat}: {config['transform']}")
        print(f"     └─ {config['reason']}")
    print()
    
    for feature, result in step1_results.items():
        if feature not in df.columns:
            continue
        
        ftype = result['type']
        zero_ratio = result['zero_ratio']
        original_data = df[feature].replace([np.inf, -np.inf], np.nan).values
        valid_data = original_data[~np.isnan(original_data)]
        
        if len(valid_data) < 10:
            continue
        
        original_skew = stats.skew(valid_data)
        
        print(f"\n{'='*55}")
        print(f"🔧 处理特征: {feature}")
        print(f"   类型: {ftype}, 零值比例: {zero_ratio:.1%}")
        print(f"   原始偏度: {original_skew:.3f}")
        
        # ========== 检查特殊规则覆盖 ==========
        lambda_opt = None
        if feature in TRANSFORM_OVERRIDES_V2:
            config = TRANSFORM_OVERRIDES_V2[feature]
            override_transform = config['transform']
            override_reason = config['reason']
            
            print(f"   ⚠️ 应用特殊规则: {override_transform}")

            if override_transform == 'sqrt':
                transformed = safe_sqrt(original_data)
            elif override_transform == 'log1p':
                transformed = safe_log1p(original_data)
            elif override_transform == 'standardize':
                transformed = standardize(original_data)
            elif override_transform == 'yeo_johnson':
                transformed, lambda_opt = safe_yeo_johnson(original_data)
                # transform_info[feature]['lambda'] = None if lambda_opt is None else float(lambda_opt)
                print(f"      Yeo-Johnson λ = {lambda_opt:.3f}" if lambda_opt else "")
            elif override_transform == 'boxcox':
                transformed, lambda_opt = safe_boxcox(original_data)
                print(f"      Box-Cox λ = {lambda_opt:.3f}" if lambda_opt else "")
            else:
                transformed = original_data
            
            # 计算变换后偏度
            trans_valid = transformed[~np.isnan(transformed)]
            trans_skew = stats.skew(trans_valid) if len(trans_valid) > 0 else original_skew
            
            df_transformed[f'{feature}_transformed'] = transformed
            transform_info[feature] = {
                'type': ftype,
                'transform': override_transform,
                'reason': override_reason,
                'original_skew': original_skew,
                'transformed_skew': trans_skew,
                'improvement': abs(original_skew) - abs(trans_skew),
                'is_override': True,
                'lambda': None if lambda_opt is None else float(lambda_opt)
            }
            
        else:
            # ========== 智能选择变换 ==========
            best_transform, transformed, info = choose_best_transform_v2(
                original_data, feature, ftype, zero_ratio
            )
            
            print(f"   ✅ 自动选择: {best_transform}")
            print(f"      {info.get('reason', '')}")
            
            df_transformed[f'{feature}_transformed'] = transformed
            
            transform_info[feature] = {
                'type': ftype,
                'transform': best_transform,
                'original_skew': info.get('original_skew', original_skew),
                'transformed_skew': info.get('transformed_skew', original_skew),
                'improvement': info.get('improvement', 0),
                'reason': info.get('reason', ''),
                'is_override': False,
                'lambda': info.get('lambda', 0.0)

            }
            
        
        # ========== 高零膨胀：添加零值指示变量 ==========
        if zero_ratio > 0.5:
            is_zero = (df[feature].values == 0).astype(int)
            df_transformed[f'{feature}_is_zero'] = is_zero
            transform_info[feature]['has_zero_indicator'] = True
            print(f"   📍 添加零值指示变量（零值比例 {zero_ratio:.1%}）")
        
        # 打印变换后偏度
        trans_col = f'{feature}_transformed'
        trans_valid = df_transformed[trans_col].replace([np.inf, -np.inf], np.nan).dropna()
        # final_skew = stats.skew(trans_valid) if len(trans_valid) > 0 else np.nan
        if len(trans_valid) > 0:
            final_mean = float(trans_valid.mean())
            final_std = float(trans_valid.std())
            final_skew = float(stats.skew(trans_valid))
        else:
            final_mean = np.nan
            final_std = np.nan
            final_skew = np.nan

        transform_info[feature]['transformed_mean'] = final_mean
        transform_info[feature]['transformed_std'] = final_std

        # print(f"   变换后偏度: {final_skew:.3f}")
        print(f"   📊 统计: Mean={final_mean:.3f}, Std={final_std:.3f}, Skew={final_skew:.3f}")
        
        # 评估
        improvement = abs(original_skew) - abs(final_skew)
        if improvement > 0:
            print(f"   📈 偏度改善: {improvement:.3f}")
        elif abs(final_skew) <= 1.0:
            print(f"   ✅ 偏度可接受 (|{final_skew:.2f}| ≤ 1)")
        else:
            print(f"   ⚠️ 偏度仍较大，但已是最优变换")
    
    # ========== 绑定绘图 ==========
    # plot_transformation_comparison_v3(df, df_transformed, step1_results, transform_info, output_dir)
    
    # 论文画图
    data_dict = {
    'total_short_comment': {
        'original': df['total_short_comment'].replace([np.inf, -np.inf], np.nan).dropna().values,
        'transformed': df_transformed['total_short_comment_transformed'].replace([np.inf, -np.inf], np.nan).dropna().values,
        'method': 'Yeo-Johnson',
        'orig_skew': 11.506,
        'trans_skew': 0.352,
        'zero_ratio': 49.5,
        'grade': 'A',
        'subtitle': '(a)',
    },
    'retweet_ratio_post': {
        'original': df['retweet_ratio_post'].replace([np.inf, -np.inf], np.nan).dropna().values,
        'transformed': df_transformed['retweet_ratio_post_transformed'].replace([np.inf, -np.inf], np.nan).dropna().values,
        'method': 'Yeo-Johnson',
        'orig_skew': 4.943,
        'trans_skew': 1.044,
        'zero_ratio': 71.2,
        'grade': 'C',
        'subtitle': '(b)',
    },
}
    plot_transform_comparison_two_features(data_dict, 'fig5_3_transform_comparison.svg')
    
    # 保存变换信息
    transform_summary = []
    for feat, info in transform_info.items():
        trans_skew = info['transformed_skew']
        orig_skew = info['original_skew']
        improvement = abs(orig_skew) - abs(trans_skew)
        
        # 评估状态
        if abs(trans_skew) <= 0.5:
            status = '✅优秀'
        elif abs(trans_skew) <= 1.0:
            status = '✅良好'
        elif improvement > 0:
            status = '⚠️改善'
        else:
            status = '❌未改善'
        
        transform_summary.append({
            '特征': feat,
            '类型': info['type'],
            '变换方法': info['transform'],
            '原始偏度': f"{orig_skew:.3f}",
            '变换偏度': f"{trans_skew:.3f}",
            '偏度改善': f"{improvement:.3f}",
            '状态': status,
            '零值指示': '✅' if info.get('has_zero_indicator') else '',
            '规则覆盖': '✅' if info.get('is_override') else ''
        })
    
    summary_df = pd.DataFrame(transform_summary)
    summary_df.to_csv(f'{output_dir}/变换信息汇总.csv', index=False, encoding='utf-8-sig')
    
    print("\n" + "="*70)
    print("✅ 步骤2（V3版）完成！")
    print(f"📁 结果保存至: {output_dir}")
    print("="*70)
    
    # 打印汇总表
    print("\n📋 变换汇总表:")
    display_cols = ['特征', '变换方法', '原始偏度', '变换偏度', '状态', '规则覆盖']
    print(summary_df[display_cols].to_string(index=False))
    
    # 统计
    n_excellent = (summary_df['状态'] == '✅优秀').sum()
    n_good = (summary_df['状态'] == '✅良好').sum()
    n_improved = (summary_df['状态'] == '⚠️改善').sum()
    n_failed = (summary_df['状态'] == '❌未改善').sum()
    
    print(f"\n📊 统计: 优秀{n_excellent} | 良好{n_good} | 改善{n_improved} | 未改善{n_failed}")
    
    return df_transformed, transform_info, summary_df


def plot_transformation_comparison_v3(df_original, df_transformed, step1_results, transform_info, output_dir):
    """绘制变换前后对比图（V3版）"""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    features = list(transform_info.keys())
    n_features = len(features)
    
    # 每页显示4个特征
    features_per_page = 4
    n_pages = (n_features + features_per_page - 1) // features_per_page
    
    for page in range(n_pages):
        start_idx = page * features_per_page
        end_idx = min(start_idx + features_per_page, n_features)
        page_features = features[start_idx:end_idx]
        
        fig, axes = plt.subplots(len(page_features), 4, figsize=(20, 5 * len(page_features)))
        if len(page_features) == 1:
            axes = axes.reshape(1, -1)
        
        for i, feature in enumerate(page_features):
            # 原始数据
            original = df_original[feature].replace([np.inf, -np.inf], np.nan).dropna()
            
            # 变换后数据
            trans_col = f'{feature}_transformed'
            if trans_col in df_transformed.columns:
                transformed = df_transformed[trans_col].replace([np.inf, -np.inf], np.nan).dropna()
            else:
                transformed = original
            
            info = transform_info.get(feature, {})
            transform_method = info.get('transform', 'unknown')
            is_override = info.get('is_override', False)
            
            orig_skew = stats.skew(original)
            trans_skew = stats.skew(transformed)
            
            # 1. 原始直方图
            axes[i, 0].hist(original, bins=50, density=True, alpha=0.7, 
                           edgecolor='black', color='steelblue')
            axes[i, 0].set_title(f'{feature}\n原始分布 (偏度={orig_skew:.2f})', fontsize=10)
            axes[i, 0].set_ylabel('密度')
            axes[i, 0].axvline(original.mean(), color='red', linestyle='--', linewidth=1, label='均值')
            
            # 2. 变换后直方图
            # 根据结果选择颜色
            if abs(trans_skew) <= 0.5:
                color = 'green'
                status = '✅优秀'
            elif abs(trans_skew) <= 1.0:
                color = 'limegreen'
                status = '✅良好'
            elif abs(trans_skew) < abs(orig_skew):
                color = 'orange'
                status = '⚠️改善'
            else:
                color = 'red'
                status = '❌'
            
            method_label = f'{transform_method}' + (' [规则]' if is_override else ' [自动]')
            axes[i, 1].hist(transformed, bins=50, density=True, alpha=0.7, 
                           edgecolor='black', color=color)
            axes[i, 1].set_title(f'{method_label}\n变换后 (偏度={trans_skew:.2f}) {status}', fontsize=10)
            axes[i, 1].axvline(transformed.mean(), color='red', linestyle='--', linewidth=1)
            
            # 3. 原始Q-Q图
            stats.probplot(original, dist="norm", plot=axes[i, 2])
            axes[i, 2].set_title('原始 Q-Q Plot', fontsize=10)
            axes[i, 2].grid(True, alpha=0.3)
            
            # 4. 变换后Q-Q图
            stats.probplot(transformed, dist="norm", plot=axes[i, 3])
            axes[i, 3].set_title('变换后 Q-Q Plot', fontsize=10)
            axes[i, 3].grid(True, alpha=0.3)
            
            # 添加网格
            for j in range(2):
                axes[i, j].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/变换对比_第{page+1}页.png', dpi=150, bbox_inches='tight')
        plt.close()
    
    # ========== 绘制偏度改善汇总图 ==========
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    features_list = list(transform_info.keys())
    orig_skews = [transform_info[f]['original_skew'] for f in features_list]
    trans_skews = [transform_info[f]['transformed_skew'] for f in features_list]
    transforms = [transform_info[f]['transform'] for f in features_list]
    
    x = np.arange(len(features_list))
    width = 0.35
    
    # 偏度绝对值对比
    bars1 = axes[0].bar(x - width/2, np.abs(orig_skews), width, label='原始|偏度|', 
                        alpha=0.7, color='steelblue')
    bars2 = axes[0].bar(x + width/2, np.abs(trans_skews), width, label='变换|偏度|', 
                        alpha=0.7, color='green')
    
    # 标注改善情况
    for j, (os_val, ts_val) in enumerate(zip(orig_skews, trans_skews)):
        if abs(ts_val) > abs(os_val):
            axes[0].scatter(x[j] + width/2, abs(ts_val) + 0.1, marker='x', color='red', s=80, zorder=5)
        elif abs(ts_val) <= 0.5:
            axes[0].scatter(x[j] + width/2, abs(ts_val) + 0.1, marker='o', color='gold', s=80, zorder=5)
    
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(features_list, rotation=45, ha='right', fontsize=8)
    axes[0].set_ylabel('|偏度|')
    axes[0].set_title('偏度变化对比\n(o=优秀, ✕=未改善)')
    axes[0].legend()
    axes[0].axhline(1.0, color='orange', linestyle='--', linewidth=1.5, label='可接受阈值')
    axes[0].axhline(0.5, color='green', linestyle='--', linewidth=1, alpha=0.5, label='优秀阈值')
    axes[0].grid(True, alpha=0.3)
    
    # 偏度变化方向（原始 vs 变换）
    colors = ['green' if abs(t) <= abs(o) else 'red' for o, t in zip(orig_skews, trans_skews)]
    axes[1].scatter(orig_skews, trans_skews, c=colors, s=60, alpha=0.7, edgecolors='black')
    
    # 添加对角线（不变线）
    lim = max(abs(min(orig_skews + trans_skews)), abs(max(orig_skews + trans_skews))) + 0.5
    axes[1].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='不变线')
    axes[1].axhline(0, color='gray', linestyle='-', alpha=0.3)
    axes[1].axvline(0, color='gray', linestyle='-', alpha=0.3)
    
    # 添加特征标签
    for j, feat in enumerate(features_list):
        axes[1].annotate(feat.split('_')[0], (orig_skews[j], trans_skews[j]), 
                        fontsize=7, alpha=0.7)
    
    axes[1].set_xlabel('原始偏度')
    axes[1].set_ylabel('变换偏度')
    axes[1].set_title('偏度变化散点图\n(绿=改善, 红=变差)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # 变换方法分布
    transform_counts = pd.Series(transforms).value_counts()
    colors = plt.cm.Set3(np.linspace(0, 1, len(transform_counts)))
    wedges, texts, autotexts = axes[2].pie(transform_counts.values, labels=transform_counts.index, 
                                            autopct='%1.0f%%', colors=colors, startangle=90)
    axes[2].set_title('变换方法分布')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/变换效果汇总.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"   📊 已生成 {n_pages} 页变换对比图 + 汇总图")


# ==================== 变换效果详细评估 ====================

def evaluate_transformations_v3(df_original, df_transformed, transform_info, output_dir='./step2_transform_v3'):
    """详细评估变换效果"""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    evaluation = []
    
    print("\n" + "="*70)
    print("📊 变换效果详细评估")
    print("="*70)
    
    for feature, info in transform_info.items():
        if feature not in df_original.columns:
            continue
        
        original = df_original[feature].replace([np.inf, -np.inf], np.nan).dropna()
        
        trans_col = f'{feature}_transformed'
        if trans_col not in df_transformed.columns:
            continue
        
        transformed = df_transformed[trans_col].replace([np.inf, -np.inf], np.nan).dropna()
        
        orig_skew = stats.skew(original)
        trans_skew = stats.skew(transformed)
        orig_kurt = stats.kurtosis(original)
        trans_kurt = stats.kurtosis(transformed)
        
        # Shapiro-Wilk正态性检验
        sample_size = min(5000, len(original))
        orig_sample = original.sample(sample_size, random_state=42) if len(original) > sample_size else original
        trans_sample = transformed.sample(sample_size, random_state=42) if len(transformed) > sample_size else transformed
        
        _, orig_shapiro_p = stats.shapiro(orig_sample)
        _, trans_shapiro_p = stats.shapiro(trans_sample)
        
        # 综合评估
        skew_score = 'A' if abs(trans_skew) <= 0.5 else ('B' if abs(trans_skew) <= 1.0 else ('C' if abs(trans_skew) < abs(orig_skew) else 'D'))
        
        evaluation.append({
            '特征': feature,
            '变换': info['transform'],
            '原始偏度': round(orig_skew, 3),
            '变换偏度': round(trans_skew, 3),
            '偏度改善': round(abs(orig_skew) - abs(trans_skew), 3),
            '原始峰度': round(orig_kurt, 3),
            '变换峰度': round(trans_kurt, 3),
            '原始正态p': f"{orig_shapiro_p:.2e}",
            '变换正态p': f"{trans_shapiro_p:.2e}",
            '评级': skew_score
        })
        
        # 打印详情
        grade_emoji = {'A': '🌟', 'B': '✅', 'C': '⚠️', 'D': '❌'}[skew_score]
        print(f"\n{grade_emoji} {feature} [{info['transform']}]:")
        print(f"   偏度: {orig_skew:.3f} → {trans_skew:.3f}")
        print(f"   峰度: {orig_kurt:.3f} → {trans_kurt:.3f}")
        print(f"   正态p: {orig_shapiro_p:.2e} → {trans_shapiro_p:.2e}")
    
    eval_df = pd.DataFrame(evaluation)
    eval_df.to_csv(f'{output_dir}/变换效果评估.csv', index=False, encoding='utf-8-sig')
    
    # 统计各评级数量
    grade_counts = eval_df['评级'].value_counts()
    
    print("\n" + "="*70)
    print("📋 评级统计:")
    print(f"   🌟 A级 (|偏度|≤0.5): {grade_counts.get('A', 0)}")
    print(f"   ✅ B级 (|偏度|≤1.0): {grade_counts.get('B', 0)}")
    print(f"   ⚠️ C级 (有改善): {grade_counts.get('C', 0)}")
    print(f"   ❌ D级 (未改善): {grade_counts.get('D', 0)}")
    print("="*70)
    
    return eval_df


# ==================== 主函数 ====================

def run_step2_transform_v4(df_feature, step1_results, TRANSFORM_OVERRIDES_V2, output_dir):
    """
    步骤2主函数（V3版）
    
    Parameters:
    -----------
    df_feature : pd.DataFrame, 原始特征数据
    step1_results : dict, 步骤1的检验结果
    output_dir : str, 输出目录
    
    Returns:
    --------
    df_transformed : pd.DataFrame, 变换后的数据
    transform_info : dict, 变换信息
    eval_df : pd.DataFrame, 变换效果评估
    """
    # 1. 应用变换
    df_transformed, transform_info, summary_df = apply_transformations_v3(
        df_feature, step1_results,TRANSFORM_OVERRIDES_V2, output_dir
    )
    
    # 2. 评估变换效果
    eval_df = evaluate_transformations_v3(
        df_feature, df_transformed, transform_info, output_dir
    )
    
    return df_transformed, transform_info, eval_df




In [ ]:

TRANSFORM_OVERRIDES_V2 = {
    # ========== 原始偏度已经很好，直接标准化 ==========
    'neg_ratio_post': {
        'transform': 'standardize',
        'reason': '原始偏度0.48已接近正态，Logit过度校正至-6.6，直接标准化'
    },
    'vis_concentration_post': {
        'transform': 'standardize', 
        'reason': '原始偏度0.93可接受，Logit过度校正至-5.1，直接标准化'
    },
    'comp_ratio_post': {
        'transform': 'standardize',
        'reason': '原始偏度0.56已接近正态，变换后反而变差，直接标准化'
    },
    
    # ========== 需要真正变换的 ==========
    'semantic_shift_post': {
        'transform': 'yeo_johnson',
        'reason': '原始偏度2.3需要变换，之前仅标准化无效，改用Yeo-Johnson'
    },
    
    # ========== 改善有限但可接受 ==========
    'neg_ratio_comment': {
        'transform': 'sqrt',
        'reason': '原始偏度1.02，使用温和的sqrt变换避免过度校正'
    },
    
    # ========== 其他需要调整的 ==========
    # 根据步骤1结果，以下特征零值比例高，需要特殊处理
    'gini_post': {
        'transform': 'log1p',
        'reason': '零值比例75%，使用log1p处理右偏'
    },
    'gini_comment': {
        'transform': 'log1p',
        'reason': '零值比例98%，使用log1p处理'
    },
    'origin_ratio_post': {
        'transform': 'sqrt',
        'reason': '零值比例59%，使用温和的sqrt变换'
    },
}



### run

In [ ]:
# df_transformed, transform_info, eval_df = run_step2_transform(
#     df_feature=df_features,
#     step1_results=step1_results,
#     output_dir='./step2_transform'
# )
# 运行V3版变换
df_transformed, transform_info, eval_df = run_step2_transform_v4(
    df_feature=df_feature_slice,  # 原始特征DataFrame
    step1_results=step1_results, # 步骤1的结果
    TRANSFORM_OVERRIDES_V2 = TRANSFORM_OVERRIDES_V2, # 特殊规则覆盖
    output_dir=os.path.join(RESULT_PATH, 'step2_transform') # 输出目录
)

## step3 周期拆解

In [ ]:
# from step1_3 import run_step3_prophet

### v2

In [ ]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from prophet import Prophet
from prophet.serialize import model_to_json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== 版本周期工具 ====================

def prepare_version_periods(version_dict,VERSION_ABBR,VERSION_COLORS):
    """
    将 version_dict 变成 DataFrame + changepoints 列表
    version_dict: { '大版本上半预热': ('2025-01-20 12:00', '2025-01-24 11:00'), ... }
    """
    periods = []
    changepoints = []

    for name, (start, end) in version_dict.items():
        start_time = pd.to_datetime(start)
        end_time = pd.to_datetime(end)
        periods.append({
            'name': name,
            'abbr': VERSION_ABBR.get(name, name[:4]),
            'start': start_time,
            'end': end_time,
            'duration_hours': (end_time - start_time).total_seconds() / 3600,
            'color': VERSION_COLORS.get(name, '#E0E0E0')
        })
        changepoints.append(start_time)

    periods_df = pd.DataFrame(periods).sort_values('start').reset_index(drop=True)

    print("="*70)
    print("📋 版本周期信息:")
    print("="*70)
    for _, row in periods_df.iterrows():
        print(f"   • {row['name']}")
        print(f"     {row['start'].strftime('%m/%d %H:%M')} ~ {row['end'].strftime('%m/%d %H:%M')} "
              f"({row['duration_hours']:.1f}小时)")

    return periods_df, changepoints


def add_version_background(ax, periods_df, y_min, y_max):
    """在图中画版本区间背景"""
    if periods_df is None or len(periods_df) == 0:
        return
    for _, row in periods_df.iterrows():
        rect = Rectangle(
            (mdates.date2num(row['start']), y_min),
            mdates.date2num(row['end']) - mdates.date2num(row['start']),
            y_max - y_min,
            facecolor=row['color'],
            alpha=0.2,
            edgecolor='none',
            zorder=0
        )
        ax.add_patch(rect)


# ==================== Prophet 拟合 & 分解 ====================

def fit_prophet_for_feature(series, changepoints=None,
                            weekly_seasonality=True,
                            daily_seasonality=True,
                            changepoint_prior_scale=0.1):
    """
    使用 Prophet 在【变换后的原始序列】上做趋势+季节性分解
    series: pd.Series, index 为 DatetimeIndex, 值为变换后的 y
    """
    # 清洗
    series_clean = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(series_clean) < 100:
        print("   ⚠️ 有效点数 < 100，跳过 Prophet 拟合")
        return None, None, None

    df_p = pd.DataFrame({'ds': series_clean.index, 'y': series_clean.values})

    # 过滤 changepoints 在数据范围内
    if changepoints:
        data_start = series_clean.index.min()
        data_end = series_clean.index.max()
        valid_changepoints = [cp for cp in changepoints if data_start <= cp <= data_end]
    else:
        valid_changepoints = None

    print(f"   有效版本变化点: {len(valid_changepoints) if valid_changepoints else 0}")

    # 初始化 Prophet
    model = Prophet(
        yearly_seasonality=False,          # 不做年季节性
        weekly_seasonality=weekly_seasonality,
        daily_seasonality=daily_seasonality,  # 日内季节性（对于 15min 数据）
        changepoints=valid_changepoints if valid_changepoints else None,
        changepoint_prior_scale=changepoint_prior_scale,
        interval_width=0.95
    )

    # 拟合
    model.fit(df_p)

    # 预测（样本内）
    forecast = model.predict(df_p[['ds']])

    # R²
    y_actual = df_p['y'].values
    y_pred = forecast['yhat'].values
    ss_res = np.sum((y_actual - y_pred) ** 2)
    ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

    print(f"   Prophet R² = {r2:.4f}")
    return model, forecast, r2


def build_decomp_from_prophet(series, forecast):
    """
    基于 Prophet 结果构建类似 STL 的分解结果字典
    输出给 Step9 使用（seasonality模板）、以及统计用。
    """
    series_clean = series.replace([np.inf, -np.inf], np.nan).dropna()
    df_fit = pd.DataFrame({'y': series_clean})
    df_fit['yhat'] = forecast['yhat'].values[:len(series_clean)]
    df_fit['trend'] = forecast['trend'].values[:len(series_clean)]

    # Prophet 的季节性列名可能有: 'weekly', 'daily'
    if 'daily' in forecast.columns:
        df_fit['seasonal_day'] = forecast['daily'].values[:len(series_clean)]
    else:
        df_fit['seasonal_day'] = 0.0

    if 'weekly' in forecast.columns:
        df_fit['seasonal_week'] = forecast['weekly'].values[:len(series_clean)]
    else:
        df_fit['seasonal_week'] = 0.0

    df_fit['resid'] = df_fit['y'] - df_fit['yhat']

    # 方差占比（仅做描述，不追求严格正交）
    total_var = df_fit['y'].var()
    if total_var <= 0:
        var_decomp = {'trend': 0, 'seasonal': 0, 'residual': 100}
    else:
        trend_var = df_fit['trend'].var()
        seas_var = (df_fit['seasonal_day'] + df_fit['seasonal_week']).var()
        resid_var = df_fit['resid'].var()
        var_decomp = {
            'trend': trend_var / total_var * 100,
            'seasonal': seas_var / total_var * 100,
            'residual': resid_var / total_var * 100
        }

    # 构造类似 STL 的结果结构
    result = {
        'trend_day': pd.Series(df_fit['trend'].values, index=series_clean.index),
        'seasonal_day': pd.Series(df_fit['seasonal_day'].values, index=series_clean.index),
        'seasonal_week': pd.Series(df_fit['seasonal_week'].values, index=series_clean.index),
        'resid_final': pd.Series(df_fit['resid'].values, index=series_clean.index),
        'var_decomposition_day': var_decomp,
        'n_points': len(series_clean),
        'date_range': (series_clean.index.min(), series_clean.index.max())
    }
    return result, df_fit


def analyze_version_effects_prophet(series, forecast, periods_df, feature_name):
    """
    对每个版本周期统计：
      - 期间内 y 均值 / std（变换域）
      - Prophet 残差均值 / std
      - 趋势起点/终点/变化量
    返回 DataFrame（多特征后面 concat）
    """
    series_clean = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(series_clean) == 0 or forecast is None:
        return pd.DataFrame([])

    df_tmp = pd.DataFrame({
        'ds': series_clean.index,
        'y': series_clean.values,
        'yhat': forecast['yhat'].values[:len(series_clean)],
        'trend': forecast['trend'].values[:len(series_clean)],
    })
    df_tmp['resid'] = df_tmp['y'] - df_tmp['yhat']

    stats_list = []
    for _, p in periods_df.iterrows():
        mask = (df_tmp['ds'] >= p['start']) & (df_tmp['ds'] < p['end'])
        sub = df_tmp[mask]
        if len(sub) == 0:
            continue
        stats_list.append({
            'feature': feature_name,
            'version': p['name'],
            'abbr': p['abbr'],
            'n_points': len(sub),
            'mean_value': sub['y'].mean(),
            'std_value': sub['y'].std(),
            'mean_resid': sub['resid'].mean(),
            'std_resid': sub['resid'].std(),
            'trend_start': sub['trend'].iloc[0],
            'trend_end': sub['trend'].iloc[-1],
            'trend_change': sub['trend'].iloc[-1] - sub['trend'].iloc[0]
        })
    return pd.DataFrame(stats_list)
# 提取季节性模板
def extract_and_save_profiles(feature, forecast, output_dir):
    """
    [修改原因]：为了在 Step 5 实现轻量级查表，不仅保存模型，而是显式提取季节性曲线。
    将 Prophet 预测结果中的 'daily' 和 'weekly' 提取为字典并保存为 JSON。
    """
    # 构造临时数据用于提取
    df_temp = pd.DataFrame({
        'ds': forecast['ds'],
        'daily': forecast['daily'] if 'daily' in forecast.columns else 0,
        'weekly': forecast['weekly'] if 'weekly' in forecast.columns else 0
    })

    # 1. 提取日内模板 (按 HH:MM 聚合，取均值)
    df_temp['time_str'] = df_temp['ds'].dt.strftime('%H:%M')
    daily_profile = df_temp.groupby('time_str')['daily'].mean()

    # 2. 提取周模板 (按 DayOfWeek 0-6 聚合，取均值)
    df_temp['dow'] = df_temp['ds'].dt.dayofweek
    weekly_profile = df_temp.groupby('dow')['weekly'].mean()

    # 构造字典
    profile_data = {
        'feature': feature,
        'daily': daily_profile.to_dict(),
        'weekly': weekly_profile.to_dict()  # key 是 int (0-6)
    }
    
    # 保存为 JSON
    # [说明] JSON 文件体积极小（几KB），任何语言都能读，且方便人工校验
    safe_name = feature.replace('/', '_')
    save_path = os.path.join(output_dir, f'profile_{safe_name}.json')
    
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(profile_data, f, ensure_ascii=False, indent=4)
    
    print(f"      💾 季节性模板已保存: {save_path}")
    return profile_data

# ==================== 可视化 ====================

def plot_prophet_decomposition(series, forecast, feature_name,
                               r_squared, periods_df, output_dir):
    """
    绘制单个特征的 Prophet 分解图（原 STL 图的 Prophet 版）
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    series_clean = series.replace([np.inf, -np.inf], np.nan).dropna()
    df_plot = pd.DataFrame({
        'ds': series_clean.index,
        'y': series_clean.values,
        'yhat': forecast['yhat'].values[:len(series_clean)],
        'trend': forecast['trend'].values[:len(series_clean)],
    })
    if 'weekly' in forecast.columns:
        df_plot['weekly'] = forecast['weekly'].values[:len(series_clean)]
    else:
        df_plot['weekly'] = 0.0
    if 'daily' in forecast.columns:
        df_plot['daily'] = forecast['daily'].values[:len(series_clean)]
    else:
        df_plot['daily'] = 0.0
    df_plot['resid'] = df_plot['y'] - df_plot['yhat']

    fig = plt.figure(figsize=(20, 14))

    # 1) 原始+拟合
    ax1 = fig.add_subplot(4, 1, 1)
    ax1.plot(df_plot['ds'], df_plot['y'], 'b-', linewidth=0.6, alpha=0.7, label='实际值')
    ax1.plot(df_plot['ds'], df_plot['yhat'], 'r-', linewidth=1.0, alpha=0.9, label='Prophet拟合')
    y_min = df_plot['y'].min() - df_plot['y'].std()
    y_max = df_plot['y'].max() + df_plot['y'].std()
    add_version_background(ax1, periods_df, y_min, y_max)
    ax1.set_title(f'{feature_name} - Prophet拟合 (R²={r_squared:.4f})', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax1.tick_params(axis='x', rotation=30)

    # 2) 趋势
    ax2 = fig.add_subplot(4, 1, 2)
    ax2.plot(df_plot['ds'], df_plot['trend'], 'orange', linewidth=1.0)
    t_min = df_plot['trend'].min() - df_plot['trend'].std()
    t_max = df_plot['trend'].max() + df_plot['trend'].std()
    add_version_background(ax2, periods_df, t_min, t_max)
    ax2.set_title('趋势 (Trend)', fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax2.tick_params(axis='x', rotation=30)

    # 3) 周季节性
    ax3 = fig.add_subplot(4, 1, 3)
    # 按星期几聚合 weekly
    weekly = df_plot.set_index('ds')['weekly']
    weekly_pattern = weekly.groupby(weekly.index.dayofweek).mean()
    dow_labels = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
    ax3.bar(range(7), weekly_pattern.values, color='steelblue', alpha=0.7)
    ax3.set_xticks(range(7))
    ax3.set_xticklabels(dow_labels)
    ax3.set_ylabel('季节性')
    ax3.set_title('周季节性 (Weekly)', fontsize=11)
    ax3.axhline(0, color='black', linewidth=0.5)
    ax3.grid(True, alpha=0.3, axis='y')

    # 4) 日内季节性
    ax4 = fig.add_subplot(4, 1, 4)
    daily = df_plot.set_index('ds')['daily']
    hourly_pattern = daily.groupby(daily.index.hour + daily.index.minute/60).mean()
    ax4.plot(hourly_pattern.index, hourly_pattern.values, 'g-', linewidth=1.5)
    ax4.fill_between(hourly_pattern.index, hourly_pattern.values, alpha=0.3, color='green')
    ax4.set_xlabel('小时')
    ax4.set_ylabel('季节性')
    ax4.set_title('日内季节性 (Daily)', fontsize=11)
    ax4.set_xticks(range(0, 25, 3))
    ax4.axhline(0, color='black', linewidth=0.5)
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/{feature_name}_Prophet分解.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'   📊 图片已保存: {output_dir}/{feature_name}_Prophet分解.png')


# ==================== 主函数：合并版 Step3_Prophet ====================

def run_step3_prophet(
    df_transformed,       # 步骤2输出（含 feature_transformed 等），索引为 DatetimeIndex
    feature_list,
    version_dict=None,   # {版本名: (start, end)}，可选；若无版本分析则传 None
    VERSION_ABBR=None,
    VERSION_COLORS=None,
    output_dir=None
):
    """
    合并版 Step3：
      - 用 Prophet 对各特征做趋势+季节性分解（替代 STL）
      - 在变换后的原始序列上做版本周期统计（替代原 Step5）
      - 生成 Prophet 残差列（供 Step4 事件冲击、Step6 GARCH 使用）
      - 生成 decomp_results（供 Step9 合成中抽取季节性模板）

    返回：
      prophet_results : {feature: {model, forecast, r_squared}}
      df_prophet_resid : DataFrame，在 df_transformed 基础上增加：
                         - feature_resid（Prophet 残差）
                         - feature_prophet_resid（同上）
      decomp_results : {feature: {...}}，包含 seasonal_day/seasonal_week 等
      all_version_stats_df : 所有特征×版本的统计明细（Step9 生成 version_offsets 用）
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    print("="*70)
    print("📌 合并版 Step3：Prophet 分解 + 版本统计")
    print("="*70)

    # 确保时间索引
    # if not isinstance(df_transformed.index, pd.DatetimeIndex):
    #     df_transformed = df_transformed.copy()
    #     df_transformed.index = pd.to_datetime(df_transformed.index)

    print(f"   数据范围: {df_transformed.index.min()} ~ {df_transformed.index.max()}")
    print(f"   数据点数: {len(df_transformed)}")

    # 版本周期信息
    if version_dict is not None:
        periods_df, version_changepoints = prepare_version_periods(version_dict,VERSION_ABBR,VERSION_COLORS)
    else:
        periods_df, version_changepoints = pd.DataFrame(), None

    prophet_results = {}
    df_prophet_resid = df_transformed.copy()
    decomp_results = {}
    all_version_stats = []

    for feature in feature_list:
        print(f"\n{'='*60}")
        print(f"🔧 Prophet分解: {feature}")
        print('='*60)

        # 选择用于 Prophet 的列：优先 *_transformed，其次原始列
        col_name = f'{feature}_transformed' if f'{feature}_transformed' in df_transformed.columns else feature
        if col_name not in df_transformed.columns:
            print(f"   ❌ 列 {col_name} 不存在，跳过")
            continue

        series = df_transformed[col_name]
        print(f"   使用列: {col_name}")
        print(f"   非空点数: {series.replace([np.inf, -np.inf], np.nan).dropna().shape[0]}")

        # Prophet 拟合
        model, forecast, r2 = fit_prophet_for_feature(
            series,
            changepoints=version_changepoints,
            weekly_seasonality=True,
            daily_seasonality=True,
            changepoint_prior_scale=0.1
        )
        if model is None:
            continue

        prophet_results[feature] = {
            'model': model,
            'forecast': forecast,
            'r_squared': r2,
            'column_used': col_name
        }
        
        safe_name = feature.replace('/', '_')
        # model_save_path = os.path.join(output_dir, f'prophet_model_{safe_name}.json')
        
        # with open(model_save_path, 'w') as f:
        #     f.write(model_to_json(model))
        extract_and_save_profiles(feature, forecast, output_dir)

        # 生成分解结果（给 Step9 用）
        decomp_result, df_fit = build_decomp_from_prophet(series, forecast)
        decomp_results[feature] = decomp_result

        # 残差序列：对齐到原 index，其他位置为 NaN
        resid_series = pd.Series(index=df_transformed.index, dtype=float)
        resid_series.loc[df_fit.index] = df_fit['resid'].values

        # 保存为两种命名：feature_resid（供 Step4）、feature_prophet_resid（供 Step6）
        df_prophet_resid[f'{feature}_resid'] = resid_series
        # df_prophet_resid[f'{feature}_prophet_resid'] = resid_series

        # 版本统计（若有 version_dict）
        if version_dict is not None and len(periods_df) > 0:
            v_stats = analyze_version_effects_prophet(
                series, forecast, periods_df, feature_name=feature
            )
            if len(v_stats) > 0:
                all_version_stats.append(v_stats)
                # 打印趋势变化最大的一段
                max_idx = v_stats['trend_change'].abs().idxmax()
                row = v_stats.loc[max_idx]
                direction = '↑上升' if row['trend_change'] > 0 else '↓下降'
                print(f"   📈 趋势变化最大: {row['version']} ({direction} Δ={row['trend_change']:.4f})")

        # 绘图
        # plot_prophet_decomposition(
        #     series, forecast, feature,
        #     r_squared=r2,
        #     periods_df=periods_df,
        #     output_dir=output_dir
        # )
    

    # 汇总版本统计
    if all_version_stats:
        all_version_stats_df = pd.concat(all_version_stats, ignore_index=True)
        all_version_stats_df.to_csv(
            f'{output_dir}/版本周期统计明细_Prophet.csv', index=False, encoding='utf-8-sig'
        )
    else:
        all_version_stats_df = pd.DataFrame()

    # 生成 Prophet 拟合汇总表
    summary_rows = []
    for feature, res in prophet_results.items():
        row = {
            '特征': feature,
            'Prophet_R²': f"{res['r_squared']:.4f}",
            '使用列': res['column_used']
        }
        # 若有版本统计，附上趋势变化最大的版本
        if not all_version_stats_df.empty:
            vs = all_version_stats_df[all_version_stats_df['feature'] == feature]
            if not vs.empty:
                max_idx = vs['trend_change'].abs().idxmax()
                r = vs.loc[max_idx]
                row['最大变化版本'] = r['abbr']
                row['趋势变化'] = f"{r['trend_change']:.4f}"
                row['变化方向'] = '↑' if r['trend_change'] > 0 else '↓'
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(f'{output_dir}/Prophet分解汇总.csv', index=False, encoding='utf-8-sig')

    print("\n" + "="*70)
    print("✅ 合并版 Step3_Prophet 完成！")
    print(f"📁 结果保存至: {output_dir}")
    print("="*70)

    if not summary_df.empty:
        print("\n📋 Prophet分解汇总:")
        print(summary_df.to_string(index=False))

    return prophet_results, df_prophet_resid, decomp_results, all_version_stats_df, summary_df

In [ ]:

# ==================== 常量配置 ====================

PERIOD_DAY = 96     # 15分钟 × 96 = 1天
PERIOD_WEEK = 672   # 15分钟 × 672 = 1周

VERSION_COLORS = {
    '大版本上半预热': '#FFCDD2',
    '大版本上半更新': '#EF5350',
    '大版本下半预热': '#FFECB3',
    '大版本下半更新': '#FFA726',
    '小版本上半预热': '#C8E6C9',
    '小版本上半更新': '#66BB6A',
    '小版本下半预热': '#BBDEFB',
    '小版本下半更新': '#42A5F5',
}

VERSION_ABBR = {
    '大版本上半预热': '大上预',
    '大版本上半更新': '大上更',
    '大版本下半预热': '大下预',
    '大版本下半更新': '大下更',
    '小版本上半预热': '小上预',
    '小版本上半更新': '小上更',
    '小版本下半预热': '小下预',
    '小版本下半更新': '小下更',
}

# 版本周期字典
version_dict = {
    '大版本上半预热': ('2025-01-20 12:00:00', '2025-01-24 11:00:00'),
    '大版本上半更新': ('2025-01-24 11:00:00', '2025-02-11 12:00:00'),
    '大版本下半预热': ('2025-02-11 12:00:00', '2025-02-14 12:45:00'),
    '大版本下半更新': ('2025-02-14 12:45:00', '2025-02-22 12:00:00'),
    '小版本上半预热': ('2025-02-22 12:00:00', '2025-02-26 11:00:00'),
    '小版本上半更新': ('2025-02-26 11:00:00', '2025-03-09 12:00:00'),
    '小版本下半预热': ('2025-03-09 12:00:00', '2025-03-12 11:00:00'),
    '小版本下半更新': ('2025-03-12 11:00:00', '2025-03-20 12:00:00')
}

#### 论文图

In [ ]:
def plot_prophet_decomposition_comparison_v2(
    series_dict,
    forecast_dict,
    r2_dict,
    periods_df=None,
    output_path=None
):
    """
    图5.4: 两个特征的Prophet分解效果对比 (2列 × 4行)
    """
    features = list(series_dict.keys())
    assert len(features) == 2, "需要恰好两个特征进行对比"
    
    # ── 字号配置 ──
    # FONTSIZE_SUPTITLE = 13
    # FONTSIZE_COL_TITLE = 11.5
    # FONTSIZE_ROW_TITLE = 10
    # FONTSIZE_TICK = 8
    # FONTSIZE_LABEL = 9
    # FONTSIZE_LEGEND = 8
    # FONTSIZE_ANNOTATION = 8
    # FONTSIZE_DOW = 8.5
    FONTSIZE_SUPTITLE = 20
    FONTSIZE_COL_TITLE = 16
    FONTSIZE_ROW_TITLE = 16
    FONTSIZE_TICK = 16
    FONTSIZE_LABEL = 16
    FONTSIZE_LEGEND = 16
    FONTSIZE_ANNOTATION = 16
    FONTSIZE_DOW = 16
    
    fig, axes = plt.subplots(4, 2, figsize=(18, 16))
    fig.suptitle('图 5.4  Prophet 分解效果对比', 
                 fontsize=FONTSIZE_SUPTITLE, fontweight='bold', y=0.98)
    
    # 颜色方案
    COLOR_ACTUAL = '#4E79A7'
    COLOR_FIT = '#E15759'
    COLOR_TREND = '#F28E2B'
    COLOR_WEEKLY_BAR = '#4E79A7'
    COLOR_DAILY_LINE = '#59A14F'
    COLOR_DAILY_FILL = '#59A14F'
    
    subtitles = ['(a)', '(b)']
    
    for col_idx, feat_name in enumerate(features):
        series = series_dict[feat_name]
        forecast = forecast_dict[feat_name]
        r2 = r2_dict[feat_name]
        
        # 数据准备
        series_clean = series.replace([np.inf, -np.inf], np.nan).dropna()
        n_pts = len(series_clean)
        
        # 【修复1】：强制将 index 转换为 datetime 格式，写死 pd.to_datetime
        df_plot = pd.DataFrame({
            'ds': pd.to_datetime(series_clean.index), 
            'y': series_clean.values,
            'yhat': forecast['yhat'].values[:n_pts],
            'trend': forecast['trend'].values[:n_pts],
        })
        df_plot['weekly'] = (forecast['weekly'].values[:n_pts] 
                             if 'weekly' in forecast.columns 
                             else 0.0)
        df_plot['daily'] = (forecast['daily'].values[:n_pts]
                            if 'daily' in forecast.columns
                            else 0.0)
        df_plot['resid'] = df_plot['y'] - df_plot['yhat']
        
        # 列标题
        grade = '强周期性' if r2 > 0.4 else ('中等周期性' if r2 > 0.2 else '弱周期性')
        axes[0, col_idx].set_title(
            f'{subtitles[col_idx]} {feat_name}\n'
            f'{grade}，$R^2$={r2:.4f}',
            fontsize=FONTSIZE_COL_TITLE, fontweight='bold', pad=10
        )
        
        # ====== 第1行: 原始 + 拟合 ======
        ax = axes[0, col_idx]
        ax.plot(df_plot['ds'], df_plot['y'],
                color=COLOR_ACTUAL, linewidth=0.5, alpha=0.6, label='实际值')
        ax.plot(df_plot['ds'], df_plot['yhat'],
                color=COLOR_FIT, linewidth=1.2, alpha=0.9, label='Prophet拟合')
        
        y_margin = df_plot['y'].std() * 0.3
        y_min_bg = df_plot['y'].min() - y_margin
        y_max_bg = df_plot['y'].max() + y_margin
        add_version_background(ax, periods_df, y_min_bg, y_max_bg)
        
        ax.set_ylabel('变换域取值', fontsize=FONTSIZE_LABEL)
        ax.legend(fontsize=FONTSIZE_LEGEND, loc='upper right', framealpha=0.85)
        ax.grid(True, alpha=0.15)
        ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        ax.tick_params(axis='x', rotation=30)
        
        # ====== 第2行: 趋势 ======
        ax = axes[1, col_idx]
        ax.plot(df_plot['ds'], df_plot['trend'],
                color=COLOR_TREND, linewidth=1.8)
        
        t_margin = df_plot['trend'].std() * 0.5
        t_min = df_plot['trend'].min() - t_margin
        t_max = df_plot['trend'].max() + t_margin
        add_version_background(ax, periods_df, t_min, t_max)
        
        # 标注趋势变化幅度
        delta_g = df_plot['trend'].iloc[-1] - df_plot['trend'].iloc[0]
        direction = '↑' if delta_g > 0 else '↓'
        ax.annotate(
            f'$\\Delta g$={delta_g:+.3f} {direction}',
            xy=(0.95, 0.9), xycoords='axes fraction',
            ha='right', va='top',
            fontsize=FONTSIZE_ANNOTATION,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor=COLOR_TREND, alpha=0.85)
        )
        
        ax.set_ylabel('趋势分量', fontsize=FONTSIZE_LABEL)
        ax.set_title('趋势项 $\\hat{g}(t)$', fontsize=FONTSIZE_ROW_TITLE)
        ax.grid(True, alpha=0.15)
        ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        ax.tick_params(axis='x', rotation=30)
        
        # ====== 第3行: 周季节性 (柱状图) ======
        ax = axes[2, col_idx]
        weekly = df_plot.set_index('ds')['weekly']
        # 现在已经是 DatetimeIndex 了，这里不会再报错
        weekly_pattern = weekly.groupby(weekly.index.dayofweek).mean()
        
        dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        bars = ax.bar(range(7), weekly_pattern.values,
                      color=COLOR_WEEKLY_BAR, alpha=0.7, edgecolor='white',
                      linewidth=0.8)
        
        # 高亮周末
        for idx_bar in [5, 6]:
            if idx_bar < len(bars):
                bars[idx_bar].set_color('#E15759')
                bars[idx_bar].set_alpha(0.7)
        
        ax.set_xticks(range(7))
        ax.set_xticklabels(dow_labels, fontsize=FONTSIZE_DOW)
        ax.set_ylabel('周期分量', fontsize=FONTSIZE_LABEL)
        ax.set_title('周季节性 $\\hat{s}_{\\mathrm{week}}(t)$',
                      fontsize=FONTSIZE_ROW_TITLE)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.grid(True, alpha=0.15, axis='y')
        ax.tick_params(axis='y', labelsize=FONTSIZE_TICK)
        
        # ====== 第4行: 日内季节性 (折线 + 填充) ======
        ax = axes[3, col_idx]
        daily = df_plot.set_index('ds')['daily']
        # 同理，提取 hour 也安全了
        hourly_frac = daily.index.hour + daily.index.minute / 60
        hourly_pattern = daily.groupby(hourly_frac).mean()
        
        ax.plot(hourly_pattern.index, hourly_pattern.values,
                color=COLOR_DAILY_LINE, linewidth=1.8)
        ax.fill_between(hourly_pattern.index, hourly_pattern.values,
                        alpha=0.2, color=COLOR_DAILY_FILL)
        
        # 标注峰值和谷值
        peak_hour = hourly_pattern.idxmax()
        trough_hour = hourly_pattern.idxmin()
        ax.annotate(
            f'峰值 {int(peak_hour)}:00',
            xy=(peak_hour, hourly_pattern.max()),
            xytext=(peak_hour - 3, hourly_pattern.max() * 1.1),
            fontsize=FONTSIZE_ANNOTATION,
            arrowprops=dict(arrowstyle='->', color=COLOR_DAILY_LINE, lw=1.2),
            color=COLOR_DAILY_LINE, fontweight='bold',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7)
        )
        ax.annotate(
            f'谷值 {int(trough_hour)}:00',
            xy=(trough_hour, hourly_pattern.min()),
            xytext=(trough_hour + 3, hourly_pattern.min() * 1.1),
            fontsize=FONTSIZE_ANNOTATION,
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.0),
            color='gray',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7)
        )
        
        ax.set_xlabel('时刻 (小时)', fontsize=FONTSIZE_LABEL)
        ax.set_ylabel('周期分量', fontsize=FONTSIZE_LABEL)
        ax.set_title('日内季节性 $\\hat{s}_{\\mathrm{day}}(t)$',
                      fontsize=FONTSIZE_ROW_TITLE)
        ax.set_xticks(range(0, 25, 3))
        ax.set_xticklabels([f'{h}:00' for h in range(0, 25, 3)],
                           fontsize=FONTSIZE_TICK)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.grid(True, alpha=0.15)
        ax.tick_params(axis='y', labelsize=FONTSIZE_TICK)
    
    # ── 版本图例 (如果有periods_df) ──
    if periods_df is not None and len(periods_df) > 0:
        from matplotlib.patches import Patch
        # 【修复2】：这里原代码写的是 'label'，但前面的 periods_df 生成函数里没有 'label'
        # 帮您改成了 'name'（或你需要的 'abbr'），避免报 KeyError
        legend_col = 'abbr' if 'abbr' in periods_df.columns else 'name'
        unique_labels = periods_df[[legend_col, 'color']].drop_duplicates()
        legend_patches = [
            Patch(facecolor=row['color'], alpha=0.25, label=row[legend_col])
            for _, row in unique_labels.iterrows()
        ]
        fig.legend(
            handles=legend_patches,
            loc='lower center', ncol=min(len(legend_patches), 6),
            fontsize=FONTSIZE_LEGEND, framealpha=0.85,
            bbox_to_anchor=(0.5, -0.02)
        )
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    
    if output_path:
        plt.savefig(output_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f'已保存: {output_path}')
    
    plt.show()
    plt.close()

#### run

In [ ]:
prophet_results, df_prophet_resid, decomp_results, all_version_stats, summary_df = run_step3_prophet(
    df_transformed.set_index(['timestamp']),       # 步骤2输出（含 feature_transformed 等），索引为 DatetimeIndex
    feature_list,
    version_dict=version_dict,   # {版本名: (start, end)}，可选；若无版本分析则传 None
    VERSION_ABBR=VERSION_ABBR,
    VERSION_COLORS=VERSION_COLORS,
    output_dir=os.path.join(RESULT_PATH, 'step3_prophet')
)


In [ ]:
periods_df, version_changepoints = prepare_version_periods(version_dict,VERSION_ABBR,VERSION_COLORS)
df_transformed_index = df_transformed.set_index('timestamp')
plot_prophet_decomposition_comparison_v2(
    series_dict={
        'total_volume_post': df_transformed_index['total_volume_post_transformed'],       # pd.Series with DatetimeIndex
        'semantic_shift_comment': df_transformed_index['semantic_shift_comment_transformed'],
    },
    forecast_dict={
        'total_volume_post': prophet_results.get('total_volume_post', {}).get('forecast', 0.0),    # Prophet forecast DataFrame
        'semantic_shift_comment': prophet_results.get('semantic_shift_comment', {}).get('forecast', 0.0),  
    },
    r2_dict={
        'total_volume_post': prophet_results.get('total_volume_post', {}).get('r_squared', 0.0),
        'semantic_shift_comment': prophet_results.get('semantic_shift_comment', {}).get('r_squared', 0.0)
    },
    periods_df=periods_df,  # 可选
    output_path='fig5_5_prophet_comparison.svg'
)

## step4 官方运营动作冲击

In [ ]:
# from step1_4 import run_step4_combo_impact

In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats
from scipy.optimize import curve_fit
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False



# ==================== 组合分析配置 ====================

# 为15种组合定义颜色
COMBO_COLORS = plt.cm.tab20(np.linspace(0, 1, 20))

# 事件类型标记样式
CATEGORY_MARKERS = {
    'promotion_other': 'o',
    'promotion_normal': 'o',      # 圆形 - 推广
    'promotion_main': 'o',      # 圆形 - 推广
                # 圆形 - 推广
    'crisis': 'X',         # X形 - 危机
    'guide': 's',          # 方形 - 引导
    'maintenance': 'D',    # 菱形 - 维护
    'daily': '^',          # 三角 - 日常
}

# 事件类型简称
CATEGORY_LABELS = {
    'promotion_other': '推广-其他',
    'promotion_normal': '推广-影响力一般',
    'promotion_main': '推广-影响力较大',

    'crisis': '道歉公告',
    'guide': '攻略',
    'maintenance': '维护公告',
    'daily': '日常互动',
}


# ==================== 衰减函数 ====================

def exponential_decay(t, a, tau):
    """指数衰减: a * exp(-t/tau)"""
    return a * np.exp(-t / (tau + 1e-6))

def power_decay(t, a, alpha):
    """幂律衰减: a * (t+1)^(-alpha)"""
    return a * np.power(t + 1, -alpha)

def fit_decay(time_points, response_values, decay_type='exponential'):
    """拟合衰减曲线"""
    if len(time_points) < 3 or len(response_values) < 3:
        return None, 0
    
    try:
        if decay_type == 'exponential':
            a0 = np.abs(response_values[0]) if len(response_values) > 0 else 1
            tau0 = len(time_points) / 3
            popt, _ = curve_fit(exponential_decay, time_points, response_values, 
                               p0=[a0, tau0], maxfev=5000, 
                               bounds=([0, 0.1], [np.inf, 100]))
            fitted = exponential_decay(time_points, *popt)
        else:  # power
            a0 = np.abs(response_values[0]) if len(response_values) > 0 else 1
            popt, _ = curve_fit(power_decay, time_points, response_values,
                               p0=[a0, 0.5], maxfev=5000,
                               bounds=([0, 0.01], [np.inf, 3]))
            fitted = power_decay(time_points, *popt)
        
        ss_res = np.sum((response_values - fitted) ** 2)
        ss_tot = np.sum((response_values - np.mean(response_values)) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        
        return popt, max(0, r_squared)
    except:
        return None, 0


# ==================== 组合分析核心函数 ====================

def prepare_combo_event_data(df_off):
    """
    准备官方事件数据，创建组合变量
    
    Parameters:
    -----------
    df_off : pd.DataFrame, 官方事件数据
             需要包含列: timestamp, category, official_author
    
    Returns:
    --------
    df_events : 处理后的事件数据（含组合变量）
    combo_stats : 组合统计信息
    """
    df_events = df_off.copy()
    
    # 确保timestamp是datetime类型
    df_events['timestamp'] = pd.to_datetime(df_events['timestamp'])
    
    # 创建组合变量: author_category
    df_events['combo'] = df_events['official_author'] + '_' + df_events['category']
    
    # 按时间排序
    df_events = df_events.sort_values('timestamp').reset_index(drop=True)
    
    # 统计各组合
    combo_stats = df_events['combo'].value_counts().reset_index()
    combo_stats.columns = ['组合', '事件数']
    
    print("="*60)
    print("📋 事件组合统计:")
    print("="*60)
    print(f"   总事件数: {len(df_events)}")
    print(f"   组合类型数: {df_events['combo'].nunique()}")
    print("\n   各组合事件数:")
    for _, row in combo_stats.iterrows():
        print(f"      • {row['组合']}: {row['事件数']}次")
    
    # 创建组合颜色映射
    combos = df_events['combo'].unique()
    print(combos)
    combo_color_map = {combo: COMBO_COLORS[i % 20] for i, combo in enumerate(combos)}
    
    return df_events, combo_stats, combo_color_map



# ==================== 动态衰减回归分析 ====================
def determine_best_decay_type(df_feature, df_events, feature_name, window_after=32):
    """
    既确定最优衰减类型，也计算冲击统计量 (兼容旧接口)
    """
    col_name = f'{feature_name}_resid' if f'{feature_name}_resid' in df_feature.columns else feature_name
    series = df_feature[col_name]
    
    # 初始化完整的返回结构
    result_structure = {
        'by_combo': {},
        'column_used': col_name
    }
    
    combo_configs = {} # 用于回归
    
    for combo in df_events['combo'].unique():
        combo_events = df_events[df_events['combo'] == combo]
        impacts = []
        
        # 1. 提取响应曲线
        for _, event in combo_events.iterrows():
            start = event['timestamp']
            end = start + pd.Timedelta(minutes=15 * window_after)
            segment = series[(series.index >= start) & (series.index < end)].values
            if len(segment) > 0:
                if len(segment) < window_after:
                    segment = np.pad(segment, (0, window_after-len(segment)), 'constant')
                impacts.append(segment[:window_after])
        
        if not impacts:
            continue

        # 计算统计量
        mean_impact = np.nanmean(impacts, axis=0)
        # 寻找峰值（绝对值最大点）
        peak_idx = np.argmax(np.abs(mean_impact))
        overall_peak = mean_impact[peak_idx]
        peak_time_hours = peak_idx * 15 / 60
        
        # 准备拟合数据 (去除基线)
        mean_impact_fit = np.abs(mean_impact - mean_impact[0])
        t = np.arange(len(mean_impact_fit))
        
        # 2. 拟合比较
        popt_exp, r2_exp = fit_decay(t, mean_impact_fit, 'exponential')
        popt_pow, r2_pow = fit_decay(t, mean_impact_fit, 'power')
        
        # 3. 择优
        decay_info = {}
        if r2_pow > r2_exp + 0.05:
            config = {'type': 'power', 'param': popt_pow[1]}
            decay_info = {
                'type': 'power', 'params': popt_pow, 'r_squared': r2_pow,
                'half_life_min': np.nan # 幂律衰减半衰期定义复杂，暂略
            }
        else:
            tau = popt_exp[1] if popt_exp is not None else 8.0
            config = {'type': 'exponential', 'param': tau}
            decay_info = {
                'type': 'exponential', 'params': popt_exp, 'r_squared': r2_exp,
                'half_life_min': tau * np.log(2) * 15
            }

        combo_configs[combo] = config
        
        # 4. 填充 result_structure (兼容绘图函数)
        result_structure['by_combo'][combo] = {
            'mean_impact': mean_impact,
            'overall_peak': overall_peak,
            'peak_time': peak_idx,
            'peak_time_hours': peak_time_hours,
            'n_events': len(impacts),
            'decay': decay_info,
            'config': config # 额外保存config供回归使用
        }
            
    return result_structure

def create_dynamic_event_indicators(df_feature, df_events, combo_configs, window_after=32):
    """
    根据每个组合的最优配置，生成对应的衰减指示变量
    """
    df = df_feature.copy()
    event_columns = []
    
    for combo, config in combo_configs.items():
        combo_events = df_events[df_events['combo'] == combo]
        
        safe_combo = combo.replace(' ', '_').replace('-', '_')
        decay_col = f'event_{safe_combo}_{config["type"]}' # 列名带上类型
        
        df[decay_col] = 0.0
        decay_param = config['param']
        
        # 预计算衰减模板 (加速)
        t_template = np.arange(window_after + 1)
        if config['type'] == 'exponential':
            # exp(-t/tau)
            decay_template = np.exp(-t_template / (decay_param + 1e-6))
        else:
            # (t+1)^(-alpha)
            decay_template = np.power(t_template + 1, -decay_param)
            
        for _, event in combo_events.iterrows():
            event_time = event['timestamp']
            
            # 找到索引位置
            try:
                start_idx = df.index.get_indexer([event_time], method='nearest')[0]
            except:
                continue
                
            if start_idx < 0 or start_idx >= len(df): continue
            
            end_idx = min(start_idx + window_after, len(df))
            length = end_idx - start_idx
            
            # 叠加模板
            df.iloc[start_idx:end_idx, df.columns.get_loc(decay_col)] += decay_template[:length]
            
        event_columns.append(decay_col)
        
    return df, event_columns


def combo_regression_analysis_dynamic(df_feature, feature_name, df_events, use_residual=True):
    """
    带组合事件变量的回归分析:动态衰减回归分析
    """
    # 确定Y变量
    if use_residual and f'{feature_name}_resid' in df_feature.columns:
        y_col = f'{feature_name}_resid'
    elif f'{feature_name}_transformed' in df_feature.columns:
        y_col = f'{feature_name}_transformed'
    else:
        y_col = feature_name
    
    if y_col not in df_feature.columns:
        return None
    
    print(f"\n   📊 回归分析 (Y: {y_col})")
    
    # 1. 确定最优衰减配置
    impact_analysis = determine_best_decay_type(df_feature, df_events, feature_name)
    # 提取 configs
    combo_configs = {}
    if 'by_combo' in impact_analysis:
        for combo, data in impact_analysis['by_combo'].items():
            combo_configs[combo] = data['config']
    
    if not combo_configs:
        return None
    
    # 2. 创建动态变量
    df_reg, event_cols = create_dynamic_event_indicators(df_feature, df_events, combo_configs)
    
    # 3. 回归 (OLS)
    y = df_reg[y_col].values
    X = df_reg[event_cols].values
    
    # 清洗
    valid_mask = ~(np.isnan(y) | np.isinf(y) | np.any(np.isnan(X) | np.isinf(X), axis=1))
    y_clean = y[valid_mask]
    X_clean = X[valid_mask]
    
    if len(y_clean) < 100 or X_clean.shape[1] == 0:
        print(f"   ⚠️ 数据不足或无有效事件变量")
        return None
    
    # 标准化X
    # X_mean = X_clean.mean(axis=0)
    # X_std = X_clean.std(axis=0) + 1e-10
    # X_scaled = (X_clean - X_mean) / X_std
    
    # 添加常数项
    # X_with_const = sm.add_constant(X_scaled)
    X_with_const = sm.add_constant(X_clean)

    
    
    model = OLS(y_clean, X_with_const).fit()
    

    results = {
            'feature': feature_name,
            'r_squared': model.rsquared,
            'intercept': model.params[0], # 获取截距
            'coefficients': {},
            'configs': combo_configs,
            'significant_combos': []
        }

    # 计算残差并保存
    resid_clean = y_clean - model.fittedvalues
    resid_series = pd.Series(np.nan, index=df_feature.index)
    valid_index = df_feature.index[valid_mask]
    resid_series.loc[valid_index] = resid_clean
    resid_col = f'{feature_name}_event_adj_resid'
    df_feature[resid_col] = resid_series
    results['event_adj_resid_col'] = resid_col # 现在可以赋值了

    for i, col in enumerate(['const'] + event_cols):
        # 确保索引不过界 (statsmodels 的 params 顺序与 exog 对应)
        # 我们的 X_with_const 列顺序是 ['const'] + event_cols
        if col == 'const': 
            continue
        
        # 确保索引不越界 (防御性编程)
        if i >= len(model.params):
            break

        # 使用位置索引获取统计量
        coef = model.params[i]
        p_val = model.pvalues[i]
        t_score = model.tvalues[i]
        
        temp = col.replace('event_', '')
        decay_type = temp.split('_')[-1]
        combo_name = temp.replace(f'_{decay_type}', '')
        
        if p_val < 0.05:
            results['significant_combos'].append(combo_name)

        results['coefficients'][combo_name] = {
            'coef': coef,
            'pvalue': p_val,
            't_score': t_score,
            'decay_type': decay_type,
            'decay_param': combo_configs[combo_name]['param']
        }
        
    return results
        


# ==================== 可视化 ====================

def plot_combo_impact(df_feature, feature_name, df_events, 
                     impact_results, combo_color_map, output_dir):
    """
    绑定绘制组合事件冲击分析图
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # 确定使用的列
    col_name = f'{feature_name}_transformed'
    if col_name not in df_feature.columns:
        col_name = feature_name
    if col_name not in df_feature.columns:
        return
    
    series = df_feature[col_name]
    
    fig = plt.figure(figsize=(24, 20))
    
    # ========== 图1: 时序图 + 所有事件标记 ==========
    ax1 = fig.add_subplot(4, 2, 1)
    ax1.plot(series.index, series.values, linewidth=0.5, alpha=0.7, color='steelblue')
    
    # 标记事件
    for _, event in df_events.iterrows():
        event_time = event['timestamp']
        combo = event['combo']
        color = combo_color_map.get(combo, 'gray')
        cat = event['category']
        marker = CATEGORY_MARKERS.get(cat, 'o')
        
        if series.index.min() <= event_time <= series.index.max():
            ax1.axvline(event_time, color=color, alpha=0.4, linewidth=0.8)
    
    ax1.set_title(f'{feature_name} - 时序图与事件标记', fontsize=12, fontweight='bold')
    ax1.set_ylabel('值')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax1.tick_params(axis='x', rotation=30)
    ax1.grid(True, alpha=0.3)
    
    # ========== 图2: 所有组合的冲击响应曲线 ==========
    ax2 = fig.add_subplot(4, 2, 2)
    
    if 'by_combo' in impact_results and impact_results['by_combo']:
        for combo, data in impact_results['by_combo'].items():
            if len(data['mean_impact']) > 0:
                time_axis = np.arange(len(data['mean_impact'])) * 15 / 60  # 小时
                color = combo_color_map.get(combo, 'gray')
                
                ax2.plot(time_axis, data['mean_impact'], linewidth=2, 
                        label=f"{combo} (n={data['n_events']})", color=color, alpha=0.8)
        
        ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
        ax2.axvline(0, color='red', linestyle='--', linewidth=1, label='事件发生')
        ax2.set_xlabel('事件后时间 (小时)')
        ax2.set_ylabel('相对变化 (%)')
        ax2.set_title('各组合的平均冲击响应', fontsize=12)
        ax2.legend(fontsize=7, loc='upper right', ncol=2)
        ax2.grid(True, alpha=0.3)
    
    # ========== 图3: 冲击强度热力图 (author × category) ==========
    ax3 = fig.add_subplot(4, 2, 3)
    
    # 构建热力图数据
    authors = df_events['official_author'].unique()
    categories = df_events['category'].unique()
    
    heatmap_data = np.zeros((len(authors), len(categories)))
    heatmap_data[:] = np.nan
    
    for i, author in enumerate(authors):
        for j, cat in enumerate(categories):
            combo = f"{author}_{cat}"
            if combo in impact_results['by_combo']:
                heatmap_data[i, j] = impact_results['by_combo'][combo]['overall_peak']
    
    # 绘制热力图
    im = ax3.imshow(heatmap_data, cmap='RdBu_r', aspect='auto', 
                    vmin=-np.nanmax(np.abs(heatmap_data)), 
                    vmax=np.nanmax(np.abs(heatmap_data)))
    
    ax3.set_xticks(np.arange(len(categories)))
    ax3.set_yticks(np.arange(len(authors)))
    ax3.set_xticklabels([CATEGORY_LABELS.get(c, c) for c in categories], fontsize=9)
    ax3.set_yticklabels(authors, fontsize=9)
    
    # 添加数值标注
    for i in range(len(authors)):
        for j in range(len(categories)):
            if not np.isnan(heatmap_data[i, j]):
                text = ax3.text(j, i, f'{heatmap_data[i, j]:.1f}%',
                               ha='center', va='center', fontsize=8,
                               color='white' if abs(heatmap_data[i, j]) > np.nanmax(np.abs(heatmap_data))/2 else 'black')
    
    ax3.set_title('冲击强度热力图 (官方来源 × 事件类型)', fontsize=12)
    plt.colorbar(im, ax=ax3, label='峰值冲击 (%)')
    
    # ========== 图4: 按事件类型聚合的冲击 ==========
    ax4 = fig.add_subplot(4, 2, 4)
    
    if 'by_category' in impact_results and impact_results['by_category']:
        categories_sorted = sorted(impact_results['by_category'].items(), 
                                   key=lambda x: -abs(x[1]['mean_peak']))
        
        cats = [c[0] for c in categories_sorted]
        peaks = [c[1]['mean_peak'] for c in categories_sorted]
        n_events = [c[1]['n_events'] for c in categories_sorted]
        
        colors = ['green' if p > 0 else 'red' for p in peaks]
        bars = ax4.bar(range(len(cats)), peaks, color=colors, alpha=0.7, edgecolor='black')
        
        ax4.set_xticks(range(len(cats)))
        ax4.set_xticklabels([CATEGORY_LABELS.get(c, c) for c in cats], rotation=45, ha='right')
        ax4.set_ylabel('平均峰值冲击 (%)')
        ax4.set_title('按事件类型的平均冲击', fontsize=12)
        ax4.axhline(0, color='black', linewidth=0.5)
        ax4.grid(True, alpha=0.3, axis='y')
        
        # 标注事件数
        for i, (bar, n) in enumerate(zip(bars, n_events)):
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'n={n}', ha='center', va='bottom', fontsize=8)
    
    # ========== 图5: 按官方来源聚合的冲击 ==========
    ax5 = fig.add_subplot(4, 2, 5)
    
    if 'by_author' in impact_results and impact_results['by_author']:
        authors_sorted = sorted(impact_results['by_author'].items(),
                               key=lambda x: -abs(x[1]['mean_peak']))
        
        auths = [a[0] for a in authors_sorted]
        peaks = [a[1]['mean_peak'] for a in authors_sorted]
        n_events = [a[1]['n_events'] for a in authors_sorted]
        
        colors = ['green' if p > 0 else 'red' for p in peaks]
        bars = ax5.barh(range(len(auths)), peaks, color=colors, alpha=0.7, edgecolor='black')
        
        ax5.set_yticks(range(len(auths)))
        ax5.set_yticklabels(auths, fontsize=9)
        ax5.set_xlabel('平均峰值冲击 (%)')
        ax5.set_title('按官方来源的平均冲击', fontsize=12)
        ax5.axvline(0, color='black', linewidth=0.5)
        ax5.grid(True, alpha=0.3, axis='x')
        
        for i, (bar, n) in enumerate(zip(bars, n_events)):
            ax5.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                    f'n={n}', ha='left', va='center', fontsize=8)
    
    # ========== 图6: 冲击响应时间分布 ==========
    ax6 = fig.add_subplot(4, 2, 6)
    
    if 'by_combo' in impact_results:
        peak_times = [data['peak_time_hours'] for data in impact_results['by_combo'].values()
                     if 'peak_time_hours' in data]
        
        if peak_times:
            ax6.hist(peak_times, bins=15, alpha=0.7, edgecolor='black', color='steelblue')
            ax6.axvline(np.mean(peak_times), color='red', linestyle='--', 
                       label=f'平均响应时间: {np.mean(peak_times):.1f}h')
            ax6.set_xlabel('峰值响应时间 (小时)')
            ax6.set_ylabel('组合数')
            ax6.set_title('冲击响应时间分布', fontsize=12)
            ax6.legend()
            ax6.grid(True, alpha=0.3)
    
    # ========== 图7: 衰减曲线对比 ==========
    ax7 = fig.add_subplot(4, 2, 7)
    
    decay_plotted = False
    for combo, data in impact_results.get('by_combo', {}).items():
        if data.get('decay') and data['decay']['r_squared'] > 0.4:
            decay_info = data['decay']
            peak_idx = data['peak_time']
            
            if peak_idx >= len(data['mean_impact']) - 3:
                continue
            
            # 原始数据
            time_axis = np.arange(len(data['mean_impact']) - peak_idx) * 15 / 60
            values = np.abs(data['mean_impact'][peak_idx:])
            
            color = combo_color_map.get(combo, 'gray')
            ax7.scatter(time_axis, values, alpha=0.3, s=15, color=color)
            
            # 拟合曲线
            t_fit = np.linspace(0, time_axis[-1], 50)
            if decay_info['type'] == 'exponential':
                y_fit = exponential_decay(t_fit * 4, *decay_info['params'])
                halflife = decay_info.get('half_life_min', 0) / 60
                label = f"{combo}: τ½={halflife:.1f}h"
            else:
                y_fit = power_decay(t_fit * 4, *decay_info['params'])
                label = f"{combo}: α={decay_info['params'][1]:.2f}"
            
            ax7.plot(t_fit, y_fit, linewidth=2, color=color, label=label)
            decay_plotted = True
    
    if decay_plotted:
        ax7.set_xlabel('峰值后时间 (小时)')
        ax7.set_ylabel('|冲击强度| (%)')
        ax7.set_title('事件冲击衰减曲线', fontsize=12)
        ax7.legend(fontsize=7, loc='upper right')
        ax7.grid(True, alpha=0.3)
    else:
        ax7.text(0.5, 0.5, '无有效衰减曲线\n(R² > 0.4)', ha='center', va='center', fontsize=12)
        ax7.axis('off')
    
    # ========== 图8: 统计汇总 ==========
    ax8 = fig.add_subplot(4, 2, 8)
    
    summary_text = f"""
    特征: {feature_name}
    使用列: {impact_results.get('column_used', 'N/A')}
    
    ═══════════════════════════════════
    组合事件冲击分析汇总
    ═══════════════════════════════════
    
    分析的组合数: {len(impact_results.get('by_combo', {}))}
    
    """
    
    # 按冲击强度排序
    if 'by_combo' in impact_results:
        sorted_combos = sorted(impact_results['by_combo'].items(),
                              key=lambda x: -abs(x[1]['overall_peak']))
        
        summary_text += "冲击强度排名 (Top 5):\n"
        for i, (combo, data) in enumerate(sorted_combos[:5]):
            direction = "↑" if data['overall_peak'] > 0 else "↓"
            summary_text += f"  {i+1}. {combo}\n"
            summary_text += f"     {direction} {abs(data['overall_peak']):.1f}% @ {data['peak_time_hours']:.1f}h\n"
            summary_text += f"     事件数: {data['n_events']}\n"
            if data.get('decay') and data['decay']['r_squared'] > 0.3:
                if data['decay']['type'] == 'exponential':
                    halflife = data['decay'].get('half_life_min', 0)
                    summary_text += f"     半衰期: {halflife/60:.1f}h\n"
    
    ax8.text(0.02, 0.98, summary_text, transform=ax8.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax8.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{feature_name}_组合事件冲击分析.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"   📊 图片已保存: {output_dir}/{feature_name}_组合事件冲击分析.png")


# ==================== 主函数 ====================

def run_step4_combo_impact_v2(df_decomposed, feature_list, df_off,
                          use_residual=True, output_dir=None):
    """
    步骤4主函数：官方来源×事件类型 组合冲击分析
    
    Parameters:
    -----------
    df_decomposed : pd.DataFrame, 步骤3的输出（含分解成分）
    feature_list : list, 特征列表
    df_off : pd.DataFrame, 官方事件数据（需要timestamp, category, official_author）
    use_residual : bool, 是否优先使用残差
    output_dir : str, 输出目录
    
    Returns:
    --------
    all_impact_results : dict, 所有特征的组合冲击结果
    all_regression_results : dict, 回归分析结果
    summary_df : pd.DataFrame, 汇总表
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*70)
    print("📌 步骤4：官方来源×事件类型 组合冲击分析")
    print("="*70)
    
    # 准备事件数据
    df_events, combo_stats, combo_color_map = prepare_combo_event_data(df_off)
    
    # 保存组合统计
    combo_stats.to_csv(f'{output_dir}/事件组合统计.csv', index=False, encoding='utf-8-sig')
    
    all_impact_results = {}
    all_regression_results = {}
    
    for feature in feature_list:
        print(f"\n{'='*60}")
        print(f"🔧 分析特征: {feature}")
        print('='*60)
        
        # 组合冲击分析
        impact_result = determine_best_decay_type(
            df_decomposed, df_events, feature, window_after=32
        )
        
        if impact_result and 'by_combo' in impact_result and impact_result['by_combo']:
            all_impact_results[feature] = impact_result
            
            # 打印关键发现
            sorted_combos = sorted(impact_result['by_combo'].items(),
                                  key=lambda x: -abs(x[1]['overall_peak']))
            
            print("\n   📈 冲击强度排名:")
            for i, (combo, data) in enumerate(sorted_combos[:3]):
                direction = "↑正向" if data['overall_peak'] > 0 else "↓负向"
                print(f"      {i+1}. {combo}: {direction} {abs(data['overall_peak']):.1f}%")
            
            # 绑定绘图
            plot_combo_impact(df_decomposed, feature, df_events,
                            impact_result, combo_color_map, output_dir)
        else:
            print(f"   ⚠️ 无有效冲击数据")
        
        # 回归分析
        reg_result = combo_regression_analysis_dynamic(
            df_decomposed, feature, df_events, use_residual=use_residual
        )
        
        if reg_result:
            all_regression_results[feature] = reg_result
    
    # ========== 生成汇总表 ==========
    # ========== 1. 生成回归模型详细评估表 (人读) ==========
    # 这是一个长表，包含每个特征、每个组合的详细回归指标
    regression_details = []
    
    for feature, reg_res in all_regression_results.items():
        # 基础模型信息
        base_info = {
            '特征': feature,
            '拟合优度(R2)': round(reg_res['r_squared'], 4),
            '截距项(Intercept)': round(reg_res['intercept'], 6),
            '显著组合数': len(reg_res['significant_combos'])
        }
        
        # 遍历该特征下的所有组合
        for combo, coef_info in reg_res['coefficients'].items():
            row = base_info.copy()
            row.update({
                '组合名称': combo,
                '回归系数(Coef)': round(coef_info['coef'], 6),
                'P值(P-value)': round(coef_info['pvalue'], 4),
                '显著性': '⭐⭐' if coef_info['pvalue'] < 0.01 else ('⭐' if coef_info['pvalue'] < 0.05 else ''),
                '衰减类型': coef_info['decay_type'],
                '衰减参数': round(coef_info['decay_param'], 2)
            })
            regression_details.append(row)
            
    if regression_details:
        df_details = pd.DataFrame(regression_details)
        # 调整列顺序，让人更容易阅读
        cols = ['特征', '组合名称', '回归系数(Coef)', 'P值(P-value)', '显著性', 
                '拟合优度(R2)', '截距项(Intercept)', '衰减类型', '衰减参数']
        df_details = df_details[cols]
        df_details.to_csv(f'{output_dir}/回归模型详细评估.csv', index=False, encoding='utf-8-sig')
        print(f"   📝 详细评估表已保存: {output_dir}/回归模型详细评估.csv")

    # ========== 2. 导出合成所需的完整参数 JSON (机器读) ==========
    # 将 regression results 转换为 JSON 友好的格式
    # 结构: {feature: {intercept: ..., combos: {combo_name: {coef, decay...}}}}
    json_params = {}
    for feature, reg_res in all_regression_results.items():
        json_params[feature] = {
            'intercept': reg_res['intercept'],
            'r_squared': reg_res['r_squared'],
            'coefficients': reg_res['coefficients'] # 包含 coef, pvalue, type, param
        }
        
    import json
    with open(f'{output_dir}/regression_params_for_synthesis.json', 'w', encoding='utf-8') as f:
        json.dump(json_params, f, indent=2, ensure_ascii=False)
    print(f"   💾 合成参数文件已保存: {output_dir}/regression_params_for_synthesis.json")
    
    summary_data = []
    
    for feature in feature_list:
        row = {'特征': feature}
        
        if feature in all_impact_results:
            impact = all_impact_results[feature]
            
            # 找最强冲击的组合
            if impact['by_combo']:
                max_combo = max(impact['by_combo'].items(),
                               key=lambda x: abs(x[1]['overall_peak']))
                row['最强组合'] = max_combo[0]
                row['最大冲击%'] = f"{max_combo[1]['overall_peak']:.1f}"
                row['响应时间h'] = f"{max_combo[1]['peak_time_hours']:.1f}"
                
                # 衰减半衰期
                if max_combo[1].get('decay') and max_combo[1]['decay'].get('half_life_min'):
                    row['半衰期h'] = f"{max_combo[1]['decay']['half_life_min']/60:.1f}"
                else:
                    row['半衰期h'] = '-'
        
        if feature in all_regression_results:
            reg = all_regression_results[feature]
            row['R²'] = f"{reg['r_squared']:.4f}"
            row['显著组合数'] = len(reg['significant_combos'])
        
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(f'{output_dir}/组合事件冲击汇总.csv', index=False, encoding='utf-8-sig')
    
    # ========== 生成组合效应对比表 ==========
    combo_effect_data = []
    for feature in all_impact_results:
        for combo, data in all_impact_results[feature]['by_combo'].items():
            combo_effect_data.append({
                '特征': feature,
                '组合': combo,
                '事件数': data['n_events'],
                '峰值冲击%': round(data['overall_peak'], 2),
                '响应时间h': round(data['peak_time_hours'], 2),
                '方向': '正向' if data['overall_peak'] > 0 else '负向'
            })
    
    combo_effect_df = pd.DataFrame(combo_effect_data)
    combo_effect_df.to_csv(f'{output_dir}/全部组合效应明细.csv', index=False, encoding='utf-8-sig')
    
    print("\n" + "="*70)
    print("✅ 步骤4完成！")
    print(f"📁 结果保存至: {output_dir}")
    print("="*70)
    
    print("\n📋 组合事件冲击汇总:")
    print(summary_df.to_string(index=False))
    
    # 统计
    if combo_effect_data:
        print(f"\n📊 统计: 分析了 {len(all_impact_results)} 个特征 × {df_events['combo'].nunique()} 个组合")
        
        # 找出影响最大的组合
        combo_avg = combo_effect_df.groupby('组合')['峰值冲击%'].apply(lambda x: np.mean(np.abs(x))).sort_values(ascending=False)
        print("\n📈 平均冲击强度最大的组合 (Top 5):")
        for combo, avg_impact in combo_avg.head(5).items():
            print(f"   • {combo}: 平均|冲击|={avg_impact:.1f}%")
    
    return all_impact_results, all_regression_results, summary_df, df_decomposed



In [ ]:
df_official['category'].unique()
df_official['official_author'].unique()


### run

In [ ]:

impact_results, regression_results, impact_summary, df_impact_resid = run_step4_combo_impact_v2(
    df_decomposed=df_prophet_resid, # V2 整合周期拆解
    feature_list=feature_list,
    df_off=slice_data(df_official,start,end),
    use_residual=True,
    output_dir=os.path.join(RESULT_PATH, 'step4_event_impact')
)

### 论文图

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# ── 1. 全局中文与字体设置 ──
mpl.rcParams['font.family'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False
import textwrap
def plot_event_heatmap():
    """
    绘制事件组合对各个特征影响的热力图
    """
    print("📌 正在准备数据并绘制热力图...")
    
    # ── 2. 准备数据 ──
    data = {
        '事件组合': [
            '官方版本宣发', 
            '官方维护公告', 
            '官方日常互动', 
            '官方危机', 
            '官方攻略'
        ],
        'total_volume_post': [0.317089, 0.533409, -0.022688, 0.196998, 0.539664],
        'total_volume_comment': [0.195575, 0.190808, 0.269188, 0.028813, -0.135836],
        'comp_ratio_post': [0.210650, 0.261042, 0.017583, 0.051882, 0.732259],
        'neg_ratio_post': [0.103687, 0.449850, -0.221609, 1.133699, 0.503467],
        'semantic_shift_comment': [-0.167964, -0.252716, -0.191121, -0.175838, -0.822442],
        'vis_concentration_post': [-0.009426, -0.174555, -0.026309, 0.059233, -0.410688]
    # }
        # '广场流量': [0.317089, 0.533409, -0.022688, 0.196998, 0.539664],
        # '评论流量': [0.195575, 0.190808, 0.269188, 0.028813, -0.135836],
        # '广场文本压缩比': [0.210650, 0.261042, 0.017583, 0.051882, 0.732259],
        # '广场负面情绪占比': [0.103687, 0.449850, -0.221609, 1.133699, 0.503467],
        # '评论语义漂移': [-0.167964, -0.252716, -0.191121, -0.175838, -0.822442],
        # '广场图片集中度': [-0.009426, -0.174555, -0.026309, 0.059233, -0.410688]
    }
    
    # 创建 DataFrame 并将 '事件组合' 设为索引
    df = pd.DataFrame(data)
    df.set_index('事件组合', inplace=True)

    # df.index = df.index.str.replace('_', '\n', n=1)  # n=1 表示只替换第一个下划线
    df.index = [textwrap.fill(label, width=4) for label in df.index]
    # ── 3. 绘制热力图 ──
    # 创建画布，设置合适的尺寸
    plt.figure(figsize=(12, 6))
    
    # 绘制热力图
    # annot=True: 显示具体数值
    # fmt=".3f": 保留3位小数
    # cmap="RdBu_r": 蓝-白-红配色 (蓝负红正)
    # center=0: 以0为颜色分界点
    # linewidths=0.5: 单元格之间的网格线宽度
    ax = sns.heatmap(df, 
                     annot=True, 
                     fmt=".3f", 
                     cmap="RdBu_r", 
                     center=0, 
                     linewidths=0.5,
                     cbar_kws={'label': '数值大小'},
                     annot_kws={
                         "size": 18,      # 👈 修改这里：控制内部数字的大小（比如改成 14 会更大）
                         "weight": "bold" # 👈 可选：如果你想让数字加粗，可以加上这行
                     },)
    cbar = ax.collections[0].colorbar
    
    # 1. 修改颜色条的【刻度数字】大小
    cbar.ax.tick_params(labelsize=18) 
    
    # 2. 修改颜色条的【标题标签】内容及字体大小
    cbar.set_label('影响程度数值', size=18, weight='normal') # size控制大小，weight控制加粗
    # df.index = df.index.str.replace('_', '\n', n=1)
    # ── 4. 细节调整 ──
    plt.title('各官方事件组合对特征的影响热力图', fontsize=20, fontweight='normal', pad=15)
    plt.xlabel('特征', fontsize=18)
    plt.ylabel('事件组合', fontsize=18)
    
    # 调整 x 轴标签的角度，防止文字重叠
    plt.xticks(rotation=15, ha='right', fontsize=18)
    plt.yticks(rotation=0, fontsize=18)
    
    plt.tight_layout()
    
    # 如果你需要保存图片，可以取消下面这行的注释
    plt.savefig('event_heatmap.svg', dpi=300, bbox_inches='tight')
    
    plt.show()

# 执行画图函数
plot_event_heatmap()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm

mpl.rcParams['font.family'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False


def plot_event_impact_heatmap(coef_matrix, output_path=None):
    """
    图5.5: 典型事件组合 × 核心特征的冲击系数热力图
    
    Parameters
    ----------
    coef_matrix : pd.DataFrame
        行=事件组合名(中文标签), 列=特征名(中文标签), 
        值=回归系数(非显著项填NaN)
    output_path : str, optional
    """
    # ── 字号配置 ──
    FONTSIZE_TITLE = 13
    FONTSIZE_AXIS_LABEL = 10
    FONTSIZE_TICK_X = 9
    FONTSIZE_TICK_Y = 10
    FONTSIZE_ANNOT = 8.5
    FONTSIZE_CBAR = 9
    FONTSIZE_NOTE = 8
    
    n_rows, n_cols = coef_matrix.shape
    
    fig_width = max(10, n_cols * 1.4 + 3)
    fig_height = max(5, n_rows * 0.9 + 2.5)
    fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_height))
    
    # ── 数据处理 ──
    display_matrix = coef_matrix.copy()
    numeric_matrix = display_matrix.values.astype(float)
    
    # 对称色标：以0为中心
    max_abs = np.nanmax(np.abs(numeric_matrix))
    if max_abs == 0:
        max_abs = 1.0
    norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
    
    # ── 绘制热力图 ──
    # 用mask处理NaN（显示为灰色背景）
    masked_matrix = np.ma.masked_invalid(numeric_matrix)
    
    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad(color='#F5F5F5')  # NaN单元格的背景色
    
    im = ax.imshow(masked_matrix, cmap=cmap, norm=norm, aspect='auto')
    
    # ── 坐标轴 ──
    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(display_matrix.columns, fontsize=FONTSIZE_TICK_X,
                       rotation=35, ha='right')
    ax.set_yticklabels(display_matrix.index, fontsize=FONTSIZE_TICK_Y)
    
    # ── 数值标注 ──
    for i in range(n_rows):
        for j in range(n_cols):
            val = numeric_matrix[i, j]
            if np.isnan(val):
                # 非显著项标注为短横线
                ax.text(j, i, '—', ha='center', va='center',
                        fontsize=FONTSIZE_ANNOT - 1, color='#BDBDBD',
                        fontstyle='italic')
            else:
                # 根据背景色深浅选择文字颜色
                text_color = 'white' if abs(val) > max_abs * 0.55 else 'black'
                
                # 格式化：小数值用3位，大数值用2位
                if abs(val) >= 0.1:
                    text = f'{val:+.3f}'
                else:
                    text = f'{val:+.4f}'
                
                fontweight = 'bold' if abs(val) > max_abs * 0.7 else 'normal'
                ax.text(j, i, text, ha='center', va='center',
                        fontsize=FONTSIZE_ANNOT, color=text_color,
                        fontweight=fontweight)
    
    # ── 网格线 ──
    ax.set_xticks(np.arange(n_cols + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(n_rows + 1) - 0.5, minor=True)
    ax.grid(which='minor', color='white', linewidth=1.5)
    ax.tick_params(which='minor', bottom=False, left=False)
    
    # ── 颜色条 ──
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.85)
    cbar.set_label('回归系数 $\\hat{\\beta}_c$（变换域尺度）',
                   fontsize=FONTSIZE_CBAR)
    cbar.ax.tick_params(labelsize=FONTSIZE_ANNOT)
    
    # ── 标题与注释 ──
    ax.set_title('图 5.5  典型事件组合 × 核心特征的冲击系数矩阵\n'
                 '（仅显示 $p < 0.05$ 的显著项，"—"表示不显著）',
                 fontsize=FONTSIZE_TITLE, fontweight='bold', pad=15)
    
    ax.set_xlabel('舆情特征', fontsize=FONTSIZE_AXIS_LABEL, labelpad=10)
    ax.set_ylabel('官方事件组合', fontsize=FONTSIZE_AXIS_LABEL, labelpad=10)
    
    # 底部注释
    fig.text(0.5, -0.02,
             '注：正值（红色）表示事件发生后该特征上升，'
             '负值（蓝色）表示下降。系数量纲为变换域标准化尺度。',
             ha='center', fontsize=FONTSIZE_NOTE, color='gray',
             fontstyle='italic')
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=200, bbox_inches='tight', facecolor='white')
        print(f'已保存: {output_path}')
    
    plt.show()
    plt.close()


# ── 构建数据并调用 ──

# 定义事件组合标签映射
event_labels = {
    '无限暖暖_promotion_main': '主推宣发',
    '无限暖暖_maintenance': '维护公告',
    '无限暖暖_daily': '日常互动',
    '无限暖暖_crisis': '官方危机',
    '无限暖暖搬砖工_crisis': '子号危机',
    '无限暖暖_guide': '官方攻略',
}

# 定义特征标签映射
feature_labels = {
    'total_volume_post': '广场流量',
    'total_volume_comment': '评论流量',
    'senti_symbol_post': '情感符号帖',
    'neg_ratio_post': '负面情绪比',
    'gini_comment': '评论基尼系数',
    'comp_ratio_post': '文本压缩比',
    'semantic_shift_comment': '语义漂移(评论)',
    'total_long_comment': '长评论量',
}

# 从原始回归结果构建矩阵
# 以下数据直接从表格提取（仅p<0.05的值，否则为NaN）
data = {
    '广场流量':     [0.317, 0.533, np.nan, 0.197, 0.681, 0.540],
    '评论流量':     [0.196, 0.191, 0.269, np.nan, np.nan, np.nan],
    '情感符号帖':   [0.206, 0.230, np.nan, np.nan, 0.313, 0.350],
    '负面情绪比':   [0.104, 0.450, -0.222, 1.134, 0.283, np.nan],
    '评论基尼系数': [np.nan, 0.019, 0.019, np.nan, np.nan, 0.034],
    '文本压缩比':   [0.211, 0.261, np.nan, np.nan, np.nan, 0.732],
    '语义漂移(评论)': [-0.168, -0.253, -0.191, np.nan, np.nan, -0.822],
    '长评论量':     [0.077, 0.066, 0.096, 0.088, np.nan, np.nan],
}

index_labels = ['主推宣发', '维护公告', '日常互动', 
                '官方危机', '子号危机', '官方攻略']

coef_matrix = pd.DataFrame(data, index=index_labels)

plot_event_impact_heatmap(coef_matrix, output_path='fig5_5_event_impact_heatmap.png')

## step4.5 提取短期自相关性

In [ ]:
from step1_4_5 import run_step1_4_5_ar_filter

### run

In [ ]:
df_ar_resid, ar_results, summary_df  = run_step1_4_5_ar_filter(
    df_impact_resid,
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'step1_4.5_ar_filter')
  )

## step5 波动率建模

In [ ]:
from step1_5 import run_step5_garch_v3

### 修改

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from arch import arch_model
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== GARCH 配置 ====================

VERSION_COLORS = {
    '大版本上半预热': '#FFCDD2', '大版本上半更新': '#EF5350',
    '大版本下半预热': '#FFECB3', '大版本下半更新': '#FFA726',
    '小版本上半预热': '#C8E6C9', '小版本上半更新': '#66BB6A',
    '小版本下半预热': '#BBDEFB', '小版本下半更新': '#42A5F5',
}

VERSION_ABBR = {
    '大版本上半预热': '大上预', '大版本上半更新': '大上更',
    '大版本下半预热': '大下预', '大版本下半更新': '大下更',
    '小版本上半预热': '小上预', '小版本上半更新': '小上更',
    '小版本下半预热': '小下预', '小版本下半更新': '小下更',
}


# ==================== 数据准备 ====================

def prepare_garch_data(df, feature_name):
    """
    准备GARCH建模数据
    修改：优先使用 AR 残差 (_ar_resid)，假设均值回归已在 Step 1.4.5 完成
    """
    # 1. 优先查找 AR 残差
    col_name = f'{feature_name}_ar_resid'
    
    # 2. 如果没有，回退查找 Prophet 残差 (兼容旧流程，但会警告)
    if col_name not in df.columns:
        print(f"   ⚠️ 未找到 {col_name}，尝试查找 Prophet 残差...")
        for suffix in ['_prophet_resid', '_resid', '']:
            alt_col = f'{feature_name}{suffix}'
            if alt_col in df.columns:
                col_name = alt_col
                print(f"   ⚠️ 使用 {col_name} 代替 (建议先运行 AR 过滤)")
                break
    
    if col_name not in df.columns:
        print(f"   ❌ 找不到 {feature_name} 的残差数据列")
        return None
    
    print(f'   使用列: {col_name}')
    series = df[col_name].copy()
    
    # 清洗数据
    series = series.replace([np.inf, -np.inf], np.nan).dropna()
    
    # 检查数据质量
    if len(series) < 200:
        print(f"   ⚠️ 数据不足200条 ({len(series)})")
        return None
    
    # 标准化（GARCH对尺度敏感，转为百分比尺度或标准化尺度）
    mean_val = series.mean()
    std_val = series.std()
    
    if std_val == 0 or np.isnan(std_val):
        print(f"   ⚠️ 标准差为0或NaN")
        return None
    
    # 缩放: (x - mu) / sigma * 100
    # 乘以100是为了让优化器更容易收敛（避免数值过小）
    series_scaled = (series - mean_val) / std_val * 100  
    
    return series_scaled, mean_val, std_val


# ==================== GARCH 模型拟合 ====================

def fit_garch_models(series, max_p=2, max_q=2):
    """
    拟合多种GARCH模型配置，并在 正态分布 和 t分布 之间进行优选
    
    Returns:
    --------
    best_model : 最优模型结果对象
    best_name : 最优模型名称
    all_results : 所有模型的结果字典
    """
    all_results = {}
    best_aic = np.inf
    best_model = None
    best_name = None
    
    # 确保数据是numpy数组
    data = series.values if hasattr(series, 'values') else series
    
    # 定义要尝试的分布类型
    # Normal: 标准正态分布
    # t: Student's t 分布 (捕获厚尾特征)
    distributions = ['Normal', 't'] 
    
    # 定义模型配置生成器
    configs = []
    
    # 1. 标准 GARCH (遍历 p, q)
    for p in range(1, max_p + 1):
        for q in range(1, max_q + 1):
            configs.append({'vol': 'Garch', 'p': p, 'q': q, 'o': 0, 'name': f'GARCH({p},{q})'})
            
    # 2. EGARCH (非对称) - 固定 p=1, q=1 以减少计算量，通常足够
    configs.append({'vol': 'EGARCH', 'p': 1, 'q': 1, 'o': 0, 'name': 'EGARCH(1,1)'})
    
    # 3. GJR-GARCH (杠杆) - 固定 p=1, o=1, q=1
    configs.append({'vol': 'Garch', 'p': 1, 'q': 1, 'o': 1, 'name': 'GJR-GARCH(1,1,1)'})

    # 开始遍历所有组合
    for config in configs:
        for dist in distributions:
            try:
                # 构建模型名称
                full_name = f"{config['name']}-{dist}"
                
                # 初始化模型
                # mean='Constant' 或 'Zero'。因为输入已经是 AR 残差，理论上均值为0，但设为 Constant 更稳健
                model = arch_model(data, 
                                   vol=config['vol'], 
                                   p=config['p'], 
                                   q=config['q'], 
                                   o=config['o'],
                                   dist=dist,      # 关键修改：传入分布类型
                                   mean='Constant', 
                                   rescale=False)
                
                # 拟合
                result = model.fit(disp='off', show_warning=False)

                # 1. 提取参数 (Series 转 dict)
                params_dict = result.params.to_dict()
                # 记录结果
                # all_results[full_name] = {
                #     'model': result,
                #     'aic': result.aic,
                #     'bic': result.bic,
                #     'loglik': result.loglikelihood,
                #     'dist': dist
                # }

                record = {
                    'model': result,
                    'aic': result.aic,
                    'bic': result.bic,
                    'loglik': result.loglikelihood,
                    'dist': dist
                }

                record.update(params_dict)
                all_results[full_name] = record
                
                # 更新最优模型 (AIC 越小越好)
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_model = result
                    best_name = full_name
                    
            except Exception as e:
                # 某些复杂模型可能无法收敛，跳过
                pass
    
    if best_model is None:
        print("   ❌ 所有GARCH模型拟合失败")
        return None, None, None
    
    print(f"   最优模型: {best_name} (AIC={best_aic:.2f})")
    
    return best_model, best_name, all_results


def analyze_garch_results(model_result, series, model_name):
    """
    分析GARCH模型结果
    """
    analysis = {}
    params = model_result.params
    
    # 1. 基础数据
    analysis['cond_vol'] = model_result.conditional_volatility
    analysis['std_resid'] = model_result.std_resid
    analysis['aic'] = model_result.aic
    analysis['bic'] = model_result.bic
    
    # 2. 识别分布类型和参数
    is_t_dist = 't' in model_name or 'nu' in params
    analysis['distribution'] = 'Student-t' if is_t_dist else 'Normal'
    if is_t_dist:
        analysis['nu'] = params.get('nu', 5.0) # 自由度
        
    # 3. 计算持续性 (Persistence)
    # 对于 GARCH/GJR: Persistence = alpha + beta + 0.5*gamma
    # 对于 EGARCH: Persistence = beta
    if 'EGARCH' in model_name:
        analysis['persistence'] = abs(params.get('beta[1]', 0))
    else:
        alpha = params.get('alpha[1]', 0)
        beta = params.get('beta[1]', 0)
        gamma = params.get('gamma[1]', 0)
        analysis['persistence'] = alpha + beta + 0.5 * gamma
        
    analysis['has_clustering'] = analysis['persistence'] > 0.7
    
    # 4. 统计特征
    analysis['mean_vol'] = analysis['cond_vol'].mean()
    analysis['max_vol'] = analysis['cond_vol'].max()
    
    # 5. Ljung-Box 检验 (检验标准化残差平方的自相关 -> 检查 ARCH 效应是否消除)
    try:
        # 检验 std_resid^2
        lb2 = acorr_ljungbox(model_result.std_resid ** 2, lags=[10], return_df=True)
        analysis['lb_p_sq'] = float(lb2['lb_pvalue'].iloc[0])
        analysis['arch_cleared'] = analysis['lb_p_sq'] > 0.05
    except:
        analysis['lb_p_sq'] = np.nan
        analysis['arch_cleared'] = False
    
    return analysis


def compute_confidence_intervals(series, model_result, confidence_levels=[0.95, 0.99]):
    """
    计算动态置信区间
    修改：根据模型分布（Normal vs t）动态选择临界值
    """
    cond_vol = model_result.conditional_volatility
    mean_pred = model_result.params.get('mu', 0)
    
    # 判断是否为 t 分布
    params = model_result.params
    if 'nu' in params:
        dist_type = 't'
        nu = params['nu']
    else:
        dist_type = 'norm'
    
    ci_dict = {'timestamp': series.index[:len(cond_vol)], 'mean': mean_pred}
    
    for cl in confidence_levels:
        alpha = 1 - cl
        # 计算双尾临界值
        if dist_type == 't':
            # t分布的分位数
            crit_val = stats.t.ppf(1 - alpha/2, df=nu)
        else:
            # 正态分布的分位数
            crit_val = stats.norm.ppf(1 - alpha/2)
            
        ci_dict[f'upper_{int(cl*100)}'] = mean_pred + crit_val * cond_vol
        ci_dict[f'lower_{int(cl*100)}'] = mean_pred - crit_val * cond_vol
    
    ci_df = pd.DataFrame(ci_dict)
    ci_df.set_index('timestamp', inplace=True)
    
    return ci_df


# ==================== 可视化 ====================

def add_version_background(ax, periods_df, y_min, y_max):
    """添加版本周期背景"""
    if periods_df is None: return
    for _, row in periods_df.iterrows():
        rect = Rectangle(
            (mdates.date2num(row['start']), y_min),
            mdates.date2num(row['end']) - mdates.date2num(row['start']),
            y_max - y_min,
            facecolor=row['color'], alpha=0.2, edgecolor='none', zorder=0
        )
        ax.add_patch(rect)

def plot_garch_results(series, model_result, model_name, analysis, 
                      ci_df, periods_df, feature_name, output_dir):
    """
    绘制GARCH分析结果（包含 t 分布拟合展示）
    """
    os.makedirs(output_dir, exist_ok=True)
    
    fig = plt.figure(figsize=(20, 15))
    cond_vol = analysis['cond_vol']
    std_resid = analysis['std_resid']
    
    # ========== 图1: 原始残差 + 动态置信区间 ==========
    ax1 = fig.add_subplot(3, 2, 1)
    
    y_data = series.values[:len(cond_vol)]
    y_min, y_max = np.nanmin(y_data) * 1.1, np.nanmax(y_data) * 1.1
    
    add_version_background(ax1, periods_df, y_min, y_max)
    
    ax1.plot(series.index, y_data, linewidth=0.5, color='steelblue', label='AR残差')
    ax1.fill_between(ci_df.index, ci_df['lower_95'], ci_df['upper_95'],
                    alpha=0.3, color='orange', label='95% CI')
    ax1.fill_between(ci_df.index, ci_df['lower_99'], ci_df['upper_99'],
                    alpha=0.15, color='red', label='99% CI')
    
    ax1.set_ylim(y_min, y_max)
    ax1.set_title(f'{feature_name} - 动态波动区间 ({model_name})', fontsize=12, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # ========== 图2: 条件波动率 ==========
    ax2 = fig.add_subplot(3, 2, 2)
    vol_min, vol_max = cond_vol.min() * 0.9, cond_vol.max() * 1.1
    add_version_background(ax2, periods_df, vol_min, vol_max)
    
    ax2.plot(series.index, cond_vol, color='red', linewidth=1, label='条件波动率')
    ax2.axhline(cond_vol.mean(), color='black', linestyle='--', label='均值')
    
    ax2.set_title('条件波动率 (Conditional Volatility)', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # ========== 图3: 标准化残差分布 vs 理论分布 ==========
    ax3 = fig.add_subplot(3, 2, 3)
    
    ax3.hist(std_resid, bins=50, density=True, alpha=0.6, color='gray', label='标准化残差')
    x_range = np.linspace(std_resid.min(), std_resid.max(), 100)
    
    # 绘制正态分布参照
    ax3.plot(x_range, stats.norm.pdf(x_range), 'g--', linewidth=1.5, label='Normal(0,1)')
    
    # 如果模型是 t 分布，绘制拟合的 t 分布
    if analysis['distribution'] == 'Student-t':
        nu = analysis['nu']
        ax3.plot(x_range, stats.t.pdf(x_range, df=nu), 'r-', linewidth=2, label=f'Model t(df={nu:.1f})')
        
    ax3.set_title(f'残差分布 ({analysis["distribution"]})', fontsize=12)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # ========== 图4: Q-Q Plot ==========
    ax4 = fig.add_subplot(3, 2, 4)
    if analysis['distribution'] == 'Student-t':
        stats.probplot(std_resid, dist="t", sparams=(analysis['nu'],), plot=ax4)
        ax4.set_title(f'Q-Q Plot (vs t-dist df={analysis["nu"]:.1f})', fontsize=12)
    else:
        stats.probplot(std_resid, dist="norm", plot=ax4)
        ax4.set_title('Q-Q Plot (vs Normal)', fontsize=12)
    ax4.grid(True, alpha=0.3)

    # ========== 图5: ARCH 效应检验 (残差平方 ACF) ==========
    ax5 = fig.add_subplot(3, 2, 5)
    plot_acf(std_resid**2, ax=ax5, lags=30, alpha=0.05, title='标准化残差平方 ACF (检验ARCH消除)')
    ax5.grid(True, alpha=0.3)
    
    # ========== 图6: 模型汇总信息 ==========
    ax6 = fig.add_subplot(3, 2, 6)
    
    summary_text = f"""
    模型汇总: {feature_name}
    ──────────────────────────
    最优模型: {model_name}
    分布类型: {analysis['distribution']}
    AIC: {analysis['aic']:.1f}
    
    参数:
    """
    if 'nu' in analysis:
        summary_text += f"  自由度 (nu): {analysis['nu']:.2f} (厚尾特征)\n"
    summary_text += f"  持续性: {analysis.get('persistence', 0):.4f}\n"
    summary_text += f"  ARCH效应消除: {'✅ 是' if analysis.get('arch_cleared') else '❌ 否'}\n"
    
    # 异常统计
    upper_99 = ci_df['upper_99'].values
    lower_99 = ci_df['lower_99'].values
    anomalies = (y_data > upper_99) | (y_data < lower_99)
    summary_text += f"\n异常点 (超出99% CI): {anomalies.sum()} ({anomalies.mean():.2%})"
    
    ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11, va='top', family='monospace')
    ax6.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{feature_name}_GARCH分析.png', dpi=120)
    plt.close()
    
    return anomalies


# ==================== 主函数 ====================

def run_step5_garch(df_input, feature_list, version_dict=None, output_dir=None):
    """
    步骤5主函数：GARCH波动率建模 (融合优化版)
    
    Parameters:
    -----------
    df_input : pd.DataFrame, 包含 AR 残差 (_ar_resid) 的数据
    feature_list : list, 特征列表
    version_dict : dict, 版本周期字典
    output_dir : str, 输出目录
    
    Returns:
    --------
    garch_results : dict, GARCH建模详细结果
    df_output : pd.DataFrame, 增加了 _garch_std_resid 和 _garch_vol 列的数据
    summary_df : pd.DataFrame, 汇总表
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*70)
    print("📌 步骤5：GARCH 波动率建模 (融合优化版)")
    print("   - 输入: AR模型残差 (_ar_resid)")
    print("   - 模型: 自动选择 GARCH/EGARCH/GJR-GARCH + Normal/t-dist")
    print("   - 目标: 获取纯净的标准化残差 (用于Copula)")
    print("="*70)
    
    # 准备版本周期数据
    periods_df = None
    if version_dict is not None:
        periods = []
        for name, (start, end) in version_dict.items():
            periods.append({
                'name': name,
                'start': pd.to_datetime(start), 'end': pd.to_datetime(end),
                'color': VERSION_COLORS.get(name, '#E0E0E0')
            })
        periods_df = pd.DataFrame(periods).sort_values('start').reset_index(drop=True)
    
    garch_results = {}
    df_output = df_input.copy()
    summary_data = []
    
    for feature in feature_list:
        print(f"\n{'='*60}")
        print(f"🔧 GARCH建模: {feature}")
        print('='*60)
        
        # 1. 准备数据 (查找 AR 残差)
        result = prepare_garch_data(df_output, feature)
        if result is None:
            continue
        
        series_scaled, mean_val, std_val = result
        
        # 2. 拟合 GARCH 模型 (对比 Normal 和 t 分布)
        best_model, model_name, all_models = fit_garch_models(series_scaled)
        
        if best_model is None:
            continue
        params_dict = best_model.params.to_dict()
        params_str = ', '.join([f"{k}={v:.4f}" for k, v in params_dict.items()])
        
        # 3. 分析结果
        analysis = analyze_garch_results(best_model, series_scaled, model_name)
        
        # 4. 计算置信区间 (基于模型分布动态计算)
        ci_df = compute_confidence_intervals(series_scaled, best_model)
        
        # 5. [核心新增] 保存用于 Copula 的关键列
        # 标准化残差 (Std Resid) -> 应该是 i.i.d 的
        # 条件波动率 (Volatility)
        col_std_resid = f'{feature}_garch_std_resid'
        col_vol = f'{feature}_garch_vol'
        
        # 对齐索引赋值
        df_output.loc[series_scaled.index, col_std_resid] = analysis['std_resid']
        df_output.loc[series_scaled.index, col_vol] = analysis['cond_vol']
        
        # 6. 绘图
        anomalies_mask = plot_garch_results(
            series_scaled, best_model, model_name, analysis,
            ci_df, periods_df, feature, output_dir
        )
        
        # 7. 保存结果
        garch_results[feature] = {
            'model': best_model,
            'model_name': model_name,
            'analysis': analysis,
            'params': best_model.params.to_dict(),
            'ci_df': ci_df,
            'scaling': {
                'mean': mean_val, 
                'std': std_val, 
                'scale_factor': 100.0,
                'garch_mu': float(best_model.params.get('mu', 0.0))} # 记录缩放参数以便还原
        }
        
        # 汇总信息
        summary_data.append({
            '特征': feature,
            '最优模型': model_name,
            '分布': analysis['distribution'],
            'AIC': f"{analysis['aic']:.1f}",
            '自由度(nu)': f"{analysis.get('nu', 0):.2f}",
            '持续性': f"{analysis.get('persistence', 0):.4f}",
            'ARCH消除': '✅' if analysis.get('arch_cleared') else '❌',
            '异常比例': f"{anomalies_mask.mean():.2%}",
        })
        
    # 导出汇总
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(f'{output_dir}/GARCH建模汇总.csv', index=False, encoding='utf-8-sig')
    
    print("\n" + "="*70)
    print("✅ 步骤5完成！")
    print(f"📁 结果保存至: {output_dir}")
    print("="*70)
    print(summary_df.to_string(index=False))
    
    return garch_results, df_output, summary_df

### run

In [ ]:
# 执行步骤5：GARCH波动率建模
garch_results, df_garch_resid, garch_summary= run_step5_garch(
    df_ar_resid,  # 步骤5的输出
    feature_list=feature_list,
    version_dict=version_dict,
    output_dir=os.path.join(RESULT_PATH, 'step5_garch')

)

### 短期波动性test

In [ ]:
for feature_name in feature_list:
    if feature_name in garch_results:
        analysis_data = garch_results[feature_name]['analysis']
        print(f"特征: {feature_name}")
        
        # 定义一个处理函数：取前5项 + 格式化4位小数
        def print_top5_acf(label, data_dict):
            if not data_dict: return
            # 1. 提取前5个项目 (items() 转换为元组列表)
            top5_items = list(data_dict.items())[:5]
            # 2. 格式化并构建显示字符串
            formatted = ", ".join([f"'{k}': {v:.4f}" for k, v in top5_items])
            print(f"{label}: {{{formatted}}}")

        acf_std = analysis_data.get('acf_signature_std_resid')
        if acf_std:
            # 只打印短期 ACF 的前5项
            print_top5_acf("短期 ACF (前5项)", acf_std['acf_short'])
            # 季节性 ACF 通常只有 3 项，可以直接打印全部或同样限制
            print_top5_acf("季节性 ACF", acf_std['acf_seasonal'])
        
        acf_sq = analysis_data.get('acf_signature_std_resid_sq')
        if acf_sq:
            print_top5_acf("平方项 ACF (前5项)", acf_sq['acf_short'])
        print("-" * 30) # 分隔线方便阅读

## step7 异常检测+噪音归纳

In [ ]:
# from step1_7 import run_step7_clean_verify

### 修改

In [ ]:
garch_results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from scipy import stats
from scipy.stats import norm, t as t_dist, kstest
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ==================== 1. 配置与工具函数 ====================

VERSION_COLORS = {
    '大版本上半预热': '#FFCDD2', '大版本上半更新': '#EF5350',
    '大版本下半预热': '#FFECB3', '大版本下半更新': '#FFA726',
    '小版本上半预热': '#C8E6C9', '小版本上半更新': '#66BB6A',
    '小版本下半预热': '#BBDEFB', '小版本下半更新': '#42A5F5',
}

def add_version_background(ax, periods_df, y_min, y_max):
    """绘制版本背景色块"""
    if periods_df is None or periods_df.empty: return
    for _, row in periods_df.iterrows():
        rect = Rectangle(
            (mdates.date2num(row['start']), y_min),
            mdates.date2num(row['end']) - mdates.date2num(row['start']),
            y_max - y_min,
            facecolor=row['color'], alpha=0.15, edgecolor='none', zorder=0
        )
        ax.add_patch(rect)

def get_std_resid(garch_result):
    """从GARCH结果中提取标准化残差并对齐时间索引"""
    std_resid = garch_result['analysis']['std_resid']
    idx = garch_result['ci_df'].index
    
    if isinstance(std_resid, pd.Series):
        if not isinstance(std_resid.index, pd.DatetimeIndex):
            if len(std_resid) == len(idx):
                std_resid.index = idx
            else:
                m = min(len(std_resid), len(idx))
                std_resid = std_resid.iloc[:m]
                std_resid.index = idx[:m]
        return std_resid
    else:
        arr = np.asarray(std_resid, dtype=float)
        m = min(len(arr), len(idx))
        return pd.Series(arr[:m], index=idx[:m])

def check_stats(series, name):
    """计算核心统计检验指标"""
    clean_s = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean_s) < 20:
        return {'LB_p': np.nan, 'ARCH_p': np.nan, 'Norm_p': np.nan}
        
    res = {}
    try:
        lb = acorr_ljungbox(clean_s, lags=[10], return_df=True)
        res['LB_p'] = lb['lb_pvalue'].iloc[0]
        res['is_white'] = res['LB_p'] > 0.05
    except:
        res['LB_p'] = 0.0
        res['is_white'] = False
        
    try:
        lm = het_arch(clean_s, ddof=4)
        res['ARCH_p'] = lm[1]
        res['no_arch'] = res['ARCH_p'] > 0.05
    except:
        res['ARCH_p'] = 0.0
        res['no_arch'] = False
        
    res['mean'] = clean_s.mean()
    res['std'] = clean_s.std()
    res['skew'] = clean_s.skew()
    res['kurt'] = clean_s.kurtosis()
    
    return res

# ==================== 2. 核心：异常识别与温莎化清洗 ====================

def identify_and_winsorize(series, ci_df, garch_result, q_low=0.005, q_high=0.995):
    """识别异常并温莎化清洗"""
    params = garch_result['params']
    nu = params.get('nu', 100)
    
    limit_upper = t_dist.ppf(q_high, df=nu)
    limit_lower = t_dist.ppf(q_low, df=nu)
    
    mask_pos = series > limit_upper
    mask_neg = series < limit_lower
    anomaly_mask = mask_pos | mask_neg
    
    cleaned_series = series.copy()
    valid_data = series[~anomaly_mask]
    
    if len(valid_data) > 0:
        win_upper = valid_data.quantile(q_high)
        win_lower = valid_data.quantile(q_low)
    else:
        win_upper = limit_upper
        win_lower = limit_lower
        
    cleaned_series[mask_pos] = win_upper
    cleaned_series[mask_neg] = win_lower
    
    return cleaned_series, anomaly_mask, (limit_lower, limit_upper)

# ==================== 3. 主流程：处理、对比与绘图 ====================

def process_cleaning_comparison(df_input, garch_results, feature_list, version_dict=None, output_dir='./step7_clean_verify'):
    os.makedirs(output_dir, exist_ok=True)
    print("="*80)
    print("📌 步骤7：GARCH残差清洗、验证与相关性分析")
    print("   目标：温莎化处理异常值，对比清洗前后统计性质，生成Copula输入")
    print("="*80)
    
    periods_df = None
    if version_dict:
        periods = [{'name': k, 'start': pd.to_datetime(v[0]), 'end': pd.to_datetime(v[1]), 
                    'color': VERSION_COLORS.get(k, '#E0E0E0')} for k, v in version_dict.items()]
        periods_df = pd.DataFrame(periods).sort_values('start')
        
    comparison_stats = []
    cleaned_data_dict = {}
    anomaly_records_dict = {}
    
    for feature in feature_list:
        if feature not in garch_results: continue
        print(f"\n🔍 处理特征: {feature}")
        
        result = garch_results[feature]
        ci_df = result['ci_df']
        
        std_resid_raw = get_std_resid(result)
        std_resid_clean, mask, (lim_l, lim_h) = identify_and_winsorize(std_resid_raw, ci_df, result)
        
        cleaned_data_dict[feature] = std_resid_clean
        result['analysis']['std_resid_clean'] = std_resid_clean
        
        stat_raw = check_stats(std_resid_raw, 'Raw')
        stat_clean = check_stats(std_resid_clean, 'Clean')
        
        comparison_stats.append({
            '特征': feature,
            '异常点数': mask.sum(),
            '异常比例': f"{mask.sum()/len(std_resid_raw):.2%}",
            'Raw_LB_p': stat_raw['LB_p'],
            'Raw_ARCH_p': stat_raw['ARCH_p'],
            'Raw_Kurt': stat_raw['kurt'],
            'Clean_LB_p': stat_clean['LB_p'],
            'Clean_ARCH_p': stat_clean['ARCH_p'],
            'Clean_Kurt': stat_clean['kurt'],
            '白噪声改善': '✅' if stat_clean['LB_p'] > stat_raw['LB_p'] else '➖',
            '分布改善(峰度)': '✅' if abs(stat_clean['kurt']) < abs(stat_raw['kurt']) else '❌'
        })
        
        # --- 新增核心逻辑：构建异常 DataFrame ---
        if mask.sum() > 0:
            # 提取异常点的索引和值
            anom_series = std_resid_raw[mask]
            
            # 构建列表，判断类型
            anom_list = []
            for ts, val in anom_series.items():
                if val > lim_h:
                    a_type = 'positive_extreme'
                elif val < lim_l:
                    a_type = 'negative_extreme'
                else:
                    a_type = 'structural_break' # 兜底类型，虽然温莎化通常只处理极值
                
                anom_list.append({
                    'timestamp': ts,
                    'type': a_type,
                    'value': val # 保留值以便后续观察，虽然 mask 函数不需要
                })
            
            # 创建 DataFrame
            anomaly_df = pd.DataFrame(anom_list)
        else:
            # 如果没有异常，创建一个空的 DataFrame，包含必要的列
            anomaly_df = pd.DataFrame(columns=['timestamp', 'type', 'value'])
            
        # 存入字典
        anomaly_records_dict[feature] = anomaly_df
        
        # --- 绘图 ---
        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(3, 2)
        
        ax1 = fig.add_subplot(gs[0, :])
        y_min, y_max = std_resid_raw.min()*1.1, std_resid_raw.max()*1.1
        add_version_background(ax1, periods_df, y_min, y_max)
        ax1.plot(std_resid_raw.index, std_resid_raw, c='gray', alpha=0.5, lw=1, label='清洗前 (Raw)')
        ax1.plot(std_resid_clean.index, std_resid_clean, c='green', alpha=0.8, lw=1, ls='--', label='清洗后 (Winsorized)')
        if mask.any():
            anom_points = std_resid_raw[mask]
            ax1.scatter(anom_points.index, anom_points, c='red', s=20, marker='x', label='被清洗的异常点', zorder=5)
        ax1.set_title(f'{feature} - 残差清洗效果 (Winsorization)', fontsize=12, fontweight='bold')
        ax1.legend()

        ax2 = fig.add_subplot(gs[1, 0])
        sns.kdeplot(std_resid_raw, ax=ax2, color='gray', fill=True, alpha=0.3, label='清洗前分布')
        sns.kdeplot(std_resid_clean, ax=ax2, color='green', fill=False, lw=2, label='清洗后分布')
        x = np.linspace(-4, 4, 100)
        ax2.plot(x, norm.pdf(x), 'r:', label='标准正态 N(0,1)')
        ax2.set_title(f'分布形态修正 (峰度: {stat_raw["kurt"]:.2f} -> {stat_clean["kurt"]:.2f})')
        ax2.legend()

        ax3 = fig.add_subplot(gs[1, 1])
        plot_acf(std_resid_clean, ax=ax3, lags=20, alpha=0.05, title='清洗后残差 ACF')

        ax4 = fig.add_subplot(gs[2, :])
        vol_proxy_raw = (std_resid_raw**2).rolling(24).mean()
        vol_proxy_clean = (std_resid_clean**2).rolling(24).mean()
        ax4.plot(vol_proxy_raw.index, vol_proxy_raw, c='gray', alpha=0.5, label='清洗前波动')
        ax4.plot(vol_proxy_clean.index, vol_proxy_clean, c='green', lw=1.5, label='清洗后波动')
        ax4.set_title(f'波动率聚集残留检查')
        ax4.legend()
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/{feature}_resid_clean_verify.png', dpi=100)
        plt.close()

    df_comp = pd.DataFrame(comparison_stats)
    df_comp.to_csv(f'{output_dir}/残差清洗前后对比报告.csv', index=False, encoding='utf-8-sig')
    print("\n📋 清洗效果对比摘要:")
    print(df_comp[['特征', '异常比例', 'Raw_Kurt', 'Clean_Kurt', '白噪声改善']].to_string(index=False))
    
    return cleaned_data_dict,anomaly_records_dict

# ==================== 4. 步骤7后续：清洗后的相关性分析 ====================

def analyze_cleaned_correlation(cleaned_data_dict, output_dir):
    print("\n" + "="*80)
    print("📌 步骤7.5：基于清洗后数据的相关性分析")
    print("="*80)
    
    if not cleaned_data_dict: return None
    
    df_clean = pd.DataFrame(cleaned_data_dict).dropna()
    
    corr_pearson = df_clean.corr(method='pearson')
    corr_kendall = df_clean.corr(method='kendall')
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    mask = np.triu(np.ones_like(corr_pearson, dtype=bool))
    
    sns.heatmap(corr_pearson, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r', center=0, 
                square=True, ax=axes[0], vmin=-1, vmax=1)
    axes[0].set_title('Pearson 相关矩阵 (线性)')
    
    sns.heatmap(corr_kendall, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r', center=0, 
                square=True, ax=axes[1], vmin=-1, vmax=1)
    axes[1].set_title('Kendall Tau 相关矩阵 (Copula输入)')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/清洗后残差相关性矩阵.png', dpi=150)
    plt.close()
    
    corr_kendall.to_csv(f'{output_dir}/corr_matrix_kendall.csv', encoding='utf-8-sig')
    
    return corr_kendall,corr_pearson

# ==================== [核心修改] 5. 边缘分布拟合 (含非参数经验分布) ====================
def fit_marginal_distributions(cleaned_data_dict, output_dir, D_threshold=0.05):
    """
    对清洗后的残差进行分布拟合，确定Copula所需的边缘分布参数
    
    逻辑升级：
    1. 比较 AIC 选出最佳参数模型 (Normal vs Student-t)。
    2. 计算 KS 检验的 p-value 和 D-statistic。
    3. 判定逻辑：
       - 优先看 p-value：若 p > 0.05，完美通过。
       - 次看 D-statistic：若样本量大导致 p 很小，但 D < D_threshold (默认0.05)，则"豁免"通过。
       - 否则：回退到 Empirical (经验分布)。
    """
    print("\n" + "="*80)
    print(f"📌 步骤7.8：边缘分布拟合 (大样本策略: D < {D_threshold} 可豁免)")
    print("="*80)
    
    fit_results = []
    
    for feature, data in cleaned_data_dict.items():
        data = data.dropna()
        n_samples = len(data)
        if n_samples < 50: continue
        
        # --- A. 拟合参数模型 ---
        
        # 1. Normal
        mu, std = norm.fit(data)
        ll_n = np.sum(norm.logpdf(data, mu, std))
        aic_n = 2*2 - 2*ll_n
        ks_n_stat, ks_n_p = kstest(data, 'norm', args=(mu, std))
        
        # 2. Student-t
        try:
            params_t = t_dist.fit(data)
            df_t, loc_t, scale_t = params_t
            ll_t = np.sum(t_dist.logpdf(data, df_t, loc_t, scale_t))
            aic_t = 2*3 - 2*ll_t
            ks_t_stat, ks_t_p = kstest(data, 't', args=params_t)
        except:
            aic_t = np.inf
            ks_t_p = 0.0
            ks_t_stat = 1.0
            params_t = (np.nan, np.nan, np.nan)

        # --- B. 决策逻辑 ---
        
        # 1. 选出 AIC 更低的模型候选
        if aic_t < aic_n:
            cand_dist = 'Student-t'
            cand_params = f"df={params_t[0]:.2f}, loc={params_t[1]:.4f}, scale={params_t[2]:.4f}"
            cand_p = ks_t_p
            cand_D = ks_t_stat
        else:
            cand_dist = 'Normal'
            cand_params = f"loc={mu:.4f}, scale={std:.4f}"
            cand_p = ks_n_p
            cand_D = ks_n_stat
            
        # 2. 双重判定 (P值 + D值)
        is_statistically_valid = cand_p > 0.05
        is_practically_valid = cand_D < D_threshold
        
        if is_statistically_valid:
            final_dist = cand_dist
            final_params = cand_params
            decision = "✅ P值通过"
        elif is_practically_valid:
            final_dist = cand_dist
            final_params = cand_params
            decision = f"⚠️ D值豁免 (D={cand_D:.3f})"
        else:
            final_dist = 'Empirical'
            final_params = "Non-parametric (KDE/ECDF)"
            decision = f"❌ 拒绝 (p={cand_p:.2e}, D={cand_D:.3f})"
            
        fit_results.append({
            '特征': feature,
            '样本量': n_samples,
            'AIC_优选': cand_dist,
            'KS_D值': cand_D,
            'KS_p值': cand_p,
            '最终选择': final_dist,
            '分布参数': final_params,
            '决策依据': decision
        })
        
    df_fit = pd.DataFrame(fit_results)
    df_fit.to_csv(f'{output_dir}/边缘分布拟合参数.csv', index=False, encoding='utf-8-sig')
    
    # 打印精简报告
    print("✅ 拟合完成。部分结果预览:")
    print(df_fit[['特征', '样本量', '最终选择', 'KS_D值', '决策依据']].head(10).to_string(index=False))
    
    return df_fit

# ==================== 入口函数 ====================

def run_step7_clean_verify_v2(df_with_ci, garch_results, feature_list, version_dict=None, output_dir='./step7_clean_verify'):
    
    # 1. 清洗与验证
    cleaned_data,all_anomalies = process_cleaning_comparison(
        df_input=df_with_ci,
        garch_results=garch_results,
        feature_list=feature_list,
        version_dict=version_dict,
        output_dir=output_dir
    )
    
    # 2. 相关性分析
    corr_matrix_kendall, corr_matrix_pearson = analyze_cleaned_correlation(cleaned_data, output_dir)
    
    # 3. 保存清洗后的数据 (Copula建模必须输入)
    if cleaned_data:
        df_cleaned_output = pd.DataFrame(cleaned_data)
        save_path = f'{output_dir}/step7_cleaned_residuals.csv'
        df_cleaned_output.to_csv(save_path, encoding='utf-8-sig')
        print(f"\n💾 清洗后的残差数据已保存: {save_path}")
        
    # 4. 边缘分布拟合 (自动选择 Normal / T / Empirical)
    dist_params = fit_marginal_distributions(cleaned_data, output_dir)
    
    return cleaned_data, corr_matrix_kendall, corr_matrix_pearson, dist_params,all_anomalies 

### run

In [ ]:
cleaned_data, corr_matrix_kendall, corr_matrix_pearson, dist_params,all_anomalies = run_step7_clean_verify_v2(
    df_garch_resid, garch_results, feature_list, version_dict=version_dict,
    output_dir=os.path.join(RESULT_PATH, 'step7_clean_and_corr'))

#### 论文图

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

mpl.rcParams['font.family'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False


def plot_pearson_correlation_matrix(
    corr_matrix,
    feature_labels=None,
    output_path=None
):
    """
    图5.6: 清洗后标准化残差的Pearson相关系数矩阵（下三角热力图）
    
    Parameters
    ----------
    corr_matrix : pd.DataFrame
        19×19 Pearson相关系数矩阵
    feature_labels : dict, optional
        {原始特征名: 中文标签} 映射
    output_path : str, optional
    """
    # ── 字号配置 ──
    FONTSIZE_TITLE = 16
    FONTSIZE_TICK = 16
    FONTSIZE_ANNOT = 10
    FONTSIZE_CBAR = 16
    FONTSIZE_NOTE = 16
    
    # ── 标签映射 ──
    if feature_labels is not None:
        display_labels = [feature_labels.get(f, f) for f in corr_matrix.columns]
        corr_display = corr_matrix.copy()
        corr_display.columns = display_labels
        corr_display.index = display_labels
    else:
        corr_display = corr_matrix.copy()
    
    n = len(corr_display)
    
    # ── 下三角掩码 ──
    mask = np.triu(np.ones((n, n), dtype=bool))
    
    # ── 图形 ──
    fig_size = max(10, n * 0.65)
    fig, ax = plt.subplots(1, 1, figsize=(fig_size, fig_size * 0.85))
    
    # ── 色标 ──
    vmax = np.max(np.abs(corr_display.values[~mask]))
    vmax = min(vmax * 1.1, 1.0)
    
    # ── 热力图 ──
    sns.heatmap(
        corr_display,
        mask=mask,
        annot=True,
        fmt='.2f',
        cmap='RdBu_r',
        center=0,
        vmin=-vmax,
        vmax=vmax,
        square=True,
        ax=ax,
        linewidths=0.5,
        linecolor='white',
        annot_kws={'size': FONTSIZE_ANNOT},
        cbar_kws={
            'label': 'Pearson $\\rho$',
            'shrink': 0.75,
            'aspect': 30,
        }
    )
    
    # ── 坐标轴 ──
    ax.set_xticklabels(ax.get_xticklabels(),
                       rotation=40, ha='right', fontsize=FONTSIZE_TICK)
    ax.set_yticklabels(ax.get_yticklabels(),
                       rotation=0, fontsize=FONTSIZE_TICK)
    
    # ── 颜色条字号 ──
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=FONTSIZE_ANNOT)
    cbar.set_label('Pearson $\\rho$', fontsize=FONTSIZE_CBAR)
    
    # ── 标题 ──
    ax.set_title('清洗后标准化残差的Pearson相关系数矩阵',
                 fontsize=FONTSIZE_TITLE, fontweight='bold', pad=15)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f'已保存: {output_path}')
    
    plt.show()
    plt.close()


# ══════════════════════════════════════
# 调用示例
# ══════════════════════════════════════

# 特征标签映射（按论文使用的简称）
FEATURE_LABELS = {
    'total_volume_post': '广场流量',
    'total_volume_comment': '评论流量',
    'gini_post': '基尼(帖)',
    'gini_comment': '基尼(评)',
    'senti_symbol_post': '情感符号(帖)',
    'senti_symbol_comment': '情感符号(评)',
    'comp_ratio_post': '压缩比(帖)',
    'comp_ratio_comment': '压缩比(评)',
    'retweet_ratio_post': '转发率',
    'total_short_post': '短帖量',
    'total_long_post': '长帖量',
    'total_short_comment': '短评量',
    'total_long_comment': '长评量',
    'neg_ratio_post': '负面比(帖)',
    'neg_ratio_comment': '负面比(评)',
    'semantic_shift_post': '语义漂移(帖)',
    'semantic_shift_comment': '语义漂移(评)',
    'vis_abs_redundancy_post': '图片冗余度',
    'vis_concentration_post': '图片集中度',
}

plot_pearson_correlation_matrix(
    corr_matrix_pearson,   # 从 analyze_cleaned_correlation 返回的 corr_pearson
    feature_labels=FEATURE_LABELS,
    output_path='fig5_6_pearson_correlation.svg'
)

# 合成数据V3

## 计算正常掩码

In [ ]:
def build_global_normal_mask(
    index: pd.DatetimeIndex,
    all_anomalies: dict,
    remove_types=('positive_extreme', 'negative_extreme', 'structural_break'),
    pad: str | pd.Timedelta = '0min'
) -> pd.Series:
    """
    根据 run_step7_anomaly_noise 输出的 all_anomalies 构造“全局正常掩码”。

    Parameters
    ----------
    index : pd.DatetimeIndex
        你要评估的真实/合成数据的时间索引（通常是 df_real.index 或 df_synth.index）
    all_anomalies : dict
        run_step7_8_anomaly_noise 的返回值之一：
        { feature_name: anomaly_df, ... }
        anomaly_df 里必须有 'timestamp' 和 'type' 列。
    remove_types : tuple
        哪些异常类型视为“需要剔除”的异常点。
        默认：正向极端、负向极端、结构突变；高波动期不过滤。
    pad : str or Timedelta
        给每个异常点左右扩展的时间窗口。
        比如 '30min' 表示把异常点前后 30 分钟内的样本都视为异常。
        若不需要扩展，设为 '0min'。

    Returns
    -------
    normal_mask : pd.Series[bool]
        与 index 对齐的布尔序列：
            True  → 正常（可以用于 SyntheticDataValidator）
            False → 异常（被任何一个特征标记为需要剔除）
    """
    # 全部先标为 True（正常）
    normal_mask = pd.Series(True, index=index)

    if not all_anomalies:
        return normal_mask

    # 统一 pad 类型
    pad = pd.to_timedelta(pad)

    for feat, anomaly_df in all_anomalies.items():
        if anomaly_df is None or len(anomaly_df) == 0:
            continue

        df = anomaly_df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        # 只保留需要剔除的类型
        df = df[df['type'].isin(remove_types)]
        if df.empty:
            continue

        for ts in df['timestamp']:
            if pad == pd.Timedelta(0):
                # 只剔除异常点本身
                if ts in normal_mask.index:
                    normal_mask.loc[ts] = False
            else:
                # 剔除异常点前后一个时间窗口
                start = ts - pad
                end = ts + pad
                mask = (normal_mask.index >= start) & (normal_mask.index <= end)
                normal_mask.loc[mask] = False

    return normal_mask

In [ ]:
index = pd.DataFrame(cleaned_data).index
normal_mask = build_global_normal_mask(
    index=index,
    all_anomalies=all_anomalies,
    remove_types=('positive_extreme', 'negative_extreme', 'structural_break'),
    pad='30min'   # 不想扩展就写 '0min'
)

## 准备完整版本信息

In [ ]:
df_version = df_official[df_official['version'].notnull()].copy()
final_end_time = pd.to_datetime('2025-11-24 18:15:00')
df_version = df_version.sort_values('timestamp')
df_version['unique_key'] = df_version['version_no'].astype(str) + '_' + df_version['version']
df_version['start_time'] = df_version['timestamp']
df_version['end_time'] = df_version['timestamp'].shift(-1).fillna(final_end_time)
version_dict_complete={}
for row in df_version.iterrows():
    # print(row[1].unique_key)
    version_dict_complete[row[1].unique_key] = (str(row[1].start_time),str(row[1].end_time))

## 质量检验-可视化

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from statsmodels.tsa.stattools import acf
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# ================= 配置 =================
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 复用之前的颜色配置
VERSION_COLORS = {
    '大版本上半预热': '#FFCDD2', '大版本上半更新': '#EF5350',
    '大版本下半预热': '#FFECB3', '大版本下半更新': '#FFA726',
    '小版本上半预热': '#C8E6C9', '小版本上半更新': '#66BB6A',
    '小版本下半预热': '#BBDEFB', '小版本下半更新': '#42A5F5',
}

def parse_version_type_simple(version_name):
    """简化的版本类型解析"""
    s = str(version_name)
    size = '大版本' if '大' in s else '小版本'
    half = '下半' if '下' in s else '上半'
    phase = '预热' if '预' in s else '更新'
    return f"{size}{half}{phase}"

# ================= 核心检验函数 =================

def run_quality_validation(
    step,
    df_hist,        # 历史真实数据 (2个月)
    df_synth,       # 合成数据 (11个月)
    version_dict,   # 版本排期表 {name: (start, end)}
    feature_list,   # 要检验的特征列表
    output_dir
):
    """
    执行四大质量检验：
    1. 可视化全景拼接与断层检查
    2. 分版本类型的均值一致性检验
    3. 周期性强度 (ACF) 检验
    4. 波动性 (Volatility) 检验
    """
    os.makedirs(output_dir, exist_ok=True)
    print("="*70)
    print("🔍 合成数据质量全方位检验")
    print("="*70)

    # 确保索引是时间类型
    df_hist.index = pd.to_datetime(df_hist.index)
    df_synth.index = pd.to_datetime(df_synth.index)
    
    # 找到历史数据的截止点 (Cut-off point)
    cut_off_date = df_hist.index.max()
    print(f"   ⏱️ 历史数据截止点: {cut_off_date}")

    stats_report = []
    
    for feature in feature_list:
        print(f"\n   👉 正在检验特征: {feature}")
        if step ==5:
            col_hist = f'{feature}_transformed' if f'{feature}_transformed' in df_hist.columns else feature
        elif step==4:
            col_hist = f'{feature}_resid' if f'{feature}_resid' in df_hist.columns else feature
        elif step==3:
            col_hist = f'{feature}_event_adj_resid' if f'{feature}_event_adj_resid' in df_hist.columns else feature
        elif step==2:
            col_hist = f'{feature}_ar_resid' if f'{feature}_ar_resid' in df_hist.columns else feature
        else:
            col_hist = feature
        
        col_synth = feature # 假设合成数据列名就是特征名

        if col_hist not in df_hist.columns or col_synth not in df_synth.columns:
            print(f"      ⚠️ 列缺失，跳过: Hist={col_hist}, Synth={col_synth}")
            continue
            
        series_h = df_hist[col_hist].dropna()
        series_s = df_synth[col_synth].dropna()

        # ------------------------------------------------------
        # 1. 可视化检验 (The "Eye Test")
        # ------------------------------------------------------
        fig, ax = plt.subplots(figsize=(18, 6))
        
        # A. 绘制合成数据 (全量)
        ax.plot(series_s.index, series_s.values, color='#1f77b4', alpha=0.6, linewidth=1, label='合成数据 (Synthetic)')
        
        # B. 绘制历史数据 (覆盖在上面，用于对比重构精度)
        ax.plot(series_h.index, series_h.values, color='black', alpha=0.8, linewidth=1.2, linestyle='--', label='历史真实 (History)')
        
        # C. 标记 Cut-off point
        ax.axvline(cut_off_date, color='red', linestyle=':', linewidth=2, label='拼接点 (Cut-off)')
        
        # D. 版本背景染色
        y_min, y_max = ax.get_ylim()
        added_labels = set()
        for v_name, (v_start, v_end) in version_dict.items():
            v_start = pd.to_datetime(v_start)
            v_end = pd.to_datetime(v_end)
            
            # 只画在数据范围内的
            if v_end < series_s.index.min() or v_start > series_s.index.max():
                continue
                
            v_type = parse_version_type_simple(v_name)
            color = VERSION_COLORS.get(v_type, '#EEEEEE')
            
            # 图例去重
            label = v_type if v_type not in added_labels else None
            added_labels.add(v_type)
            
            rect = Rectangle(
                (mdates.date2num(v_start), y_min),
                mdates.date2num(v_end) - mdates.date2num(v_start),
                y_max - y_min,
                facecolor=color, alpha=0.2, edgecolor='none', zorder=-1
            )
            ax.add_patch(rect)
            
            # 在顶部标注版本名
            mid_point = mdates.date2num(v_start) + (mdates.date2num(v_end) - mdates.date2num(v_start))/2
            ax.text(mid_point, y_max, v_name.split('_')[-1][:2], ha='center', va='bottom', fontsize=8, rotation=90, color='gray')

        ax.set_title(f'step{step}特征: {feature} - 历史与合成全景拼接校验\n(观察红线处是否有断崖，以及大版本背景色处水位是否上升)', fontsize=14)
        ax.legend(loc='upper left')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.tight_layout()
        plt.savefig(f'{output_dir}/step{step}_{feature}.png', dpi=150)
        print(f"      📷 可视化检验图已保存: {output_dir}")
        plt.close()

        # ------------------------------------------------------
        # 2. 分组统计检验 (均值对比)
        # ------------------------------------------------------
        # 将数据打标版本类型
        def get_version_stats(series, v_dict):
            temp_df = pd.DataFrame({'val': series})
            temp_df['v_type'] = np.nan
            for v_name, (s, e) in v_dict.items():
                s, e = pd.to_datetime(s), pd.to_datetime(e)
                mask = (temp_df.index >= s) & (temp_df.index < e)
                temp_df.loc[mask, 'v_type'] = parse_version_type_simple(v_name)
            return temp_df.groupby('v_type')['val'].mean()

        mean_hist = get_version_stats(series_h, version_dict)
        mean_synth = get_version_stats(series_s, version_dict)
        
        # 计算差异
        comparison = pd.concat([mean_hist, mean_synth], axis=1, keys=['History_Mean', 'Synth_Mean'])
        comparison['Diff_Abs'] = comparison['Synth_Mean'] - comparison['History_Mean']
        comparison['Diff_Pct'] = (comparison['Diff_Abs'] / comparison['History_Mean'].abs()) * 100
        
        # 保存统计
        comp_csv = comparison.reset_index()
        comp_csv['feature'] = feature
        stats_report.append(comp_csv)
        
        print(f"      📊 分版本均值差异 (Top 3):")
        print(comparison[['History_Mean', 'Synth_Mean', 'Diff_Pct']].head(3).to_string())

        # ------------------------------------------------------
        # 3. 周期性强度检验 (ACF)
        # ------------------------------------------------------
        # 设定周期滞后：1天=96点 (15min), 1周=672点
        lag_day = 96
        lag_week = 672
        
        def safe_acf(x, lags):
            if len(x) < lags: return 0
            return acf(x, nlags=lags, fft=True)[-1]

        acf_h_day = safe_acf(series_h, lag_day)
        acf_s_day = safe_acf(series_s, lag_day)
        acf_h_week = safe_acf(series_h, lag_week)
        acf_s_week = safe_acf(series_s, lag_week)

        # ------------------------------------------------------
        # 4. 波动性检验 (Volatility)
        # ------------------------------------------------------
        std_h = series_h.std()
        std_s = series_s.std()
        std_ratio = std_s / std_h if std_h != 0 else 0

        # 汇总该特征的指标
        meta_row = pd.DataFrame([{
            'feature': feature,
            'v_type': 'GLOBAL_METRICS',
            'History_Mean': series_h.mean(),
            'Synth_Mean': series_s.mean(),
            'Diff_Pct': (series_s.mean() - series_h.mean())/series_h.mean()*100,
            'ACF_Day_Hist': acf_h_day, 'ACF_Day_Synth': acf_s_day,
            'ACF_Week_Hist': acf_h_week, 'ACF_Week_Synth': acf_s_week,
            'Std_Hist': std_h, 'Std_Synth': std_s, 'Std_Ratio': std_ratio
        }])
        stats_report.append(meta_row)

    # ================= 汇总报告 =================
    if stats_report:
        final_report = pd.concat(stats_report, ignore_index=True)
        # 调整列顺序便于阅读
        cols = ['feature', 'v_type', 'History_Mean', 'Synth_Mean', 'Diff_Pct', 
                'ACF_Day_Hist', 'ACF_Day_Synth', 'Std_Ratio']
        # 只保留存在的列
        cols = [c for c in cols if c in final_report.columns]
        
        save_path = f'{output_dir}/validation_report_full.csv'
        final_report.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"\n✅ 检验完成！完整报告已保存至: {save_path}")
        
        # 简单打印警报
        print("\n🚨 潜在异常预警 (Criteria: Mean Diff > 20% or Std Ratio > 1.5):")
        abnormal = final_report[
            (final_report['v_type'] == 'GLOBAL_METRICS') & 
            ((final_report['Diff_Pct'].abs() > 20) | (final_report['Std_Ratio'] > 1.5) | (final_report['Std_Ratio'] < 0.5))
        ]
        if not abnormal.empty:
            print(abnormal[['feature', 'Diff_Pct', 'Std_Ratio']].to_string(index=False))
        else:
            print("   无明显异常。")
    else:
        print("❌ 未生成任何检验报告，请检查输入数据。")

# ================= 调用示例 =================
# run_quality_validation(df_hist_transformed, df_synthetic_transformed, version_dict_complete, ['active_users', 'latency'])

## 质量检验-评分

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, wasserstein_distance, pearsonr
from statsmodels.tsa.stattools import acf
import os

# ==================== 配置常量 ====================
# 评分阈值配置 (可根据业务容忍度调整)
THRESHOLDS = {
    'mean_error': {'S': 0.05, 'A': 0.10, 'B': 0.20},  # 均值偏差比例
    'std_error':  {'S': 0.10, 'A': 0.20, 'B': 0.30},  # 标准差偏差比例
    'skew_diff':  {'S': 0.20, 'A': 0.50, 'B': 1.00},  # 偏度绝对差
    'acf_corr':   {'S': 0.95, 'A': 0.85, 'B': 0.70},  # ACF曲线相关性 (越高越好)
    'wass_dist':  {'S': 0.10, 'A': 0.30, 'B': 0.50},  # 归一化后的Wasserstein距离
}

def get_grade(value, metric_name, reverse=False):
    """
    根据阈值计算评级
    reverse=True 表示值越大越好 (如相关性)
    """
    th = THRESHOLDS.get(metric_name)
    if not th: return 'N/A'
    
    val = abs(value)
    
    if not reverse:
        # 越小越好 (Error, Diff, Dist)
        if val <= th['S']: return 'S'
        if val <= th['A']: return 'A'
        if val <= th['B']: return 'B'
        return 'C'
    else:
        # 越大越好 (Correlation)
        if val >= th['S']: return 'S'
        if val >= th['A']: return 'A'
        if val >= th['B']: return 'B'
        return 'C'

def calculate_numeric_quality(
    step,
    df_hist,          # 历史真实数据 (Step 1.3 输入)
    df_synth,         # Prophet合成数据 (Step 2.5 输出)
    feature_list,
    output_dir,
    normal_mask
):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("=" * 70)
    print("📊 Step 6: 数值化质量评级 (Numeric Evaluation)")
    print("=" * 70)

    results = []

    
    # 1. 数据准备与对齐
    # 1. 确保索引是时间类型以便对齐
    if not isinstance(df_hist.index, pd.DatetimeIndex):
        df_hist.index = pd.to_datetime(df_hist.index)
    if not isinstance(df_synth.index, pd.DatetimeIndex):
        df_synth.index = pd.to_datetime(df_synth.index)
        
    for feature in feature_list:
        print(f"\n   👉 正在检验特征: {feature}")
        if step ==5:
            col_hist = f'{feature}_transformed' if f'{feature}_transformed' in df_hist.columns else feature
        elif step==4:
            col_hist = f'{feature}_resid' if f'{feature}_resid' in df_hist.columns else feature
        elif step==3:
            col_hist = f'{feature}_event_adj_resid' if f'{feature}_event_adj_resid' in df_hist.columns else feature
        elif step==2:
            col_hist = f'{feature}_ar_resid' if f'{feature}_ar_resid' in df_hist.columns else feature
        else:
            col_hist = feature
        
        col_synth = feature
        
        if col_hist not in df_hist.columns or col_synth not in df_synth.columns:
            print(f"   ⚠️ 跳过 {feature}: 列缺失")
            continue

        series_h = df_hist[col_hist]
        series_s = df_synth[col_synth]

        if normal_mask is not None:
            # 1. 对齐 mask 和 history
            # reindex 确保 mask 覆盖 history 的时间范围，缺失填 True (默认正常)
            aligned_mask = normal_mask.reindex(series_h.index).fillna(True).astype(bool)
            
            # 2. 过滤历史数据 (只取正常点)
            h_data = series_h.loc[aligned_mask].dropna().values
            
            # 3. 处理合成数据
            # 如果是“重构/回测”任务 (索引重叠)，不仅要过滤历史，也要过滤合成数据的对应时刻，
            # 这样才是 Apple-to-Apple 的比较 (正常 vs 正常)。
            common_idx = series_s.index.intersection(series_h.index)
            if len(common_idx) > 0:
                # 索引有重叠，说明是重构对比，需要同步过滤
                mask_for_s = normal_mask.reindex(series_s.index).fillna(True).astype(bool)
                s_data = series_s.loc[mask_for_s].dropna().values
            else:
                # 索引无重叠 (如生成未来数据)，则认为合成数据全是“正常”的
                s_data = series_s.dropna().values
        else:
            h_data = series_h.dropna().values
            s_data = series_s.dropna().values
        # ==========================================================

        if len(h_data) == 0 or len(s_data) == 0:
            print(f"   ⚠️ 跳过 {feature}: 过滤后数据为空")
            continue

        # ----------------------------------------------------
        # 2. 计算统计指标 (保持原有逻辑)
        # ----------------------------------------------------
        
        # A. 基础统计量
        h_mean, s_mean = np.mean(h_data), np.mean(s_data)
        h_std, s_std = np.std(h_data), np.std(s_data)
        
        mean_err = (s_mean - h_mean) / (abs(h_mean) + 1e-6)
        std_err = (s_std - h_std) / (h_std + 1e-6)

        # B. 分布形态
        h_skew, s_skew = skew(h_data), skew(s_data)
        skew_diff = s_skew - h_skew
        
        # Wasserstein 距离
        wd = wasserstein_distance(h_data, s_data)
        wd_norm = wd / (h_std + 1e-6)

        # C. 周期性结构 (ACF)
        # 注意：ACF 对数据连续性敏感。剔除异常点会造成时间空洞。
        # 但相比于异常值带来的巨大偏差，这种空洞对 ACF 形状的影响通常较小。
        # 只要数据量足够大，主要周期性依然能体现。
        lags = min(96 * 2, len(h_data) // 2)
        if lags > 10:
            h_acf = acf(h_data, nlags=lags, fft=True)
            s_acf = acf(s_data, nlags=lags, fft=True)
            if np.std(h_acf) > 0 and np.std(s_acf) > 0:
                acf_corr, _ = pearsonr(h_acf, s_acf)
            else:
                acf_corr = 0.0
        else:
            acf_corr = 0.0

        # ----------------------------------------------------
        # 3. 评分 (保持原有逻辑)
        # ----------------------------------------------------
        grade_mean = get_grade(mean_err, 'mean_error')
        grade_std  = get_grade(std_err, 'std_error')
        grade_skew = get_grade(skew_diff, 'skew_diff')
        grade_acf  = get_grade(acf_corr, 'acf_corr', reverse=True)
        grade_dist = get_grade(wd_norm, 'wass_dist')
        
        grades = [grade_mean, grade_std, grade_skew, grade_acf, grade_dist]
        if 'C' in grades:
            final_grade = 'C (Risk)'
        elif 'B' in grades:
            final_grade = 'B (Fair)'
        elif 'A' in grades:
            final_grade = 'A (Good)'
        else:
            final_grade = 'S (Perfect)'

        # ----------------------------------------------------
        # 4. 记录结果
        # ----------------------------------------------------
        res_row = {
            '特征': feature,
            '综合评级': final_grade,
            '均值误差%': f"{mean_err*100:.1f}%",
            '波动误差%': f"{std_err*100:.1f}%",
            '偏度差异': f"{skew_diff:.2f}",
            '分布距离(WD)': f"{wd_norm:.3f}",
            '周期相似度': f"{acf_corr:.3f}",
            'G_Mean': grade_mean, 'G_Std': grade_std,
            'G_Skew': grade_skew, 'G_Dist': grade_dist, 'G_ACF': grade_acf,
            'Raw_Mean_H': h_mean, 'Raw_Mean_S': s_mean,
            'Raw_Std_H': h_std, 'Raw_Std_S': s_std
        }
        results.append(res_row)
        
        print(f"   👉 {feature:<20} | 评级: {final_grade:<10} | 均值err: {mean_err*100:5.1f}% | 周期Corr: {acf_corr:.2f}")

    # 5. 输出文件
    if results:
        df_res = pd.DataFrame(results)
        df_res['sort_key'] = df_res['综合评级'].map({'C (Risk)': 0, 'B (Fair)': 1, 'A (Good)': 2, 'S (Perfect)': 3})
        df_res = df_res.sort_values('sort_key').drop(columns=['sort_key'])
        
        mask_suffix = "_masked" if normal_mask is not None else ""
        save_path = os.path.join(output_dir, f'step{step}_quality_scorecard{mask_suffix}.csv')
        df_res.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        print(f"\n✅ 评级报告已生成: {save_path}")
        return df_res
    else:
        return pd.DataFrame()

## step1 底层数据生成

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t as t_dist, norm
from scipy.linalg import cholesky
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
from statsmodels.stats.diagnostic import acorr_ljungbox

# 配置
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== 核心工具函数 ====================

def fix_correlation_matrix(corr_matrix, min_eigval=1e-8):
    """
    修复非正定相关矩阵（确保可以进行Cholesky分解）
    使用最近正定矩阵投影方法
    """
    # 确保对称性
    corr_matrix = (corr_matrix + corr_matrix.T) / 2
    
    # 确保对角线为1
    np.fill_diagonal(corr_matrix, 1.0)
    
    # 特征值分解
    eigvals, eigvecs = np.linalg.eigh(corr_matrix)
    
    # 如果已经正定，直接返回
    if np.min(eigvals) >= min_eigval:
        return corr_matrix, False
    
    # 将负特征值和过小的特征值修复为正值
    eigvals_fixed = np.maximum(eigvals, min_eigval)
    
    # 重构矩阵
    corr_fixed = eigvecs @ np.diag(eigvals_fixed) @ eigvecs.T
    
    # 重新归一化对角线为1
    d = np.sqrt(np.diag(corr_fixed))
    corr_fixed = corr_fixed / np.outer(d, d)
    
    # 确保对角线精确为1
    np.fill_diagonal(corr_fixed, 1.0)
    
    return corr_fixed, True


# def empirical_quantile_transform(u_samples, historical_resid):
#     """
#     经验分位数映射：将U[0,1]样本转换为经验分布样本
#     使用 plotting position 分位点 (i-0.5)/n
#     """
#     sorted_resid = np.sort(historical_resid)
#     n = len(sorted_resid)
    
#     if n == 0:
#         return np.full_like(u_samples, np.nan, dtype=float)
    
#     # 构建经验CDF的分位点 (0,1) 内部点
#     empirical_quantiles = (np.arange(1, n + 1) - 0.5) / n
    
#     # 线性插值
#     transformed = np.interp(u_samples, empirical_quantiles, sorted_resid)
    
#     return transformed

def empirical_quantile_transform(u_samples, historical_resid):
    """
    经验分位数映射：将U[0,1]样本转换为经验分布样本
    修正版：使用 np.quantile 进行更平滑的插值，处理边界问题
    """
    historical_resid = np.asarray(historical_resid)
    valid_mask = ~np.isnan(historical_resid) & ~np.isinf(historical_resid)
    clean_resid = historical_resid[valid_mask]
    
    if len(clean_resid) < 10:
        # 样本太少，回退到标准正态分布映射
        # 保持均值和方差的大致刻度
        mu, std = np.mean(clean_resid), np.std(clean_resid)
        return norm.ppf(u_samples) * std + mu
        
    # 使用 numpy 的 quantile 函数进行逆变换
    # method='linear' 是默认的 Type 7 插值，适合连续分布模拟
    # u_samples 必须在 [0, 1] 之间
    
    # 为了防止 u=0 或 u=1 导致无穷大或越界（虽然 np.quantile 能处理，但最好钳位）
    u_clamped = np.clip(u_samples, 1e-9, 1 - 1e-9)
    
    transformed = np.quantile(clean_resid, u_clamped, method='linear')
    
    return transformed


def generate_synthetic_step1_integrated(
    noise_analysis_dict,      # 各特征的噪音分析结果
    corr_matrix,              # Pearson相关矩阵
    std_resid_dict,           # 各特征的历史标准化残差池
    feature_list,             # 特征列表
    start_date,               # 合成数据起始日期
    months=11,                # 合成月数
    freq='15min',             # 时间频率
    seed=42,                  # 随机种子
    clip_mode='smooth',       # 截断模式: 'smooth' (默认) 或 'conservative'
    output_dir='./result/step1_synthetic'
):
    """
    整合版第一步：生成相关白噪声
    """
    np.random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    
    print("═" * 70)
    print(f"📌 第一步：生成 {months} 个月的相关白噪声 (Mode: {clip_mode})")
    print("═" * 70)
    
    # ══════════════════════════════════════════════════════════════════════
    # 1. 构建时间轴
    # ══════════════════════════════════════════════════════════════════════
    start_ts = pd.Timestamp(start_date)
    end_ts = start_ts + pd.DateOffset(months=months)
    
    time_index = pd.date_range(
        start=start_ts, 
        end=end_ts, 
        freq=freq, 
        inclusive='left'
    )
    
    n_samples = len(time_index)
    n_features = len(feature_list)
    
    print(f"   📅 时间范围: {start_ts} 至 {end_ts}")
    print(f"   📊 样本总数: {n_samples:,} | 特征数: {n_features}")
    
    # ══════════════════════════════════════════════════════════════════════
    # 2. 处理相关矩阵 (默认输入为 Pearson)
    # ══════════════════════════════════════════════════════════════════════
    # 对齐特征
    rho_target = corr_matrix.loc[feature_list, feature_list].values.astype(float).copy()
    
    # 修复非正定性 (Copula 需要 Σ 为正定)
    rho_fixed, was_fixed = fix_correlation_matrix(rho_target)
    
    if was_fixed:
        print(f"   ⚠️ 输入 Pearson 矩阵非正定，已自动修复")
    
    # ══════════════════════════════════════════════════════════════════════
    # 3. Cholesky 分解生成相关正态噪声
    # ══════════════════════════════════════════════════════════════════════
    # 尝试 Cholesky 分解，如果失败不再捕获异常，直接报错
    L = cholesky(rho_fixed, lower=True)
    
    # 生成独立标准正态随机数 Z ~ N(0, I)
    z_independent = np.random.standard_normal((n_samples, n_features))
    
    # 引入相关性: X = Z @ L.T
    z_correlated = z_independent @ L.T
    
    # ══════════════════════════════════════════════════════════════════════
    # 4. 转换为均匀分布 U[0,1]
    # ══════════════════════════════════════════════════════════════════════
    u_correlated = norm.cdf(z_correlated)
    
    # 避免边界值 (防止 ppf 返回 ±∞)
    epsilon = 1e-10
    u_correlated = np.clip(u_correlated, epsilon, 1 - epsilon)
    
    # ══════════════════════════════════════════════════════════════════════
    # 5. 边际分布逆变换 & 截断
    # ══════════════════════════════════════════════════════════════════════
    synthetic_noise = np.zeros((n_samples, n_features))
    generation_info = {}
    
    print(f"\n   {'特征':<25} {'分布':<15} {'截断策略'}")
    print("   " + "─" * 60)
    
    for i, feature in enumerate(feature_list):
        noise_analysis = noise_analysis_dict.get(feature, {})
        std_resid_pool = std_resid_dict.get(feature, np.array([]))
        
        # 清洗残差池
        std_resid_pool = np.asarray(std_resid_pool)
        valid_mask = ~np.isnan(std_resid_pool) & ~np.isinf(std_resid_pool)
        std_resid_pool = std_resid_pool[valid_mask]
        
        best_dist = noise_analysis.get('best_distribution', 'Non-parametric')
        col_data = None
        dist_info = {}

        # --- A. 分布逆变换 (代码A逻辑) ---
        if 'N(0,1)' in best_dist:
            col_data = norm.ppf(u_correlated[:, i])
            dist_info = {'type': 'Normal', 'params': 'N(0,1)'}
            
        elif 't(' in best_dist or noise_analysis.get('is_t_dist', False):
            df = max(float(noise_analysis.get('t_df', 5.0)), 5.0)
            loc = noise_analysis.get('t_loc', 0.0)
            scale = max(float(noise_analysis.get('t_scale', 1.0)), 1e-6)
            
            col_data = t_dist.ppf(u_correlated[:, i], df, loc, scale)
            dist_info = {'type': 't-dist', 'params': f'df={df:.1f}'}
            
        else:
            # 经验分布 (代码A逻辑)
            if len(std_resid_pool) >= 100:
                col_data = empirical_quantile_transform(u_correlated[:, i], std_resid_pool)
                dist_info = {'type': 'Empirical', 'params': f'n={len(std_resid_pool)}'}
            else:
                # 降级为 t 分布
                df = max(float(noise_analysis.get('t_df', 5.0)), 5.0)
                col_data = t_dist.ppf(u_correlated[:, i], df, 0, 1)
                dist_info = {'type': 't-dist(fallback)', 'params': f'df={df:.1f}'}

        # --- B. 数据截断 (整合逻辑) ---
        # 如果残差池太小，无法计算分位数，强制使用固定阈值
        if len(std_resid_pool) < 10:
            clip_min, clip_max = -6.0, 6.0
            clip_desc = "Fixed[-6,6]"
        else:
            if clip_mode == 'smooth':
                # 代码B逻辑: 平滑策略 (0.05% ~ 99.95% + 10% Margin)
                # 允许比历史极值稍微大一点，保留尾部活力
                lower = np.percentile(std_resid_pool, 0.05)
                upper = np.percentile(std_resid_pool, 99.95)
                margin = (upper - lower) * 0.1
                clip_min = lower - margin
                clip_max = upper + margin
                clip_desc = "Smooth(w/Margin)"
                
            elif clip_mode == 'conservative':
                # 代码A逻辑: 保守策略 (0.1% ~ 99.9%)
                # 严格限制在历史范围内
                clip_min = np.quantile(std_resid_pool, 0.001)
                clip_max = np.quantile(std_resid_pool, 0.999)
                clip_desc = "Conservative(Strict)"
            else:
                # 默认 fallback
                clip_min, clip_max = -10.0, 10.0
                clip_desc = "Loose"

        # 执行截断
        col_data = np.clip(col_data, clip_min, clip_max)
        synthetic_noise[:, i] = col_data
        
        # 记录信息
        generation_info[feature] = {
            'distribution': dist_info,
            'clipping': {'mode': clip_mode, 'min': float(clip_min), 'max': float(clip_max)}
        }
        
        if i < 5:  # 只打印前5个
            print(f"   {feature:<25} {dist_info['type']:<15} {clip_desc}")

    # ══════════════════════════════════════════════════════════════════════
    # 6. 构建结果 & 验证
    # ══════════════════════════════════════════════════════════════════════
    df_synthetic_noise = pd.DataFrame(
        synthetic_noise,
        index=time_index,
        columns=feature_list
    )
    df_synthetic_noise.index.name = 'timestamp'
    
    # 验证相关性 (代码A逻辑，适配 Pearson 输入)
    print("\n   📈 相关性保持度验证:")
    
    # 计算合成数据的 Pearson 相关系数
    synth_corr_pearson = df_synthetic_noise.corr(method='pearson').values
    
    # 计算偏差
    corr_diff = np.abs(synth_corr_pearson - rho_fixed)
    np.fill_diagonal(corr_diff, 0)
    
    print(f"     • Pearson MAE: {corr_diff.mean():.4f}")
    print(f"     • Pearson Max Error: {corr_diff.max():.4f}")
    
    # 简单的白噪音检验 (Ljung-Box) - 无 try/except
    print("\n   📈 白噪音检验 (前3个特征):")
    for feature in feature_list[:3]:
        lb_res = acorr_ljungbox(df_synthetic_noise[feature], lags=[10], return_df=True)
        p_val = lb_res['lb_pvalue'].iloc[0]
        status = "✅" if p_val > 0.05 else "⚠️"
        print(f"     • {feature}: p={p_val:.4f} {status}")

    # ══════════════════════════════════════════════════════════════════════
    # 7. 可视化 & 保存
    # ══════════════════════════════════════════════════════════════════════
    print("\n【生成可视化报告】")
    
    # 7.1 热力图对比
    fig = plt.figure(figsize=(18, 6))
    
    ax1 = fig.add_subplot(1, 3, 1)
    sns.heatmap(rho_fixed, ax=ax1, cmap='RdBu_r', center=0, cbar=False,
                xticklabels=feature_list, yticklabels=feature_list)
    ax1.set_title('Target Correlation (Pearson)')
    
    ax2 = fig.add_subplot(1, 3, 2)
    sns.heatmap(synth_corr_pearson, ax=ax2, cmap='RdBu_r', center=0, cbar=False,
                xticklabels=feature_list, yticklabels=feature_list)
    ax2.set_title('Synthetic Correlation (Pearson)')
    
    ax3 = fig.add_subplot(1, 3, 3)
    sns.heatmap(corr_diff, ax=ax3, cmap='Reds', vmin=0, vmax=0.1,
                xticklabels=feature_list, yticklabels=feature_list)
    ax3.set_title(f'Difference (MAE={corr_diff.mean():.4f})')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/correlation_check.png')
    plt.close()
    
    # 7.2 保存数据
    df_synthetic_noise.to_csv(f'{output_dir}/synthetic_noise.csv', encoding='utf-8-sig')
    
    # Parquet 需要 pyarrow 引擎，假设环境已有
    df_synthetic_noise.to_parquet(f'{output_dir}/synthetic_noise.parquet')

    # 7.3 保存报告
    report = {
        'params': {'months': months, 'clip_mode': clip_mode},
        'quality': {'pearson_mae': float(corr_diff.mean())},
        'details': generation_info
    }
    with open(f'{output_dir}/generation_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print(f"   ✅ 完成。数据已保存至: {output_dir}")
    
    return df_synthetic_noise, report

### run

In [ ]:
df_synthetic_noise, generation_report = generate_synthetic_step1_integrated(
    noise_analysis_dict=dist_params,
    corr_matrix=corr_matrix_pearson,
    std_resid_dict=cleaned_data,
    feature_list=feature_list,
    start_date='2024-12-27',   
    months=11,                  # 合成11个月
    freq='15min',              # 15分钟窗口
    seed=42,                   # 固定种子确保可复现
    clip_mode='smooth',        # 平滑截断策略
    # clip_mode='conservative',        # 平滑截断策略

    output_dir=os.path.join(RESULT_PATH, 'g_step1_synthetic_noise'))

### 质量检验

In [ ]:
run_quality_validation(
    1,
    pd.DataFrame(cleaned_data), 
    df_synthetic_noise, 
    version_dict_complete,
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g1_noise_validation'))

## step2 注入波动率

### final version

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import json
import os
import warnings
from statsmodels.stats.diagnostic import acorr_ljungbox

# 配置
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 辅助函数：波动率重采样
# ============================================================
def generate_bootstrap_volatility(hist_vol, n_samples, block_size=96):
    """
    终极保底方案：历史波动率分块重采样
    block_size=96 约等于 1 天 (15min * 4 * 24)，保留日内波动模式
    """
    # 移除无效值
    valid_vol = hist_vol[~np.isnan(hist_vol) & ~np.isinf(hist_vol)]
    if len(valid_vol) < block_size:
        return np.ones(n_samples) # 数据太少，退化为常数波动率
        
    generated_vol = []
    current_len = 0
    
    rng = np.random.default_rng()
    
    while current_len < n_samples:
        # 随机选一个起始点
        start_idx = rng.integers(0, len(valid_vol) - block_size)
        # 取出一个块
        block = valid_vol[start_idx : start_idx + block_size]
        generated_vol.append(block)
        current_len += len(block)
        
    # 拼接并裁剪
    final_vol = np.concatenate(generated_vol)[:n_samples]
    
    # 加上微小的随机扰动 (0.9 ~ 1.1)，防止完全重复
    noise = rng.uniform(0.9, 1.1, size=n_samples)
    return final_vol * noise


# ============================================================
# 引入 Tanh 饱和函数
# ============================================================
def smooth_tanh_saturation(x, limit):
    """
    双曲正切饱和：平滑地将数据限制在 [-limit, limit] 范围内
    - x 小的时候，近似线性 (y ≈ x)
    - x 大的时候，平滑逼近 limit
    """
    # 保护 limit 防止除零或无效
    limit = max(float(limit), 1e-6)
    
    # 核心公式: limit * tanh(x / limit)
    # tanh 的定义域是 (-inf, inf)，值域是 (-1, 1)
    # 所以结果严格限制在 (-limit, limit) 之间
    return limit * np.tanh(x / limit)
# ============================================================
# Core Logic (From Code B): Bootstrap Burn-in
# ============================================================
def _make_burnin_z_from_series(z_t: np.ndarray, burnin: int, rng: np.random.Generator) -> np.ndarray:
    """
    用 bootstrap 的方式从 z_t 中抽样生成 burn-in 段，保证与 z_t 同分布。
    """
    z_t = np.asarray(z_t, dtype=float)
    if burnin <= 0:
        return z_t

    z_t_clean = z_t[~np.isnan(z_t) & ~np.isinf(z_t)]
    # 如果有效数据太少，退化为标准正态
    if z_t_clean.size < 10:
        z_pre = rng.standard_normal(burnin)
    else:
        # 有放回抽样
        idx = rng.integers(0, z_t_clean.size, size=burnin)
        z_pre = z_t_clean[idx]

    return np.concatenate([z_pre, z_t], axis=0)

# ============================================================
# Core Logic (From Code B): Volatility Generators
# ============================================================
def generate_gjr_garch_volatility_from_z(
    z_full: np.ndarray,
    omega: float,
    alpha: float,
    gamma: float,
    beta: float,
    unconditional_var: float | None = None,
    burnin: int = 500
):
    """
    GJR-GARCH(1,1,1) 生成器 (Code B逻辑：带持久性限制与极值保护)
    """
    z_full = np.asarray(z_full, dtype=float)
    total = z_full.size
    
    # 鲁棒性修正
    omega = float(max(omega, 1e-12))
    alpha = float(max(alpha, 0.0))
    gamma = float(gamma)  
    beta  = float(max(beta, 0.0))

    # 持续性检查与缩放 (Code B: 阈值 0.995)
    persistence = alpha + beta + 0.5 * gamma
    if persistence >= 0.995:
        scale = 0.995 / max(persistence, 1e-12)
        alpha *= scale
        beta  *= scale
        gamma *= scale
        persistence = alpha + beta + 0.5 * gamma

    # 无条件方差初始化
    if unconditional_var is None:
        denom = max(1.0 - persistence, 1e-6)
        unconditional_var = float(max(omega / denom, 1e-10))
    else:
        unconditional_var = float(max(unconditional_var, 1e-10))

    sigma_sq = np.zeros(total, dtype=float)
    eps      = np.zeros(total, dtype=float)

    # 初始状态
    sigma_sq[0] = unconditional_var
    eps[0] = np.sqrt(sigma_sq[0]) * z_full[0]

    for t in range(1, total):
        indicator = 1.0 if eps[t-1] < 0 else 0.0
        
        # GJR 核心公式
        sigma_sq[t] = (omega
                       + alpha * (eps[t-1]**2)
                       + gamma * (eps[t-1]**2) * indicator
                       + beta  * sigma_sq[t-1])
        
        # 极值保护 (Code B: Log Soft Clip的前置保护)
        sigma_sq[t] = min(max(sigma_sq[t], 1e-12), 1e8) 
        
        eps[t] = np.sqrt(sigma_sq[t]) * z_full[t]

    sigma = np.sqrt(sigma_sq)
    return sigma[burnin:], sigma_sq[burnin:]


def generate_garch_volatility_from_z(
    z_full: np.ndarray,
    omega: float,
    alpha_list,
    beta_list,
    unconditional_var: float | None = None,
    burnin: int = 500
):
    """
    通用 GARCH(p,q) 生成器 (Code B逻辑：支持高阶)
    """
    z_full = np.asarray(z_full, dtype=float)
    total = z_full.size

    omega = float(max(omega, 1e-12))
    alpha_list = [max(a, 0.0) for a in alpha_list]
    beta_list = [max(b, 0.0) for b in beta_list]

    p = len(alpha_list)
    q = len(beta_list)
    max_lag = max(p, q, 1)

    persistence = sum(alpha_list) + sum(beta_list)

    # 持续性限制 (Code B: 阈值 0.995)
    if persistence >= 0.995:
        scale = 0.995 / max(persistence, 1e-12)
        alpha_list = [a * scale for a in alpha_list]
        beta_list = [b * scale for b in beta_list]
        persistence = sum(alpha_list) + sum(beta_list)

    if unconditional_var is None:
        unconditional_var = omega / max(1.0 - persistence, 1e-6)
    unconditional_var = float(max(unconditional_var, 1e-10))

    sigma_sq = np.zeros(total, dtype=float)
    eps_sq = np.zeros(total, dtype=float)

    # 初始化
    sigma_sq[:max_lag] = unconditional_var
    eps_sq[:max_lag] = sigma_sq[:max_lag] * (z_full[:max_lag] ** 2)

    # 递推
    for t in range(max_lag, total):
        arch_term = 0.0
        for i in range(p):
            arch_term += alpha_list[i] * eps_sq[t - 1 - i]

        garch_term = 0.0
        for j in range(q):
            garch_term += beta_list[j] * sigma_sq[t - 1 - j]

        sigma_sq[t] = omega + arch_term + garch_term
        sigma_sq[t] = min(max(sigma_sq[t], 1e-12), 1e8)

        eps_sq[t] = sigma_sq[t] * (z_full[t] ** 2)

    sigma = np.sqrt(sigma_sq)
    return sigma[burnin:], sigma_sq[burnin:]


def generate_egarch_volatility_from_z(
    z_full: np.ndarray,
    omega: float,
    alpha: float,
    gamma: float,
    beta: float,
    unconditional_var: float | None = None,
    burnin: int = 500
):
    """
    EGARCH(1,1) 生成器 (Code B逻辑：Beta限制与初始值保护)
    """
    z_full = np.asarray(z_full, dtype=float)
    total = z_full.size

    # Code B: Beta 保护
    if abs(beta) >= 0.995:
        beta = np.sign(beta) * 0.995

    # 计算经验 E|z|
    z_clean = z_full[~np.isnan(z_full) & ~np.isinf(z_full)]
    E_abs_z = float(np.mean(np.abs(z_clean))) if z_clean.size > 10 else np.sqrt(2 / np.pi)

    # 初始化 (E[log σ^2] = ω / (1 - β))
    if unconditional_var is None:
        unc_log_var = omega / (1.0 - beta)
    else:
        unc_log_var = np.log(max(unconditional_var, 1e-10))
    
    # Code B: 初始值限幅 [-10, 10]
    unc_log_var = np.clip(unc_log_var, -10, 10)

    log_sigma_sq = np.zeros(total, dtype=float)
    log_sigma_sq[0] = unc_log_var

    for t in range(1, total):
        z_prev = z_full[t-1]
        g_z = alpha * (np.abs(z_prev) - E_abs_z) + gamma * z_prev
        
        log_sigma_sq[t] = omega + beta * log_sigma_sq[t-1] + g_z
        
        # 强力裁剪防止溢出
        log_sigma_sq[t] = np.clip(log_sigma_sq[t], -20, 20)

    sigma_sq = np.exp(log_sigma_sq)
    sigma = np.sqrt(sigma_sq)
    return sigma[burnin:], sigma_sq[burnin:]


# ============================================================
# Core Logic (From Code B): Parameter Extraction
# ============================================================
def extract_garch_params_dynamic(garch_results, feature_name):
    """
    Code B+ 修改版：提取参数并应用 0.985 强阻尼 (Simulation Regularization)
    """
    if feature_name not in garch_results:
        return None

    result = garch_results[feature_name]
    params = result['params']
    model_name = str(result['model_name'])
    # params = model_result.params

    params_dict = {
        'model_type': model_name,
        'omega': float(params.get('omega', 1e-6)),
    }

    # ================= 提取原始参数 =================
    
    # --- 分支 A: EGARCH ---
    if 'EGARCH' in model_name:
        params_dict['model_class'] = 'EGARCH'
        params_dict['alpha'] = float(params.get('alpha[1]', 0.1))
        params_dict['gamma'] = float(params.get('gamma[1]', 0.0))
        params_dict['beta']  = float(params.get('beta[1]', 0.9))

    # --- 分支 B: GJR-GARCH ---
    elif 'GJR' in model_name or 'GJR-GARCH' in model_name:
        params_dict['model_class'] = 'GJR'
        params_dict['alpha'] = float(params.get('alpha[1]', 0.05))
        params_dict['gamma'] = float(params.get('gamma[1]', 0.0)) 
        params_dict['beta']  = float(params.get('beta[1]', 0.90))

    # --- 分支 C: 标准 GARCH (动态 p,q) ---
    else:
        params_dict['model_class'] = 'GARCH'
        alpha_list = []
        for i in range(1, 5): 
            key = f'alpha[{i}]'
            if key in params.keys(): alpha_list.append(float(params[key]))
            else: break
        
        beta_list = []
        for i in range(1, 5):
            key = f'beta[{i}]'
            if key in params.keys(): beta_list.append(float(params[key]))
            else: break

        if not alpha_list: alpha_list = [0.05]
        if not beta_list: beta_list = [0.90]

        params_dict['alpha_list'] = alpha_list
        params_dict['beta_list'] = beta_list

    # ================= 修改重点：强制阻尼逻辑 (0.985) =================
    # 针对 11 个月长周期，降低持续性上限，防止发散
    MAX_PERSISTENCE = 0.985
    
    current_persistence = 0.0
    cls = params_dict['model_class']

    if cls == 'EGARCH':
        # EGARCH 的持续性主要由 Beta 决定
        current_persistence = abs(params_dict['beta'])
        if current_persistence > MAX_PERSISTENCE:
            # print(f"   🔧 {feature_name} (EGARCH): Beta {current_persistence:.4f} -> {MAX_PERSISTENCE}")
            damping = MAX_PERSISTENCE / current_persistence
            params_dict['beta'] *= damping

    elif cls == 'GJR':
        # GJR Persistence = alpha + beta + 0.5 * gamma
        current_persistence = params_dict['alpha'] + params_dict['beta'] + 0.5 * params_dict['gamma']
        if current_persistence > MAX_PERSISTENCE:
            # print(f"   🔧 {feature_name} (GJR): Persistence {current_persistence:.4f} -> {MAX_PERSISTENCE}")
            damping = MAX_PERSISTENCE / current_persistence
            params_dict['alpha'] *= damping
            params_dict['beta']  *= damping
            params_dict['gamma'] *= damping

    else: # GARCH
        # GARCH Persistence = sum(alpha) + sum(beta)
        sum_alpha = sum(params_dict['alpha_list'])
        sum_beta = sum(params_dict['beta_list'])
        current_persistence = sum_alpha + sum_beta
        
        if current_persistence > MAX_PERSISTENCE:
            # print(f"   🔧 {feature_name} (GARCH): Persistence {current_persistence:.4f} -> {MAX_PERSISTENCE}")
            damping = MAX_PERSISTENCE / current_persistence
            params_dict['alpha_list'] = [a * damping for a in params_dict['alpha_list']]
            params_dict['beta_list']  = [b * damping for b in params_dict['beta_list']]

    # ================= 提取统计量 =================
    cond_vol = np.asarray(result['analysis']['cond_vol'], dtype=float)
    valid_vol = cond_vol[~np.isnan(cond_vol) & ~np.isinf(cond_vol)]
    
    if valid_vol.size > 10:
        params_dict['unconditional_var'] = float(np.mean(valid_vol ** 2))
        params_dict['vol_max'] = float(np.max(valid_vol))
        params_dict['vol_min'] = float(np.min(valid_vol))
        params_dict['vol_std'] = float(np.std(valid_vol))
    else:
        params_dict['unconditional_var'] = 1.0
        params_dict['vol_max'] = 5.0
        params_dict['vol_min'] = 0.1
        params_dict['vol_std'] = 1.0

    if 'scaling' in result:
        params_dict['scaling_mean'] = float(result['scaling']['mean'])
        params_dict['scaling_std']  = float(result['scaling']['std'])
    if 'params' in result:
        params_dict['garch_mu'] = float(result['params'].get('mu', 0.0))
   

    return params_dict
# ============================================================
# 质量验证
# ============================================================
def verify_step2_quality(df_noise, df_final, df_volatility, feature_list):
    """
    Step 2 专项质量验证：生成分特征的详细质量报告
    """
    print("\n【Step 2 深度质量检测 (Per-Feature)】")
    
    quality_records = []
    
    # 计算相关性矩阵差异 (全局指标，无法拆分到单特征，但可计算该特征的平均相关性偏差)
    corr_noise = df_noise.corr()
    corr_final = df_final.corr()
    corr_diff = np.abs(corr_final - corr_noise)
    # 计算每个特征与其他特征的相关性偏差的平均值
    mean_corr_diff_per_feature = corr_diff.mean(axis=1)

    for feature in feature_list:
        # 1. 提取序列
        s_noise = df_noise[feature]
        s_final = df_final[feature]
        
        # 2. 厚尾性 (Kurtosis Change)
        k_noise = s_noise.kurtosis()
        k_final = s_final.kurtosis()
        k_change = k_final - k_noise
        
        # 3. 极值合理性 (Max Sigma Check)
        # 检查相对于标准差的倍数
        std_val = s_final.std()
        max_abs_val = s_final.abs().max()
        sigma_ratio = max_abs_val / std_val if std_val > 1e-9 else 0.0
        
        # 4. 波动率聚集性 (Ljung-Box on squared residuals)
        # 这里集成 Ljung-Box 检验，不再在主函数里单独做
        s_final_clean = s_final[np.isfinite(s_final)]
        if len(s_final_clean) > 20:
            lb_res = acorr_ljungbox(s_final_clean ** 2, lags=[10], return_df=True)
            lb_p = float(lb_res['lb_pvalue'].iloc[0])
        else:
            lb_p = 1.0
            
        # 记录
        quality_records.append({
            'feature': feature,
            'kurtosis_noise': k_noise,
            'kurtosis_final': k_final,
            'kurtosis_increase': k_change,
            'max_sigma_ratio': sigma_ratio,
            'corr_diff_mean': mean_corr_diff_per_feature[feature],
            'lb_pvalue': lb_p,
            'is_heavy_tail': k_change > 0.1,  # 阈值可调
            'is_vol_clustered': lb_p < 0.05,
            'is_extreme_safe': sigma_ratio < 30.0 # 阈值可调
        })

    # 转为 DataFrame
    df_quality = pd.DataFrame(quality_records).set_index('feature')
    
    # 打印一些摘要
    print(f"   📊 峰度增加特征占比: {(df_quality['kurtosis_increase'] > 0).mean():.1%}")
    print(f"   📊 显著波动率聚集占比: {(df_quality['lb_pvalue'] < 0.05).mean():.1%}")
    print(f"   📏 最大 Sigma 倍数: {df_quality['max_sigma_ratio'].max():.2f}")
    
    # 找出潜在异常特征
    risky_features = df_quality[~df_quality['is_extreme_safe']].index.tolist()
    if risky_features:
        print(f"   ⚠️ 极值风险特征 ({len(risky_features)}个): {risky_features[:5]}...")
    else:
        print("   ✅ 所有特征极值均在安全范围内")

    return df_quality
# ============================================================
# Engineering Shell (From Code A): Main Orchestrator
# ============================================================
def inject_volatility_step2(
    df_synthetic_noise,
    garch_results,
    feature_list,
    risky_features=None,
    burnin=500,
    seed=42,
    output_dir=None
):
    """
    第二步：注入波动率
    架构：Code A (日志/报告/绘图/IO)
    内核：Code B (生成器/软截断/零均值还原)
    """
    os.makedirs(output_dir, exist_ok=True)
    rng_global = np.random.default_rng(seed)

    print("═" * 70)
    print("📌 第二步：注入 GARCH/EGARCH 条件波动率 (Soft Clipping Mode)")
    print("═" * 70)

    n_samples = len(df_synthetic_noise)
    time_index = df_synthetic_noise.index

    # 1. 提取参数 (使用 Code B 的动态提取器)
    print("\n【1. 提取模型参数】")
    all_params = {}
    model_summary = {'GARCH': 0, 'GJR': 0, 'EGARCH': 0, 'fallback': 0}

    for feature in feature_list:
        params = extract_garch_params_dynamic(garch_results, feature)
        # print(f"   {feature:<25} -> ", params if params else "Fallback")
        if params is None:
            params = {
                'model_class': 'EGARCH', 'model_type': 'Default',
                'omega': 0.01, 'alpha': 0.1, 'gamma': -0.05, 'beta': 0.9,
                'vol_max': 5.0, 'scaling_std': 1.0
            }
            model_summary['fallback'] += 1
        else:
            model_summary[params['model_class']] += 1
            
        all_params[feature] = params
        
        # 简单日志
        # if len(all_params) <= 5 or len(all_params) > len(feature_list) - 2:
        #     print(f"   {feature:<25} -> {params['model_class']}")

    print(f"\n   模型汇总: {model_summary}")

    # 2. 生成与注入
    print("\n【2. 生成波动率 & 注入 (Code B Kernel)】")
    volatility_dict = {}
    df_with_volatility = df_synthetic_noise.copy()

    for i, feature in enumerate(feature_list):
        params = all_params[feature]
        
        persistence = params.get('alpha', 0) + params.get('beta', 0) + 0.5 * params.get('gamma', 0)
        is_risky_feature = False

        # 强制指定某些字段走保底逻辑
        
        if any(k in feature for k in risky_features) or persistence > 0.98:
            is_risky_feature = True
            print(f"   ⚠️ {feature} 被标记为潜在风险特征 (Persistence: {persistence:.4f})")
            
        # if feature == "comp_ratio_post":is_risky_feature = False
        #准备历史波动率数据
        hist_vol_raw = np.array(garch_results[feature]['analysis']['cond_vol'])

        sigma = None # 初始化
        # === 分支 1: 启动保底方案 (Bootstrap) ===
        if is_risky_feature and len(hist_vol_raw) > 100:
            print(f"   🛡️ {feature} 启用历史重采样保底 (避免爆炸/零膨胀)")# 测试20260208
            sigma = generate_bootstrap_volatility(hist_vol_raw, n_samples)
            # print(f"   2️⃣ sigma: 均值={np.mean(sigma):.6f}, 最大={np.max(sigma):.6f}")# 测试20260208


        # === 分支 2: 正常 GARCH 递归 ===
        else:
            # print(f"    {feature} 启用正常GARCH生成器 (持久性: {persistence:.4f})")# 测试20260208

            # --- A. Bootstrap Burn-in (Code B) ---
            z_t = df_synthetic_noise[feature].to_numpy(dtype=float)

            # print(f"   1️⃣ z_t: 均值={np.mean(z_t):.6f}, 标准差={np.std(z_t):.6f}")# 测试20260208
            rng = np.random.default_rng(seed + i * 1000)
            z_full = _make_burnin_z_from_series(z_t, burnin, rng)

            # --- B. 调用生成器 (Code B) ---
            cls = params['model_class']
            args = {
                'z_full': z_full, 'omega': params['omega'], 
                'unconditional_var': params.get('unconditional_var'), 'burnin': burnin
            }
            
            if cls == 'GJR':
                sigma, _ = generate_gjr_garch_volatility_from_z(
                    alpha=params['alpha'], gamma=params['gamma'], beta=params['beta'], **args
                )
            elif cls == 'EGARCH':
                sigma, _ = generate_egarch_volatility_from_z(
                    alpha=params['alpha'], gamma=params['gamma'], beta=params['beta'], **args
                )
            else: # GARCH
                sigma, _ = generate_garch_volatility_from_z(
                    alpha_list=params['alpha_list'], beta_list=params['beta_list'], **args
                )

        # print(f"   2️⃣ sigma: 均值={np.mean(sigma):.6f}, 最大={np.max(sigma):.6f}")# 测试20260208
        # --- C. 截断策略: Soft Clipping (Code B 核心优势) ---
        # 逻辑：允许极值存在，但通过对数压缩防止其无限发散
        hist_max = params.get('vol_max', 10.0)
        limit_upper = max(hist_max * 4.0, 10.0) # 允许比历史最大值大4倍
        # 应用 Tanh 饱和
        sigma = smooth_tanh_saturation(sigma, limit_upper)
        volatility_dict[feature] = sigma
        
        if np.max(sigma) > limit_upper:
            # 软截断公式: limit + log(1 + (x - limit))
            mask = sigma > limit_upper
            # print(f"      🔧 {feature} 触发软截断 (Max: {np.max(sigma):.1f} -> {limit_upper:.1f}+log)")
            sigma[mask] = limit_upper + np.log1p(sigma[mask] - limit_upper)
        
        # print(f"   2️⃣ 截断sigma: 均值={np.mean(sigma):.6f}, 最大={np.max(sigma):.6f}")# 测试20260208

        volatility_dict[feature] = sigma

        # --- D. 注入与还原 (Code B 逻辑: 加 scaling_mean) ---
        # ε_scaled = σ_scaled * z_t
        eps_scaled = sigma * z_t
        
        # print(f"   3️⃣ eps_scaled (sigma*z): 均值={np.mean(eps_scaled):.6f}") # 测试20260208
        garch_mu = params.get('garch_mu', 0.0)
        scaled_with_mu = garch_mu + eps_scaled
        # print(f"   4️⃣ garch_mu = {garch_mu:.6f}")# 测试20260208
        # print(f"   5️⃣ scaled_with_mu (mu + sigma*z): 均值={np.mean(scaled_with_mu):.6f}")# 测试20260208
        # 还原: raw = scaled * std / 100 
        # 需要加入均值，因为数据并非零均值
        scaling_std = params.get('scaling_std', 1.0)
        scaling_mean = params.get('scaling_mean', 0.0)

        # eps_raw = eps_scaled * scaling_std / 100.0 + scaling_mean
        eps_raw = scaled_with_mu * scaling_std / 100.0 + scaling_mean
        # eps_raw = scaled_with_mu
        if 'redundancy' in feature:
            current_std = np.std(eps_scaled) + 1e-9
            correction_factor = 100.0 / current_std
            eps_scaled_final = eps_scaled * correction_factor
            eps_raw = (eps_scaled_final * scaling_std / 100.0) + scaling_mean
            # print(f"   注意: {feature} 是冗余特征，未加均值还原")# 测试20260208


        # print(f"   6️⃣ 最终 eps_raw:")# 测试20260208
        # print(f"      均值 = {np.mean(eps_raw):.6f}")
        # print(f"      标准差 = {np.std(eps_raw):.6f}")
        # print(f"      期望均值 ≈ {scaling_mean:.6f} (scaling_mean)")
        
        df_with_volatility[feature] = eps_raw

    # 结果转 DataFrame
    volatility_df = pd.DataFrame(volatility_dict, index=time_index)

    # 3. 质量验证 (保留 Code A 的 Ljung-Box 报告)
    print("\n【3. 质量验证】")
    # 调用新的验证函数，获取分特征报告
    df_quality_report = verify_step2_quality(
        df_synthetic_noise, 
        df_with_volatility, 
        volatility_df, 
        feature_list
    )
    
    # 保存分特征验证报告
    quality_csv_path = os.path.join(output_dir, 'quality_check_per_feature.csv')
    df_quality_report.to_csv(quality_csv_path, encoding='utf-8-sig')
    print(f"   📝 分特征质量报告已保存: {quality_csv_path}")

    # 计算全局汇总指标 (用于 JSON 报告)
    global_metrics = {
        'avg_kurtosis_increase': float(df_quality_report['kurtosis_increase'].mean()),
        'avg_corr_diff': float(df_quality_report['corr_diff_mean'].mean()),
        'max_sigma_ratio': float(df_quality_report['max_sigma_ratio'].max()),
        'vol_clustered_ratio': float((df_quality_report['lb_pvalue'] < 0.05).mean()),
        'heavy_tail_ratio': float((df_quality_report['kurtosis_increase'] > 0).mean())
    }

    # 4. 可视化 (修改版：逐个特征绘图)
    print("\n【4. 生成可视化 (逐特征)】")
    
    # 创建专门存放图片的子文件夹，防止文件过多杂乱
    plots_dir = output_dir
    # os.makedirs(plots_dir, exist_ok=True)
    
    for idx, feature in enumerate(feature_list):
        # 为每个特征创建一个独立的画布
        # 设置 figsize 为长条形，适合左右对比
        fig = plt.figure(figsize=(16, 6))
        
        # --- 左图: 波动率 (1行2列的第1个) ---
        ax1 = fig.add_subplot(1, 2, 1)
        vol_data = volatility_df[feature].iloc[:2000] # 只画前2000个点，避免过密
        ax1.plot(vol_data.index, vol_data.values, linewidth=1.0, color='red', alpha=0.8)
        ax1.set_title(f'{feature}\nConditional Volatility (Scaled)', fontsize=12)
        ax1.grid(True, alpha=0.3)
        # 优化时间轴显示
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        
        # --- 右图: 注入效果对比 (1行2列的第2个) ---
        ax2 = fig.add_subplot(1, 2, 2)
        
        # 画原始白噪声 (背景)
        noise_data = df_synthetic_noise[feature].iloc[:2000]
        ax2.plot(noise_data.index, noise_data.values, linewidth=0.5, alpha=0.4, color='gray', label='z_t (White Noise)')
        
        # 画注入后的残差 (前景)
        injected_data = df_with_volatility[feature].iloc[:2000]
        ax2.plot(injected_data.index, injected_data.values, linewidth=0.8, alpha=0.9, color='steelblue', label='ε_t (GARCH Raw)')
        
        ax2.set_title(f'{feature}\nVolatility Injection Effect', fontsize=12)
        ax2.legend(fontsize=9, loc='upper right')
        ax2.grid(True, alpha=0.3)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

        plt.tight_layout()
        
        # 处理文件名中可能存在的非法字符 (如 / 或空格)
        safe_fname = str(feature).replace('/', '_').replace('\\', '_').replace(' ', '_')
        save_path = os.path.join(plots_dir, f'step2_volatility_{safe_fname}.png')
        
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig) # 关键：画完一个关闭一个，释放内存
        
        if (idx + 1) % 5 == 0:
            print(f"   ...已保存 {idx + 1}/{len(feature_list)} 张图片")

    print(f"   ✅ 所有特征对比图已保存至: {plots_dir}")

    plt.tight_layout()
    plt.savefig(f'{output_dir}/step2_volatility_injection.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 5. 保存结果 (Code A 的 IO 逻辑)
    print("\n【5. 保存结果】")
    df_with_volatility.to_csv(f'{output_dir}/synthetic_with_volatility.csv', encoding='utf-8-sig')
    volatility_df.to_csv(f'{output_dir}/synthetic_volatility_scaled.csv', encoding='utf-8-sig')
    
    try:
        df_with_volatility.to_parquet(f'{output_dir}/synthetic_with_volatility.parquet')
        volatility_df.to_parquet(f'{output_dir}/synthetic_volatility_scaled.parquet')
        print("   ✅ Parquet 保存成功")
    except:
        pass

    # 生成 JSON 报告
    injection_report = {
        'metadata': {'n_samples': int(n_samples), 'burnin': int(burnin), 'seed': int(seed)},
        'strategy': {
            'clipping': 'Tanh Saturation (Strict Bound)', # 更新描述
            'damping': 'Max Persistence 0.985',           # 更新描述
            'rescaling': 'Zero-Mean (No Trend Addition)'},
        'model_summary': model_summary,
        'global_quality_metrics': global_metrics, # 使用新的汇总指标
        # 'clustering_check': {k: bool(v) for k, v in clustering_results.items()}
    }
    with open(f'{output_dir}/volatility_injection_report.json', 'w', encoding='utf-8') as f:
        json.dump(injection_report, f, ensure_ascii=False, indent=2)

    print(f"   ✅ 完成。数据已保存至: {output_dir}")
    return df_with_volatility, volatility_df, injection_report


def run_step2_inject_volatility_v4(
    df_synthetic_noise,
    garch_results,
    feature_list,
    risky_features,
    burnin=500,
    seed=42,
    output_dir=None
):
    """
    流程封装Wrapper
    """
    return inject_volatility_step2(
        df_synthetic_noise, garch_results, feature_list, risky_features, burnin, seed, output_dir
    )

### run

In [ ]:
df_with_volatility, volatility_df, vol_report = run_step2_inject_volatility_v4(
    df_synthetic_noise=df_synthetic_noise,  # 第一步输出
    garch_results=garch_results,            # 步骤6的GARCH结果
    feature_list=feature_list,              # 19个特征
    risky_features = ['gini', 'comp_ratio','retweet_ratio_post','semantic_shift_post', 'vis_abs'],
    burnin=500,                             # 预热期
    seed=42,
    output_dir=os.path.join(RESULT_PATH, 'g_step2_inject_volatility')
)

# 查看结果
print(f"注入波动率后数据形状: {df_with_volatility.shape}")
print(f"时间范围: {df_with_volatility.index[0]} ~ {df_with_volatility.index[-1]}")


### 质量检验

In [ ]:
run_quality_validation(
    2,
    df_ar_resid, 
    df_with_volatility, 
    version_dict_complete,
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g2_garch_validation'))

## step3 AR

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import lfilter
import matplotlib.pyplot as plt
import os
import json
import warnings

# 屏蔽可能的未来警告
warnings.filterwarnings('ignore')

def parse_ar_params(feature_name, feature_ar_data):
    """
    辅助函数：解析复杂格式的 AR 结果字典
    
    Args:
        feature_name: 特征名
        feature_ar_data: 字典，格式如 {'params': {...}, 'lags': [1,2], 'input_col': '...'}
    
    Returns:
        coeffs_array: 按顺序排列的 AR 系数数组 (缺失 lag 补 0)
        ar_order: 最大滞后阶数
    """
    # 1. 安全检查
    if not feature_ar_data or 'lags' not in feature_ar_data or not feature_ar_data['lags']:
        return np.array([]), 0
        
    params = feature_ar_data.get('params', {})
    lags = feature_ar_data.get('lags', [])
    input_col = feature_ar_data.get('input_col', '') # 获取原始列名用于拼凑 key
    
    # 2. 确定最大阶数 (处理稀疏滞后，例如只选了 lag 1 和 5)
    max_lag = max(lags)
    
    # 3. 初始化系数数组 (索引 i 对应 lag i+1)
    coeffs_array = np.zeros(max_lag)
    
    # 4. 填充系数
    # 尝试两种 key 格式：
    # 格式 A: "{input_col}.L{lag}" (Statsmodels 标准输出)
    # 格式 B: "L{lag}.{input_col}" (部分旧版本格式)
    for lag in lags:
        # 尝试构建可能的键名
        key_candidates = [
            f"{input_col}.L{lag}",
            f"L{lag}.{input_col}"
        ]
        
        found = False
        for key in key_candidates:
            if key in params:
                coeffs_array[lag-1] = params[key] # lag 1 存入 index 0
                found = True
                break
        
        if not found:
            print(f"   ⚠️ {feature_name}: 未找到 Lag {lag} 的系数 (Key 尝试: {key_candidates})")

    return coeffs_array, max_lag
from statsmodels.tsa.ar_model import AutoReg


def verify_step3_quality(df_input, df_output, ar_params_dict):
    """
    Step 3 专项验证：记忆性植入检查与参数回测
    """
    print("\n【Step 3 深度质量检测 (Parameter Recovery)】")
    
    results = []
    
    features = df_output.columns
    for feature in features:
        # ================= 改动点：适配复杂输入格式 =================
        # 使用之前定义的解析函数，将字典转换为稠密数组
        # 例如: {'L1': 0.5, 'L3': 0.2} -> [0.5, 0.0, 0.2]
        if feature in ar_params_dict:
            target_coeffs, max_lag = parse_ar_params(feature, ar_params_dict[feature])
        else:
            target_coeffs = np.array([])
            
        if len(target_coeffs) == 0:
            continue
        # ==========================================================
            
        # 1. 自相关性变化 (ACF Check)
        # 理论上 Step 3 的 ACF(1) 应该显著提升（对于正相关）或改变
        acf_in = df_input[feature].autocorr(lag=1)
        acf_out = df_output[feature].autocorr(lag=1)
        acf_change = acf_out - acf_in
        
        # 2. 参数回测 (Coefficient Recovery)
        # 用生成的数据反向训练一个 AR 模型，看能不能还原出参数
        # target_coeffs 是稠密数组 (包含0)，长度即为最大滞后阶数
        lags = len(target_coeffs)
        
        try:
            # 仅使用前 5000 个数据进行快速拟合，防止数据量过大变慢
            train_data = df_output[feature].values[:5000]
            
            # AutoReg fit: lags=int 意味着拟合 lag 1 到 lag k
            # 这与我们 target_coeffs 的结构 (稠密数组) 是一致的
            model = AutoReg(train_data, lags=lags, old_names=False)
            res = model.fit()
            
            # 提取拟合出的系数 (params[0]是const intercept, 后面才是phi)
            fitted_coeffs = res.params[1:] 
            
            # 计算系数误差 (MAE)
            if len(fitted_coeffs) == len(target_coeffs):
                coeff_error = np.mean(np.abs(fitted_coeffs - target_coeffs))
            else:
                coeff_error = 999.0
                
        except Exception as e:
            # print(f"Fit failed for {feature}: {e}")
            fitted_coeffs = []
            coeff_error = 999.0
            
        results.append({
            'feature': feature,
            'acf_input': acf_in,
            'acf_output': acf_out,
            'acf_delta': acf_change,
            'target_phi1': target_coeffs[0], # 目标第一阶系数
            'fitted_phi1': fitted_coeffs[0] if len(fitted_coeffs)>0 else 0,
            'coeff_recovery_mae': coeff_error
        })
        
    if not results:
        print("   ⚠️ 没有有效的 AR 特征进行验证")
        return pd.DataFrame()

    df_res = pd.DataFrame(results).set_index('feature')
    
    # --- 打印摘要 ---
    print(f"   📈 平均自相关提升: {df_res['acf_delta'].mean():.4f}")
    print(f"   🎯 系数还原误差 (MAE): {df_res['coeff_recovery_mae'].mean():.4f}")
    
    # 筛选还原失败的特征 (误差 > 0.1 视为偏差较大，对于合成数据通常允许宽松一点)
    bad_recovery = df_res[df_res['coeff_recovery_mae'] > 0.15]
    if not bad_recovery.empty:
        print(f"   ⚠️ 以下特征参数还原度较低 ({len(bad_recovery)}个):")
        print(bad_recovery[['target_phi1', 'fitted_phi1', 'coeff_recovery_mae']].head())
    else:
        print("   ✅ 所有特征的 AR 参数均成功还原 (Error < 0.15)")
        
    return df_res



def apply_ar_filter_step3(
    df_step2_input,   # 来自 Step 2 的输出 (df_with_volatility)
    ar_params_dict,   # 分解流程 Step 4 的 AR 结果字典
    feature_list,
    output_dir='./result/step3_ar'
):
    print("═" * 70)
    print("📌 第三步：叠加 AR(p) 短期线性自相关 (Inertia Injection)")
    print("═" * 70)
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 容器
    df_ar_output = pd.DataFrame(index=df_step2_input.index)
    ar_info_log = {}
    
    # 统计计数
    stats_count = {'processed': 0, 'skipped': 0, 'unstable': 0}

    for feature in feature_list:
        input_series = df_step2_input[feature].values
        
        # 1. 获取并解析 AR 参数
        if feature in ar_params_dict:
            coeffs, ar_order = parse_ar_params(feature, ar_params_dict[feature])
        else:
            coeffs, ar_order = np.array([]), 0
        
        # 2. 检查系数有效性与稳定性
        is_stable = True
        
        if len(coeffs) == 0:
            # 无 AR 参数
            output_series = input_series
            stats_count['skipped'] += 1
        else:
            # 简单的平稳性检查 (充分非必要条件: sum(|phi|) < 1)
            # 对于合成数据，严格一点比较安全，防止长周期发散
            # 如果仅仅略大于1，可能导致数据缓慢飘走，建议在这里做个阻尼或者跳过
            sum_abs = np.sum(np.abs(coeffs))
            
            if sum_abs >= 1.0:
                print(f"   ⚠️ {feature}: AR 系数非平稳 (Sum Abs={sum_abs:.4f} >= 1.0)，已降级为无自相关")
                output_series = input_series
                is_stable = False
                stats_count['unstable'] += 1
                ar_order = 0 # 重置以便记录
                coeffs = []
            else:
                # 3. 应用 AR 滤波 (lfilter)
                # y[t] = phi_1*y[t-1] + phi_2*y[t-2] + ... + x[t]
                # 变换为: y[t] - phi_1*y[t-1] - ... = x[t]
                # Scipy lfilter: a[0]*y[t] + a[1]*y[t-1]... = b[0]*x[t]
                # 对应: a = [1, -phi_1, -phi_2, ...], b = [1]
                
                b = [1.0]
                a = np.concatenate(([1.0], -coeffs))
                
                # 注意：此处输入的是 Step 2 的带波动率残差 (Innovation)
                # lfilter 会将其作为白噪声输入，生成具有自相关的序列
                output_series = lfilter(b, a, input_series)
                stats_count['processed'] += 1

        df_ar_output[feature] = output_series
        
        # 记录日志 (转换 numpy array 为 list 以便 JSON 序列化)
        ar_info_log[feature] = {
            'order': int(ar_order), 
            'coeffs': list(coeffs) if len(coeffs) > 0 else [],
            'stable': bool(is_stable)
        }

    # 4. 验证 (ACF 检查)
    print("\n   📈 自相关性 (Lag-1) 变化检查 (Top 5 Features):")
    print(f"   {'Feature':<30} {'Step2(Raw)':<12} {'Step3(AR)':<12} {'Change'}")
    print("   " + "-" * 65)
    
    count = 0
    for feat in feature_list:
        if feat not in ar_info_log or ar_info_log[feat]['order'] == 0:
            continue
        
        acf2 = df_step2_input[feat].autocorr(lag=1)
        acf3 = df_ar_output[feat].autocorr(lag=1)
        print(f"   {feat:<30} {acf2:.4f}       {acf3:.4f}       {acf3-acf2:+.4f}")
        
        count += 1
        if count >= 5: break
        
    print(f"\n   📊 处理统计: 成功叠加 {stats_count['processed']}, 跳过/无参 {stats_count['skipped']}, 非平稳剔除 {stats_count['unstable']}")

    # 5. 可视化对比 (生成一张图，对比前后的 ACF 或 时序)
    plot_ar_effect(df_step2_input, df_ar_output, feature_list, output_dir)
    
    # 6. 质量验证
    df_quality_report = verify_step3_quality(
        df_step2_input, 
        df_ar_output, 
        ar_params_dict # 传入原始字典，函数内部会调用 parse_ar_params
    )
    if not df_quality_report.empty:
        df_quality_report.to_csv(f'{output_dir}/quality_check_ar_recovery.csv')
        
        # 计算汇总指标
        recovery_metrics = {
            'avg_acf_delta': float(df_quality_report['acf_delta'].mean()),
            'avg_coeff_mae': float(df_quality_report['coeff_recovery_mae'].mean()),
            'failed_recovery_count': int((df_quality_report['coeff_recovery_mae'] > 0.15).sum())
        }
    else:
        recovery_metrics = {}
        
    # 7. 保存
    df_ar_output.to_csv(f'{output_dir}/step3_ar_output.csv', encoding='utf-8-sig')
    try:
        df_ar_output.to_parquet(f'{output_dir}/step3_ar_output.parquet')
    except:
        pass
    
    with open(f'{output_dir}/step3_ar_params.json', 'w', encoding='utf-8') as f:
        json.dump(ar_info_log, f, indent=2, ensure_ascii=False)

    print(f"   ✅ Step 3 完成。结果已保存至 {output_dir}")
    return df_ar_output

def plot_ar_effect(df_in, df_out, feature_list, output_dir):
    """
    绘制 AR 效果对比图 (选择前 3 个有 AR 参数的特征)
    """
    # 筛选出真正应用了 AR 的特征
    valid_features = []
    for f in feature_list:
        if not df_in[f].equals(df_out[f]):
            valid_features.append(f)
    
    if not valid_features:
        return

    n_plot = min(len(valid_features), 3)
    fig, axes = plt.subplots(n_plot, 2, figsize=(15, 4 * n_plot))
    if n_plot == 1: axes = [axes] # 统一维度

    for i in range(n_plot):
        feat = valid_features[i]
        
        # 时序图 (局部)
        ax_ts = axes[i][0] if n_plot > 1 else axes[0]
        ax_ts.plot(df_in[feat].iloc[:200], label='Step2 (Uncorrelated)', alpha=0.5, color='gray', lw=1)
        ax_ts.plot(df_out[feat].iloc[:200], label='Step3 (AR Injected)', alpha=0.8, color='blue', lw=1)
        ax_ts.set_title(f"{feat} - Time Series (First 200 pts)")
        ax_ts.legend()
        
        # ACF 图 (自相关图)
        ax_acf = axes[i][1] if n_plot > 1 else axes[1]
        
        # 手动计算简单的 ACF 用于绘图
        lags = range(11)
        acf_in = [df_in[feat].autocorr(lag=l) for l in lags]
        acf_out = [df_out[feat].autocorr(lag=l) for l in lags]
        
        ax_acf.plot(lags, acf_in, 'o--', label='Step2 ACF', color='gray')
        ax_acf.plot(lags, acf_out, 'o-', label='Step3 ACF', color='red')
        ax_acf.axhline(0, color='black', lw=0.5)
        ax_acf.set_title(f"{feat} - Autocorrelation Function")
        ax_acf.set_xlabel("Lag")
        ax_acf.legend()

    plt.tight_layout()
    plt.savefig(f'{output_dir}/step3_ar_check.png')
    plt.close()

# 示例调用 (假设环境已准备好)
# df_step3 = apply_ar_filter_step3(df_step2, ar_results, feature_list)

### run

In [ ]:
df_synthetic_ar = apply_ar_filter_step3(
    df_with_volatility,   # 来自 Step 2 的输出 (df_with_volatility)
    ar_results,   # 分解流程 Step 4 的 AR 系数
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g_step3_ar')
)


### 质量检验

In [ ]:
run_quality_validation(
    3,
    df_impact_resid, 
    df_synthetic_ar, 
    version_dict_complete,
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g3_ar_validation'))

## step4 叠加时间冲击

In [ ]:
print(df_official.head(5))

In [ ]:
df_official.category.unique()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# 1. 基础工具函数
# ============================================================

def generate_decay_wave(length, impact_coef, decay_type, decay_param):
    """
    生成衰减波形
    """
    if length <= 0: return np.array([])
    t = np.arange(length)
    
    decay_param = float(decay_param)
    if decay_param < 1e-6: decay_param = 1e-6

    if decay_type == 'exponential':
        # exp(-t/tau)
        wave = np.exp(-t / decay_param)
    elif decay_type == 'power':
        # (t+1)^(-alpha)
        wave = np.power(t + 1, -decay_param)
    else:
        # 默认
        wave = np.exp(-t / 8.0)
        
    return impact_coef * wave

def load_decomposition_params(json_path):
    """从 JSON 加载分解参数"""
    if not os.path.exists(json_path):
        print(f"⚠️ 警告: 参数文件 {json_path} 不存在")
        return {}
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# ============================================================
# 2. 过滤逻辑
# ============================================================

def check_modeling_criteria(feature, combo_name, coef, p_val, model_selection=None):
    """
    【双重过滤】判断是否对该特征的该组合事件进行建模
    """
    if model_selection is None:
        model_selection = {}
        
    # 参数解包
    threshold_p = model_selection.get('threshold_p_value', 0.1) 
    threshold_coef = model_selection.get('threshold_min_coef', 0.001) # Sigma单位
    force_positive_features = model_selection.get('force_positive_features', 
                                                ['volume', 'post', 'comment', 'view', 'share'])
    
    # 0. Crisis 硬过滤 (双重保险)
    if 'crisis' in combo_name:
        return False, "Crisis event excluded"

    # 1. 幅度过滤
    if abs(coef) < threshold_coef:
        return False, f"Impact too small ({abs(coef):.4f} < {threshold_coef})"
        
    # 2. 显著性过滤
    if p_val > threshold_p:
        return False, f"Not significant (p={p_val:.4f} > {threshold_p})"
        
    # 3. 业务逻辑过滤 (方向性)
    is_volume_metric = any(kw in feature for kw in force_positive_features)
    if is_volume_metric:
        if coef < 0:
            return False, f"Negative impact on volume metric ({coef:.4f})"

    return True, "Pass"

# ============================================================
# 3. 质量验证函数 (生成 quality_check_event_impact)
# ============================================================

def verify_step4_quality(df_pre, df_post, df_events, feature_list, output_dir):
    """
    Step 4 专项验证：检查冲击覆盖率和极值风险
    输出: quality_check_event_impact.csv
    """
    print("\n【Step 4 深度质量检测 (Post-Synthesis Validation)】")
    
    results = []
    check_window = 16 # 4小时窗口
    
    # 计算纯冲击成分
    df_shock_only = df_post - df_pre
    
    # 筛选有效时间范围内的事件
    sim_start = df_post.index.min()
    sim_end = df_post.index.max()
    valid_events = df_events[(df_events['timestamp'] >= sim_start) & 
                             (df_events['timestamp'] <= sim_end)]
    
    for feature in feature_list:
        if feature not in df_shock_only.columns: continue

        shock_series = df_shock_only[feature]
        max_impact = shock_series.max()
        min_impact = shock_series.min()
        
        # 状态判断
        if abs(max_impact) < 1e-9 and abs(min_impact) < 1e-9:
            results.append({
                'feature': feature, 'status': 'No Impact',
                'max_impact': 0, 'min_impact': 0,
                'event_response_rate': 0, 'max_sigma_jump': 0, 'is_safe': True
            })
            continue
            
        # 计算事件响应覆盖率
        active_events = 0
        total_events = 0
        
        # 采样检查事件响应 (加速)
        check_events = valid_events['timestamp']
        if len(check_events) > 500:
            check_events = check_events.sample(500, random_state=42)

        for t in check_events:
            try:
                locs = shock_series.index.get_indexer([t], method='nearest', tolerance=pd.Timedelta('30min'))
                if locs[0] == -1: continue
                
                start_loc = locs[0]
                end_loc = min(start_loc + check_window, len(shock_series))
                local_shock = shock_series.iloc[start_loc:end_loc]
                
                # 如果窗口内有明显非零值
                if local_shock.abs().max() > 1e-5:
                    active_events += 1
                total_events += 1
            except: continue
            
        response_rate = active_events / total_events if total_events > 0 else 0.0
        
        # 极值风险检查
        pre_std = df_pre[feature].std()
        if pre_std < 1e-6: pre_std = 1e-6
        max_sigma_jump = max(abs(max_impact), abs(min_impact)) / pre_std
        
        results.append({
            'feature': feature,
            'status': 'Active',
            'max_impact': round(max_impact, 4),
            'min_impact': round(min_impact, 4),
            'event_response_rate': round(response_rate, 2),
            'max_sigma_jump': round(max_sigma_jump, 2),
            'is_safe': max_sigma_jump < 20.0 # 20倍标准差阈值
        })
        
    # 保存报告
    df_res = pd.DataFrame(results)
    if not df_res.empty:
        save_path = os.path.join(output_dir, 'quality_check_event_impact.csv')
        df_res.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        active_df = df_res[df_res['status']=='Active']
        print(f"   📊 平均事件响应率: {active_df['event_response_rate'].mean():.1%}")
        
        risky = active_df[~active_df['is_safe']]
        if not risky.empty:
            print(f"   ⚠️ 警告: {len(risky)} 个特征冲击幅度过大 (>20 std):")
            print(risky[['feature', 'max_sigma_jump']].head(3))
        else:
            print("   ✅ 极值检查通过 (Max Sigma Jump < 20)")

# ============================================================
# 4. 核心合成主函数 (含包络与埋点)
# ============================================================

def apply_event_shocks_step4_envelope(
    df_step3_input,       # Step 3 输出 (AR 残差)
    df_official_schedule, # 官方排期表
    decomp_params,        # 分解参数
    feature_list,
    model_selection=None, 
    output_dir='./result/step4_event_envelope'
):
    print("═" * 70)
    print("📌 第四步：官方运营事件冲击 (Envelope Strategy & Deep Debug)")
    print("═" * 70)
    
    os.makedirs(output_dir, exist_ok=True)
    

    feature_impact_adjustments = {
        # 🔴 红色警报：严重过冲 (Tail Diff > 200%)
        # 行动：大幅削减，建议打 5-6 折
        'comp_ratio_post': 0.5, 

        # 🟡 黄色警报：评论类普遍过冲 (Tail Diff 50%~80%)
        # 行动：因为是累加效应导致的，建议打 7 折
        'total_volume_comment': 0.7,
        'gini_comment': 0.7,
        'senti_symbol_comment': 0.7,
        'comp_ratio_comment': 0.7,
        'total_short_comment': 0.7,
        'total_long_comment': 0.7,
        'neg_ratio_comment': 0.7,
        'semantic_shift_comment': 0.7,

        # 🟢 绿色通行：表现良好 (Tail Diff < 20%)
        # 行动：保持原样 (默认 1.0)，显式写出来是为了方便管理
        'total_volume_post': 1.0,
        'gini_post': 1.0,
        'retweet_ratio_post': 1.0,
        'neg_ratio_post': 1.0,
        'vis_abs_redundancy_post': 1.0,
        'vis_concentration_post': 1.0
    }

    # --- 0. 配置缺省值 ---
    if model_selection is None:
        model_selection = {
            'threshold_p_value': 0.1,
            'threshold_min_coef': 0.001,
            'simulation_strength': 1.0 # 强度调节因子
        }
    
    # --- 1. 预处理排期 (强制过滤 Crisis) ---
    df_sch = df_official_schedule.copy()
    df_sch['timestamp'] = pd.to_datetime(df_sch['timestamp'])
    
    # 强制过滤 category == 'crisis'
    original_len = len(df_sch)
    df_sch = df_sch[df_sch['category'] != 'crisis']
    print(f"   🛡️ 已过滤 {original_len - len(df_sch)} 个 Crisis 事件")
        
    df_sch['combo'] = df_sch['official_author'] + '_' + df_sch['category']
    
    # 筛选时间范围
    sim_start = df_step3_input.index.min()
    sim_end = df_step3_input.index.max()
    df_sch = df_sch[(df_sch['timestamp'] >= sim_start) & (df_sch['timestamp'] <= sim_end)]
    
    print(f"   📅 有效模拟事件数: {len(df_sch)}")
    
    df_output = df_step3_input.copy()
    
    # 全局日志容器
    all_filter_logs = []
    all_modeling_stats = []
    
    # --- 2. 逐特征处理 ---
    for feature in feature_list:
        if feature not in decomp_params:
            continue
            
        feat_adj_factor = feature_impact_adjustments.get(feature, 1.0)
        
        # 打印一下日志，确保你知道它生效了
        if feat_adj_factor != 1.0:
            print(f"   🔧 微调生效: {feature} 冲击强度 x {feat_adj_factor}")
        feat_data = decomp_params[feature]
        coeffs_map = feat_data.get('coefficients', {})
        
        # 初始化双通道包络层 & 计数层
        n_points = len(df_step3_input)
        total_shock_accumulator = np.zeros(n_points)
        pos_envelope = np.zeros(n_points)  # 正向最大值层
        neg_envelope = np.zeros(n_points)  # 负向最小值层 (存负值)
        event_counts = np.zeros(n_points, dtype=int) # 堆积计数器
        
        # 特征级调试日志
        wave_debug_log = []
        stacking_log = []
        shock_magnitudes = []
        
        # 预计算组合决策 (生成 filter_logs)
        combo_decisions = {}
        for combo, param in coeffs_map.items():
            coef = param['coef']
            p_val = param.get('pvalue', 1.0)
            
            should_model, reason = check_modeling_criteria(
                feature, combo, coef, p_val, model_selection
            )
            
            combo_decisions[combo] = {'pass': should_model, 'param': param}
            
            if not should_model:
                all_filter_logs.append({
                    'feature': feature, 
                    'combo': combo, 
                    'coef': coef, 
                    'p_val': p_val, 
                    'reason': reason
                })

        # 计算基准波动率 (用于量纲还原)
        # [关键修正]: 分解系数是Sigma单位，合成需要还原为绝对值
        current_std = df_step3_input[feature].std()
        if current_std < 1e-6: current_std = 1e-6

        # --- 事件遍历 ---
        cnt_applied = 0
        
        for _, event in df_sch.iterrows():
            combo = event['combo']
            
            # 检查决策
            if combo not in combo_decisions or not combo_decisions[combo]['pass']:
                continue
                
            param = combo_decisions[combo]['param']
            
            # [量纲还原] Sigma Coef -> Absolute Impact
            raw_coef = param['coef']
            sim_strength = model_selection.get('simulation_strength', 1.0)
            # 核心公式：基础冲击 * 全局强度 * 特征专属微调
            actual_impact = raw_coef * current_std * sim_strength * feat_adj_factor
            
            # 波形参数
            d_type = param['decay_type']
            d_param = param['decay_param']
            
            # 定位时间
            try:
                locs = df_step3_input.index.get_indexer([event['timestamp']], method='nearest', tolerance=pd.Timedelta('30min'))
                if locs[0] == -1: continue
                idx_loc = locs[0]
            except: continue
            
            # # 决定长度 (限制最长48小时)
            # if d_type == 'exponential':
            #     duration = int(d_param * 5)
            # else: # power
            #     try: duration = int(np.power(0.05, -1.0/(d_param+1e-6)))
            #     except: duration = 96
            # duration = min(duration, 96*2) 
            
            # ================= [新增] 参数锁死与保护 =================
            # 1. 修正衰减参数：防止指数衰减过慢
            # 对于指数分布，d_param 是 tau (时间尺度)，限制最大为 32 (约8小时半衰期)
            # 对于幂律分布，d_param 是 alpha (衰减速率)，通常 < 3，min(x, 32) 无副作用
            real_d_param = min(d_param, 32.0)
            
            # 2. 计算时长并应用硬截断
            if d_type == 'exponential':
                # 指数衰减：5倍 tau 约等于衰减到 0.6%
                duration = int(real_d_param * 5)
            else:
                # 幂律衰减：直接锁死 24 小时 (96 个点)
                # 幂律的长尾太长，通过计算衰减到 1% 的时间往往不靠谱
                duration = 96 
                
            # 3. 二次保护：绝对时长不超过 48 小时 (192点)
            duration = min(duration, 192)
            # =======================================================

            end_loc = min(idx_loc + duration, n_points)
            length = end_loc - idx_loc
            if length <= 0: continue
            
            # 生成波形
            wave = generate_decay_wave(length, actual_impact, d_type, d_param)
            
            # 如果是强制截断的幂律波形，对最后 4 个点做线性淡出，避免断崖
            if d_type == 'power' and length >= 10:
                fade_len = 4
                fade_factors = np.linspace(1, 0, fade_len)
                wave[-fade_len:] *= fade_factors
            
            # === [埋点 1] 波形末端检查 ===
            end_val_ratio = abs(wave[-1]) / (abs(actual_impact) + 1e-9)
            if end_val_ratio > 0.1: 
                wave_debug_log.append({
                    'time': str(event['timestamp']),
                    'end_ratio': round(end_val_ratio, 3)
                })

            # === [核心逻辑] 双通道包络 ===
            # 更新计数器
            event_counts[idx_loc:end_loc] += 1
            
            # 分通道取极值
            target_slice = slice(idx_loc, end_loc)
            slice_len = end_loc - idx_loc
            if len(wave) != slice_len:
                wave = wave[:slice_len]
            # [修复] 直接累加
            total_shock_accumulator[target_slice] += wave

            if actual_impact > 0:
                pos_envelope[target_slice] = np.maximum(pos_envelope[target_slice], wave)
            else:
                neg_envelope[target_slice] = np.minimum(neg_envelope[target_slice], wave)
                
            cnt_applied += 1
            shock_magnitudes.append(np.max(np.abs(wave)))
            
        # === [埋点 2] 堆积风险分析 ===
        max_overlap = np.max(event_counts) if len(event_counts) > 0 else 0
        
        # 合并双通道
        total_shock = pos_envelope + neg_envelope
        
        # 双通道注入
        # df_output[feature] = df_output[feature] + total_shock
        
        # 改用累加
        df_output[feature] = df_output[feature] + total_shock_accumulator
        
        # 记录 modeling_stats
        max_shock_val = np.max(np.abs(total_shock)) if len(total_shock) > 0 else 0
        all_modeling_stats.append({
            'feature': feature,
            'events_applied': cnt_applied,
            'max_overlap': max_overlap,
            'max_shock_val': max_shock_val,
            'avg_single_shock': np.mean(shock_magnitudes) if shock_magnitudes else 0,
            'wave_truncation_issues': len(wave_debug_log)
        })
        
        # 打印日志
        if cnt_applied > 0:
            print(f"   ⚡ {feature:<25}: 注入 {cnt_applied} 事件 | Max Overlap: {max_overlap} | Peak: {max_shock_val:.4f}")

    # --- 3. 保存所有输出文件 ---
    print("\n   💾 保存输出文件...")
    
    # 1. 最终数据
    df_output.to_csv(f'{output_dir}/step4_event_output.csv', encoding='utf-8-sig')
    
    # 2. 过滤日志 filter_logs_dropped
    if all_filter_logs:
        pd.DataFrame(all_filter_logs).to_csv(f'{output_dir}/filter_logs_dropped.csv', index=False, encoding='utf-8-sig')
        print(f"      📄 已生成 filter_logs_dropped.csv ({len(all_filter_logs)} 条记录)")
    
    # 3. 建模统计 modeling_stats
    if all_modeling_stats:
        pd.DataFrame(all_modeling_stats).to_csv(f'{output_dir}/modeling_stats.csv', index=False, encoding='utf-8-sig')
        print(f"      📄 已生成 modeling_stats.csv")

    # --- 4. 调用质量验证 ---
    # 生成 quality_check_event_impact.csv
    verify_step4_quality(df_step3_input, df_output, df_sch, feature_list, output_dir)
    print(f"      📄 已生成 quality_check_event_impact.csv")
    
    print(f"\n   ✅ Step 4 完成。结果已保存至 {output_dir}")
    return df_output



### 确认量纲

In [ ]:
# def diagnose_scaling_mismatch(
#     df_decomposed_data,   # 分解流程 Step 4 使用的数据表 (df_prophet_resid 或 df_decomposed)
#     df_synthesis_step3,   # 合成流程 Step 3 的输出 (df_ar_output)
#     regression_results,   # 分解得到的回归结果字典
#     feature_list
# ):
#     print("═" * 70)
#     print("🔍 量纲/缩放比例 深度诊断 (Scaling Mismatch Diagnosis)")
#     print("═" * 70)
    
#     mismatch_log = []
    
#     print(f"{'特征':<25} | {'分解端 Std':<12} | {'合成端 Std':<12} | {'比例 (分解/合成)':<15} | {'平均冲击系数':<12} | {'冲击/合成Std (倍数)'}")
#     print("-" * 110)
    
#     for feature in feature_list:
#         # 1. 获取分解端的数据统计
#         # 尝试找对应的列名
#         decomp_col = f'{feature}_resid' if f'{feature}_resid' in df_decomposed_data.columns else feature
#         if decomp_col not in df_decomposed_data.columns:
#             decomp_std = np.nan
#         else:
#             decomp_std = df_decomposed_data[decomp_col].std()
            
#         # 2. 获取合成端的数据统计
#         if feature in df_synthesis_step3.columns:
#             synth_std = df_synthesis_step3[feature].std()
#         else:
#             synth_std = np.nan
            
#         # 3. 获取回归系数的平均量级
#         avg_coef = 0.0
#         if feature in regression_results:
#             coeffs = [c['coef'] for c in regression_results[feature].get('coefficients', {}).values()]
#             if coeffs:
#                 avg_coef = np.mean(np.abs(coeffs))
                
#         # 4. 计算比例
#         ratio = decomp_std / synth_std if (synth_std and synth_std > 1e-9) else 0.0
#         shock_sigma = avg_coef / synth_std if (synth_std and synth_std > 1e-9) else 0.0
        
#         print(f"{feature:<25} | {decomp_std:<12.4f} | {synth_std:<12.4f} | {ratio:<15.2f} | {avg_coef:<12.4f} | {shock_sigma:.1f}x")
        
#         # 诊断逻辑
#         if ratio > 10.0 or ratio < 0.1:
#             mismatch_log.append(f"⚠️ {feature}: 量纲严重不匹配 (差异 {ratio:.1f} 倍)")
#         if shock_sigma > 10.0:
#             mismatch_log.append(f"⚠️ {feature}: 冲击幅度过大 (平均为 {shock_sigma:.1f} 倍合成端标准差)")

#     print("\n【诊断结论】")
#     if not mismatch_log:
#         print("✅ 量纲基本一致，数据处于同一尺度。")
#     else:
#         print("❌ 发现量纲错位，这解释了为什么冲击看起来那么大：")
#         for log in mismatch_log[:10]:
#             print("  " + log)
#         if len(mismatch_log) > 10: print("  ... (更多省略)")
        
#         # 自动推测原因
#         ratios = [df_decomposed_data[f'{f}_resid'].std()/df_synthesis_step3[f].std() 
#                   for f in feature_list if f in df_synthesis_step3.columns and f'{f}_resid' in df_decomposed_data.columns]
#         avg_ratio = np.nanmean(ratios)
        
#         print(f"\n💡 线索分析: 分解端的数据波动幅度大约是合成端的 {avg_ratio:.1f} 倍。")
#         if 90 < avg_ratio < 110:
#             print("👉 极大概率：合成端在 Step 2 还原时多除了一次 100，或者分解端用了 x100 后的数据。")
#             print("   (GARCH 建模常用 rescale=True 会将数据 x100)")
#         elif 0.9 < avg_ratio < 1.1:
#             print("👉 量纲看似一致，但个别特征冲击过大，可能是回归过拟合了异常值。")
#         else:
#             print(f"👉 可能是 Step 2 的 scaling_std 参数还原逻辑有误。")

# # 调用
# diagnose_scaling_mismatch(df_prophet_resid, df_synthetic_ar, regression_results, feature_list)

### run

In [ ]:
# 定义您的业务规则
selection_rules = {
    'threshold_p_value': 0.1,        # 显著性放宽到 0.1
    'threshold_min_coef': 0.005,     # 忽略小于 0.5% 的微小冲击
    # 'force_positive_features': ['total_volume', 'comment_count', 'interact_num']
    'force_positive_features': [''] # 必须正向的特征
      # 必须正向的特征
}

# 运行合成 (内部会自动调用验证函数)
df_step4 = apply_event_shocks_step4_envelope(
    df_synthetic_ar, 
    df_official, 
    regression_results, 
    feature_list, 
    model_selection=selection_rules,
    output_dir=os.path.join(RESULT_PATH, 'g_step4_event')
)

### 质量检验

In [ ]:
run_quality_validation(
    4,
    df_prophet_resid, 
    df_step4, 
    version_dict_complete,
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g4_impact_validation'))

## step5 叠加周期性成分

### prepare version

In [ ]:
# df_version
version_dict_complete
# # version_dict

### main

In [ ]:
import pandas as pd
import numpy as np
from prophet.serialize import model_from_json
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. 辅助配置 ====================

# 用来解析 version key 到标准类型的逻辑
def parse_version_type(version_name: str):
    """
    将 'v1.2_大上预' 解析为 '大版本上半预热'
    """
    s = str(version_name)
    size = '大版本' if '大' in s else '小版本'
    half = '上半' if '上' in s else '下半'
    phase = '预热' if '预' in s else '更新'
    return f'{size}{half}{phase}'

# ==================== 2. 核心逻辑：构建基准趋势 ====================

def learn_baselines_from_history(df_history, version_dict_history, feature_name):
    """
    从历史数据中学习每种版本类型（如'大版本上半更新'）的平均值
    """
    # 1. 准备容器
    type_values = {}  # { '大版本上半更新': [1.2, 1.3, ...], ... }
    
    # 2. 遍历历史版本字典
    for v_name, (start, end) in version_dict_history.items():
        std_type = parse_version_type(v_name)
        
        # 截取该时间段的历史数据
        mask = (df_history.index >= start) & (df_history.index < end)
        period_data = df_history.loc[mask, feature_name].dropna()
        
        if len(period_data) > 0:
            if std_type not in type_values:
                type_values[std_type] = []
            type_values[std_type].extend(period_data.values)
            
    # 3. 计算均值
    baselines = {}
    global_mean = df_history[feature_name].mean()
    
    # 定义所有可能的类型
    all_types = [
        f'{s}{h}{p}' 
        for s in ['大版本', '小版本'] 
        for h in ['上半', '下半'] 
        for p in ['预热', '更新']
    ]
    
    for t in all_types:
        if t in type_values and len(type_values[t]) > 0:
            baselines[t] = np.mean(type_values[t])
        else:
            # 如果历史没出现过这个类型（比如历史只有小版本，未来有大版本）
            # 这里需要定义简单的推演规则，例如：大版本 = 全局均值 * 1.2 (如果是原始域)
            # 但因为是在【变换域】(Log/BoxCox)，加减法即可
            # 策略：如果没有数据，暂且回退到全局均值，或者根据规则微调
            if '大版本' in t:
                baselines[t] = global_mean + 0.1  # 假设大版本比均值高一点
            else:
                baselines[t] = global_mean
                
    print(f"   📊 学习到的版本基准 ({feature_name}):")
    for k, v in baselines.items():
        print(f"      - {k}: {v:.4f}")
        
    return baselines, global_mean

def construct_schedule_trend(future_index, version_dict_complete, baselines, global_mean,smooth_window=None):
    """
    根据未来排期表，构建阶梯式趋势线
    """
    trend_series = pd.Series(index=future_index, data=global_mean) # 默认填全局均值
    
    for v_name, (start, end) in version_dict_complete.items():
        std_type = parse_version_type(v_name)
        # 获取该类型的基准值
        val = baselines.get(std_type, global_mean)
        
        # 填入时间段
        mask = (trend_series.index >= start) & (trend_series.index < end)
        trend_series.loc[mask] = val
        
    # 平滑处理：消除版本切换时的生硬台阶
    # 使用 Gaussian 或 Rolling Mean 平滑接缝，窗口设为 12小时 (48点)
    # trend_smooth = trend_series.rolling(window=48, min_periods=1, center=True, win_type='gaussian').mean(std=10)
    # trend_smooth = trend_smooth.fillna(trend_series) # 补全边缘
    if smooth_window and smooth_window > 0:
        trend_smooth = trend_series.rolling(
            window=smooth_window, min_periods=1, center=True, win_type='gaussian'
        ).mean(std=smooth_window/3)
        return trend_smooth.fillna(trend_series)
    else:
        return trend_series

def apply_seasonality_from_profile(future_index, profile_path):
    """
    [修改原因]：Step 5 核心替换逻辑。
    不加载模型，而是读取 JSON，根据 future_index 的时间特征（几点几分、星期几）
    直接查找对应的 seasonality 数值。
    """
    if not os.path.exists(profile_path):
        print(f"      ⚠️ 警告：未找到季节性模板 {profile_path}，将使用0填充")
        return np.zeros(len(future_index))
    
    with open(profile_path, 'r', encoding='utf-8') as f:
        profile = json.load(f)
    
    # 1. 映射日内季节性 (Daily)
    # JSON key 是 "HH:MM", value 是 float
    # 使用 map 极速查找，比 predict 快几十倍
    time_keys = future_index.strftime('%H:%M')
    daily_map = profile['daily']
    daily_component = pd.Series(time_keys).map(daily_map).fillna(0).values
    
    # 2. 映射周季节性 (Weekly)
    # JSON key 是 string "0", "1"... 但 map 需要匹配
    # 注意：json load 进来 key 可能是字符串，需要确保类型匹配
    weekly_map = {int(k): v for k, v in profile['weekly'].items()}
    dow_keys = future_index.dayofweek
    weekly_component = pd.Series(dow_keys).map(weekly_map).fillna(0).values
    
    return daily_component + weekly_component
# ==================== 3. 主流程 Step 5 ====================

def run_step5_schedule_synthesis(
    df_step4_input,          # 包含噪声和冲击的残差
    df_history,              # 原始历史数据（变换后）
    version_dict_history,    # 历史数据的版本字典
    version_dict_future,     # 你的 version_dict_complete
    prophet_model_dir,
    feature_list,
    output_dir,
    prophet_results,   # [新增参数] 传入 Step 3 返回的 prophet_results 字典
    r2_threshold=0.1   # [新增参数] R² 阈值，默认 0.1
):
    print("=" * 70)
    print("📌 Step 5: 基于排期表的趋势重构 (Schedule-Based Synthesis)")
    print("=" * 70)
    
    os.makedirs(output_dir, exist_ok=True)
    
    df_final = pd.DataFrame(index=df_step4_input.index)
    
    for feature in feature_list:
        print(f"\n   Processing {feature}...")
        
        # --- A. 学习基准 ---
        # 确保 feature 在历史数据中存在
        hist_col = f'{feature}_transformed' if f'{feature}_transformed' in df_history.columns else feature
        baselines, global_mean = learn_baselines_from_history(
            df_history, version_dict_history, hist_col
        )
        
        # --- B. 构建趋势 (Trend) ---
        # 根据 version_dict_complete 铺设地基
        trend_component = construct_schedule_trend(
            df_step4_input.index, version_dict_future, baselines, global_mean,smooth_window=0
        )
        
        # --- C. 获取季节性 (Seasonality) ---
        # 使用 Prophet 模型，只预测 seasonality，令 trend=0
        safe_name = feature.replace('/', '_')
        if feature in prophet_results:
            r2 = prophet_results[feature]['r_squared']
        else:
            r2 = 0.0
            print(f"      ⚠️ 未找到 R² 数据，默认为 0")
        if r2 < r2_threshold:
            print(f"      📉 R² ({r2:.4f}) < {r2_threshold}，忽略季节性 (Seasonality=0)")
            seasonality_component = 0.0
        else:
            profile_path = os.path.join(prophet_model_dir, f'profile_{safe_name}.json')
            seasonality_component = apply_seasonality_from_profile(df_step4_input.index, profile_path)
        # model_path = os.path.join(prophet_model_dir, f'prophet_model_{safe_name}.json')
        
        # seasonality_component = 0.0
        # if os.path.exists(model_path):
        #     try:
        #         m = model_from_json(open(model_path, 'r').read())
        #         future_df = pd.DataFrame({'ds': df_step4_input.index})
        #         # 将所有额外回归量置0
        #         if m.extra_regressors:
        #             for reg in m.extra_regressors:
        #                 future_df[reg] = 0.0
                        
        #         forecast = m.predict(future_df)
        #         # Prophet 的 additive_terms 包含 weekly + daily
        #         seasonality_component = forecast['additive_terms'].values
        #     except Exception as e:
        #         print(f"      ⚠️ Prophet 模型加载失败，跳过季节性: {e}")
        
        # --- D. 最终合成 ---
        # Final = 排期趋势 + 周期性 + (残差+冲击)
        resid_component = df_step4_input[feature].values
        
        final_series = trend_component.values + seasonality_component + resid_component
        df_final[feature] = final_series
        
        print(f"      趋势均值: {trend_component.mean():.4f} | 周期幅度: {np.ptp(seasonality_component):.4f}")

    # 保存
    save_path = f'{output_dir}/synthetic_transformed_schedule_based.csv'
    df_final.to_csv(save_path, encoding='utf-8-sig')
    print("\n" + "="*70)
    print(f"✅ 合成完成！结果已保存至: {save_path}")
    print("现在可以进行 Step 6 (逆变换) 了。")
    print("="*70)
    
    return df_final

### runv2

In [ ]:
prophet_results['total_volume_post']

In [ ]:
df_step5 = run_step5_schedule_synthesis(
    df_step4_input=df_step4,
    df_history=df_transformed.set_index('timestamp'),
    version_dict_history=version_dict, # 必须提供历史字典供学习
    version_dict_future=version_dict_complete, # 必须提供未来字典供构建
    prophet_model_dir=os.path.join(RESULT_PATH, 'step3_prophet'),
    feature_list=feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g_step5_trend'),
    prophet_results=prophet_results, # 传入 Prophet 的结果字典
    r2_threshold=0.1 # 可以调整这个阈值来控制季节)
)

### 质量检验

### run validation

In [ ]:
run_quality_validation(
    5,
    df_transformed.set_index('timestamp'), 
    df_step5, 
    version_dict_complete,
    feature_list,
    output_dir=os.path.join(RESULT_PATH, 'g5_prophet_validation'))



### 质检2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

def debug_step2_5_components(
    df_history_transformed,   # 历史Step2结果 (原始变换数据)
    df_synth_final,           # 合成Step2.5结果
    df_step4_input,           # 合成Step2.5的输入残差 (Step4结果)
    prophet_decomp_results,   # 历史Step3分解结果 (字典)
    feature_name,
    output_dir
):
    os.makedirs(output_dir, exist_ok=True)
    print(f"🔍 正在诊断特征: {feature_name} ...")

    # 1. 获取历史组件
    # 注意：历史数据比较短 (2个月)，合成数据长 (11个月)
    # 我们主要对比“分布形态” (Histogram/KDE) 而非时间轴对齐
    
    if feature_name not in prophet_decomp_results:
        print(f"   ⚠️ 历史分解结果中找不到 {feature_name}")
        return

    hist_decomp = prophet_decomp_results[feature_name]
    # 历史组件序列
    h_trend = hist_decomp['trend_day']  # Prophet trend
    h_seas = hist_decomp['seasonal_day'] + hist_decomp['seasonal_week'] # Total Seasonality
    h_resid = hist_decomp['resid_final'] # Prophet resid
    h_total = df_history_transformed[f'{feature_name}_transformed'] if f'{feature_name}_transformed' in df_history_transformed else df_history_transformed[feature_name]

    # 2. 获取合成组件
    # 这里的难点是 Step2.5 并没有保存中间变量。
    # 我们需要通过简单的减法反推，或者你修改 Step2.5 返回中间变量。
    # 这里假设：
    # Synth_Total = df_synth_final
    # Synth_Resid = df_step4_input (这是Step2.5的输入)
    # Synth_Trend_Seas = Synth_Total - Synth_Resid (这是合成出来的确定性部分)
    
    s_total = df_synth_final[feature_name]
    s_resid = df_step4_input[feature_name]
    s_deterministic = s_total - s_resid # 这里包含了 Trend + Seasonality
    
    # 3. 绘图诊断：分布对比 (KDE Plot)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'特征: {feature_name} - 历史(Step1.3) vs 合成(Step2.5) 组件分布诊断', fontsize=16)

    # --- A. 总数据对比 ---
    sns.kdeplot(h_total, ax=axes[0,0], fill=True, color='black', label='历史总数据')
    sns.kdeplot(s_total, ax=axes[0,0], fill=True, color='blue', label='合成总数据')
    axes[0,0].set_title('A. 总数据分布 (Total)')
    axes[0,0].legend()

    # --- B. 残差对比 (核心嫌疑人) ---
    sns.kdeplot(h_resid, ax=axes[0,1], fill=True, color='gray', linestyle='--', label='历史残差 (Prophet Resid)')
    sns.kdeplot(s_resid, ax=axes[0,1], fill=True, color='green', label='合成残差 (Step4 Input)')
    axes[0,1].set_title('B. 残差分布 (Residuals)\n如果不重合，说明Step2/4有问题')
    axes[0,1].legend()

    # --- C. 确定性部分 (Trend+Seas) 对比 ---
    # 由于历史和未来排期不同，Trend均值不同是正常的。
    # 我们关注的是“形状宽度” (Seas幅度) 和 “多峰性” (Trend台阶)
    sns.kdeplot(h_trend + h_seas, ax=axes[0,2], fill=True, color='orange', alpha=0.3, label='历史(Trend+Seas)')
    sns.kdeplot(s_deterministic, ax=axes[0,2], fill=True, color='red', alpha=0.3, label='合成(Trend+Seas)')
    axes[0,2].set_title('C. 确定性部分 (Trend + Seasonality)\n看宽度是否一致(季节性强度)')
    axes[0,2].legend()

    # --- D, E, F: 时间序列表现 (取前1000个点对比) ---
    # 历史
    axes[1,0].plot(h_total.values, color='black', alpha=0.8, linewidth=1)
    axes[1,0].set_title('历史时序 (全量)')
    
    # 合成
    axes[1,1].plot(s_total.values, color='blue', alpha=0.8, linewidth=0.5)
    axes[1,1].set_title('合成时序 (全量)')
    
    # 残差特写
    axes[1,2].plot(h_resid.values[:500], color='black', label='History')
    axes[1,2].plot(s_resid.values[:500], color='green', alpha=0.7, label='Synth')
    axes[1,2].set_title('残差细节 (前500点)')
    axes[1,2].legend()

    plt.tight_layout()
    save_path = f'{output_dir}/Diagnosis_{feature_name}.png'
    plt.savefig(save_path)
    print(f"   📊 诊断图已保存: {save_path}")
    plt.close()

# 使用示例 (伪代码):
# debug_step2_5_components(
#    df_history_transformed, 
#    df_synth_final,     # Step 2.5 的输出
#    df_step4_output,    # Step 2.5 的输入 (残差)
#    prophet_decomp_results, # Step 1.3 的输出
#    'vis_abs_redundancy_post'
# )

In [ ]:
for i in feature_list:
    print(f"正在诊断特征: {i}")
    debug_step2_5_components(
        df_transformed, 
        df_step5,     # Step 2.5 的输出
        df_step4,    # Step 2.5 的输入 (残差)
        decomp_results, # Step 1.3 的输出
        i,
        output_dir=os.path.join(RESULT_PATH, 'g5_debug')
    )

## step 3.1 特殊变量处理

#### main

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import yeojohnson, beta as beta_dist
from typing import Dict, Tuple


def _slot_index_15min(idx: pd.DatetimeIndex) -> np.ndarray:
    """将时间索引映射到日内 15 分钟槽位 [0..95]"""
    return idx.hour * 4 + idx.minute // 15


def estimate_daily_baseline(
    df_real_raw: pd.DataFrame,
    feature: str,
    time_index: pd.DatetimeIndex,
    min_baseline: float = 1e-6
) -> np.ndarray:
    """
    按日内 15min 槽位估计 baseline（原始空间）。
    """
    s = df_real_raw[feature].reindex(time_index).fillna(0).astype(float)
    slot = _slot_index_15min(time_index)

    df_tmp = pd.DataFrame({'slot': slot, 'y': s.values})
    slot_mean = df_tmp.groupby('slot')['y'].mean()
    slot_mean = slot_mean.reindex(np.arange(96)).fillna(0.0)

    baseline = slot_mean.values[slot]
    baseline = np.clip(baseline, min_baseline, None)
    return baseline


def build_event_boost_profile(
    time_index: pd.DatetimeIndex,
    feature: str,
    events_df: pd.DataFrame | None = None,
    impact_results: Dict | None = None,
    window_after: int = 32,         # 32*15min=8小时
    half_life_hours: float = 1.5,   # 半衰期
    max_boost: float = 3.0          # λ_t 最大倍数 (1+boost<=max_boost)
) -> np.ndarray:
    """
    基于事件和 Step4 impact_results，为单个特征生成事件增强曲线 boost_t（相对提升比例）。
    λ_t = baseline_t * (1 + boost_t)
    """
    n = len(time_index)
    boost = np.zeros(n, dtype=float)
    if events_df is None or len(events_df) == 0:
        return boost

    # 从 impact_results 估一个“典型峰值增幅”（百分比）
    peak_ratio = 1.0  # 默认 +100%
    if impact_results and feature in impact_results:
        by_combo = impact_results[feature].get('by_combo', {})
        if by_combo:
            peaks = [d.get('overall_peak', 0.0) for d in by_combo.values()]
            if peaks:
                peak_ratio = np.percentile(np.abs(peaks), 75) / 100.0
                peak_ratio = float(np.clip(peak_ratio, 0.2, 3.0))

    df_ev = events_df.copy()
    df_ev['timestamp'] = pd.to_datetime(df_ev['timestamp'])
    df_ev = df_ev.sort_values('timestamp')

    tau = max(half_life_hours * 4, 1.0)  # 以 15min 为单位的半衰期

    for _, ev in df_ev.iterrows():
        t0 = ev['timestamp']
        pos = time_index.searchsorted(t0)
        if pos >= n:
            continue
        for k in range(window_after):
            t = pos + k
            if t >= n:
                break
            boost[t] += peak_ratio * np.exp(-k / tau)

    boost = np.clip(boost, 0.0, max_boost - 1.0)
    return boost

In [ ]:
def generate_count_poisson_yj(
    df_real_raw: pd.DataFrame,
    time_index: pd.DatetimeIndex,
    feature: str,
    transform_info: Dict,
    events_df: pd.DataFrame | None = None,
    impact_results: Dict | None = None,
    max_boost: float = 3.0,
    seed: int = 2025
) -> Tuple[pd.Series, Dict]:
    """
    为稀疏计数特征生成合成序列（Poisson + 事件驱动）。
    原始空间生成，再做 yeo_johnson 变换。

    自动按真实均值做一次 λ_t 的尺度校准。
    """
    rng = np.random.default_rng(seed)

    # 1) baseline λ_t
    baseline = estimate_daily_baseline(df_real_raw, feature, time_index, min_baseline=1e-4)

    # 2) 事件增强 boost_t
    boost = build_event_boost_profile(
        time_index=time_index,
        feature=feature,
        events_df=events_df,
        impact_results=impact_results,
        window_after=32,
        half_life_hours=1.5,
        max_boost=max_boost
    )
    lam0 = baseline * (1.0 + boost)

    # 3) 用真实均值校准 λ 尺度
    real_mean = float(df_real_raw[feature].dropna().mean() or 1e-4)
    syn_mean0 = float(lam0.mean() or 1e-4)
    lam_scale = real_mean / syn_mean0
    lam_scale = float(np.clip(lam_scale, 0.5, 3.0))  # 不让过分放大/缩小
    lam_t = lam0 * lam_scale
    lam_t = np.clip(lam_t, 1e-4, None)

    # 4) Poisson 采样
    counts = rng.poisson(lam_t)

    # 限制最大值不过分：以真实 99 分位为上限
    real_vals = df_real_raw[feature].dropna().values
    if len(real_vals) > 0:
        q99 = np.percentile(real_vals, 99)
        max_cap = max(q99 * 1.5, q99 + 3)
        counts = np.clip(counts, 0, max_cap)
    counts = counts.astype(float)

    # 5) Yeo-Johnson 变换
    ti = transform_info.get(feature, {})
    lam_yj = ti.get('lambda', None)
    if lam_yj is None:
        yj, lam_est = yeojohnson(counts + 1e-6)
        lam_yj = lam_est
    else:
        yj = yeojohnson(counts + 1e-6, lam_yj)

    synthetic_yj = pd.Series(yj, index=time_index, name=feature)

    debug = {
        'baseline_lambda': baseline,
        'boost': boost,
        'lam0': lam0,
        'lam_scale': lam_scale,
        'lam_t': lam_t,
        'counts': counts,
        'yj_lambda': lam_yj
    }
    return synthetic_yj, debug

In [ ]:
def generate_ratio_beta_yj(
    df_real_raw: pd.DataFrame,
    time_index: pd.DatetimeIndex,
    feature: str,
    transform_info: Dict,
    events_df: pd.DataFrame | None = None,
    impact_results: Dict | None = None,
    eps: float = 1e-4,
    max_boost: float = 1.8,   # 比例不宜太大提升
    k_mult: float = 2.0,      # 提高浓度，减小方差
    seed: int = 3035
) -> Tuple[pd.Series, Dict]:
    """
    为 [0,1] 比例特征生成合成序列（Beta + 事件）。
    原始空间生成，再做 yeo_johnson 变换。
    """
    rng = np.random.default_rng(seed)
    s_real = df_real_raw[feature].astype(float).clip(eps, 1 - eps)

    # 全局 Beta 拟合
    m = float(s_real.mean())
    v = float(s_real.var(ddof=1) or 1e-6)
    k = m * (1 - m) / v - 1.0
    k = float(np.clip(k, 4.0, 80.0)) * k_mult  # 提高浓度，减少波动

    # 日内 baseline
    slot = _slot_index_15min(df_real_raw.index)
    df_tmp = pd.DataFrame({'slot': slot, 'y': s_real.values})
    slot_mean = df_tmp.groupby('slot')['y'].mean()
    slot_mean = slot_mean.reindex(np.arange(96)).fillna(m)
    mu_t = slot_mean.values[_slot_index_15min(time_index)]

    # 事件增强
    boost = build_event_boost_profile(
        time_index=time_index,
        feature=feature,
        events_df=events_df,
        impact_results=impact_results,
        window_after=32,
        half_life_hours=1.5,
        max_boost=max_boost
    )
    mu_ev0 = mu_t * (1.0 + boost)
    mu_ev0 = np.clip(mu_ev0, eps, 1 - eps)

    # 用真实均值校准 mu_ev
    syn_mean0 = float(mu_ev0.mean())
    real_mean = float(m)
    mu_scale = real_mean / max(syn_mean0, 1e-6)
    mu_scale = float(np.clip(mu_scale, 0.5, 1.5))
    mu_ev = np.clip(mu_ev0 * mu_scale, eps, 1 - eps)

    alpha_t = mu_ev * k
    beta_t = (1 - mu_ev) * k

    vals = np.empty(len(time_index), dtype=float)
    for i in range(len(time_index)):
        a_i = max(alpha_t[i], 1e-3)
        b_i = max(beta_t[i], 1e-3)
        vals[i] = beta_dist.rvs(a_i, b_i, random_state=rng)

    # YJ 变换
    ti = transform_info.get(feature, {})
    lam_yj = ti.get('lambda', None)
    if lam_yj is None:
        yj, lam_est = yeojohnson(vals)
        lam_yj = lam_est
    else:
        yj = yeojohnson(vals, lam_yj)

    synthetic_yj = pd.Series(yj, index=time_index, name=feature)

    debug = {
        'mu_t': mu_t,
        'boost': boost,
        'mu_ev0': mu_ev0,
        'mu_scale': mu_scale,
        'mu_ev': mu_ev,
        'k': k,
        'alpha_t': alpha_t,
        'beta_t': beta_t,
        'vals': vals,
        'yj_lambda': lam_yj
    }
    return synthetic_yj, debug

In [ ]:
def generate_continuous_gaussian_yj(
    df_real_raw: pd.DataFrame,
    time_index: pd.DatetimeIndex,
    feature: str,
    transform_info: Dict,
    events_df: pd.DataFrame | None = None,
    impact_results: Dict | None = None,
    noise_scale: float = 0.4,
    pulse_std_mult: float = 0.8,
    seed: int = 4045
) -> Tuple[pd.Series, Dict]:
    """
    为连续型特征生成合成序列（baseline + 事件脉冲 + 高斯噪声）。
    原始空间生成，再做 yeo_johnson 变换。
    """
    rng = np.random.default_rng(seed)
    s_real = df_real_raw[feature].reindex(time_index).astype(float).fillna(0.0)

    # baseline μ_t
    slot = _slot_index_15min(time_index)
    df_tmp = pd.DataFrame({'slot': slot, 'y': s_real.values})
    slot_mean = df_tmp.groupby('slot')['y'].mean()
    slot_mean = slot_mean.reindex(np.arange(96)).fillna(s_real.mean())
    mu_t = slot_mean.values[slot]

    # 事件脉冲
    boost = build_event_boost_profile(
        time_index=time_index,
        feature=feature,
        events_df=events_df,
        impact_results=impact_results,
        window_after=32,
        half_life_hours=1.5,
        max_boost=3.0
    )
    real_std = float(df_real_raw[feature].std() or 1.0)
    event_pulse = boost * (pulse_std_mult * real_std)

    # 噪声
    eps = rng.standard_normal(len(time_index)) * (noise_scale * real_std)

    raw0 = mu_t + event_pulse + eps

    # 用真实 std 校准一次幅度
    cur_std = float(raw0.std(ddof=1) or 1e-6)
    scale_std = real_std / cur_std
    scale_std = float(np.clip(scale_std, 0.5, 2.0))
    raw = mu_t + (raw0 - mu_t) * scale_std

    # YJ 变换
    ti = transform_info.get(feature, {})
    lam_yj = ti.get('lambda', None)
    if lam_yj is None:
        yj, lam_est = yeojohnson(raw)
        lam_yj = lam_est
    else:
        yj = yeojohnson(raw, lam_yj)

    synthetic_yj = pd.Series(yj, index=time_index, name=feature)

    debug = {
        'mu_t': mu_t,
        'boost': boost,
        'event_pulse': event_pulse,
        'eps': eps,
        'raw0': raw0,
        'raw': raw,
        'scale_std': scale_std,
        'yj_lambda': lam_yj
    }
    return synthetic_yj, debug

In [ ]:
def apply_special_generators_for_sparse_features(
    df_synthetic_final: pd.DataFrame,
    df_real_raw: pd.DataFrame,
    transform_info: Dict,
    events_df: pd.DataFrame,
    impact_results: Dict,
    seed: int = 2025
):
    time_index = df_synthetic_final.index
    debug_all = {}

    # 1) 计数特征
    count_features = ['total_short_post', 'total_long_post','total_short_comment',  'total_long_comment','senti_symbol_comment']
    for j, feat in enumerate(count_features):
        if feat not in df_real_raw.columns:
            continue
        syn, dbg = generate_count_poisson_yj(
            df_real_raw=df_real_raw,
            time_index=time_index,
            feature=feat,
            transform_info=transform_info,
            events_df=events_df,
            impact_results=impact_results,
            max_boost=3.0,
            seed=seed + j * 11
        )
        df_synthetic_final[feat] = syn
        debug_all[feat] = dbg
        print(f"[SPECIAL-COUNT] 覆盖特征: {feat}")

    # 2) 比例特征
    ratio_features = ['retweet_ratio_post']
    for j, feat in enumerate(ratio_features):
        if feat not in df_real_raw.columns:
            continue
        syn, dbg = generate_ratio_beta_yj(
            df_real_raw=df_real_raw,
            time_index=time_index,
            feature=feat,
            transform_info=transform_info,
            events_df=events_df,
            impact_results=impact_results,
            eps=1e-4,
            max_boost=1.8,    # 注意这里降低 max_boost
            k_mult=2.0,
            seed=seed + 100 + j * 11
        )
        df_synthetic_final[feat] = syn
        debug_all[feat] = dbg
        print(f"[SPECIAL-RATIO] 覆盖特征: {feat}")

    # 3) 连续强度特征
    cont_features = ['vis_abs_redundancy_post','semantic_shift_comment']  # 如需也可加 semantic_shift_comment
    for j, feat in enumerate(cont_features):
        if feat not in df_real_raw.columns:
            continue
        syn, dbg = generate_continuous_gaussian_yj(
            df_real_raw=df_real_raw,
            time_index=time_index,
            feature=feat,
            transform_info=transform_info,
            events_df=events_df,
            impact_results=impact_results,
            noise_scale=0.4,
            pulse_std_mult=0.8,
            seed=seed + 200 + j * 11
        )
        df_synthetic_final[feat] = syn
        debug_all[feat] = dbg
        print(f"[SPECIAL-CONT] 覆盖特征: {feat}")

    return df_synthetic_final, debug_all

#### run

In [ ]:
# df_final_trans, debug_special = apply_special_generators_for_sparse_features(
#     df_synthetic_final=df_final_trans,
#     df_real_raw=df_features.set_index('timestamp'),   # 原始 3 个月真实数据（含 11 个月时间段时，缺口用 0 或向前填充）
#     transform_info=transform_info,
#     events_df=df_official,
#     impact_results=impact_results,       # Step4 结果；若没有，可传 {}
#     seed=2025
# )

## step6 逆变换与零值注入

### 准备正常数据统计量

In [ ]:
def calculate_clean_stats(df_real, normal_mask, feature_list):
    """
    计算剔除异常后的真实数据统计量（作为校准目标）
    """
    clean_stats = {}
    print(f"📊 正在计算清洗后的分布目标 (正常样本占比: {normal_mask.mean():.2%})...")

    for feat in feature_list:
        if feat not in df_real.columns:
            continue
            
        # 只取正常时间段的数据
        raw_data = df_real[feat].values
        clean_data = df_real[normal_mask][feat].dropna().values
        
        if len(clean_data) == 0:
            # 如果剔除完了（极其罕见），就回退到原始数据
            clean_data = raw_data
        
        # 计算关键统计量
        stats = {
            'mean': float(np.mean(clean_data)),
            'std': float(np.std(clean_data)),
            'min': float(np.min(clean_data)),
            'max': float(np.max(clean_data)), # 这里的 max 就是“正常情况下的最大值”
            'zero_ratio': float((clean_data == 0).mean()),
            # [核心] 计算 101 个百分位点
            'percentiles': np.percentile(clean_data, np.linspace(0, 100, 101)).tolist()
        }
        clean_stats[feat] = stats
        
    return clean_stats

In [ ]:
# garch_results['total_volume_post']['ci_df']

In [ ]:
df_feature_slice.set_index('timestamp')[normal_mask]

In [ ]:
clean_stats = calculate_clean_stats(df_feature_slice.set_index('timestamp'), normal_mask, feature_list)

### v1

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from scipy import stats
# import json
# import os
# import warnings
# warnings.filterwarnings('ignore')

# plt.rcParams['font.sans-serif'] = ['SimHei']
# plt.rcParams['axes.unicode_minus'] = False


# # ============================================================
# # 1) 逆变换函数库（修正版）
# #    - [MOD-YJ] 修正 Yeo-Johnson 逆变换：分段应按 y>=0 / y<0，而不是按 lambda 正负
# # ============================================================

# def inverse_yeo_johnson_correct(y, lmbda):
#     """
#     正确的 Yeo-Johnson 逆变换（向量化）
#     分段由 y 的正负决定：
#       y>=0  <-> x>=0
#       y<0   <-> x<0
#     """
#     y = np.asarray(y, dtype=float)
#     x = np.empty_like(y, dtype=float)
#     eps = 1e-12

#     pos = y >= 0
#     neg = ~pos

#     # x >= 0 branch
#     if abs(lmbda) < 1e-10:
#         # y = log1p(x) -> x = expm1(y)
#         x[pos] = np.expm1(y[pos])
#     else:
#         base = lmbda * y[pos] + 1.0
#         base = np.maximum(base, eps)
#         x[pos] = np.power(base, 1.0 / lmbda) - 1.0

#     # x < 0 branch
#     if abs(lmbda - 2.0) < 1e-10:
#         # y = -log1p(-x) -> -y = log1p(-x) -> -x = expm1(-y)
#         x[neg] = -np.expm1(-y[neg])
#     else:
#         # y = -(( (1-x)^(2-l) - 1)/(2-l))  -> (1-x)^(2-l) = 1 - (2-l)*y
#         base = 1.0 - (2.0 - lmbda) * y[neg]   # y[neg] < 0 => base > 1 generally
#         base = np.maximum(base, eps)
#         x[neg] = 1.0 - np.power(base, 1.0 / (2.0 - lmbda))

#     return x


# def inverse_log1p(y):
#     """y = log1p(x) -> x = expm1(y)"""
#     y = np.asarray(y, dtype=float)
#     return np.expm1(y)


# def inverse_sqrt(y):
#     """y = sqrt(x) -> x = y^2（sqrt 一般用于非负）"""
#     y = np.asarray(y, dtype=float)
#     return np.square(y)


# def inverse_standardize(y, original_mean, original_std):
#     """y = (x-mean)/std -> x = y*std + mean"""
#     y = np.asarray(y, dtype=float)
#     return y * float(original_std) + float(original_mean)


# def inverse_boxcox(y, lambda_param, shift=0.0):
#     """
#     Box-Cox 逆变换（允许 shift）
#       y = ( (x+shift)^λ - 1)/λ , λ!=0
#       y = log(x+shift)         , λ=0
#     -> x = inv(...) - shift
#     """
#     y = np.asarray(y, dtype=float)
#     lmbda = float(lambda_param)
#     shift = float(shift)
#     eps = 1e-12

#     if abs(lmbda) < 1e-10:
#         x_shifted = np.exp(y)
#     else:
#         base = lmbda * y + 1.0
#         base = np.maximum(base, eps)
#         x_shifted = np.power(base, 1.0 / lmbda)

#     return x_shifted - shift

# # ============================================================
# # 核心新增：分布校准函数
# # ============================================================

# def calibrate_distribution(synth_data, hist_mean, hist_std, hist_max, feature_name):
#     """
#     强制校准合成数据的分布，使其统计特性与历史数据对齐。
#     用于解决逆变换后的数值漂移问题。
#     """
#     s_mean = np.mean(synth_data)
#     s_std = np.std(synth_data)
    
#     # 如果合成数据全是常数或NaN，无法校准
#     if s_std < 1e-9 or np.isnan(s_std):
#         return synth_data

#     # 1. 计算偏差
#     mean_bias = abs(s_mean - hist_mean) / (abs(hist_mean) + 1e-6)
#     std_bias = abs(s_std - hist_std) / (hist_std + 1e-6)
    
#     # 2. 判定是否需要校准
#     # 阈值：如果均值或方差偏差超过 50%，或者最大值极其离谱，则触发校准
#     # 对于 "BAD" 的特征，这个阈值通常都会被触发
#     # needs_calibration = (mean_bias > 0.5) or (std_bias > 0.5) or (np.max(synth_data) > 10 * hist_max)
#     needs_calibration = (mean_bias > 0.2) or (std_bias > 0.2) or (np.max(synth_data) > 10 * hist_max)

    
#     if needs_calibration:
#         print(f"  🔧 校准触发 {feature_name:<20}: Mean {s_mean:.2f}->{hist_mean:.2f} | Std {s_std:.2f}->{hist_std:.2f}")
        
#         # 3. Z-Score 归一化重构 (Moment Matching)
#         # S_new = (S - mu_s) / sigma_s * sigma_h + mu_h
#         # 这会保留波形形状，但强行将统计量拉回历史水平
        
#         # 考虑到 Step 5 可能添加了趋势（Trend），我们可能希望保留一点 Mean 的变化
#         # 但如果是严重的漂移，完全保留 Trend 也是错误的
#         # 折中方案：校准到 (Historical Mean * 1.1) 允许微涨，或者完全对齐
#         # 这里选择完全对齐 Mean 和 Std，这是最安全的“洗白”方式
        
#         calibrated = (synth_data - s_mean) / s_std * hist_std + hist_mean
        
#         # 4. 再次确保非负
#         calibrated = np.maximum(calibrated, 0.0)
        
#         return calibrated
#     else:
#         return synth_data


# # ============================================================
# # 2) 零值注入策略（修正版）
# #    - [MOD-Z] 阈值注入改为“精确选前 k 个最小值”，避免分位数重复值导致超注入
# #    - [MOD-Z2] 支持“只补足零值”（不尝试减少已经存在的零）
# # ============================================================

# def inject_zeros_threshold_exact_only_add(data, target_zero_ratio):
#     """
#     只“增加零值”以达到目标比例（不会减少现有零）
#     - 先计算当前零比例
#     - 若不足，则把最小的若干非零值置 0
#     """
#     data = np.asarray(data, dtype=float)
#     n = data.size
#     if n == 0:
#         return data.copy(), np.zeros(0, dtype=bool)

#     target_zero_ratio = float(target_zero_ratio)
#     target_zero_ratio = min(max(target_zero_ratio, 0.0), 1.0)

#     current_mask = (data == 0)
#     current_zeros = int(current_mask.sum())
#     target_zeros = int(round(n * target_zero_ratio))

#     if target_zeros <= current_zeros:
#         # 已经达到或超过目标：不减少零（保持非负约束/物理约束结果）
#         return data.copy(), current_mask

#     need = target_zeros - current_zeros

#     # 在非零位置里选最小的 need 个
#     idx_nonzero = np.where(~current_mask)[0]
#     if idx_nonzero.size == 0:
#         return data.copy(), current_mask

#     vals = data[idx_nonzero]
#     order = np.argsort(vals)  # 小到大
#     chosen = idx_nonzero[order[:need]]

#     out = data.copy()
#     mask = current_mask.copy()
#     out[chosen] = 0.0
#     mask[chosen] = True
#     return out, mask


# # ============================================================
# # 3) 物理约束（按你的新要求定制）
# #    - [MOD-NONNEG] 所有特征最终必须非负
# #    - [MOD-CONPR] conpr* 为压缩比：默认值 1，且建议下界为 1（更符合“压缩比”定义）--已调整为1-压缩比，取值范围[0,1]，越小表示压缩比越大
# # ============================================================

# def apply_physical_constraints_nonneg(data, feature_name, feature_type, feature_limits=None):
#     """
#     feature_type:
#       - 'count'            : 非负、四舍五入为整数
#       - 'ratio_01'         : 截断到 [0,1]
#     #   - 'compression_ratio': conpr* 压缩比，默认 1，下界 1（可选上界）--并入ratio
#       - 'continuous'       : 连续非负（>=0），可选上界
#     """
#     data = np.asarray(data, dtype=float)
#     out = data.copy()
#     info = {
#         'n_nan_inf_filled': 0,
#         'n_negative_clipped': 0,
#         'n_lower_clipped': 0,
#         'n_upper_clipped': 0
#     }

#     # conpr 特殊：默认填充值为 1，其它默认 0
#     # default_fill = 1.0 if str(feature_name).startswith('conpr') else 0.0
#     default_fill = 0.0


#     # 先处理 NaN/Inf（用默认值）
#     bad = ~np.isfinite(out)
#     if bad.any():
#         out[bad] = default_fill
#         info['n_nan_inf_filled'] = int(bad.sum())

#     # 全局非负要求：先把 <0 的裁到 0（或后面按类型裁到 1）
#     neg = out < 0
#     if neg.any():
#         out[neg] = 0.0
#         info['n_negative_clipped'] = int(neg.sum())

#     # 类型约束
#     feature_limits = feature_limits or {}

#     if feature_type == 'count':
#         # 非负整数
#         out = np.round(out)
#         # 可选上界
#         maxv = feature_limits.get('max_value', None)
#         if maxv is not None:
#             maxv = float(maxv)
#             upper = out > maxv
#             if upper.any():
#                 out[upper] = maxv
#                 info['n_upper_clipped'] += int(upper.sum())

#     elif feature_type == 'ratio_01':
#         lower = out < 0
#         upper = out > 1
#         if lower.any():
#             out[lower] = 0.0
#             info['n_lower_clipped'] += int(lower.sum())
#         if upper.any():
#             out[upper] = 1.0
#             info['n_upper_clipped'] += int(upper.sum())

#     # elif feature_type == 'compression_ratio':
#     #     # [MOD-CONPR] 默认 1，且建议压缩比下界为 1（不允许小于 1）
#     #     lower_bound = float(feature_limits.get('min_value', 1.0))
#     #     lower = out < lower_bound
#     #     if lower.any():
#     #         out[lower] = lower_bound
#     #         info['n_lower_clipped'] += int(lower.sum())

#         maxv = feature_limits.get('max_value', None)
#         if maxv is not None:
#             maxv = float(maxv)
#             upper = out > maxv
#             if upper.any():
#                 out[upper] = maxv
#                 info['n_upper_clipped'] += int(upper.sum())

#     else:  # 'continuous'
#         # 连续非负（>=0）
#         lower_bound = float(feature_limits.get('min_value', 0.0))
#         lower = out < lower_bound
#         if lower.any():
#             out[lower] = lower_bound
#             info['n_lower_clipped'] += int(lower.sum())

#         maxv = feature_limits.get('max_value', None)
#         if maxv is not None:
#             maxv = float(maxv)
#             upper = out > maxv
#             if upper.any():
#                 out[upper] = maxv
#                 info['n_upper_clipped'] += int(upper.sum())

#     # 再次确保非负
#     out = np.maximum(out, 0.0)

#     # conpr：如果还出现 0（例如全为 NaN），强制成 1
#     if str(feature_name).startswith('conpr'):
#         out = np.where(out <= 0, 1.0, out)

#     return out, info


# # ============================================================
# # 4) 主流程：逆变换 -> 物理约束 -> 零注入（只补足） -> 再约束
# #    - [MOD-ORDER] 零注入放在约束之后，避免 rounding/clip 改变零比例
# #    - [MOD-CONPR-ZERO] conpr* 不注入零（默认 1）
# # ============================================================

# def inverse_transform_and_inject_zeros_nonneg(
#     df_synthetic_final,       # Step3 输出（变换域）
#     feature_list,
#     step1_results,            # 需含 zero_ratio（目标零比例）、可含 mean/std 等
#     transform_info,           # 你的 transform_info（含 transform, lambda, original_mean/std 等）
#     feature_config=None,      # {'count':[...], 'ratio_01':[...], 'continuous':[...]}；conpr 自动识别
#     zero_injection_method='threshold_exact',  # 这里只实现最稳的 exact-only-add
#     output_dir=None,
#     safety_margin=1.5  # [新增] 安全系数，允许合成数据是历史最大值的 1.5 倍
# ):
#     os.makedirs(output_dir, exist_ok=True)

#     # 默认类型映射（你可覆盖）
#     if feature_config is None:
#         feature_config = {
#             'count': [],
#             'ratio_01': [],
#             'continuous': []
#         }

#     # 构建 feature -> type 映射
#     feature_type_map = {}
#     for t, feats in feature_config.items():
#         for f in feats:
#             feature_type_map[str(f)] = t

#     # conpr* 强制指定为 compression_ratio
#     # for f in feature_list:
#     #     if str(f).startswith('conpr'):
#     #         feature_type_map[str(f)] = 'compression_ratio'

#     print("═" * 70)
#     print("📌 Step4：逆变换 + 零值注入 + 物理约束（全非负 + conpr默认1）")
#     print("═" * 70)

#     time_index = df_synthetic_final.index
#     n = len(time_index)

#     df_physical = pd.DataFrame(index=time_index)
#     report = {
#         'inverse_stats': [],
#         'zero_stats': [],
#         'constraint_stats': []
#     }

#     for i, feature in enumerate(feature_list):
#         if feature not in df_synthetic_final.columns:
#             print(f"  ⚠️ {feature} 不在 df_synthetic_final，跳过")
#             continue
#         # 1. 获取变换参数
#         trans = transform_info.get(feature, {}) if transform_info else {}
#         method = trans.get('transform', 'none')
#         y = df_synthetic_final[feature].to_numpy(dtype=float)

#         # 2. 计算安全上界 (Safety Upper Bound)
#         # 从 step1_results 获取历史最大值
#         hist_stats = step1_results.get(feature, {})
#         hist_max = hist_stats.get('max', 1e6) 
#         if pd.isna(hist_max): hist_max = 1e6


#         # 历史统计量
#         hist_mean = float(hist_stats.get('mean', 0.0))
#         hist_std = float(hist_stats.get('std', 1.0))
#         hist_max = float(hist_stats.get('max', 1.0))
#         hist_zero_ratio = float(hist_stats.get('zero_ratio', 0.0))
        
#         # 设定硬顶：历史最大值 * 系数 (防止 10^18 爆炸)
#         # 对于比率类(ratio)，最大值不应超过 1.0太多(如果是0-1)，或者由历史决定
#         # 给一个保底值 10.0，防止全0数据的倍数依然是0
#         safe_max_val = max(float(hist_max) * safety_margin, 10.0)
        
#         # 特殊处理 ratio_01 类型，物理上限是 1.0
#         ftype = feature_type_map.get(feature, 'continuous')
#         if ftype == 'ratio_01':
#             safe_max_val = 1.0
                

#         # 4. 执行逆变换
#         # ---------- (A) 输入安全裁剪（只对 log1p/boxcox 做 log 类裁剪） ----------
#         # [MOD] 不把 yeo_johnson 当 log 类截断
#         if method == 'log1p':
#             y = np.clip(y, -15.0, 15.0)
#         elif method == 'boxcox':
#             # boxcox 若 lambda≈0 等价 log
#             lmbda = float(trans.get('lambda', 1.0))
#             if abs(lmbda) < 1e-10:
#                 y = np.clip(y, -15.0, 15.0)
#             # 其它 lambda 不强行 log 截断（但 inverse 里会确保 base>0）
#         elif method == 'standardize':
#             y = np.clip(y, -50.0, 50.0)

#         # ---------- (B) 逆变换 ----------
#         if method == 'yeo_johnson':
#             lmbda = trans.get('lambda', 0.0)
#             lmbda = 0.0 if lmbda is None else float(lmbda)
#             x = inverse_yeo_johnson_correct(y, lmbda)

#         elif method == 'log1p':
#             x = inverse_log1p(y)

#         elif method == 'sqrt':
#             x = inverse_sqrt(y)

#         elif method == 'standardize':
#             orig_mean = trans.get('original_mean', step1_results.get(feature, {}).get('mean', 0.0))
#             orig_std = trans.get('original_std', step1_results.get(feature, {}).get('std', 1.0))
#             orig_mean = 0.0 if orig_mean is None else float(orig_mean)
#             orig_std = 1.0 if (orig_std is None or float(orig_std) == 0.0) else float(orig_std)
#             x = inverse_standardize(y, orig_mean, orig_std)

#         elif method == 'boxcox':
#             lmbda = float(trans.get('lambda', 1.0))
#             shift = float(trans.get('shift', 0.0))
#             x = inverse_boxcox(y, lmbda, shift)

#         else:
#             x = y.copy()
#         df_physical[feature] = x
#     return df_physical, report
        

#     def function_tmp():    
#         # 5. 后截断 (Post-clipping): 再次确保不爆炸 20260212new
#         # 如果逆变换后的值超过了安全上界，强行拉回
#         mask_explode = x > safe_max_val
#         if mask_explode.any():
#             # 记录一下被截断的数量
#             n_clip = mask_explode.sum()
#             # 仅在大量截断或极值过大时打印，避免刷屏
#             if x[mask_explode].max() > safe_max_val * 2:
#                 print(f"  ⚠️ {feature:<25}: 触发防爆截断! Max {x.max():.2e} -> {safe_max_val:.2f} ({n_clip} pts)")
#             x[mask_explode] = safe_max_val

#         # ---------- (C) 第一次物理约束（保证全非负） ----------
#         ftype = feature_type_map.get(feature, 'continuous')
#         # 你可以在这里为个别特征加上 max_value 等限制
#         if ftype != 'ratio_01': 
#             x = calibrate_distribution(x, hist_mean, hist_std, hist_max, feature)
#         limits = {}
#         x_constrained, c_info = apply_physical_constraints_nonneg(
#             x, feature_name=feature, feature_type=ftype, feature_limits=limits
#         )

#         # ---------- (D) 零值注入（只补足；conpr 不注入零） ----------
#         # target_zero = float(step1_results.get(feature, {}).get('zero_ratio', 0.0))
#         target_zero = hist_zero_ratio
#         target_zero = min(max(target_zero, 0.0), 1.0)

#         # if str(feature).startswith('conpr'):
#         #     # [MOD-CONPR-ZERO] conpr 默认 1，不注入零
#         #     injected = x_constrained
#         #     zmask = (injected == 0)
#         #     actual_zero = float(zmask.mean())
#         #     note = "skip (conpr default=1)"
#         # else:
#         # 只在目标零比例>0 时尝试补足
#         if target_zero > 0 and zero_injection_method.startswith('threshold'):
#             injected, zmask = inject_zeros_threshold_exact_only_add(x_constrained, target_zero)
#         else:
#             injected, zmask = x_constrained.copy(), (x_constrained == 0)

#         actual_zero = float(zmask.mean())
#         note = "only-add exact"
        
#         # ---------- (E) 再次物理约束（避免注入后边界问题；count 需要保持整数） ----------
#         limits['max_value'] = max(hist_max * safety_margin, 10.0)  # 注入零后仍可能有极大值，确保后续约束能处理
#         injected2, c_info2 = apply_physical_constraints_nonneg(
#             injected, feature_name=feature, feature_type=ftype, feature_limits=limits
#         )

#         # df_physical[feature] = injected2

#         # ---------- 记录统计 ----------
#         yv = y[np.isfinite(y)]
#         xv = x_constrained[np.isfinite(x_constrained)]
#         pv = injected2[np.isfinite(injected2)]

#         report['inverse_stats'].append({
#             'feature': feature,
#             'transform': method,
#             'y_mean': float(np.mean(yv)) if yv.size else np.nan,
#             'y_std': float(np.std(yv)) if yv.size else np.nan,
#             'x_mean_after_inverse_before_zero': float(np.mean(xv)) if xv.size else np.nan,
#             'x_std_after_inverse_before_zero': float(np.std(xv)) if xv.size else np.nan
#         })
#         report['zero_stats'].append({
#             'feature': feature,
#             'target_zero_ratio': float(target_zero),
#             'actual_zero_ratio': float((pv == 0).mean()) if pv.size else np.nan,
#             'note': note
#         })
#         report['constraint_stats'].append({
#             'feature': feature,
#             'type': ftype,
#             **c_info,
#             # 第二次约束也记录一下（可选）
#             'n_nan_inf_filled_after_zero': int(c_info2.get('n_nan_inf_filled', 0)),
#             'n_negative_clipped_after_zero': int(c_info2.get('n_negative_clipped', 0)),
#             'n_lower_clipped_after_zero': int(c_info2.get('n_lower_clipped', 0)),
#             'n_upper_clipped_after_zero': int(c_info2.get('n_upper_clipped', 0)),
#         })

#         if (i + 1) % 5 == 0 or (i + 1) == len(feature_list):
#             print(f"  ✅ 已处理 {i+1}/{len(feature_list)}")

#     # 保存
#     df_physical.to_csv(f'{output_dir}/synthetic_physical.csv', encoding='utf-8-sig')
#     with open(f'{output_dir}/inverse_report.json', 'w', encoding='utf-8') as f:
#         json.dump(report, f, ensure_ascii=False, indent=2)

#     print("\n" + "═" * 70)
#     print("✅ Step6 完成：已输出 synthetic_physical.csv（全非负；conpr>=1）")
#     print("═" * 70)

#     return df_physical, report


# # ============================================================
# # 5) 运行封装
# # ============================================================

# def run_step6_inverse_transform(
#     df_synthetic_final,
#     feature_list,
#     step1_results,
#     transform_info,
#     feature_config=None,
#     output_dir=None
# ):
#     """
#     说明：
#     - df_synthetic_final 必须是“变换域”的合成结果（来自你修正版 Step3）
#     - transform_info 里每个 feature 至少应有 transform 字段；yeo_johnson/boxcox 要有 lambda
#     - step1_results 里每个 feature 推荐有 zero_ratio（原始数据的零比例）
#     - 所有最终数据强制非负
#     - conpr* 强制为1-压缩比：默认 0
#     """
#     if feature_config is None:
#         # 你可以把自己的 list 放进来；conpr 会自动覆盖为 compression_ratio
#         feature_config = {
#             'count': [
#                 'total_volume_post', 'total_volume_comment',
#                 'total_short_post', 'total_long_post',
#                 'total_short_comment', 'total_long_comment',
#                 'vis_abs_redundancy_post'
#             ],
#             'ratio_01': [
#                 'gini_post', 'gini_comment','retweet_ratio_post',
#                 'origin_ratio_post', 'neg_ratio_post', 'neg_ratio_comment',
#                 'vis_concentration_post', 'comp_ratio_post', 'comp_ratio_comment'
#             ],
#             'continuous': [
#                 'semantic_shift_post', 'semantic_shift_comment',
#                 'senti_symbol_post', 'senti_symbol_comment'
#             ]
#         }

#     return inverse_transform_and_inject_zeros_nonneg(
#         df_synthetic_final=df_synthetic_final,
#         feature_list=feature_list,
#         step1_results=step1_results,
#         transform_info=transform_info,
#         feature_config=feature_config,
#         zero_injection_method='threshold_exact',
#         output_dir=output_dir
#     )



### v2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import json
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ============================================================
# 1) 逆变换函数库（修正版）
#    - [MOD-YJ] 修正 Yeo-Johnson 逆变换：分段应按 y>=0 / y<0，而不是按 lambda 正负
# ============================================================

def inverse_yeo_johnson_correct(y, lmbda):
    """
    正确的 Yeo-Johnson 逆变换（向量化）
    分段由 y 的正负决定：
      y>=0  <-> x>=0
      y<0   <-> x<0
    """
    y = np.asarray(y, dtype=float)
    x = np.empty_like(y, dtype=float)
    eps = 1e-12

    pos = y >= 0
    neg = ~pos

    # x >= 0 branch
    if abs(lmbda) < 1e-10:
        # y = log1p(x) -> x = expm1(y)
        x[pos] = np.expm1(y[pos])
    else:
        base = lmbda * y[pos] + 1.0
        base = np.maximum(base, eps)
        x[pos] = np.power(base, 1.0 / lmbda) - 1.0

    # x < 0 branch
    if abs(lmbda - 2.0) < 1e-10:
        # y = -log1p(-x) -> -y = log1p(-x) -> -x = expm1(-y)
        x[neg] = -np.expm1(-y[neg])
    else:
        # y = -(( (1-x)^(2-l) - 1)/(2-l))  -> (1-x)^(2-l) = 1 - (2-l)*y
        base = 1.0 - (2.0 - lmbda) * y[neg]   # y[neg] < 0 => base > 1 generally
        base = np.maximum(base, eps)
        x[neg] = 1.0 - np.power(base, 1.0 / (2.0 - lmbda))

    return x


def inverse_log1p(y):
    """y = log1p(x) -> x = expm1(y)"""
    y = np.asarray(y, dtype=float)
    return np.expm1(y)


def inverse_sqrt(y):
    """y = sqrt(x) -> x = y^2（sqrt 一般用于非负）"""
    y = np.asarray(y, dtype=float)
    return np.square(y)


def inverse_standardize(y, original_mean, original_std):
    """y = (x-mean)/std -> x = y*std + mean"""
    y = np.asarray(y, dtype=float)
    return y * float(original_std) + float(original_mean)


def inverse_boxcox(y, lambda_param, shift=0.0):
    """
    Box-Cox 逆变换（允许 shift）
      y = ( (x+shift)^λ - 1)/λ , λ!=0
      y = log(x+shift)         , λ=0
    -> x = inv(...) - shift
    """
    y = np.asarray(y, dtype=float)
    lmbda = float(lambda_param)
    shift = float(shift)
    eps = 1e-12

    if abs(lmbda) < 1e-10:
        x_shifted = np.exp(y)
    else:
        base = lmbda * y + 1.0
        base = np.maximum(base, eps)
        x_shifted = np.power(base, 1.0 / lmbda)

    return x_shifted - shift
# ============================================================
# 强制均值校准
# ============================================================

def force_mean_correction(data, target_mean, tolerance=0.05):
    """
    最后一道防线：如果均值偏差过大，通过缩放强制拉回。
    """
    current_mean = np.mean(data)
    if current_mean == 0 or target_mean == 0:
        return data
        
    bias = (current_mean - target_mean) / target_mean
    
    # 只有偏差超过阈值（如 5%）才修正，避免微调导致过拟合
    if abs(bias) > tolerance:
        scale = target_mean / current_mean
        # print(f"    ⚖️ 均值修正触发: {current_mean:.4f} -> {target_mean:.4f} (x{scale:.4f})")
        return data * scale
    return data


# ============================================================
#  分位数校准函数 (核心新增)
# ============================================================
def quantile_calibrate(feature,synth_data, hist_stats, robust=True):
    """
    鲁棒分位数校准：
    防止历史数据中的极端离群值（Outliers）破坏合成数据分布。
    """
    s_data = np.asarray(synth_data, dtype=float)
    n = len(s_data)
    if n == 0:
        return s_data

    # 1. 获取/构造历史分布骨架
    if 'percentiles' in hist_stats:
        # Step 1 计算好的 101 个点 (0% ~ 100%)
        hist_p_values = np.array(hist_stats['percentiles'])
        p_points = np.linspace(0, 100, len(hist_p_values))
    else:
        # 简易兜底
        hist_mean = hist_stats.get('mean', 0)
        hist_min = hist_stats.get('min', 0)
        hist_max = hist_stats.get('max', hist_mean * 2)
        p_points = np.array([0, 50, 100])
        hist_p_values = np.array([hist_min, hist_stats.get('median', hist_mean), hist_max])

    # =======================================================
    # [核心修改] 鲁棒处理：削弱极值的影响
    # =======================================================
    if robust and len(hist_p_values) >= 100:
        # 策略：如果最大值 (P100) 远大于 P99，说明可能是异常值
        # 这里的 "远大于" 定义为：(P100 - P99) > 3 * (P99 - P95)
        # 我们用 P99 加上一个合理的增量来替代真实的 P100
        
        p95 = hist_p_values[95]
        p99 = hist_p_values[99]
        p100 = hist_p_values[100] # 也就是 max
        
        # 计算尾部斜率
        tail_gap = p99 - p95
        if tail_gap == 0: tail_gap = 1e-6
        
        # 检查 P100 是否离谱
        # 如果 P100 比 P99 大出 5 倍的 (P99-P95) 区间，我们认为它是脏数据或极端离群点
        threshold_val = p99 + 5.0 * tail_gap
        
        if p100 > threshold_val:
            print(f"  🛡️ 触发鲁棒截断{feature} :Max值 {p100:.2f} 被限制为 {threshold_val:.2f} (基于P99推断)")
            # 修改映射表中的最大值，防止合成数据被拉得太远
            hist_p_values[-1] = threshold_val 

    # 2. 计算合成数据的百分位秩
    ranks = stats.rankdata(s_data, method='average')
    s_percentiles = (ranks - 1) / (n - 1) * 100

    # 3. 线性插值映射
    calibrated = np.interp(s_percentiles, p_points, hist_p_values)
    
    return calibrated

# ============================================================
# 核心新增：分布校准函数
# ============================================================

def calibrate_distribution(synth_data, hist_mean, hist_std, hist_max, feature_name):
    """
    强制校准合成数据的分布，使其统计特性与历史数据对齐。
    用于解决逆变换后的数值漂移问题。
    """
    s_mean = np.mean(synth_data)
    s_std = np.std(synth_data)
    
    # 如果合成数据全是常数或NaN，无法校准
    if s_std < 1e-9 or np.isnan(s_std):
        return synth_data

    # 1. 计算偏差
    mean_bias = abs(s_mean - hist_mean) / (abs(hist_mean) + 1e-6)
    std_bias = abs(s_std - hist_std) / (hist_std + 1e-6)
    
    # 2. 判定是否需要校准
    # 阈值：如果均值或方差偏差超过 50%，或者最大值极其离谱，则触发校准
    # 对于 "BAD" 的特征，这个阈值通常都会被触发
    # needs_calibration = (mean_bias > 0.5) or (std_bias > 0.5) or (np.max(synth_data) > 10 * hist_max)
    needs_calibration = (mean_bias > 0.2) or (std_bias > 0.2) or (np.max(synth_data) > 10 * hist_max)

    
    if needs_calibration:
        print(f"  🔧 校准触发 {feature_name:<20}: Mean {s_mean:.2f}->{hist_mean:.2f} | Std {s_std:.2f}->{hist_std:.2f}")
        
        # 3. Z-Score 归一化重构 (Moment Matching)
        # S_new = (S - mu_s) / sigma_s * sigma_h + mu_h
        # 这会保留波形形状，但强行将统计量拉回历史水平
        
        # 考虑到 Step 5 可能添加了趋势（Trend），我们可能希望保留一点 Mean 的变化
        # 但如果是严重的漂移，完全保留 Trend 也是错误的
        # 折中方案：校准到 (Historical Mean * 1.1) 允许微涨，或者完全对齐
        # 这里选择完全对齐 Mean 和 Std，这是最安全的“洗白”方式
        
        calibrated = (synth_data - s_mean) / s_std * hist_std + hist_mean
        
        # 4. 再次确保非负
        calibrated = np.maximum(calibrated, 0.0)
        
        return calibrated
    else:
        return synth_data


# ============================================================
# 2) 零值注入策略（修正版）
#    - [MOD-Z] 阈值注入改为“精确选前 k 个最小值”，避免分位数重复值导致超注入
#    - [MOD-Z2] 支持“只补足零值”（不尝试减少已经存在的零）
# ============================================================

def inject_zeros_threshold_exact_only_add(data, target_zero_ratio):
    """
    只“增加零值”以达到目标比例（不会减少现有零）
    - 先计算当前零比例
    - 若不足，则把最小的若干非零值置 0
    """
    data = np.asarray(data, dtype=float)
    n = data.size
    if n == 0:
        return data.copy(), np.zeros(0, dtype=bool)

    target_zero_ratio = float(target_zero_ratio)
    target_zero_ratio = min(max(target_zero_ratio, 0.0), 1.0)

    current_mask = (data == 0)
    current_zeros = int(current_mask.sum())
    target_zeros = int(round(n * target_zero_ratio))

    if target_zeros <= current_zeros:
        # 已经达到或超过目标：不减少零（保持非负约束/物理约束结果）
        return data.copy(), current_mask

    need = target_zeros - current_zeros

    # 在非零位置里选最小的 need 个
    idx_nonzero = np.where(~current_mask)[0]
    if idx_nonzero.size == 0:
        return data.copy(), current_mask

    vals = data[idx_nonzero]
    order = np.argsort(vals)  # 小到大
    chosen = idx_nonzero[order[:need]]

    out = data.copy()
    mask = current_mask.copy()
    out[chosen] = 0.0
    mask[chosen] = True
    return out, mask
def inject_zeros_latent_threshold(data, target_zero_ratio):
    """
    潜变量阈值法 (Latent Thresholding) 注入零值：
    借用数据的相对大小（分布）来决定零值位置，保留时间序列的自相关性。
    
    逻辑：
    - 找出数据的第 P 分位数 (Threshold)，其中 P = target_zero_ratio。
    - 所有小于等于 Threshold 的值置为 0。
    - 这等价于：将最小的 N * target_ratio 个数置为 0。
    """
    data = np.asarray(data, dtype=float)
    n = data.size
    if n == 0:
        return data.copy(), np.zeros(0, dtype=bool)

    target_zero_ratio = float(target_zero_ratio)
    target_zero_ratio = min(max(target_zero_ratio, 0.0), 1.0)
    
    # 目标零值数量
    target_zeros = int(round(n * target_zero_ratio))
    
    # 当前零值数量 (经过分位数校准后，可能已经有一些 0 或 接近 0 的值)
    current_mask = (data <= 1e-9) # 视为 0
    current_zeros = int(current_mask.sum())

    # 如果目标是 0，直接返回 (可能做一些清理)
    if target_zeros == 0:
        return data.copy(), np.zeros(n, dtype=bool)

    # 潜变量排序：获取所有数据的索引，按值从小到大排序
    # 注意：我们对整个序列排序，而不是只针对非零值，这样能保证全局的分布一致性
    sorted_indices = np.argsort(data)
    
    # 确定需要置 0 的索引：最小的前 target_zeros 个
    # 即使 data 中已经有 0，它们也会排在最前面，被包含在 indices_to_zero 中
    # 这样确保了这一步是 "Enforce" (强制) 零值比例，无论是增加还是修剪
    indices_to_zero = sorted_indices[:target_zeros]
    
    out = data.copy()
    
    # 执行注入
    out[indices_to_zero] = 0.0
    
    # 生成掩码
    final_mask = np.zeros(n, dtype=bool)
    final_mask[indices_to_zero] = True
    
    return out, final_mask

# ============================================================
# 3) 物理约束（按你的新要求定制）
#    - [MOD-NONNEG] 所有特征最终必须非负
#    - [MOD-CONPR] conpr* 为压缩比：默认值 1，且建议下界为 1（更符合“压缩比”定义）--已调整为1-压缩比，取值范围[0,1]，越小表示压缩比越大
# ============================================================

def apply_physical_constraints_nonneg(data, feature_name, feature_type, feature_limits=None):
    """
    feature_type:
      - 'count'            : 非负、四舍五入为整数
      - 'ratio_01'         : 截断到 [0,1]
    #   - 'compression_ratio': conpr* 压缩比，默认 1，下界 1（可选上界）--并入ratio
      - 'continuous'       : 连续非负（>=0），可选上界
    """
    data = np.asarray(data, dtype=float)
    out = data.copy()
    info = {
        'n_nan_inf_filled': 0,
        'n_negative_clipped': 0,
        'n_lower_clipped': 0,
        'n_upper_clipped': 0
    }

    # conpr 特殊：默认填充值为 1，其它默认 0
    # default_fill = 1.0 if str(feature_name).startswith('conpr') else 0.0
    default_fill = 0.0


    # 先处理 NaN/Inf（用默认值）
    bad = ~np.isfinite(out)
    if bad.any():
        out[bad] = default_fill
        info['n_nan_inf_filled'] = int(bad.sum())

    # 全局非负要求：先把 <0 的裁到 0（或后面按类型裁到 1）
    neg = out < 0
    if neg.any():
        out[neg] = 0.0
        info['n_negative_clipped'] = int(neg.sum())

    # 类型约束
    feature_limits = feature_limits or {}

    if feature_type == 'count':
        # 非负整数
        out = np.round(out)
        # 可选上界
        maxv = feature_limits.get('max_value', None)
        if maxv is not None:
            maxv = float(maxv)
            upper = out > maxv
            if upper.any():
                out[upper] = maxv
                info['n_upper_clipped'] += int(upper.sum())

    elif feature_type == 'ratio_01':
        lower = out < 0
        upper = out > 1
        if lower.any():
            out[lower] = 0.0
            info['n_lower_clipped'] += int(lower.sum())
        if upper.any():
            out[upper] = 1.0
            info['n_upper_clipped'] += int(upper.sum())

    # elif feature_type == 'compression_ratio':
    #     # [MOD-CONPR] 默认 1，且建议压缩比下界为 1（不允许小于 1）
    #     lower_bound = float(feature_limits.get('min_value', 1.0))
    #     lower = out < lower_bound
    #     if lower.any():
    #         out[lower] = lower_bound
    #         info['n_lower_clipped'] += int(lower.sum())

        maxv = feature_limits.get('max_value', None)
        if maxv is not None:
            maxv = float(maxv)
            upper = out > maxv
            if upper.any():
                out[upper] = maxv
                info['n_upper_clipped'] += int(upper.sum())

    else:  # 'continuous'
        # 连续非负（>=0）
        lower_bound = float(feature_limits.get('min_value', 0.0))
        lower = out < lower_bound
        if lower.any():
            out[lower] = lower_bound
            info['n_lower_clipped'] += int(lower.sum())

        maxv = feature_limits.get('max_value', None)
        if maxv is not None:
            maxv = float(maxv)
            upper = out > maxv
            if upper.any():
                out[upper] = maxv
                info['n_upper_clipped'] += int(upper.sum())

    # 再次确保非负
    out = np.maximum(out, 0.0)

    # conpr：如果还出现 0（例如全为 NaN），强制成 1
    if str(feature_name).startswith('conpr'):
        out = np.where(out <= 0, 1.0, out)

    return out, info


# ============================================================
# 4) 主流程：逆变换 -> 物理约束 -> 零注入（只补足） -> 再约束
#    - [MOD-ORDER] 零注入放在约束之后，避免 rounding/clip 改变零比例
#    - [MOD-CONPR-ZERO] conpr* 不注入零（默认 1）
# ============================================================

def inverse_transform_and_inject_zeros_nonneg(
    df_synthetic_final,       # Step5 输出（变换域）
    feature_list,
    step1_results,            # 需含 zero_ratio（目标零比例）、可含 mean/std 等
    clean_stats,              # 用于分位数校准
    transform_info,           # 你的 transform_info（含 transform, lambda, original_mean/std 等）
    feature_config=None,      # {'count':[...], 'ratio_01':[...], 'continuous':[...]}；conpr 自动识别
    zero_injection_method=None,  # 这里只实现最稳的 exact-only-add
    # zero_injection_method='latent',  # 这里只实现最稳的 exact-only-add

    output_dir=None,
    safety_margin=1.5  # [新增] 安全系数，允许合成数据是历史最大值的 1.5 倍
):
    os.makedirs(output_dir, exist_ok=True)

    # 默认类型映射（你可覆盖）
    if feature_config is None:
        feature_config = {
            'count': [],
            'ratio_01': [],
            'continuous': []
        }

    # 构建 feature -> type 映射
    feature_type_map = {}
    for t, feats in feature_config.items():
        for f in feats:
            feature_type_map[str(f)] = t

    # conpr* 强制指定为 compression_ratio
    # for f in feature_list:
    #     if str(f).startswith('conpr'):
    #         feature_type_map[str(f)] = 'compression_ratio'

    print("═" * 70)
    print("📌 Step4：逆变换 + 零值注入 + 物理约束（全非负）")
    print("═" * 70)

    time_index = df_synthetic_final.index
    n = len(time_index)

    df_physical = pd.DataFrame(index=time_index)
    report = {
        'inverse_stats': [],
        'zero_stats': [],
        'constraint_stats': []
    }

    print(f"开始逆变换与分位数校准，共 {len(feature_list)} 个特征...")

    for i, feature in enumerate(feature_list):
        if feature not in df_synthetic_final.columns:
            print(f"  ⚠️ {feature} 不在 df_synthetic_final，跳过")
            continue
        # 1. 获取变换参数
        trans = transform_info.get(feature, {}) if transform_info else {}
        method = trans.get('transform', 'none')
        y = df_synthetic_final[feature].to_numpy(dtype=float)

        # 2. 计算安全上界 (Safety Upper Bound)
        # 从 step1_results 获取历史最大值
        hist_stats = step1_results.get(feature, {})
        hist_max = hist_stats.get('max', 1e6) 
        if pd.isna(hist_max): hist_max = 1e6


        # 历史统计量
        hist_mean = float(hist_stats.get('mean', 0.0))
        hist_std = float(hist_stats.get('std', 1.0))
        hist_max = float(hist_stats.get('max', 1.0))
        hist_zero_ratio = float(hist_stats.get('zero_ratio', 0.0))
        
        # 设定硬顶：历史最大值 * 系数 (防止 10^18 爆炸)
        # 对于比率类(ratio)，最大值不应超过 1.0太多(如果是0-1)，或者由历史决定
        # 给一个保底值 10.0，防止全0数据的倍数依然是0
        safe_max_val = float(hist_max) * safety_margin
        
        # 特殊处理 ratio_01 类型，物理上限是 1.0
        ftype = feature_type_map.get(feature, 'continuous')
        if ftype == 'ratio_01':
            safe_max_val = 1.0
                

        # 4. 执行逆变换
        # ---------- (A) 输入安全裁剪（只对 log1p/boxcox 做 log 类裁剪） ----------
        # [MOD] 不把 yeo_johnson 当 log 类截断
        if method == 'log1p':
            y = np.clip(y, -15.0, 15.0)
        elif method == 'boxcox':
            # boxcox 若 lambda≈0 等价 log
            lmbda = float(trans.get('lambda', 1.0))
            if abs(lmbda) < 1e-10:
                y = np.clip(y, -15.0, 15.0)
            # 其它 lambda 不强行 log 截断（但 inverse 里会确保 base>0）
        elif method == 'standardize':
            y = np.clip(y, -50.0, 50.0)

        # ---------- (B) 逆变换 ----------
        if method == 'yeo_johnson':
            lmbda = trans.get('lambda', 0.0)
            lmbda = 0.0 if lmbda is None else float(lmbda)
            x = inverse_yeo_johnson_correct(y, lmbda)

        elif method == 'log1p':
            x = inverse_log1p(y)

        elif method == 'sqrt':
            x = inverse_sqrt(y)

        elif method == 'standardize':
            orig_mean = trans.get('original_mean', step1_results.get(feature, {}).get('mean', 0.0))
            orig_std = trans.get('original_std', step1_results.get(feature, {}).get('std', 1.0))
            orig_mean = 0.0 if orig_mean is None else float(orig_mean)
            orig_std = 1.0 if (orig_std is None or float(orig_std) == 0.0) else float(orig_std)
            x = inverse_standardize(y, orig_mean, orig_std)

        elif method == 'boxcox':
            lmbda = float(trans.get('lambda', 1.0))
            shift = float(trans.get('shift', 0.0))
            x = inverse_boxcox(y, lmbda, shift)

        else:
            x = y.copy()
        print(f"  🔄 逆变换 {feature:<20}: Method={method}, Safety Max={safe_max_val:.2f}")
    #     df_physical[feature] = x
    # return df_physical, report
    # def function_tmp():    
        
        
        # --- (add) 分位数校准 (Quantile Calibration) ---
        target_stats = clean_stats.get(feature, step1_results.get(feature, {}))
        is_bad_feature = feature in ['total_long_post', 'retweet_ratio_post', 'total_short_comment']
        use_robust = not is_bad_feature
        x = quantile_calibrate(feature, x, target_stats, robust=use_robust)

        # if ftype != 'ratio_01': 
        #     x = calibrate_distribution(x, hist_mean, hist_std, hist_max, feature)

        # 后截断 (Post-clipping): 再次确保不爆炸 20260212new
        # 如果逆变换后的值超过了安全上界，强行拉回
        mask_explode = x > safe_max_val
        if mask_explode.any():
            # 记录一下被截断的数量
            n_clip = mask_explode.sum()
            # 仅在大量截断或极值过大时打印，避免刷屏
            if x[mask_explode].max() > safe_max_val * 2:
                print(f"  ⚠️ {feature:<25}: 触发防爆截断! Max {x.max():.2e} -> {safe_max_val:.2f} ({n_clip} pts)")
            x[mask_explode] = safe_max_val

        

        # ---------- (C) 第一次物理约束（保证全非负） ----------
        ftype = feature_type_map.get(feature, 'continuous')
        # 为个别特征加上 max_value 等限制
        limits = {}
        if feature in ['total_short_comment', 'total_long_post' ,'retweet_ratio_post']:    
            limits['max_value'] = target_stats['percentiles'][99]
        elif ftype == 'ratio_01' or 'ratio' in feature:
            limits['max_value'] =  min(hist_max,1.0)
            # Ratio 类型绝对不能超过 1.0 (或 clean_stats 里的最大值)
            # 如果 clean_stats['max'] < 1.0 (比如 0.8)，就截断到 0.8，防止尾部过冲
        else:
            limits['max_value'] = hist_max * safety_margin
           # 注入零后仍可能有极大值，确保后续约束能处理
        x, c_info = apply_physical_constraints_nonneg(
            x, feature_name=feature, feature_type=ftype, feature_limits=limits
        )
        

        # ---------- (D) 零值注入（只补足；conpr 不注入零） ----------
        target_zero = float(step1_results.get(feature, {}).get('zero_ratio', 0.0))
        # target_zero = hist_zero_ratio
        target_zero = min(max(target_zero, 0.0), 1.0)

        # 只在目标零比例>0 时尝试补足
        if target_zero > 0 and zero_injection_method.startswith('threshold'):
            injected, zmask = inject_zeros_threshold_exact_only_add(x, target_zero)
            note = f"threshold_extract (target={target_zero:.1%})"


        else:
            injected, zmask = inject_zeros_latent_threshold(x, target_zero)
            note = f"latent_threshold (target={target_zero:.1%})"

        actual_zero = float(zmask.mean())


        if ftype != 'ratio_01':
        # 第一轮修正
            injected = force_mean_correction(injected, hist_mean, tolerance=0.01)
        
        # # 如果是 Count 类型，取整后均值会变，需要迭代检查
        # if ftype == 'count':
        #     # 预先取整看效果
        #     injected_rounded = np.round(injected)
        #     current_mean = np.mean(injected_rounded)
            
        #     # 如果取整后均值依然偏差 > 2%，进行第二轮反向补偿
        #     if hist_mean > 1e-3 and abs(current_mean - hist_mean) / hist_mean > 0.02:
        #         # 计算补偿系数。例如：目标 2.0，取整后变成 2.2。说明我要把未取整的数据再压低一点
        #         # scale = 2.0 / 2.2 = 0.909
        #         scale = hist_mean / (current_mean + 1e-9)
        #         injected = injected * scale # 对未取整的数据应用补偿
        #         # print(f"  🔄 迭代均值修正 {feature}: Scale {scale:.4f}")

        
        # ---------- (E) 再次物理约束（避免注入后边界问题；count 需要保持整数） ----------
        limits['max_value'] = hist_max * safety_margin  # 注入零后仍可能有极大值，确保后续约束能处理
        injected2, c_info2 = apply_physical_constraints_nonneg(
            injected, feature_name=feature, feature_type=ftype, feature_limits=limits
        )
        print(f"  🧾 再次物理约束{feature:<20}: {np.max(injected):.2f}->{np.max(injected2):.2f}")

        if ftype == 'count':
            injected2 = np.round(injected2)
        
        # 最后的防线：确保非负
        injected2 = np.maximum(injected2, 0.0)
        if ftype == 'ratio_01': injected2 = np.minimum(injected2, 1.0)

        df_physical[feature] = injected2

        # ---------- 记录统计 ----------
        yv = y[np.isfinite(y)]
        xv = x[np.isfinite(x)]
        pv = injected2[np.isfinite(injected2)]

        report['inverse_stats'].append({
            'feature': feature,
            'transform': method,
            'y_mean': float(np.mean(yv)) if yv.size else np.nan,
            'y_std': float(np.std(yv)) if yv.size else np.nan,
            'x_mean_after_inverse_before_zero': float(np.mean(xv)) if xv.size else np.nan,
            'x_std_after_inverse_before_zero': float(np.std(xv)) if xv.size else np.nan
        })
        report['zero_stats'].append({
            'feature': feature,
            'target_zero_ratio': float(target_zero),
            'actual_zero_ratio': float((pv == 0).mean()) if pv.size else np.nan,
            'note': note
        })
        report['constraint_stats'].append({
            'feature': feature,
            'type': ftype,
            **c_info,
            # 第二次约束也记录一下（可选）
            'n_nan_inf_filled_after_zero': int(c_info2.get('n_nan_inf_filled', 0)),
            'n_negative_clipped_after_zero': int(c_info2.get('n_negative_clipped', 0)),
            'n_lower_clipped_after_zero': int(c_info2.get('n_lower_clipped', 0)),
            'n_upper_clipped_after_zero': int(c_info2.get('n_upper_clipped', 0)),
        })

        if (i + 1) % 5 == 0 or (i + 1) == len(feature_list):
            print(f"  ✅ 已处理 {i+1}/{len(feature_list)}")

    # 保存
    df_physical.to_csv(f'{output_dir}/synthetic_physical.csv', encoding='utf-8-sig')
    with open(f'{output_dir}/inverse_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print("\n" + "═" * 70)
    print("✅ Step6 完成：已输出 synthetic_physical.csv（全非负）")
    print("═" * 70)

    return df_physical, report


# ============================================================
# 5) 运行封装
# ============================================================

def run_step6_inverse_transform(
    df_synthetic_final,
    feature_list,
    step1_results,
    clean_stats,
    transform_info,
    zero_injection_method=None,
    feature_config=None,
    output_dir=None
):
    """
    说明：
    - df_synthetic_final 必须是“变换域”的合成结果（来自你修正版 Step3）
    - transform_info 里每个 feature 至少应有 transform 字段；yeo_johnson/boxcox 要有 lambda
    - step1_results 里每个 feature 推荐有 zero_ratio（原始数据的零比例）
    - 所有最终数据强制非负
    - conpr* 强制为1-压缩比：默认 0
    """
    if feature_config is None:
        feature_config = {
            'count': [
                'total_volume_post', 'total_volume_comment',
                'total_short_post', 'total_long_post',
                'total_short_comment', 'total_long_comment',
                'vis_abs_redundancy_post',
                'senti_symbol_post', 'senti_symbol_comment'
            ],
            'ratio_01': [
                'gini_post', 'gini_comment','retweet_ratio_post',
                'origin_ratio_post', 'neg_ratio_post', 'neg_ratio_comment',
                'vis_concentration_post', 'comp_ratio_post', 'comp_ratio_comment'
            ],
            'continuous': [
                'semantic_shift_post', 'semantic_shift_comment'
                
            ]
        }

    return inverse_transform_and_inject_zeros_nonneg(
        df_synthetic_final=df_synthetic_final,
        feature_list=feature_list,
        step1_results=step1_results,
        clean_stats=clean_stats,
        transform_info=transform_info,
        feature_config=feature_config,
        zero_injection_method=zero_injection_method,
        output_dir=output_dir
    )



### v2核心

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

# ============================================================
# 1) 分位数校准函数 (核心新增)
# ============================================================
def quantile_calibrate(synth_data, hist_stats, robust=True):
    """
    鲁棒分位数校准：
    防止历史数据中的极端离群值（Outliers）破坏合成数据分布。
    """
    s_data = np.asarray(synth_data, dtype=float)
    n = len(s_data)
    if n == 0:
        return s_data

    # 1. 获取/构造历史分布骨架
    if 'percentiles' in hist_stats:
        # Step 1 计算好的 101 个点 (0% ~ 100%)
        hist_p_values = np.array(hist_stats['percentiles'])
        p_points = np.linspace(0, 100, len(hist_p_values))
    else:
        # 简易兜底
        hist_mean = hist_stats.get('mean', 0)
        hist_min = hist_stats.get('min', 0)
        hist_max = hist_stats.get('max', hist_mean * 2)
        p_points = np.array([0, 50, 100])
        hist_p_values = np.array([hist_min, hist_stats.get('median', hist_mean), hist_max])

    # =======================================================
    # [核心修改] 鲁棒处理：削弱极值的影响
    # =======================================================
    if robust and len(hist_p_values) >= 100:
        # 策略：如果最大值 (P100) 远大于 P99，说明可能是异常值
        # 这里的 "远大于" 定义为：(P100 - P99) > 3 * (P99 - P95)
        # 我们用 P99 加上一个合理的增量来替代真实的 P100
        
        p95 = hist_p_values[95]
        p99 = hist_p_values[99]
        p100 = hist_p_values[100] # 也就是 max
        
        # 计算尾部斜率
        tail_gap = p99 - p95
        if tail_gap == 0: tail_gap = 1e-6
        
        # 检查 P100 是否离谱
        # 如果 P100 比 P99 大出 5 倍的 (P99-P95) 区间，我们认为它是脏数据或极端离群点
        threshold_val = p99 + 5.0 * tail_gap
        
        if p100 > threshold_val:
            print(f"  🛡️ 触发鲁棒截断: Max值 {p100:.2f} 被限制为 {threshold_val:.2f} (基于P99推断)")
            # 修改映射表中的最大值，防止合成数据被拉得太远
            hist_p_values[-1] = threshold_val 

    # 2. 计算合成数据的百分位秩
    ranks = stats.rankdata(s_data, method='average')
    s_percentiles = (ranks - 1) / (n - 1) * 100

    # 3. 线性插值映射
    calibrated = np.interp(s_percentiles, p_points, hist_p_values)
    
    return calibrated

# ============================================================
# 2) 逆变换辅助函数 (保持不变)
# ============================================================

def inverse_yeo_johnson_correct(y, lmbda):
    y = np.asarray(y, dtype=float)
    x = np.empty_like(y, dtype=float)
    eps = 1e-12
    pos = y >= 0
    neg = ~pos
    
    if abs(lmbda) < 1e-10:
        x[pos] = np.expm1(y[pos])
    else:
        base = np.maximum(lmbda * y[pos] + 1.0, eps)
        x[pos] = np.power(base, 1.0 / lmbda) - 1.0

    if abs(lmbda - 2.0) < 1e-10:
        x[neg] = -np.expm1(-y[neg])
    else:
        base = np.maximum(1.0 - (2.0 - lmbda) * y[neg], eps)
        x[neg] = 1.0 - np.power(base, 1.0 / (2.0 - lmbda))
    return x

def inverse_log1p(y): return np.expm1(y)
def inverse_sqrt(y): return np.square(y)
def inverse_standardize(y, mean, std): return y * float(std) + float(mean)
def inverse_boxcox(y, lmbda, shift):
    lmbda = float(lmbda)
    if abs(lmbda) < 1e-10:
        return np.exp(y) - shift
    else:
        return np.power(np.maximum(lmbda * y + 1.0, 1e-12), 1.0 / lmbda) - shift

# ============================================================
# 3) 物理约束与零值处理
# ============================================================

def apply_physical_constraints(data, feature_type, limits=None):
    data = np.asarray(data, dtype=float)
    out = data.copy()
    limits = limits or {}
    
    # 全局非负
    out = np.maximum(out, 0.0)
    
    # 类型约束
    if feature_type == 'count':
        out = np.round(out)
        if 'max_value' in limits:
            out = np.minimum(out, limits['max_value'])
            
    elif feature_type == 'ratio_01':
        out = np.clip(out, 0.0, 1.0)
        
    else: # continuous
        if 'max_value' in limits:
             out = np.minimum(out, limits['max_value'])
             
    return out

def inject_zeros_threshold_exact(data, target_zero_ratio):
    """只增加零值，不减少零值"""
    n = data.size
    target_zeros = int(round(n * target_zero_ratio))
    current_zeros = int((data == 0).sum())
    
    if target_zeros <= current_zeros:
        return data

    need = target_zeros - current_zeros
    # 找到非零值中最小的 need 个，置为 0
    idx_nonzero = np.where(data != 0)[0]
    vals = data[idx_nonzero]
    # argsort 得到从小到大的索引
    smallest_indices = idx_nonzero[np.argsort(vals)[:need]]
    
    out = data.copy()
    out[smallest_indices] = 0.0
    return out

# ============================================================
# 4) 主流程
# ============================================================

def inverse_transform_and_calibrate(
    df_synthetic_final,
    feature_list,
    step1_results,
    clean_stats,
    transform_info,
    feature_config=None,
    output_dir=None
):
    os.makedirs(output_dir, exist_ok=True)
    
    # 自动识别 senti_symbol 为 count 类型（重要修复）
    # 如果用户没传 config，初始化默认
    if feature_config is None:
        feature_config = {'count': [], 'ratio_01': [], 'continuous': []}
        
    # 构建映射表
    feature_type_map = {}
    for t, feats in feature_config.items():
        for f in feats: feature_type_map[str(f)] = t
    
    # 强制修正 senti_symbol 的类型
    for f in feature_list:
        if 'senti_symbol' in f or 'total_' in f:
            feature_type_map[f] = 'count'

    df_physical = pd.DataFrame(index=df_synthetic_final.index)
    
    print(f"开始逆变换与分位数校准，共 {len(feature_list)} 个特征...")

    for feature in feature_list:
        if feature not in df_synthetic_final.columns:
            continue
            
        # --- A. 获取参数 ---
        trans = transform_info.get(feature, {})
        method = trans.get('transform', 'none')
        hist_stats = step1_results.get(feature, {})
        
        y = df_synthetic_final[feature].to_numpy(dtype=float)
        
        # --- B. 基础逆变换 ---
        if method == 'yeo_johnson':
            x = inverse_yeo_johnson_correct(y, trans.get('lambda', 0))
        elif method == 'log1p':
            x = inverse_log1p(np.clip(y, -15, 15))
        elif method == 'sqrt':
            x = inverse_sqrt(y)
        elif method == 'standardize':
            x = inverse_standardize(y, trans.get('original_mean', 0), trans.get('original_std', 1))
        elif method == 'boxcox':
            x = inverse_boxcox(y, trans.get('lambda', 1), trans.get('shift', 0))
        else:
            x = y
        print(f"  🔄 {feature:<25}: 已完成基础逆变换 ({method})")
            
        # --- C. 分位数校准 (Quantile Calibration) ---
        # 这是解决 "BAD" 评价的关键步骤
        # 它会忽略逆变换后的具体数值，直接根据排名映射回历史分布
        target_stats = clean_stats.get(feature, step1_results.get(feature, {}))
        x_calibrated = quantile_calibrate(x, target_stats, robust=False)
        # x_calibrated = quantile_calibrate(x, hist_stats)
        
        # --- D. 物理约束与类型修正 ---
        ftype = feature_type_map.get(feature, 'continuous')
        hist_max = hist_stats.get('max', 1e6)
        
        # 设定物理上限 (ratio类型强制1.0，其他类型放宽到历史最大值的1.2倍)
        limits = {'max_value': 1.0 if ftype == 'ratio_01' else hist_max * 1.2}
        
        # 应用约束 (此时 count 类型会被取整)
        x_constrained = apply_physical_constraints(x_calibrated, ftype, limits)
        
        # --- E. 零值注入 ---
        # 对于 count 类型或 continuous 类型，如果有大量零值需求，在此处理
        # 注意：分位数校准其实已经很好地处理了零值（如果历史数据有50%是0，映射后也会有50%是0）
        # 所以这里的注入只是一个“保险”，防止插值产生的微小浮点数误差
        target_zero = hist_stats.get('zero_ratio', 0.0)
        
        # Conpr (压缩比) 通常没有零值，默认不注入
        if 'conpr' not in feature and target_zero > 0.01:
             x_final = inject_zeros_threshold_exact(x_constrained, target_zero)
        else:
             x_final = x_constrained

        # conpr 特殊处理：不能为 0，最小为 1 (假设是压缩比)
        if str(feature).startswith('conpr'):
            x_final = np.maximum(x_final, 1.0)

        df_physical[feature] = x_final

    # 保存结果
    df_physical.to_csv(f'{output_dir}/synthetic_physical.csv', encoding='utf-8-sig')
    print("✅ 处理完成。")
    return df_physical

### run

In [ ]:
print(step1_results['total_long_post'].keys())

In [ ]:
print(transform_info['total_long_post'].keys())

In [ ]:
# df_synthetic  = inverse_transform_and_calibrate(
#     df_synthetic_final=df_step5,
#     feature_list=feature_list,
#     step1_results=step1_results,
#     clean_stats=clean_stats,
#     transform_info=transform_info,
#     feature_config=FEATURE_CONFIG,
#     # zero_injection_method='threshold',  # 或 'probabilistic', 'mixture'
#     output_dir=os.path.join(RESULT_PATH, 'g_step6_inverse')
# )

In [ ]:
df_synthetic, inverse_report = run_step6_inverse_transform(
    df_synthetic_final=df_step5,
    feature_list=feature_list,
    step1_results=step1_results,
    clean_stats=clean_stats,
    transform_info=transform_info,
    feature_config=FEATURE_CONFIG,
    zero_injection_method='threshold_exact',  
    # zero_injection_method='latent',  #
    output_dir=os.path.join(RESULT_PATH, 'g_step6_inverse')
)

In [ ]:
for feature in feature_list:
    if feature in df_synthetic.columns:
        zero_ratio = (df_synthetic[feature] == 0).mean()
        target = step1_results[feature]['zero_ratio']
        print(f"{feature}: 目标{target:.2%} vs 实际{zero_ratio:.2%}")

## step 4.1

### main part2 v3


In [ ]:
import numpy as np
import pandas as pd
from typing import Dict


def _slot_index_15min(idx: pd.DatetimeIndex) -> np.ndarray:
    """将时间索引映射到日内 15 分钟槽位 [0..95]"""
    return idx.hour * 4 + idx.minute // 15


def build_event_boost_profile(
    time_index: pd.DatetimeIndex,
    feature: str,
    events_df: pd.DataFrame | None = None,
    impact_results: Dict | None = None,
    window_after: int = 32,         # 32*15min=8小时
    half_life_hours: float = 1.5,   # 半衰期
    max_boost: float = 3.0          # 1+boost <= max_boost
) -> np.ndarray:
    """
    为某个特征在 time_index 上生成事件强度 boost_t (>=0)，
    boost_t 越大表示事件影响越强。
    """
    n = len(time_index)
    boost = np.zeros(n, dtype=float)
    if events_df is None or len(events_df) == 0:
        return boost

    # 从 Step4 impact_results 估一个“典型峰值增幅”（百分比）
    peak_ratio = 1.0  # 默认 +100%
    if impact_results and feature in impact_results:
        by_combo = impact_results[feature].get('by_combo', {})
        if by_combo:
            peaks = [d.get('overall_peak', 0.0) for d in by_combo.values()]
            if peaks:
                peak_ratio = np.percentile(np.abs(peaks), 75) / 100.0
                peak_ratio = float(np.clip(peak_ratio, 0.2, 3.0))

    df_ev = events_df.copy()
    df_ev['timestamp'] = pd.to_datetime(df_ev['timestamp'])
    df_ev = df_ev.sort_values('timestamp')

    tau = max(half_life_hours * 4, 1.0)  # 15min 为一单位

    for _, ev in df_ev.iterrows():
        t0 = ev['timestamp']
        pos = time_index.searchsorted(t0)
        if pos >= n:
            continue
        for k in range(window_after):
            t = pos + k
            if t >= n:
                break
            boost[t] += peak_ratio * np.exp(-k / tau)

    boost = np.clip(boost, 0.0, max_boost - 1.0)
    return boost

In [ ]:
def bootstrap_feature_event_vs_background(
    df_real_raw: pd.DataFrame,
    df_synth_index: pd.DatetimeIndex,
    feature: str,
    events_df: pd.DataFrame,
    impact_results: Dict | None = None,
    normal_mask: pd.Series | None = None,
    non_event_high_q: float = 0.8,
    event_low_q: float = 0.5,
    by_slot: bool = True,
    min_pool: int = 5,
    seed: int = 0
) -> pd.Series:
    """
    二元事件条件 bootstrap：
      - boost_real>0 的时刻视为“事件窗口”，属于事件池；
      - boost_real=0 的时刻视为“背景窗口”，属于背景池；
      - 背景池中剔除高于 non_event_high_q 分位的高峰（只保留背景水平）；
      - 事件池中只保留高于 event_low_q 分位的值（主要体现事件峰）。
    合成时：
      - 合成位置 boost_syn(t)>0 → 从事件池抽；
      - 合成位置 boost_syn(t)=0 → 只从背景池抽（不会抽到事件峰）。

    Parameters
    ----------
    df_real_raw : DataFrame
        真实 3 个月原始数据（index 为时间）
    df_synth_index : DatetimeIndex
        目标 11 个月时间轴
    feature : str
    events_df : DataFrame
    impact_results : dict or None
    normal_mask : Series[bool], optional
        正常时段掩码；只用正常样本构造池
    non_event_high_q : float
        背景池上分位阈值（如 0.8: 只保留 <=P80 的非事件值）
    event_low_q : float
        事件池下分位阈值（如 0.5: 只保留 >=P50 的事件值）
    by_slot : bool
        是否按日内槽位进一步分桶
    min_pool : int
        每个池的最小样本数，不足则回退到更粗的池
    seed : int

    Returns
    -------
    synth_series : Series
        合成序列（原始空间），index=df_synth_index
    """
    rng = np.random.default_rng(seed)

    if feature not in df_real_raw.columns:
        raise KeyError(f"{feature} not in df_real_raw")

    real = df_real_raw[feature].astype(float)

    # 只用正常时段的真实样本
    if normal_mask is not None:
        nm = normal_mask.reindex(real.index).fillna(False).astype(bool)
        real = real[nm]
    real = real.dropna()
    if len(real) < 20:
        return pd.Series(0.0, index=df_synth_index, name=feature)

    idx_real = real.index
    boost_real = build_event_boost_profile(idx_real, feature, events_df, impact_results)

    # 事件标记：True=事件时段
    is_event_real = boost_real > 0

    # 背景池：非事件时段，裁掉高尾
    bg_vals = real[~is_event_real]
    if len(bg_vals) < min_pool:
        # 几乎没背景，退化为普通 bootstrap
        bg_vals = real
    else:
        q_bg = np.quantile(bg_vals, non_event_high_q)
        bg_vals = bg_vals[bg_vals <= q_bg]

    # 事件池：事件时段，保留高值部分
    ev_vals = real[is_event_real]
    if len(ev_vals) < min_pool:
        # 事件太少，直接用全体
        ev_vals = real
    else:
        q_ev = np.quantile(ev_vals, event_low_q)
        ev_vals = ev_vals[ev_vals >= q_ev]

    # 按槽位再分桶
    if by_slot:
        slot_real = _slot_index_15min(idx_real)
        df_pool = pd.DataFrame({
            'slot': slot_real,
            'is_event': is_event_real,
            'y': real.values
        })

        # 背景池按 slot
        bg_pool_by_slot = {}
        for s, grp in df_pool[~df_pool['is_event']].groupby('slot'):
            y = grp['y'].values
            if len(y) >= min_pool:
                q_bg = np.quantile(y, non_event_high_q)
                bg_pool_by_slot[s] = y[y <= q_bg]
        # 事件池按 slot
        ev_pool_by_slot = {}
        for s, grp in df_pool[df_pool['is_event']].groupby('slot'):
            y = grp['y'].values
            if len(y) >= min_pool:
                q_ev = np.quantile(y, event_low_q)
                ev_pool_by_slot[s] = y[y >= q_ev]
    else:
        bg_pool_by_slot = {}
        ev_pool_by_slot = {}

    # 合成时刻的事件强度
    boost_syn = build_event_boost_profile(df_synth_index, feature, events_df, impact_results)
    is_event_syn = boost_syn > 0
    slot_syn = _slot_index_15min(df_synth_index)

    synth_vals = np.empty(len(df_synth_index), dtype=float)
    for i, (flag_ev, s) in enumerate(zip(is_event_syn, slot_syn)):
        if flag_ev:
            # 事件时刻：优先用事件池
            pool = ev_pool_by_slot.get(s, None) if by_slot else None
            if pool is None or len(pool) < min_pool:
                pool = ev_vals.values
        else:
            # 背景时刻：优先用背景池
            pool = bg_pool_by_slot.get(s, None) if by_slot else None
            if pool is None or len(pool) < min_pool:
                pool = bg_vals.values
        synth_vals[i] = pool[rng.integers(0, len(pool))]

    synth = pd.Series(synth_vals, index=df_synth_index, name=feature)
    return synth

In [ ]:
def overwrite_peaks_event_only(
    HARD_FEATURES_EVENT_ONLY,
    df_real_raw: pd.DataFrame,
    df_synth_raw: pd.DataFrame,
    events_df: pd.DataFrame,
    impact_results: Dict | None = None,
    normal_mask: pd.Series | None = None,
    seed: int = 1234
) -> pd.DataFrame:
    df_out = df_synth_raw.copy()
    idx_syn = df_out.index

    for j, feat in enumerate(HARD_FEATURES_EVENT_ONLY):
        if feat not in df_real_raw.columns:
            print(f"[EVENT-ONLY] {feat} 不在 df_real_raw 中，跳过")
            continue

        print(f"[EVENT-ONLY] 覆盖特征: {feat}")
        syn = bootstrap_feature_event_vs_background(
            df_real_raw=df_real_raw,
            df_synth_index=idx_syn,
            feature=feat,
            events_df=events_df,
            impact_results=impact_results,
            normal_mask=normal_mask,
            non_event_high_q=0.8,   # 无事件背景只取 <= P80 的值
            event_low_q=0.5,        # 事件池只取 >= P50 的值
            by_slot=True,
            min_pool=5,
            seed=seed + j * 37
        )
        df_out[feat] = syn

    return df_out

#### run

In [ ]:
# bad_features = [
#     'senti_symbol_comment',
#     # 'retweet_ratio_post',
#     # 'total_short_post',
#     # 'total_long_post',
#     # 'total_short_comment',
#     'total_long_comment',
#     'semantic_shift_comment',
#     # 'vis_abs_redundancy_post'
# ]
# index = df_synthetic.index

# normal_mask = build_global_normal_mask(
#     index=index,
#     all_anomalies=all_anomalies,
#     remove_types=('positive_extreme', 'negative_extreme', 'structural_break'),
#     pad='30min'   # 不想扩展就写 '0min'
# )

# df_synthetic = apply_quantile_mapping_for_features(
#     df_real_raw=df_features.set_index('timestamp'),          # 3 个月真实
#     df_synth_raw=df_synthetic,        # 11 个月合成(原始空间)
#     feature_list=bad_features,
#     normal_mask=normal_mask,          # 如果有 GARCH 异常掩码，用它限制 target 分布
#     clip_quantile=0.99
# )

In [ ]:
# df_synthetic = overwrite_hard_features_with_bootstrap(
#     BAD_HARD_FEATURES = bad_features,
#     df_real_raw=df_features.set_index('timestamp'),
#     df_synth_raw=df_synthetic,   # 这是已经 inverse-transform 后的 11 个月合成数据
#     normal_mask=normal_mask,     # 如果不想区分正常/异常，可以传 None
#     seed=2025
# )

In [ ]:
# df_synthetic = overwrite_hard_features_with_event_cond_bootstrap(
#     HARD_EVENT_SENSITIVE_FEATURES= bad_features,
#     df_real_raw=df_features.set_index('timestamp'), # 真实 3 个月(原始空间)
#     df_synth_raw=df_synthetic,       # 合成 11 个月(原始空间)
#     events_df=df_official,                # 官方事件(11 个月)
#     impact_results=impact_results,   # Step4 输出，可为 {}
#     normal_mask=normal_mask,         # 正常时段掩码，可为 None
#     seed=2025
# )

In [ ]:
# bad_features = [
#     'semantic_shift_comment',
#     'senti_symbol_comment',
#     'total_short_comment',

#     # 'total_long_comment',
#     # 'total_short_post',
#     # 'total_long_post',
#     # 'retweet_ratio_post',
#     # 'vis_abs_redundancy_post'
# ]


# df_synthetic = overwrite_peaks_event_only(
#     bad_features,
#     df_real_raw=df_features.set_index('timestamp'), # 真实 3 个月(原始空间)
#     df_synth_raw=df_synthetic,       # 合成 11 个月(原始空间)
#     events_df=df_official,                # 官方事件(11 个月)
#     impact_results=impact_results,   # Step4 输出，可为 {}
#     normal_mask=normal_mask,         # 正常时段掩码，可为 None
#     seed=2025
# )

## 质量检验

### main

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import acf

def stage_report(name, x, lags=(1,2,4,8,12,24,48,96)):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 200:
        print(name, "too short:", x.size)
        return
    a = acf(x, nlags=max(lags), fft=True)
    qs = np.quantile(x, [0.0, 0.5, 0.9, 0.99, 1.0])
    print(f"\n[{name}] n={x.size}")
    print(f"  mean={x.mean():.3f} std={x.std():.3f} zero={(x==0).mean():.3%}")
    print(f"  q0={qs[0]:.3f} q50={qs[1]:.3f} q90={qs[2]:.3f} q99={qs[3]:.3f} q100={qs[4]:.3f}")
    print("  acf:", {k: float(a[k]) for k in lags})

# 用法示例（自己替换成你的变量/列）
stage_report("Real_raw", df_features['total_volume_post'].values)
stage_report("Detection2_trans", df_transformed['total_volume_post_transformed'].values)
# stage_report("Detection3_resid", df_prophet_resid['total_volume_post_resid'].values)
# stage_report("Detection4_resid", df_impact_resid['total_volume_post_event_adj_resid'].values)
# stage_report("Detection6_resid", garch_results['total_volume_post']['analysis']['std_resid'])
# stage_report("Step1_res", df_synthetic_noise['total_volume_post'].values)
# stage_report("Step2_res", df_with_volatility['total_volume_post'].values)
# stage_report("Step3_res", df_final_trans['total_volume_post'].values)
# stage_report("Step4_res", df_synthetic['total_volume_post'].values)



In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt


class SyntheticDataValidator:
    """
    合成数据质量验证器（适配“只合成正常数据”的场景）

    主要特性：
    - 支持传入 normal_mask，只在“正常时段”上做比较（例如排除 GARCH 标出的异常点）
    - 统计/分布比较默认对真实和合成都做尾部裁剪（winsorize），减少极端异常的影响
    - 输出多种指标：mean/std/median/IQR 差异、KS p 值、ACF 差异、tail 分位差等
    """

    def __init__(self,
                 step,
                 df_real: pd.DataFrame,
                 df_synthetic: pd.DataFrame,
                 normal_mask: pd.Series | None = None,
                 clip_quantile: float = 0.99):
        """
        Parameters
        ----------
        df_real : DataFrame
            真实数据（通常是“含异常”的原始数据，或你预先处理后的正常数据）
        df_synthetic : DataFrame
            合成数据
        normal_mask : Series[bool], optional
            正常时段掩码，index 应与 df_real 对齐。
            True 表示“正常”，False 表示“异常”（将被排除在评估之外）。
        clip_quantile : float
            尾部裁剪分位数，例如 0.99 表示在 [1%, 99%] 范围内 winsorize。
        """
        self.step=step
        # 索引对齐：只保留两边都有的时间点
        common_index = df_real.index.intersection(df_synthetic.index)
        self.df_real = df_real.loc[common_index].copy()
        self.df_synth = df_synthetic.loc[common_index].copy()

        # 正常掩码
        if normal_mask is not None:
            mask = normal_mask.reindex(common_index).fillna(False).astype(bool)
            self.normal_mask = mask
        else:
            self.normal_mask = None

        self.clip_quantile = float(clip_quantile)

    # ---------- 内部工具 ----------

    def _apply_mask(self, series: pd.Series) -> pd.Series:
        """应用 normal_mask，只保留正常时段"""
        if self.normal_mask is None:
            return series
        return series[self.normal_mask]

    def _align_series(self, feature: str):
        """对齐单个特征的真实 & 合成序列，并应用 normal_mask"""
        if feature not in self.df_real.columns or feature not in self.df_synth.columns:
            return None, None
        if self.step ==5:
            col_hist = f'{feature}_transformed' if f'{feature}_transformed' in self.df_real.columns else None
        elif self.step==4:
            col_hist = f'{feature}_resid' if f'{feature}_resid' in self.df_real.columns else None
        elif self.step==3:
            col_hist = f'{feature}_event_adj_resid' if f'{feature}_event_adj_resid' in self.df_real.columns else None
        elif self.step==2:
            col_hist = f'{feature}_ar_resid' if f'{feature}_ar_resid' in self.df_real.columns else None
        else:
            col_hist = feature
        if not col_hist:
            print(f"⚠️ 在 df_real 中未找到 {feature} 的历史列，无法对齐")
            return None, None   
        real = self.df_real[col_hist].astype(float)
        synth = self.df_synth[feature].astype(float)

        real = self._apply_mask(real).dropna()
        synth = self._apply_mask(synth).dropna()

        # 再对齐一次索引（防止其中一方有 NaN 被删后错位）
        common_index = real.index.intersection(synth.index)
        real = real.loc[common_index]
        synth = synth.loc[common_index]

        if len(real) < 10 or len(synth) < 10:
            return None, None

        return real, synth

    def _winsorize_pair(self, real: pd.Series, synth: pd.Series):
        """对真实和合成各自做尾部裁剪（winsorize）"""
        q_low = 1 - self.clip_quantile
        q_high = self.clip_quantile

        r_low, r_high = real.quantile(q_low), real.quantile(q_high)
        s_low, s_high = synth.quantile(q_low), synth.quantile(q_high)

        real_clip = real.clip(r_low, r_high)
        synth_clip = synth.clip(s_low, s_high)
        return real_clip, synth_clip

    # ---------- 统计量比较 ----------

    def compute_statistics_comparison(self, feature: str) -> dict:
        """
        比较单个特征的统计量（在 winsorize 之后）
        返回：
            mean / std / median / IQR 及其百分比差异
        """
        real, synth = self._align_series(feature)
        if real is None:
            print(f"⚠️ 无法对齐 {feature}，跳过统计比较")
            return {}

        real_clip, synth_clip = self._winsorize_pair(real, synth)

        def iqr(x):
            return np.quantile(x, 0.75) - np.quantile(x, 0.25)

        stats_real = {
            'mean': real_clip.mean(),
            'std': real_clip.std(),
            'median': real_clip.median(),
            'iqr': iqr(real_clip)
        }
        stats_synth = {
            'mean': synth_clip.mean(),
            'std': synth_clip.std(),
            'median': synth_clip.median(),
            'iqr': iqr(synth_clip)
        }

        comparison = {}
        for key in stats_real:
            r = stats_real[key]
            s = stats_synth[key]
            if r != 0:
                diff_pct = abs(s - r) / abs(r) * 100
            else:
                diff_pct = abs(s - r) * 100
            diff_abs = abs(s - r)
            comparison[key] = {'real': r, 'synthetic': s, 'diff_pct': diff_pct, 'diff_abs': diff_abs}

        return comparison

    # ---------- 分布检验（去尾部 + 可下采样） ----------

    def compute_distribution_tests(self, feature: str,
                                   max_samples: int = 5000) -> dict:
        """
        在 winsorize 后对真实 & 合成做 KS / MW 检验。
        为避免样本过大导致 p 值过于敏感，若样本量 > max_samples，则随机下采样。
        """
        real, synth = self._align_series(feature)
        if real is None:
            return {}

        real_clip, synth_clip = self._winsorize_pair(real, synth)

        # 下采样
        n = min(len(real_clip), len(synth_clip), max_samples)
        if n < 10:
            return {}

        real_sample = real_clip.sample(n, random_state=42)
        synth_sample = synth_clip.sample(n, random_state=43)

        ks_stat, ks_pval = stats.ks_2samp(real_sample, synth_sample)
        mw_stat, mw_pval = stats.mannwhitneyu(real_sample,
                                             synth_sample,
                                             alternative='two-sided')

        return {
            'ks_test': {'statistic': ks_stat, 'p_value': ks_pval},
            'mw_test': {'statistic': mw_stat, 'p_value': mw_pval}
        }

    # ---------- 自相关结构比较 ----------

    def compute_autocorrelation_comparison(self, feature: str,
                                           max_lag: int = 20) -> dict:
        """
        比较真实 & 合成在 1..max_lag 的自相关（不过滤 tails，用的是正常时段上的原序列）
        返回：
            acf_real / acf_synthetic / mean_acf_diff
        """
        real, synth = self._align_series(feature)
        if real is None:
            return {}

        acf_real = [real.autocorr(lag=i) for i in range(1, max_lag + 1)]
        acf_synth = [synth.autocorr(lag=i) for i in range(1, max_lag + 1)]

        acf_real = [x if np.isfinite(x) else 0.0 for x in acf_real]
        acf_synth = [x if np.isfinite(x) else 0.0 for x in acf_synth]

        acf_diff = float(np.mean(np.abs(np.array(acf_real) - np.array(acf_synth))))

        return {
            'acf_real': acf_real,
            'acf_synthetic': acf_synth,
            'mean_acf_diff': acf_diff
        }

    # ---------- 尾部分位数差异 ----------

    def compute_tail_diff(self, feature: str,
                          quantiles=(0.95, 0.99)) -> dict:
        """
        比较高分位数（tail）差异，衡量极端值是否被过度压制或放大。
        返回：{q: diff_pct}，例如 0.95 / 0.99
        """
        real, synth = self._align_series(feature)
        if real is None:
            return {}

        tail_diffs = {}
        for q in quantiles:
            r_q = real.quantile(q)
            s_q = synth.quantile(q)
            if r_q != 0:
                diff_pct = (s_q - r_q) / abs(r_q) * 100
            else:
                diff_pct = (s_q - r_q) * 100
            tail_diffs[q] = {
                'real': r_q,
                'synthetic': s_q,
                'diff_pct': diff_pct
            }
        return tail_diffs

    # ---------- 质量打分规则 ----------

    @staticmethod
    def _compute_quality_flag(row) -> str:
        """
        简单的规则版质量打分：
        主要针对“正常部分”的拟合，不要求完美复刻原始异常。
        你可以按业务再调整这些阈值。
        """
        mean_diff_pct = row.get('mean_diff_pct', np.nan)
        mean_diff_abs = row.get('mean_diff_pct', np.nan)

        std_diff = row.get('std_diff_pct', np.nan)
        median_diff = row.get('median_diff_pct', np.nan)
        iqr_diff = row.get('iqr_diff_pct', np.nan)
        acf_d = row.get('acf_diff', np.nan)
        tail_99 = row.get('tail_99_diff_pct', np.nan)

        # GOOD：整体水平和动态都比较接近，尾部差异不过分
        if ((mean_diff_pct < 20 or mean_diff_abs < 0.1) and std_diff < 30 and
                median_diff < 20 and iqr_diff < 30 and
                acf_d < 0.06 and abs(tail_99) < 80):
            return 'GOOD'

        # OK：中等偏差，整体形态还算合理
        if (mean_diff_pct < 50 and std_diff < 60 and
                acf_d < 0.12 and abs(tail_99) < 150):
            return 'OK'

        # 其余视作 BAD：要么量级严重不对，要么动态结构很不同，要么尾巴差太多
        return 'BAD'

    # ---------- 汇总所有特征 ----------

    def validate_all_features(self, feature_list: list) -> pd.DataFrame:
        """
        对多个特征做汇总评估，返回 DataFrame。
        指标包括：
            mean_diff_pct / std_diff_pct / median_diff_pct / iqr_diff_pct
            ks_pvalue
            acf_diff
            tail_95_diff_pct / tail_99_diff_pct
            quality (GOOD / OK / BAD)
        """
        results = []

        for feature in feature_list:
            real, synth = self._align_series(feature)
            if real is None:
                continue

            stats_comp = self.compute_statistics_comparison(feature)
            dist_tests = self.compute_distribution_tests(feature)
            acf_comp = self.compute_autocorrelation_comparison(feature)
            tail_comp = self.compute_tail_diff(feature, quantiles=(0.95, 0.99))

            row = {'feature': feature}

            # 统计量差异
            for key in ['mean', 'std', 'median', 'iqr']:
                row[f'{key}_diff_pct'] = stats_comp.get(key, {}).get('diff_pct', np.nan)
                row[f'{key}_real'] = stats_comp.get(key, {}).get('real', np.nan)
                row[f'{key}_synthetic'] = stats_comp.get(key, {}).get('synthetic', np.nan)
                row[f'{key}_diff_abs'] = stats_comp.get(key, {}).get('diff_abs', np.nan)



            # KS p 值
            row['ks_pvalue'] = dist_tests.get('ks_test', {}).get('p_value', np.nan)

            # ACF
            row['acf_diff'] = acf_comp.get('mean_acf_diff', np.nan)

            # tail
            row['tail_95_diff_pct'] = tail_comp.get(0.95, {}).get('diff_pct', np.nan)
            row['tail_99_diff_pct'] = tail_comp.get(0.99, {}).get('diff_pct', np.nan)

            # 质量标记
            row['quality'] = self._compute_quality_flag(row)

            results.append(row)

        return pd.DataFrame(results)

    # ---------- 可视化比较（考虑 normal_mask & 去尾） ----------

    def plot_comparison(self, feature: str,
                        figsize: tuple = (15, 10),
                        save_path: str | None = None,
                        stop: int = 1000):
        """
        可视化比较：时间序列 / 分布 / ACF / QQ
        说明：
          - 时间序列：只画前 stop 个点，且只用 normal_mask 过滤（不裁尾）
          - 分布和 QQ：在 winsorize 后画图，更符合“正常数据”的比较
        """
        real, synth = self._align_series(feature)
        if real is None:
            print(f"特征 {feature} 不存在或有效数据不足")
            return

        # 时间序列用原值（正常时段）
        real_ts = real.iloc[:stop]
        synth_ts = synth.iloc[:stop]

        # 分布 & QQ 用 winsorize 后的值
        real_clip, synth_clip = self._winsorize_pair(real, synth)

        fig, axes = plt.subplots(2, 2, figsize=figsize)

        # 1) 时间序列
        ax1 = axes[0, 0]
        ax1.plot(real_ts.index, real_ts.values, alpha=0.7,
                 label='真实数据', linewidth=0.8)
        ax1.plot(synth_ts.index, synth_ts.values, alpha=0.7,
                 label='合成数据', linewidth=0.8)
        ax1.set_title(f'{feature} - 时间序列对比（前{len(real_ts)}点，仅正常时段）')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2) 分布
        ax2 = axes[0, 1]
        ax2.hist(real_clip, bins=50, alpha=0.5,
                 label='真实数据(裁尾后)', density=True)
        ax2.hist(synth_clip, bins=50, alpha=0.5,
                 label='合成数据(裁尾后)', density=True)
        ax2.set_title(f'{feature} - 分布对比（winsorize@{self.clip_quantile:.2f}）')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3) ACF 对比
        ax3 = axes[1, 0]
        acf_comp = self.compute_autocorrelation_comparison(feature)
        lags = range(1, len(acf_comp.get('acf_real', [])) + 1)
        ax3.plot(lags, acf_comp.get('acf_real', []), 'o-',
                 label='真实数据', markersize=4)
        ax3.plot(lags, acf_comp.get('acf_synthetic', []), 's-',
                 label='合成数据', markersize=4)
        ax3.set_title(f'{feature} - 自相关函数对比（正常时段）')
        ax3.set_xlabel('滞后阶数')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        # 4) QQ 图（裁尾后）
        ax4 = axes[1, 1]
        quantiles = np.linspace(0.01, 0.99, 50)
        real_q = np.quantile(real_clip, quantiles)
        synth_q = np.quantile(synth_clip, quantiles)
        ax4.scatter(real_q, synth_q, alpha=0.6)
        min_val = min(real_q.min(), synth_q.min())
        max_val = max(real_q.max(), synth_q.max())
        ax4.plot([min_val, max_val], [min_val, max_val],
                 'r--', label='理想线')
        ax4.set_xlabel('真实数据分位数(裁尾后)')
        ax4.set_ylabel('合成数据分位数(裁尾后)')
        ax4.set_title(f'{feature} - QQ图（winsorize@{self.clip_quantile:.2f}）')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

### 检验1

In [ ]:
df_score_1 = calculate_numeric_quality(1,pd.DataFrame(cleaned_data), slice_data(df_synthetic_noise,start,end), feature_list,os.path.join(RESULT_PATH, '0_quality_score'),normal_mask)
df_score_2 = calculate_numeric_quality(2,df_impact_resid, slice_data(df_with_volatility,start,end), feature_list,os.path.join(RESULT_PATH, '0_quality_score'),normal_mask)
df_score_3 = calculate_numeric_quality(3,df_ar_resid, slice_data(df_synthetic_ar,start,end), feature_list,os.path.join(RESULT_PATH, '0_quality_score'),normal_mask)
df_score_4 = calculate_numeric_quality(4,df_prophet_resid, slice_data(df_step4,start,end), feature_list,os.path.join(RESULT_PATH, '0_quality_score'),normal_mask)
df_score_5 = calculate_numeric_quality(5,df_transformed, slice_data(df_step5,start,end), feature_list,os.path.join(RESULT_PATH, '0_quality_score'),normal_mask)
# print(df_score)

### 检验2

In [ ]:
# validator_1 = SyntheticDataValidator(
#     step=1,
#     df_real=pd.DataFrame(cleaned_data),
#     df_synthetic=slice_data(df_synthetic_noise,start,end),
#     normal_mask=normal_mask,
#     clip_quantile=0.99
# )

# eval_df_1 = validator_1.validate_all_features(feature_list)
# eval_df_1.round(4).to_csv(os.path.join(RESULT_PATH, '0_quality_score', 'quality_report_step1.csv'), index=False)


In [ ]:
# validator_2 = SyntheticDataValidator(
#     step=2,
#     df_real=df_ar_resid, 
#     df_synthetic=slice_data(df_with_volatility,start,end),
#     normal_mask=normal_mask,
#     clip_quantile=0.99
# )

# eval_df_2 = validator_2.validate_all_features(feature_list)
# eval_df_2.round(4).to_csv(os.path.join(RESULT_PATH, '0_quality_score', 'quality_report_step2.csv'), index=False)


In [ ]:
# validator_3 = SyntheticDataValidator(
#     step=3,
#     df_real=df_impact_resid, 
#     df_synthetic=slice_data(df_synthetic_ar,start,end),
#     normal_mask=normal_mask,
#     clip_quantile=0.99
# )

# eval_df_3 = validator_3.validate_all_features(feature_list)
# eval_df_3.round(4).to_csv(os.path.join(RESULT_PATH, '0_quality_score', 'quality_report_step3.csv'), index=False)


In [ ]:
# # 此步骤bad不适用
# validator_4 = SyntheticDataValidator(
#     step=4,
#     df_real=df_prophet_resid, 
#     df_synthetic=slice_data(df_step4,start,end),
#     normal_mask=normal_mask,
#     clip_quantile=0.99
# )

# eval_df_4 = validator_4.validate_all_features(feature_list)
# eval_df_4.round(4).to_csv(os.path.join(RESULT_PATH, '0_quality_score', 'quality_report_step4.csv'), index=False)

In [ ]:
validator_5 = SyntheticDataValidator(
    step=5,
    df_real=df_transformed.set_index('timestamp'), 
    df_synthetic=slice_data(df_step5,start,end),
    normal_mask=normal_mask,
    clip_quantile=0.99
)

eval_df_5 = validator_5.validate_all_features(feature_list)
eval_df_5.round(4).to_csv(os.path.join(RESULT_PATH, '0_quality_score', 'quality_report_step5.csv'), index=False)


### 检验 最后一步

In [ ]:
validator_6 = SyntheticDataValidator(
    step=6,
    df_real=df_feature_slice.set_index('timestamp'),  
    df_synthetic=slice_data(df_synthetic,start,end),
    normal_mask=normal_mask,
    clip_quantile=0.99
)

eval_df_6 = validator_6.validate_all_features(feature_list)
eval_df_6.round(4).to_csv(os.path.join(RESULT_PATH, '0_quality_score', 'quality_report_step6.csv'), index=False)


In [ ]:
eval_df_6

In [ ]:
bad_features = ['senti_symbol_post',
 'senti_symbol_comment',
 'comp_ratio_comment',
 'retweet_ratio_post',
 'total_long_post',
 'total_short_comment',
 'total_long_comment',
 'neg_ratio_comment']

In [ ]:
eval_df_6[eval_df_6['quality']=='BAD']
# eval_df_6[eval_df_6['feature'].isin(bad_features)].round(4)


In [ ]:

# 针对单个特征看图
for feature in eval_df_6[eval_df_6['quality']=='BAD']['feature']:
# for feature in eval_df_6['feature']:

# for feature in bad_features:

    validator_6.plot_comparison(feature)
# validator.plot_comparison(feature)

# 弃用

## step5 合成数据质量验证

In [ ]:
# # ==================== 第六部分：合成数据质量验证器 ====================

# class SyntheticDataValidator:
#     """
#     验证合成数据质量
#     """
    
#     def __init__(self, df_real: pd.DataFrame, df_synthetic: pd.DataFrame):
#         """
#         Parameters:
#         -----------
#         df_real : pd.DataFrame, 真实数据
#         df_synthetic : pd.DataFrame, 合成数据
#         """
#         self.df_real = df_real.copy()
#         self.df_synthetic = df_synthetic.copy()
    
#     def compute_statistics_comparison(self, feature: str) -> dict:
#         """比较单个特征的统计量"""
#         if feature not in self.df_real.columns or feature not in self.df_synthetic.columns:
#             return {}
        
#         real = self.df_real[feature].dropna()
#         synth = self.df_synthetic[feature].dropna()
        
#         comparison = {
#             'mean': {'real': real.mean(), 'synthetic': synth.mean()},
#             'std': {'real': real.std(), 'synthetic': synth.std()},
#             'median': {'real': real.median(), 'synthetic': synth.median()},
#             'min': {'real': real.min(), 'synthetic': synth.min()},
#             'max': {'real': real.max(), 'synthetic': synth.max()},
#             'skewness': {'real': stats.skew(real), 'synthetic': stats.skew(synth)},
#             'kurtosis': {'real': stats.kurtosis(real), 'synthetic': stats.kurtosis(synth)}
#         }
        
#         # 计算差异度
#         for stat in comparison:
#             r = comparison[stat]['real']
#             s = comparison[stat]['synthetic']
#             if r != 0:
#                 comparison[stat]['diff_pct'] = abs(s - r) / abs(r) * 100
#             else:
#                 comparison[stat]['diff_pct'] = abs(s - r) * 100
        
#         return comparison
    
#     def compute_distribution_tests(self, feature: str) -> dict:
#         """分布检验"""
#         if feature not in self.df_real.columns or feature not in self.df_synthetic.columns:
#             return {}
        
#         real = self.df_real[feature].dropna()
#         synth = self.df_synthetic[feature].dropna()
        
#         # KS检验
#         ks_stat, ks_pval = stats.ks_2samp(real, synth)
        
#         # MW检验
#         mw_stat, mw_pval = stats.mannwhitneyu(real, synth, alternative='two-sided')
        
#         return {
#             'ks_test': {'statistic': ks_stat, 'p_value': ks_pval},
#             'mw_test': {'statistic': mw_stat, 'p_value': mw_pval}
#         }
    
#     def compute_autocorrelation_comparison(self, feature: str, max_lag: int = 20) -> dict:
#         """自相关比较"""
#         if feature not in self.df_real.columns or feature not in self.df_synthetic.columns:
#             return {}
        
#         real = self.df_real[feature].dropna()
#         synth = self.df_synthetic[feature].dropna()
        
#         acf_real = [real.autocorr(lag=i) for i in range(1, max_lag+1)]
#         acf_synth = [synth.autocorr(lag=i) for i in range(1, max_lag+1)]
        
#         # 去除NaN
#         acf_real = [x if not np.isnan(x) else 0 for x in acf_real]
#         acf_synth = [x if not np.isnan(x) else 0 for x in acf_synth]
        
#         # 计算ACF差异
#         acf_diff = np.mean(np.abs(np.array(acf_real) - np.array(acf_synth)))
        
#         return {
#             'acf_real': acf_real,
#             'acf_synthetic': acf_synth,
#             'mean_acf_diff': acf_diff
#         }
    
   

#     def validate_all_features(self, feature_list: list) -> pd.DataFrame:
#         """验证所有特征"""
#         results = []
        
#         for feature in feature_list:
#             if feature not in self.df_real.columns or feature not in self.df_synthetic.columns:
#                 continue
            
#             stats_comp = self.compute_statistics_comparison(feature)
#             dist_tests = self.compute_distribution_tests(feature)
#             acf_comp = self.compute_autocorrelation_comparison(feature)
            
#             results.append({
#                 'feature': feature,
#                 'mean_diff_pct': stats_comp.get('mean', {}).get('diff_pct', np.nan),
#                 'std_diff_pct': stats_comp.get('std', {}).get('diff_pct', np.nan),
#                 'ks_pvalue': dist_tests.get('ks_test', {}).get('p_value', np.nan),
#                 'acf_diff': acf_comp.get('mean_acf_diff', np.nan)
#             })
        
#         return pd.DataFrame(results)
    
#     def plot_comparison(self, feature: str, figsize: tuple = (15, 10)):
#         """可视化比较"""
#         if feature not in self.df_real.columns or feature not in self.df_synthetic.columns:
#             print(f"特征 {feature} 不存在")
#             return
        
#         fig, axes = plt.subplots(2, 2, figsize=figsize)
        
#         real = self.df_real[feature].dropna()
#         synth = self.df_synthetic[feature].dropna()
#         stop = 1000

#         # 时间序列对比
#         ax1 = axes[0, 0]
#         ax1.plot(real.index[:stop], real.values[:stop], alpha=0.7, label='真实数据', linewidth=0.8)
#         ax1.plot(synth.index[:stop], synth.values[:stop], alpha=0.7, label='合成数据', linewidth=0.8)
#         ax1.set_title(f'{feature} - 时间序列对比（前{str(stop)}点）')
#         ax1.legend()
#         ax1.grid(True, alpha=0.3)
        
#         # 分布对比
#         ax2 = axes[0, 1]
#         ax2.hist(real, bins=50, alpha=0.5, label='真实数据', density=True)
#         ax2.hist(synth, bins=50, alpha=0.5, label='合成数据', density=True)
#         ax2.set_title(f'{feature} - 分布对比')
#         ax2.legend()
#         ax2.grid(True, alpha=0.3)
        
#         # ACF对比
#         ax3 = axes[1, 0]
#         acf_comp = self.compute_autocorrelation_comparison(feature)
#         lags = range(1, len(acf_comp['acf_real'])+1)
#         ax3.plot(lags, acf_comp['acf_real'], 'o-', label='真实数据', markersize=4)
#         ax3.plot(lags, acf_comp['acf_synthetic'], 's-', label='合成数据', markersize=4)
#         ax3.set_title(f'{feature} - 自相关函数对比')
#         ax3.set_xlabel('滞后阶数')
#         ax3.legend()
#         ax3.grid(True, alpha=0.3)
        
#         # QQ图
#         ax4 = axes[1, 1]
#         # 使用分位数对比
#         quantiles = np.linspace(0.01, 0.99, 50)
#         real_quantiles = np.quantile(real, quantiles)
#         synth_quantiles = np.quantile(synth, quantiles)
#         ax4.scatter(real_quantiles, synth_quantiles, alpha=0.6)
#         min_val = min(real_quantiles.min(), synth_quantiles.min())
#         max_val = max(real_quantiles.max(), synth_quantiles.max())
#         ax4.plot([min_val, max_val], [min_val, max_val], 'r--', label='理想线')
#         ax4.set_xlabel('真实数据分位数')
#         ax4.set_ylabel('合成数据分位数')
#         ax4.set_title(f'{feature} - QQ图')
#         ax4.legend()
#         ax4.grid(True, alpha=0.3)
        
#         plt.tight_layout()
#         plt.savefig(feature+'2.jpg')
#         plt.show()


### run

In [ ]:
df_synthetic_slice = df_synthetic.copy()
df_synthetic_slice['timestamp'] = df_synthetic_slice.index
df_feature = df_features.copy().set_index('timestamp')

In [ ]:
# df_synthetic.to_csv(f'./step4_inverse/合成数据_normal.csv', index=True, encoding='utf-8-sig')

In [ ]:
def add_quality_flag(df_result):
    flags = []
    for _, row in df_result.iterrows():
        mean_diff = row['mean_diff_pct']
        std_diff = row['std_diff_pct']
        ks_p = row['ks_pvalue']
        acf_d = row['acf_diff']
        
        # 简单规则版，可按特征类型细化
        if (mean_diff < 30 and std_diff < 30 and 
            ks_p > 0.05 and acf_d < 0.08):
            flag = 'GOOD'        # 分布+动态都比较接近
        elif (mean_diff < 60 and std_diff < 60 and
            ks_p > 0.01 and acf_d < 0.15):
            flag = 'OK'          # 大致合理，但有偏
        else:
            flag = 'BAD'         # 明显不匹配，需要重调
        flags.append(flag)
    df_result['quality'] = flags
    return df_result

In [ ]:
validator = SyntheticDataValidator(df_feature,slice_data(df_synthetic_slice,start,end))
validation_results = validator.validate_all_features(feature_list)
validation_results = add_quality_flag(validation_results) 

In [ ]:
# print(validation_results)

In [ ]:
for feature in feature_list:
    validator.plot_comparison(feature)

## step6 剔除异常值

In [ ]:
import numpy as np
import pandas as pd


def detect_extreme_spikes(
    df_synth_raw: pd.DataFrame,
    df_real_raw: pd.DataFrame,
    feature_list: list,
    q_low: float = 0.01,
    q_high: float = 0.99,
    factor_hi: float = 10.0,
    factor_lo: float = 10.0
) -> pd.DataFrame:
    """
    检测合成数据中“远超真实合理范围”的极端爆点。

    参数
    ----
    df_synth_raw : 合成数据（原始空间，已逆变换）
    df_real_raw  : 真实数据（原始空间；用于估计合理范围）
    feature_list : 要检查的特征列表
    q_low, q_high : 用真实数据的分位数构造基准区间 [q_low,q_high]
    factor_hi, factor_lo : 把这个区间放大多少倍作为允许范围

    返回
    ----
    spikes_df : DataFrame，包含所有检测出的爆点：
        ['feature', 'timestamp', 'value', 'lower_thr', 'upper_thr']
    """
    records = []

    for feat in feature_list:
        if feat not in df_synth_raw.columns or feat not in df_real_raw.columns:
            continue

        real = df_real_raw[feat].dropna().astype(float)
        synth = df_synth_raw[feat].astype(float)

        if len(real) < 20:
            continue

        # 用真实数据的中间分布估计“合理范围”
        r_q_low, r_q_high = np.quantile(real, [q_low, q_high])
        # 避免 q_low ≈ q_high 导致下界=上界
        span = r_q_high - r_q_low
        if span <= 0:
            # 退化成“基于最大值”的阈值
            r_max = real.max()
            r_min = real.min()
            lower_thr = r_min * factor_lo
            upper_thr = r_max * factor_hi
        else:
            lower_thr = r_q_low - factor_lo * span
            upper_thr = r_q_high + factor_hi * span

        # 检测合成数据中的爆点
        mask = (synth < lower_thr) | (synth > upper_thr)
        idx_bad = synth.index[mask]
        for ts in idx_bad:
            records.append({
                'feature': feat,
                'timestamp': ts,
                'value': float(synth.loc[ts]),
                'lower_thr': float(lower_thr),
                'upper_thr': float(upper_thr)
            })

    spikes_df = pd.DataFrame(records)
    return spikes_df


def fix_spikes_with_neighbor_mean(
    df_synth_raw: pd.DataFrame,
    spikes_df: pd.DataFrame,
    use_linear: bool = True
) -> pd.DataFrame:
    """
    对 spikes_df 中列出的爆点进行修复，考虑“连续极端段”的情况。

    思路：
      - 对每个特征，先根据时间顺序找到所有爆点的位置；
      - 将相邻的爆点合并成若干连续段 [start_idx, end_idx]；
      - 对每个段，用该段两端的正常点做插值 / 均值填充：
          * 若两边都有正常点：线性插值（或整体均值）
          * 若只有一边有正常点：全部填该一侧的值
          * 若整条序列都是爆点：填全局中位数

    参数
    ----
    df_synth_raw : DataFrame
        合成数据（原始空间，已逆变换）
    spikes_df : DataFrame
        detect_extreme_spikes 的输出，至少包含 ['feature','timestamp']

    use_linear : bool
        True  -> 在线性插值两侧正常值之间填充；
        False -> 整段都填左右正常值的平均数。

    返回
    ----
    df_fixed : DataFrame
        修复后的合成数据
    """
    df_fixed = df_synth_raw.copy()

    if spikes_df is None or spikes_df.empty:
        return df_fixed

    # 按特征逐个处理
    for feat in spikes_df['feature'].unique():
        sub = spikes_df[spikes_df['feature'] == feat]
        if feat not in df_fixed.columns:
            continue

        s = df_fixed[feat].astype(float)
        idx_all = s.index
        pos_map = {ts: i for i, ts in enumerate(idx_all)}

        # 将该特征的爆点索引位置转为整数位置并排序
        bad_pos = sorted(
            [pos_map[ts] for ts in sub['timestamp'] if ts in pos_map]
        )
        if not bad_pos:
            continue

        # 将相邻的爆点位置合并为连续段 [start, end]
        segments = []
        start = prev = bad_pos[0]
        for p in bad_pos[1:]:
            if p == prev + 1:
                prev = p
            else:
                segments.append((start, prev))
                start = prev = p
        segments.append((start, prev))

        # 逐个段进行修复
        for a, b in segments:
            seg_len = b - a + 1

            # 段的左侧与右侧正常点索引（可能不存在）
            left = a - 1 if a > 0 else None
            right = b + 1 if b < len(s) - 1 else None

            if left is None and right is None:
                # 整条序列都是 spike：用全局中位数填
                fill_vals = [float(np.nanmedian(s.values))] * seg_len

            elif left is None:
                # 序列头部连续 spike：全段用第一个右侧正常点
                v = s.iloc[right]
                fill_vals = [v] * seg_len

            elif right is None:
                # 序列尾部连续 spike：全段用最后一个左侧正常点
                v = s.iloc[left]
                fill_vals = [v] * seg_len

            else:
                # 两边都有正常点：线性插值或整体均值
                v_left = s.iloc[left]
                v_right = s.iloc[right]

                # 若左右本身是 NaN，退化为全局中位数
                if np.isnan(v_left) and np.isnan(v_right):
                    fill_vals = [float(np.nanmedian(s.values))] * seg_len
                elif np.isnan(v_left):
                    fill_vals = [v_right] * seg_len
                elif np.isnan(v_right):
                    fill_vals = [v_left] * seg_len
                else:
                    if use_linear and seg_len >= 1:
                        # 在 [v_left, v_right] 之间插值 seg_len 个点
                        # 相当于：v_left + (v_right-v_left)*(k+1)/(seg_len+1)
                        ratios = (np.arange(1, seg_len + 1) / (seg_len + 1))
                        fill_vals = v_left + (v_right - v_left) * ratios
                    else:
                        m = (v_left + v_right) / 2.0
                        fill_vals = [m] * seg_len

            # 写回该段
            for offset, pos in enumerate(range(a, b + 1)):
                s.iloc[pos] = fill_vals[offset]

        if 'total' in feat:
            df_fixed[feat] = np.int32(s)
        else:    
            df_fixed[feat] = s

    return df_fixed

### run

In [ ]:
# 1. 选择要检查的特征（可以是所有合成特征）
feature_list = df_synthetic.columns.tolist()

# 2. 先检测爆点
spikes_df = detect_extreme_spikes(
    df_synth_raw=df_synthetic,
    df_real_raw=df_feature_slice.set_index('timestamp'),      # 真实三个月
    feature_list=feature_list,
    q_low=0.01, q_high=0.99,
    factor_hi=10.0, factor_lo=10.0
)

print("检测到的爆点数：", len(spikes_df))
if not spikes_df.empty:
    print(spikes_df.head(20))

# 3. 用前后均值进行修复
df_synthetic = fix_spikes_with_neighbor_mean(
    df_synth_raw=df_synthetic,
    spikes_df=spikes_df
)

In [ ]:
df_synthetic.to_csv(RESULT_PATH+os.sep+'合成数据_normal_final.csv', index=True, encoding='utf-8-sig')